# MBSE Mapping:
## Overview:
### High-Level Class Objects:

In [1]:
from dataclasses import dataclass, field
from typing import Optional
from collections import Counter, defaultdict
from urllib.parse import urlsplit, unquote

@dataclass
class ElementRecord:
    resource: str
    xmi_id: str
    tag: str
    metaclass: Optional[str] = None
    name: Optional[str] = None
    attributes: dict = field(default_factory=dict)
    parent: Optional[tuple] = None
    children: list = field(default_factory=list)
    registry_key: Optional[tuple] = field(default=None, init=False)

    # Normalized metadata
    element_category: Optional[str] = None
    qualified_name: Optional[str] = None
    namespace: Optional[str] = None

    @property
    def key(self):
        if self.registry_key is None:
            raise RuntimeError("Element has not been registered.")
        return self.registry_key

@dataclass
class ReferenceRecord:
    source: tuple
    element_tag: str
    attribute: str
    value: str
    target: Optional[tuple] = None
    resolved: bool = False
    resolution_status: str = "unprocessed"

@dataclass
class TransformationRule:
    rule_id: str
    name: Optional[str]

    rule_type: str

    source_elements: list = field(default_factory=list)
    target_initializers: list = field(default_factory=list)

    conditions: list = field(default_factory=list)
    inherited_rules: list = field(default_factory=list)

    source_disposition: str = "unclassified"
    target_cardinality: Optional[str] = None

@dataclass
class CrosswalkRecord:
    source: tuple
    target: tuple

    mapping_type: str

    transformation_rule: Optional[tuple] = None
    condition: Optional[str] = None

    confidence: str = "explicit"
    evidence: list = field(default_factory=list)

class MetamodelRegistry:

    def __init__(self):
        # Internal registry key -> ElementRecord
        self.elements = {}
        # Original XMI ID -> list of internal registry keys
        self.elements_by_xmi_id = defaultdict(list)
        self.references = []
        self.resources = {}
        self.uri_aliases = {}

    # Resources
    def add_resource(self, name, path):
        self.resources[name] = str(path)

    # Elements
    def add_element(self, element):
        """
        Register an element without requiring its XMI ID
        to be unique within the resource.

        Internal key:
            (resource, xmi_id, occurrence)

        Occurrence is a 1-based index for that XMI ID
        within the resource.
        """
        occurrence = (len(self.elements_by_xmi_id[(element.resource, element.xmi_id)]) + 1)
        key = (element.resource, element.xmi_id, occurrence)
        element.registry_key = key
        self.elements[key] = element
        self.elements_by_xmi_id[(element.resource, element.xmi_id)].append(key)
        # Maintain parent-child relationships.
        if element.parent is not None:
            parent = self.elements.get(element.parent)
            if parent is None:
                raise KeyError(f"Parent element not found: {element.parent}")
            parent.children.append(key)
        return key

    def get_element(self, resource, xmi_id, occurrence=None):
        """
        Retrieve an element by its original XMI ID.
        If the ID is duplicated, an occurrence must be
        specified to avoid silently selecting an element.
        """

        keys = self.elements_by_xmi_id.get((resource, xmi_id), [])
        if not keys:
            return None
        if occurrence is not None:
            if occurrence < 1 or occurrence > len(keys):
                return None
            return self.elements[keys[occurrence - 1]]
        if len(keys) > 1:
            raise ValueError(
                f"Ambiguous XMI ID '{xmi_id}' in '{resource}'. "
                f"Found {len(keys)} elements. "
                f"Specify occurrence=1 through {len(keys)}."
            )
        return self.elements[keys[0]]
    
    def get_element_by_key(self, key):
        return self.elements.get(key)
    
    def get_elements_by_xmi_id(self, resource, xmi_id):
        keys = self.elements_by_xmi_id.get((resource, xmi_id), [])
        return [self.elements[key] for key in keys]

    def get_elements(self, resource=None):
        if resource is None:
            return list(self.elements.values())
        return [element for element in self.elements.values() if element.resource == resource]


    # References
    def get_references(self, source=None):
        if source is None:
            return self.references
        return [ref for ref in self.references if ref.source == source]


    # Queries
    def get_by_name(self, name, resource=None):
        return [element for element in self.elements.values() if element.name == name and (resource is None or element.resource == resource)]

    def get_by_metaclass(self, metaclass, resource=None):
        return [element for element in self.elements.values() if element.metaclass == metaclass and (resource is None or element.resource == resource)]

    # Duplicate ID diagnostics
    def get_duplicate_ids(self, resource=None):
        duplicates = {}
        for (res, xmi_id), keys in self.elements_by_xmi_id.items():
            if len(keys) > 1 and (resource is None or resource == res):
                duplicates[(res, xmi_id)] = [self.elements[key] for key in keys]
        return duplicates

    def build_qualified_names(self):
        """Build qualified names from named containment ancestors."""
        for element in self.elements.values():
            components = []
            ancestors = []
            current = element
            visited = set()
            while current is not None:
                if current.key in visited:
                    raise ValueError(f"Containment cycle detected at {current.key}")

                visited.add(current.key)
                ancestors.append(current)
                current = (self.elements.get(current.parent) if current.parent is not None else None)
            # Reverse so names run from outermost package to element.
            ancestors.reverse()
            for ancestor in ancestors:
                if ancestor.name:
                    components.append(ancestor.name)
            element.qualified_name = "::".join(components)
            # Namespace is the qualified name of named ancestors excluding the element itself.
            parent_components = [ancestor.name for ancestor in ancestors[:-1] if ancestor.name]
            element.namespace = "::".join(parent_components) or None

    def _set_reference_result(self, ref, target=None, status="unresolved"):
        ref.target = target
        ref.resolved = target is not None
        ref.resolution_status = status

    def _resolve_idref(self, ref):
        """Resolve an idref within the source resource."""
        source_resource = ref.source[0]
        candidates = self.elements_by_xmi_id.get((source_resource, ref.value), [])
        if len(candidates) == 1:
            self._set_reference_result(ref, target=candidates[0], status="resolved_local")
        elif len(candidates) > 1:
            self._set_reference_result(ref, status="ambiguous_target")
        else:
            self._set_reference_result(ref, status="missing_local_target")

    def _resolve_href(self, ref):
        """Resolve an href using explicitly registered URI aliases."""
        parsed = urlsplit(ref.value)
        fragment = unquote(parsed.fragment)
        base_uri = parsed._replace(fragment="").geturl()
        if not fragment:
            self._set_reference_result(ref, status="href_missing_fragment") 
            return

        # A fragment-only href refers to the source resource.
        if not base_uri:
            target_resource = ref.source[0]
        else:
            target_resource = self.uri_aliases.get(base_uri)
        if target_resource is None:
            self._set_reference_result(ref, status="external_resource_unregistered")
            return
        candidates = self.elements_by_xmi_id.get((target_resource, fragment), [])
        if len(candidates) == 1:
            self._set_reference_result(ref, target=candidates[0], status="resolved_external")
        elif len(candidates) > 1:
            self._set_reference_result(ref, status="ambiguous_external_target")
        else:
            self._set_reference_result(ref, status="missing_external_target")

    def resolve_references(self):
        """Resolve all idref and href references."""
        for ref in self.references:
            if ref.attribute == "idref":
                self._resolve_idref(ref)
            elif ref.attribute == "href":
                self._resolve_href(ref)
            else:
                self._set_reference_result(ref, status="unsupported_reference_attribute")

    def reference_summary(self):
        """Summarize reference-resolution outcomes by resource."""
        summary = defaultdict(Counter)
        for ref in self.references:
            resource = ref.source[0]
            summary[resource][ref.resolution_status] += 1
        return dict(summary)

    def href_uri_inventory(self):
        """Count href base URIs by source resource."""
        inventory = defaultdict(Counter)
        for ref in self.references:
            if ref.attribute != "href":
                continue
            parsed = urlsplit(ref.value)
            base_uri = parsed._replace(fragment="").geturl()
            inventory[ref.source[0]][base_uri] += 1
        return {resource: dict(counts) for resource, counts in inventory.items()}

## XMI Extraction Function:

In [2]:
import xml.etree.ElementTree as ET

def local_name(value):
    """Remove an XML namespace from a tag or attribute name."""
    return value.split("}", 1)[-1]

def get_xmi_attribute(element, attribute_name):
    """Retrieve an attribute from the XMI namespace."""
    for key, value in element.attrib.items():
        if (key.startswith("{") and "XMI" in key.split("}", 1)[0].upper() and local_name(key) == attribute_name):
            return value
    return None

class XMIExtractor:

    def __init__(self, registry):
        self.registry = registry

    def extract(self, resource, path):
        root = ET.parse(path).getroot()
        self.registry.add_resource(resource, path)
        self._visit(root, resource)
        return root

    def _visit(self, element, resource, parent_key=None):
        xmi_id = get_xmi_attribute(element, "id")
        current_key = parent_key
        if xmi_id is not None:
            record = ElementRecord(
                resource=resource,
                xmi_id=xmi_id,
                tag=local_name(element.tag),
                metaclass=get_xmi_attribute(element, "type"),
                name=element.attrib.get("name"),
                attributes=dict(element.attrib),
                parent=parent_key
            )
            current_key = self.registry.add_element(record)
        # Capture references.
        for attr, value in element.attrib.items():
            attr_name = local_name(attr)
            if attr_name in {"idref", "href"}:
                if current_key is None:
                    continue
                self.registry.references.append(
                    ReferenceRecord(source=current_key, element_tag=local_name(element.tag), attribute=attr_name, value=value)
                )
        # Recursively traverse children.
        for child in element:
            self._visit(child, resource, current_key)

## Population and Counting Function:

In [3]:
from pathlib import Path

parent = Path(
    "C:/Users/Drew/Documents/MyFiles/repo/sysml/lang_files/"
)

processed = parent / "processed"

paths = {
    # UML resources
    "UML_Metamodel": processed / "uml_metamodel.xmi",
    "UML_PrimitiveTypes": processed / "uml_primativetypes.xmi",
    "UML_StandardProfile": processed / "uml_standardprofile.xmi",
    "UML_DiagramInterchange": processed / "uml_diagraminterchange.xmi",

    # SysML and KerML resources
    "SysML_v1": processed / "sysml_1.xmi",
    "KerML": processed / "kerml.xmi",
    "SysML_v2": processed / "sysml_2.xmi",

    # Transformation resource
    "SysML_v1_to_v2": processed / "sysml_1_to_2.xmi",
}

registry = MetamodelRegistry()
extractor = XMIExtractor(registry)

for resource, path in paths.items():

    if not path.is_file():
        raise FileNotFoundError(path)

    extractor.extract(resource, path)

    print(
        f"{resource}: "
        f"{len(registry.get_elements(resource))} elements"
    )

print("Total elements:", len(registry.elements))
print("Total references:", len(registry.references))

registry.build_qualified_names()

# Resolve local idrefs and any hrefs with currently registered aliases.
registry.resolve_references()

print("REFERENCE SUMMARY")
for resource, counts in registry.reference_summary().items():
    print(f"\n{resource}: {dict(counts)}")


UML_Metamodel: 5201 elements
UML_PrimitiveTypes: 12 elements
UML_StandardProfile: 177 elements
UML_DiagramInterchange: 319 elements
SysML_v1: 682 elements
KerML: 2294 elements
SysML_v2: 2463 elements
SysML_v1_to_v2: 14689 elements
Total elements: 25837
Total references: 18918
REFERENCE SUMMARY

UML_Metamodel: {'resolved_local': 2293, 'external_resource_unregistered': 200, 'ambiguous_target': 6}

UML_StandardProfile: {'external_resource_unregistered': 35}

UML_DiagramInterchange: {'external_resource_unregistered': 67}

SysML_v1: {'external_resource_unregistered': 127, 'resolved_local': 474}

KerML: {'resolved_local': 1610, 'external_resource_unregistered': 102}

SysML_v2: {'resolved_local': 1804, 'external_resource_unregistered': 249}

SysML_v1_to_v2: {'resolved_local': 9615, 'external_resource_unregistered': 2336}


In [4]:
from collections import defaultdict
from lxml import etree


def find_duplicate_xmi_ids(path):
    root = etree.parse(str(path)).getroot()

    xmi_id_attr = "{http://www.omg.org/spec/XMI/20131001}id"

    occurrences = defaultdict(list)

    for element in root.iter():
        xmi_id = element.get(xmi_id_attr)

        if xmi_id is None:
            continue

        parent = element.getparent()

        occurrences[xmi_id].append({
            "tag": etree.QName(element).localname,
            "name": element.get("name"),
            "parent": (
                parent.get("name")
                if parent is not None
                else None
            ),
            "line": element.sourceline,
        })

    return {
        xmi_id: records
        for xmi_id, records in occurrences.items()
        if len(records) > 1
    }
for resource, path in paths.items():
    duplicates = find_duplicate_xmi_ids(path)

    print(
        f"{resource}: "
        f"{len(duplicates)} duplicated XMI IDs"
    )

    for xmi_id, records in duplicates.items():
        print(f"  {xmi_id}: {records}")

UML_Metamodel: 7 duplicated XMI IDs
  State-isConsistentWith: [{'tag': 'ownedOperation', 'name': 'isConsistentWith', 'parent': 'State', 'line': 3814}, {'tag': 'ownedOperation', 'name': 'isConsistentWith', 'parent': 'Vertex', 'line': 4288}]
  State-isConsistentWith-spec: [{'tag': 'ownedRule', 'name': 'spec', 'parent': 'isConsistentWith', 'line': 3818}, {'tag': 'ownedRule', 'name': 'spec', 'parent': 'isConsistentWith', 'line': 4299}]
  State-isConsistentWith-spec-_specification: [{'tag': 'specification', 'name': None, 'parent': 'spec', 'line': 3820}, {'tag': 'specification', 'name': None, 'parent': 'spec', 'line': 4301}]
  State-isConsistentWith-pre: [{'tag': 'ownedRule', 'name': 'pre', 'parent': 'isConsistentWith', 'line': 3825}, {'tag': 'ownedRule', 'name': 'pre', 'parent': 'isConsistentWith', 'line': 4292}]
  State-isConsistentWith-pre-_specification: [{'tag': 'specification', 'name': None, 'parent': 'pre', 'line': 3827}, {'tag': 'specification', 'name': None, 'parent': 'pre', 'line':

## Qualified Names, Resolution, URI Inventory:

In [5]:
# 1. Qualified-name examples
for resource in registry.resources:
    print(f"\n{'=' * 70}")
    print(resource)
    elements = registry.get_elements(resource)
    named = [e for e in elements if e.name and e.qualified_name]
    for element in named[:20]:
        print(f"{element.qualified_name:<65}", f"[{element.tag}]")

# 2. Reference resolution summary
for resource, counts in registry.reference_summary().items():
    print(f"\n{resource}")
    for status, count in counts.most_common():
        print(f"  {status:<40} {count:>6}")

# 3. External URI inventory
for resource, uris in registry.href_uri_inventory().items():
    print(f"\n{resource}")

    for uri, count in sorted(uris.items()):
        print(f"  {count:>6}  {uri}")


UML_Metamodel
UML                                                               [Package]
UML::Activities                                                   [packagedElement]
UML::Activities::Activity                                         [packagedElement]
UML::Activities::Activity::maximum_one_parameter_node             [ownedRule]
UML::Activities::Activity::maximum_two_parameter_nodes            [ownedRule]
UML::Activities::Activity::edge                                   [ownedAttribute]
UML::Activities::Activity::group                                  [ownedAttribute]
UML::Activities::Activity::isReadOnly                             [ownedAttribute]
UML::Activities::Activity::isSingleExecution                      [ownedAttribute]
UML::Activities::Activity::node                                   [ownedAttribute]
UML::Activities::Activity::partition                              [ownedAttribute]
UML::Activities::Activity::structuredNode                         [ownedAttribute]
UML:

## Alias Validation:

In [6]:
from collections import defaultdict, Counter
from urllib.parse import urlsplit, unquote


def audit_uri_aliases(registry, candidate_aliases):
    """ Test candidate URI -> resource aliases against extracted XMI IDs. Does not modify registry.uri_aliases or resolve references. """
    results = defaultdict(lambda: {"unique": [], "missing": [], "ambiguous": [],})
    for ref in registry.references:
        if ref.attribute != "href":
            continue
        parsed = urlsplit(ref.value)
        base_uri = parsed._replace(fragment="").geturl()
        fragment = unquote(parsed.fragment)
        target_resource = candidate_aliases.get(base_uri)
        if target_resource is None:
            continue
        if not fragment:
            results[base_uri]["missing"].append((ref.value, "Missing fragment"))
            continue
        candidates = registry.elements_by_xmi_id.get((target_resource, fragment), [])
        if len(candidates) == 1:
            results[base_uri]["unique"].append(ref.value)
        elif len(candidates) > 1:
            results[base_uri]["ambiguous"].append((ref.value, candidates))
        else:
            results[base_uri]["missing"].append(ref.value)
    for uri, result in results.items():
        counts = {key: len(value) for key, value in result.items()}
        total = sum(counts.values())
        print(f"\n{uri}")
        print(f"  Candidate resource: {candidate_aliases[uri]}")
        print(f"  Total references:   {total}")
        print(f"  Unique matches:     {counts['unique']}")
        print(f"  Missing:            {counts['missing']}")
        print(f"  Ambiguous:          {counts['ambiguous']}")

        if result["missing"]:
            print("  Missing examples:")
            for value in result["missing"][:5]:
                print(f"    {value}")

        if result["ambiguous"]:
            print("  Ambiguous examples:")
            for value in result["ambiguous"][:5]:
                print(f"    {value}")
                
    return dict(results)

In [ ]:
candidate_aliases = {
    "http://www.omg.org/spec/UML/20131001/UML.xmi": "UML_Metamodel",
    "http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi": "UML_PrimitiveTypes",
    "http://www.omg.org/spec/UML/20131001/StandardProfile.xmi": "UML_StandardProfile",
    "http://www.omg.org/spec/UML/20161101/UML.xmi": "UML_Metamodel",
    "http://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi": "UML_PrimitiveTypes",
    "https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi": "UML_PrimitiveTypes",
    "http://www.omg.org/spec/DD/20131001/DI.xmi": "UML_DiagramInterchange",
    "https://www.omg.org/spec/KerML/20250201/KerML.xmi": "KerML",
    "https://www.omg.org/spec/SysML/20250201/SysML.xmi": "SysML_v2",
}



alias_audit = audit_uri_aliases(registry, candidate_aliases)


http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi
  Candidate resource: UML_PrimitiveTypes
  Total references:   251
  Unique matches:     251
  Missing:            0
  Ambiguous:          0

http://www.omg.org/spec/UML/20161101/UML.xmi
  Candidate resource: UML_Metamodel
  Total references:   51
  Unique matches:     51
  Missing:            0
  Ambiguous:          0

http://www.omg.org/spec/DD/20131001/DI.xmi
  Candidate resource: UML_DiagramInterchange
  Total references:   22
  Unique matches:     1
  Missing:            21
  Ambiguous:          0
  Missing examples:
    http://www.omg.org/spec/DD/20131001/DI.xmi#DiagramElement-ownedElement
    http://www.omg.org/spec/DD/20131001/DI.xmi#DiagramElement-ownedElement
    http://www.omg.org/spec/DD/20131001/DI.xmi#Diagram
    http://www.omg.org/spec/DD/20131001/DI.xmi#DiagramElement
    http://www.omg.org/spec/DD/20131001/DI.xmi#DiagramElement-modelElement

http://www.omg.org/spec/UML/20131001/UML.xmi
  Candidate resource: UML_M

### Alias Writing:

In [ ]:
registry.uri_aliases.update({
    "http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi": "UML_PrimitiveTypes",
    "http://www.omg.org/spec/UML/20161101/UML.xmi": "UML_Metamodel",
    "http://www.omg.org/spec/UML/20131001/UML.xmi": "UML_Metamodel",
    
    "http://www.omg.org/spec/UML/20131001/StandardProfile.xmi": "UML_StandardProfile",
    "https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi": "UML_PrimitiveTypes",
    "https://www.omg.org/spec/KerML/20250201/KerML.xmi": "KerML",
    "https://www.omg.org/spec/SysML/20250201/SysML.xmi": "SysML_v2",
})
registry.uri_aliases["https://www.omg.org/spec/UML/20161101/UML.xmi"] = "UML_Metamodel"
registry.uri_aliases["http://www.omg.org/spec/UML/20161101/UML.xmi"] = "UML_Metamodel"
registry.uri_aliases["https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi"] = "UML_PrimitiveTypes"
registry.uri_aliases["http://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi"] = "UML_PrimitiveTypes"

In [9]:
registry.resolve_references()
for resource, counts in registry.reference_summary().items():
    print(f"\n{resource}: {dict(counts)}")


UML_Metamodel: {'resolved_local': 2293, 'resolved_external': 200, 'ambiguous_target': 6}

UML_StandardProfile: {'resolved_external': 35}

UML_DiagramInterchange: {'external_resource_unregistered': 22, 'resolved_external': 45}

SysML_v1: {'resolved_external': 127, 'resolved_local': 474}

KerML: {'resolved_local': 1610, 'resolved_external': 102}

SysML_v2: {'resolved_local': 1804, 'resolved_external': 249}

SysML_v1_to_v2: {'resolved_local': 9615, 'resolved_external': 1436, 'external_resource_unregistered': 900}


## Transformation ID:

In [10]:
from collections import Counter
transformation_classes = [element for element in registry.get_elements("SysML_v1_to_v2") if element.metaclass == "uml:Class"]
class_packages = Counter(element.namespace for element in transformation_classes)
for package, count in class_packages.most_common():
    print(f"{count:5}  {package}")

   67  SysMLv1Tov2::Mappings::UML4SysML::Activities
   63  SysMLv1Tov2::Mappings::SysMLv1::ModelElements
   57  SysMLv1Tov2::Mappings::UML4SysML::CommonBehavior
   49  SysMLv1Tov2::Mappings::SysMLv1::Requirements
   48  SysMLv1Tov2::Mappings::UML4SysML::Classification
   46  SysMLv1Tov2::Mappings::Initializers::KerMLInitializers
   43  SysMLv1Tov2::Mappings::Factories
   42  SysMLv1Tov2::Mappings::UML4SysML::Actions::ObjectActions
   40  SysMLv1Tov2::Mappings::UML4SysML::Actions::InvocationActions
   39  SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers
   34  SysMLv1Tov2::Mappings::UML4SysML::Packages
   34  SysMLv1Tov2::Mappings::UML4SysML::Actions::StructuralFeatureActions
   33  SysMLv1Tov2::Mappings::UML4SysML::Values
   32  SysMLv1Tov2::Mappings::Initializers::SystemInitializers
   31  SysMLv1Tov2::Mappings::UML4SysML::Actions::VariableActions
   26  SysMLv1Tov2::Mappings::UML4SysML::StateMachines
   25  SysMLv1Tov2::Mappings::UML4SysML::Actions::AcceptEventActions
   22  S

### Evaluate Inheritance Structure:

In [11]:
# Id transformation base classes
resource = "SysML_v1_to_v2"
classes = [e for e in registry.get_elements(resource) if e.metaclass == "uml:Class"]
base_names = {"Mapping", "Initializer", "Factory", "MainMapping"}
for element in sorted(classes, key=lambda e: e.qualified_name or ""):
    if element.name in base_names:
        print(f"{element.name:<20}", element.qualified_name, element.key)

def extract_inheritance(registry, resource):
    """Extract class inheritance from resolved XMI generalizations."""
    edges = []
    unresolved = []
    for gen in registry.get_elements(resource):
        if gen.tag != "generalization":
            continue
        child = registry.elements.get(gen.parent)
        if child is None:
            continue
        refs = [ref for ref in registry.get_references(gen.key) if ref.attribute in {"idref", "href"}]
        general_refs = [ref for ref in refs if ref.resolved and ref.target is not None]
        if not general_refs:
            unresolved.append(gen)
            continue
        for ref in general_refs:
            parent = registry.elements.get(ref.target)
            if parent is None:
                unresolved.append(gen)
                continue
            edges.append({"child": child, "parent": parent, "generalization": gen,})
    return edges, unresolved

inheritance_edges, unresolved_generalizations = (extract_inheritance(registry, "SysML_v1_to_v2"))
print("Inheritance relationships:", len(inheritance_edges))
print("Unresolved generalizations:", len(unresolved_generalizations))

Factory              SysMLv1Tov2::Mappings::Foundations::Factory ('SysML_v1_to_v2', 'Mappings-Foundations-Factory', 1)
Initializer          SysMLv1Tov2::Mappings::Foundations::Initializer ('SysML_v1_to_v2', 'Mappings-Foundations-Initializer', 1)
MainMapping          SysMLv1Tov2::Mappings::Foundations::MainMapping ('SysML_v1_to_v2', 'Mappings-Foundations-MainMapping', 1)
Mapping              SysMLv1Tov2::Mappings::Foundations::Mapping ('SysML_v1_to_v2', 'Mappings-Foundations-Mapping', 1)
Inheritance relationships: 1588
Unresolved generalizations: 0


### Verify edges and targets, inspect inheritance:

In [12]:
from collections import Counter
print("Extracted inheritance edges:", len(inheritance_edges))
print("Generalization elements:", sum(e.tag == "generalization" for e in registry.get_elements("SysML_v1_to_v2")))
print("Target metaclasses:", Counter(edge["parent"].metaclass for edge in inheritance_edges))
print("Child metaclasses:", Counter(edge["child"].metaclass for edge in inheritance_edges))

base_names = {"Mapping", "Initializer", "Factory", "MainMapping"}
base_classes = {e.name: e for e in registry.get_elements("SysML_v1_to_v2") if e.metaclass == "uml:Class" and e.name in base_names}
for edge in inheritance_edges:
    if edge["child"].key in {element.key for element in base_classes.values()}:
        print(
            f"{edge['child'].qualified_name}"
            f"  ->  {edge['parent'].qualified_name}"
        )

Extracted inheritance edges: 1588
Generalization elements: 1588
Target metaclasses: Counter({'uml:Class': 1588})
Child metaclasses: Counter({'uml:Class': 1588})
SysMLv1Tov2::Mappings::Foundations::MainMapping  ->  SysMLv1Tov2::Mappings::Foundations::UniqueMapping
SysMLv1Tov2::Mappings::Foundations::Mapping  ->  SysMLv1Tov2::Mappings::Foundations::Initializer
SysMLv1Tov2::Mappings::Foundations::Factory  ->  SysMLv1Tov2::Mappings::Foundations::Initializer


## Goals:

- Which classes in the transformation file specify "to" and "from"
- Is initializer for inputs?
- Mapping vs initializer?
- What is factory doing?
- How do unique mapping and main mapping interact?

In [13]:
from collections import defaultdict, deque

def build_inheritance_index(inheritance_edges):
    parents = defaultdict(list)
    children = defaultdict(list)
    for edge in inheritance_edges:
        child = edge["child"]
        parent = edge["parent"]
        parents[child.key].append(parent)
        children[parent.key].append(child)
    return parents, children

def get_ancestors(element, parents):
    """Return all transitive ancestors, preserving their defining records."""
    ancestors = {}
    queue = deque(parents[element.key])
    while queue:
        current = queue.popleft()
        if current.key in ancestors:
            continue
        ancestors[current.key] = current
        queue.extend(parents[current.key])
    return list(ancestors.values())

def get_direct_members(element, registry):
    """Return elements directly contained by a class."""
    return [member for member in registry.get_elements(element.resource) if member.parent == element.key and member.tag != "generalization"]

def inspect_inherited_members(element, registry, parents):
    """List direct and inherited members, retaining their defining class."""
    classes = [element] + sorted(get_ancestors(element, parents), key=lambda e: e.qualified_name or "")
    rows = []
    for defining_class in classes:
        for member in get_direct_members(defining_class, registry):
            rows.append({
                "requested_class": element.qualified_name,
                "defined_in": defining_class.qualified_name,
                "inherited": defining_class.key != element.key,
                "member_name": member.name,
                "member_tag": member.tag,
                "member_metaclass": member.metaclass,
                "member_qualified_name": member.qualified_name,
                "member_key": member.key,
            })
    return rows

parents, children = build_inheritance_index(inheritance_edges)
for name in ["Mapping", "Initializer", "Factory", "MainMapping"]:
    element = base_classes[name]
    print(f"\n{'=' * 80}")
    print(element.qualified_name)
    rows = inspect_inherited_members(element, registry, parents)
    for row in rows:
        status = "inherited" if row["inherited"] else "direct"
        print(
            f"{status:10} | "
            f"{row['defined_in']} | "
            f"{row['member_tag']:20} | "
            f"{row['member_name']}"
        )


SysMLv1Tov2::Mappings::Foundations::Mapping
direct     | SysMLv1Tov2::Mappings::Foundations::Mapping | ownedAttribute       | from
direct     | SysMLv1Tov2::Mappings::Foundations::Mapping | ownedOperation       | filter
direct     | SysMLv1Tov2::Mappings::Foundations::Mapping | ownedOperation       | getMappedColl
direct     | SysMLv1Tov2::Mappings::Foundations::Mapping | ownedOperation       | getMapped
direct     | SysMLv1Tov2::Mappings::Foundations::Mapping | ownedOperation       | getMapped
inherited  | SysMLv1Tov2::Mappings::Foundations::Initializer | ownedAttribute       | inputs
inherited  | SysMLv1Tov2::Mappings::Foundations::Initializer | ownedAttribute       | to

SysMLv1Tov2::Mappings::Foundations::Initializer
direct     | SysMLv1Tov2::Mappings::Foundations::Initializer | ownedAttribute       | inputs
direct     | SysMLv1Tov2::Mappings::Foundations::Initializer | ownedAttribute       | to

SysMLv1Tov2::Mappings::Foundations::Factory
inherited  | SysMLv1Tov2::Mappings::Found

In [14]:
def inspect_attribute(attribute, registry):
    print(f"\n{attribute.qualified_name}")
    print("  XMI ID:", attribute.xmi_id)
    print("  Attributes:", attribute.attributes)

    for child_key in attribute.children:
        child = registry.elements.get(child_key)
        if child is not None:
            print(" Child:", child.tag, child.metaclass, child.name, child.attributes)

def inspect_property_references(attribute, registry):
    print(f"\n{attribute.qualified_name}")
    refs = registry.get_references(attribute.key)
    if not refs:
        print("  No references recorded directly on property.")
        return
    for ref in refs:
        target = registry.elements.get(ref.target) if ref.target else None
        print(f" attribute={ref.attribute}, " f"value={ref.value}, " f"status={ref.resolution_status}")
        if target:
            print(f" target={target.qualified_name}, " f"metaclass={target.metaclass}")
"""
for class_name in ["Mapping", "Initializer"]:
    cls = base_classes[class_name]
    print(f"\n{'=' * 70}")
    print(cls.qualified_name)

    for member in get_direct_members(cls, registry):
        if member.tag == "ownedAttribute":
            inspect_attribute(member, registry)
"""
for class_name in ["Mapping", "Initializer"]:
    cls = base_classes[class_name]

    for member in get_direct_members(cls, registry):
        if member.tag == "ownedAttribute":
            inspect_property_references(member, registry)


SysMLv1Tov2::Mappings::Foundations::Mapping::from
 attribute=idref, value=Mappings-Foundations-A_Mapping_from, status=resolved_local
 target=SysMLv1Tov2::Mappings::Foundations::A_Mapping_from, metaclass=uml:Association
 attribute=href, value=https://www.omg.org/spec/UML/20161101/UML.xmi#Element, status=external_resource_unregistered

SysMLv1Tov2::Mappings::Foundations::Initializer::inputs
  No references recorded directly on property.

SysMLv1Tov2::Mappings::Foundations::Initializer::to
 attribute=idref, value=Mappings-Foundations-A_Initializer_to, status=resolved_local
 target=SysMLv1Tov2::Mappings::Foundations::A_Initializer_to, metaclass=uml:Association
 attribute=href, value=https://www.omg.org/spec/KerML/20250201/KerML.xmi#Root-Elements-Element, status=resolved_external
 target=KerML::Root::Elements::Element, metaclass=uml:Class


In [15]:
# There is one missing reference
uml_element_candidates = registry.elements_by_xmi_id.get(
    ("UML_Metamodel", "Element"), []
)

print("UML Element candidates:", uml_element_candidates)

for key in uml_element_candidates:
    element = registry.elements[key]
    print(element.qualified_name, element.metaclass)

registry.uri_aliases["https://www.omg.org/spec/UML/20161101/UML.xmi"] = "UML_Metamodel"
registry.resolve_references()

UML Element candidates: [('UML_Metamodel', 'Element', 1)]
UML::CommonStructure::Element uml:Class


## Initializer:

In [16]:
def inspect_class_members(cls, registry):
    members = get_direct_members(cls, registry)
    print(f"\n{'=' * 80}")
    print(f"CLASS: {cls.qualified_name}")
    print(f"{'=' * 80}")
    for member in members:
        print(f"\n{member.name or '<unnamed>'}")
        print(f"  Tag: {member.tag}")
        print(f"  Metaclass: {member.metaclass}")
        print(f"  XMI ID: {member.xmi_id}")
        print(f"  Attributes: {member.attributes}")
        # Directly contained elements
        children = [element for element in registry.get_elements(member.resource) if element.parent == member.key]
        for child in children:
            print(
                f"    Child: {child.tag} "
                f"| metaclass={child.metaclass} "
                f"| name={child.name} "
                f"| attributes={child.attributes}"
            )

        # References originating from this member
        refs = registry.get_references(member.key)

        for ref in refs:
            target = registry.elements.get(ref.target) if ref.target else None

            print(
                f"    Reference: {ref.element_tag} "
                f"| value={ref.value} "
                f"| status={ref.resolution_status} "
                f"| target={target.qualified_name if target else None}"
            )

matches = registry.get_by_name("Initializer")
initializer = next(element for element in matches if element.qualified_name == "SysMLv1Tov2::Mappings::Foundations::Initializer")
inspect_class_members(initializer, registry)


CLASS: SysMLv1Tov2::Mappings::Foundations::Initializer

inputs
  Tag: ownedAttribute
  Metaclass: uml:Property
  XMI ID: Mappings-Foundations-Initializer-inputs
  Attributes: {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Foundations-Initializer-inputs', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Property', 'isDerived': 'true', 'isDerivedUnion': 'true', 'name': 'inputs'}
    Child: lowerValue | metaclass=uml:LiteralInteger | name= | attributes={'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Foundations-Initializer-inputs-lowerValue', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:LiteralInteger', 'name': ''}
    Child: upperValue | metaclass=uml:LiteralUnlimitedNatural | name= | attributes={'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Foundations-Initializer-inputs-upperValue', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:LiteralUnlimitedNatural', 'name': '', 'value': '-1'}

to
  Tag: ownedAttribute
  Metaclass: uml:Property
  XMI ID: Mappings

In [17]:
mapping_matches = [
    e for e in registry.get_elements()
    if e.qualified_name ==
    "SysMLv1Tov2::Mappings::Foundations::Mapping"
]
mapping = next(
    e for e in registry.get_elements()
    if e.qualified_name ==
       "SysMLv1Tov2::Mappings::Foundations::Mapping"
    and e.metaclass == "uml:Class"
)

print("Matches:", len(mapping_matches))

for e in mapping_matches:
    print(
        "Resource:", e.resource,
        "| Name:", e.qualified_name,
        "| Metaclass:", e.metaclass,
        "| XMI ID:", e.xmi_id
    )

Matches: 2
Resource: SysML_v1_to_v2 | Name: SysMLv1Tov2::Mappings::Foundations::Mapping | Metaclass: uml:Class | XMI ID: Mappings-Foundations-Mapping
Resource: SysML_v1_to_v2 | Name: SysMLv1Tov2::Mappings::Foundations::Mapping | Metaclass: uml:Generalization | XMI ID: Mappings-Foundations-Mapping-_generalization.Mappings-Foundations-Initializer


In [18]:
def get_class(registry, qualified_name):
    matches = [e for e in registry.get_elements() if e.qualified_name == qualified_name and e.metaclass == "uml:Class"]
    if len(matches) != 1:
        raise ValueError(f"Expected one class for {qualified_name}; " f"found {len(matches)}")
    return matches[0]

mapping = get_class(registry, "SysMLv1Tov2::Mappings::Foundations::Mapping")
initializer = get_class(registry, "SysMLv1Tov2::Mappings::Foundations::Initializer")

# Mapping Takeaways:

- Initializer: Defines the mapping through inputs and to 
- Mapping: Adds source filterand and general mapping
- Concrete mapping subclasses: Can specialize filtering

# Inspections:

In [19]:

parents_index, children_index = build_inheritance_index(inheritance_edges)
class_names = ["MainMapping", "UniqueMapping", "Factory"]
for name in class_names:
    cls = get_class(registry, f"SysMLv1Tov2::Mappings::Foundations::{name}")
    print(f"\n{'=' * 70}")
    print(cls.qualified_name)
    print("Direct parents:")
    for parent in parents_index[cls.key]:
        print("  ", parent.qualified_name)
    print("Direct children:")
    for child in children_index[cls.key]:
        print("  ", child.qualified_name)




SysMLv1Tov2::Mappings::Foundations::MainMapping
Direct parents:
   SysMLv1Tov2::Mappings::Foundations::UniqueMapping
Direct children:
   SysMLv1Tov2::Mappings::UML4SysML::CommonStructure::ElementMain_Mapping

SysMLv1Tov2::Mappings::Foundations::UniqueMapping
Direct parents:
   SysMLv1Tov2::Mappings::Foundations::Mapping
Direct children:
   SysMLv1Tov2::Mappings::Foundations::MainMapping
   SysMLv1Tov2::Mappings::SysMLv1::Requirements::TestCaseVerifyObjectiveRequirementUsage_Mapping
   SysMLv1Tov2::Mappings::SysMLv1::Requirements::TestCaseVerifyObjectiveMembership_Mapping
   SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationTargetReferenceUsage_Mapping
   SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationSourceReferenceUsage_Mapping
   SysMLv1Tov2::Mappings::UML4SysML::StateMachines::TimeTriggerReferenceUsage2_Mapping
   SysMLv1Tov2::Mappings::UML4SysML::StateMachines::ChangeTriggerReferenceUsage2_Mapping
   SysMLv1Tov2::Mappings::UML4SysML::StateMachines::TransitionTriggerFe

In [20]:
sample_classes = [
    "SysMLv1Tov2::Mappings::UML4SysML::CommonStructure::ElementMain_Mapping",
    "SysMLv1Tov2::Mappings::Factories::ReturnParameterFeature_Factory",
    "SysMLv1Tov2::Mappings::Factories::LiteralString_Factory",
]

for qualified_name in sample_classes:
    cls = get_class(registry, qualified_name)
    print(f"\n{'=' * 75}")
    print(cls.qualified_name)
    print("Direct parents:")
    for parent in parents_index[cls.key]:
        print("  ", parent.qualified_name)
    print("Direct members:")
    for member in get_direct_members(cls, registry):
        print(f" {member.tag}: {member.name or '<unnamed>'}" f" | {member.metaclass}")


SysMLv1Tov2::Mappings::UML4SysML::CommonStructure::ElementMain_Mapping
Direct parents:
   SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init
   SysMLv1Tov2::Mappings::Foundations::MainMapping
Direct members:
 ownedAttribute: from | uml:Property
 ownedOperation: ownedRelationship | uml:Operation
 ownedOperation: elementId | uml:Operation

SysMLv1Tov2::Mappings::Factories::ReturnParameterFeature_Factory
Direct parents:
   SysMLv1Tov2::Mappings::Foundations::Factory
   SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init
Direct members:
 ownedOperation: create | uml:Operation
 ownedOperation: direction | uml:Operation

SysMLv1Tov2::Mappings::Factories::LiteralString_Factory
Direct parents:
   SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToExpression_Init
   SysMLv1Tov2::Mappings::Foundations::Factory
Direct members:
 ownedAttribute: string | uml:Property
 ownedAttribute: to | uml:Property
 ownedOperation: create | uml:Operation
 ownedOpera

In [21]:
initializer_names = [
    "SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init",
    "SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init",
    "SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToExpression_Init",
]

for qualified_name in initializer_names:
    cls = get_class(registry, qualified_name)

    print(f"\n{'=' * 75}")
    print(cls.qualified_name)

    print("Direct parents:")
    for parent in parents_index[cls.key]:
        print("  ", parent.qualified_name)

    print("Direct members:")
    for member in get_direct_members(cls, registry):
        print(f" {member.tag}: {member.name or '<unnamed>'}" f" | {member.metaclass}")


SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init
Direct parents:
   SysMLv1Tov2::Mappings::Foundations::Initializer
Direct members:
 ownedAttribute: to | uml:Property
 ownedOperation: elementId | uml:Operation
 ownedOperation: ownedRelationship | uml:Operation
 ownedOperation: aliasId | uml:Operation
 ownedOperation: shortName | uml:Operation
 ownedOperation: declaredName | uml:Operation
 ownedRule: from_and_to_types | uml:Constraint

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init
Direct parents:
   SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToType_Init
Direct members:
 ownedAttribute: to | uml:Property
 ownedOperation: isUnique | uml:Operation
 ownedOperation: isOrdered | uml:Operation
 ownedOperation: isComposite | uml:Operation
 ownedOperation: isPortion | uml:Operation
 ownedOperation: isDerived | uml:Operation
 ownedOperation: isReadOnly | uml:Operation
 ownedOperation: isEnd | uml:Operation
 ownedOperation: direction | u

# OCL Extraction:

In [22]:
from collections import defaultdict, deque

def get_containment_children(element, registry):
    resource = element.key[0]
    return [candidate for candidate in registry.get_elements(resource) if candidate.parent == element.key]

def get_containment_descendants(element, registry):
    resource = element.key[0]
    children_by_parent = defaultdict(list)
    for candidate in registry.get_elements(resource):
        if candidate.parent is not None:
            children_by_parent[candidate.parent].append(candidate)
    descendants = []
    stack = list(children_by_parent[element.key])
    while stack:
        current = stack.pop()
        descendants.append(current)
        stack.extend(children_by_parent[current.key])
    return descendants

def get_operation_parameters(operation):
    parameters = []
    for member in direct_members(operation):
        if member.metaclass != "uml:Parameter":
            continue
        refs = registry.get_references(member.key)
        parameters.append({
            "name": member.name,
            "direction": member.attributes.get("direction"),
            "type_references": [
                {
                    "value": ref.value,
                    "target": (registry.elements[ref.target].qualified_name if ref.resolved and ref.target in registry.elements else None),
                    "status": ref.resolution_status,
                }
                for ref in refs
            ],
        })
    return parameters

def get_inheritance_descendants(root, inheritance_edges):
    """Return all transitive subclasses of a class."""
    children = defaultdict(list)
    for edge in inheritance_edges:
        children[edge["parent"].key].append(edge["child"])
    descendants = {}
    queue = deque([root.key])
    while queue:
        current = queue.popleft()
        for child in children[current]:
            if child.key == root.key:
                continue
            if child.key in descendants:
                continue
            descendants[child.key] = child
            queue.append(child.key)
    return list(descendants.values())

In [23]:
cls = get_class(
    registry,
    "SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init"
)
for member in get_direct_members(cls, registry):
    if member.metaclass not in {"uml:Operation", "uml:Constraint"}:
        continue
    print(f"\n{'=' * 70}")
    print(f"{member.name} ({member.metaclass})")
    for descendant in get_containment_descendants(member, registry):
        if descendant.metaclass == "uml:OpaqueExpression":
            print("Language:", descendant.attributes.get("language"))
            print("Body:", descendant.attributes.get("body"))


elementId (uml:Operation)
Language: OCL2.0
Body: result = Helper.createUUID()

ownedRelationship (uml:Operation)
Language: OCL2.0
Body: result = Set{}

aliasId (uml:Operation)
Language: OCL2.0
Body: result = Set{}

shortName (uml:Operation)
Language: OCL2.0
Body: result = null

declaredName (uml:Operation)
Language: OCL2.0
Body: result = null

from_and_to_types (uml:Constraint)
Language: OCL2.0
Body: from.oclIsKindOf(factory.srcType) and to.oclIsKindOf(factory.tgtType)


In [24]:
cls = get_class(registry, "SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToType_Init")
print(cls.qualified_name)
for member in get_direct_members(cls, registry):
    print(member.tag, member.name, member.metaclass, member.attributes)

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToType_Init
ownedAttribute to uml:Property {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Initializers-KerMLInitializers-ToType_Init-to', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Property', 'name': 'to'}
ownedOperation isAbstract uml:Operation {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Initializers-KerMLInitializers-ToType_Init-isAbstract_', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Operation', 'name': 'isAbstract'}
ownedOperation isSufficient uml:Operation {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Initializers-KerMLInitializers-ToType_Init-isSufficient_', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Operation', 'name': 'isSufficient'}


In [25]:
# OCL expressions in ToType_Init operations
cls = get_class(registry, "SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToType_Init")
for member in get_direct_members(cls, registry):
    if member.metaclass != "uml:Operation":
        continue

    print(f"\n{'=' * 70}")
    print(f"Operation: {member.name}")

    for descendant in get_containment_descendants(member, registry):
        if descendant.metaclass == "uml:OpaqueExpression":
            print("Language:", descendant.attributes.get("language"))
            print("Body:", descendant.attributes.get("body"))


Operation: isAbstract
Language: OCL2.0
Body: result = false

Operation: isSufficient
Language: OCL2.0
Body: result = false


In [26]:
cls = get_class(
    registry,
    "SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToExpression_Init"
)

for member in get_direct_members(cls, registry):
    print(
        member.tag,
        member.name,
        member.metaclass,
        member.attributes
    )

ownedAttribute to uml:Property {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Initializers-KerMLInitializers-ToExpression_Init-to', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Property', 'name': 'to'}


In [27]:


for root in classes:
    if root.name not in base_names:
        continue
    descendants = get_inheritance_descendants(root, inheritance_edges)
    print(f"\n{'=' * 90}")
    print(root.qualified_name)
    print("Descendants:", len(descendants))
    for descendant in sorted(descendants, key=lambda e: e.qualified_name or ""):
        print("  ", descendant.qualified_name)

get_inheritance_descendants(root, inheritance_edges)


SysMLv1Tov2::Mappings::Foundations::Initializer
Descendants: 908
   SysMLv1Tov2::Mappings::CommonMappings::CommonFeatureReferenceExpression_Mapping
   SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping
   SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInFeatureTyping_Mapping
   SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInMembership_Mapping
   SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInUntyped_Mapping
   SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageIn_Mapping
   SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInFeatureMembership_Mapping
   SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInFeatureTyping_Mapping
   SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInUntyped_Mapping
   SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageIn_Mapping
   SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterFeatureMembership_Mapping
   SysML

[ElementRecord(resource='SysML_v1_to_v2', xmi_id='Mappings-UML4SysML-StateMachines-StateBehaviorPerformActionUsage_Mapping', tag='packagedElement', metaclass='uml:Class', name='StateBehaviorPerformActionUsage_Mapping', attributes={'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-UML4SysML-StateMachines-StateBehaviorPerformActionUsage_Mapping', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Class', 'name': 'StateBehaviorPerformActionUsage_Mapping'}, parent=('SysML_v1_to_v2', 'Mappings-UML4SysML-StateMachines', 1), children=[('SysML_v1_to_v2', 'Mappings-UML4SysML-StateMachines-StateBehaviorPerformActionUsage_Mapping-_generalization.Mappings-Foundations-Mapping', 1), ('SysML_v1_to_v2', 'Mappings-UML4SysML-StateMachines-StateBehaviorPerformActionUsage_Mapping-_generalization.Mappings-Initializers-SystemInitializers-ToPerformActionUsage_Init', 1), ('SysML_v1_to_v2', 'Mappings-UML4SysML-StateMachines-StateBehaviorPerformActionUsage_Mapping-from', 1), ('SysML_v1_to_v2', 'Mappings-UML

# Build Initializer Inventory:

In [28]:
import pandas as pd
from collections import defaultdict

INITIALIZER_PREFIX = "SysMLv1Tov2::Mappings::Initializers"

def direct_members(element):
    return [member for member in members_by_parent[element.key] if member.tag != "generalization"]

# Index directly contained elements by parent.
members_by_parent = defaultdict(list)

for element in registry.get_elements("SysML_v1_to_v2"):
    if element.parent is not None:
        members_by_parent[element.parent].append(element)

# Find initializer classes.
initializer_classes = [
    element
    for element in registry.get_elements("SysML_v1_to_v2")
    if element.metaclass == "uml:Class"
    and element.qualified_name
    and element.qualified_name.startswith(INITIALIZER_PREFIX + "::")
]

initializer_classes.sort(key=lambda e: e.qualified_name)

# Build inventory.
initializer_inventory = []

for cls in initializer_classes:
    members = direct_members(cls)
    attributes = [m.name or "<unnamed>" for m in members if m.metaclass == "uml:Property"]
    operations = [m.name or "<unnamed>" for m in members if m.metaclass == "uml:Operation"]
    constraints = [m.name or "<unnamed>" for m in members if m.metaclass == "uml:Constraint"]
    initializer_inventory.append({
        "qualified_name": cls.qualified_name,
        "parents": [p.qualified_name for p in parents_index[cls.key]],
        "attributes": attributes,
        "operations": operations,
        "constraints": constraints,
        "num_attributes": len(attributes),
        "num_operations": len(operations),
        "num_constraints": len(constraints),
    })


initializer_df = pd.DataFrame(initializer_inventory)

print("Initializer classes:", len(initializer_df))
print("Total properties:", initializer_df["num_attributes"].sum())
print("Total operations:", initializer_df["num_operations"].sum())
print("Total constraints:", initializer_df["num_constraints"].sum())

Initializer classes: 78
Total properties: 78
Total operations: 78
Total constraints: 1


In [29]:
for _, row in initializer_df.iterrows():
    print(f"\n{row['qualified_name']}")

    for parent in row["parents"]:
        print(f"  Parent: {parent}")

    print(f"  Properties:  {row['attributes']}")
    print(f"  Operations:  {row['operations']}")
    print(f"  Constraints: {row['constraints']}")


SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToAnnotatingElement_Init
  Parent: SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init
  Properties:  ['to']
  Operations:  ['annotation']
  Constraints: []

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToAnnotation_Init
  Parent: SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init
  Properties:  ['to']
  Operations:  ['annotatingElement', 'annotatedElement', 'owningAnnotatedElement']
  Constraints: []

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToAssociation_Init
  Parent: SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init
  Parent: SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToClassifier_Init
  Properties:  ['to']
  Operations:  []
  Constraints: []

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToBehavior_Init
  Parent: SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToClassifier_Init
  Properties:  ['to']


In [30]:
ocl_candidates = initializer_df.loc[(initializer_df["num_operations"] > 0) | (initializer_df["num_constraints"] > 0)].copy()
ocl_candidates = ocl_candidates.sort_values(["num_constraints", "num_operations"], ascending=False)

## Extract Operation Specifications Across the Whole Framework:

## Look at Operation Specification:

In [31]:
operation_audit = []
for cls in initializer_classes:
    for operation in direct_members(cls):
        if operation.metaclass != "uml:Operation":
            continue
        expressions = [d for d in get_containment_descendants(operation, registry) if d.metaclass == "uml:OpaqueExpression"]
        specifications = []
        for expr in expressions:
            body = expr.attributes.get("body")
            language = expr.attributes.get("language")
            if body is None:
                status = "body_attribute_missing"
            elif not body.strip():
                status = "body_empty"
            elif body.strip().rstrip().endswith("="):
                status = "incomplete_assignment"
            else:
                status = "body_present"
            specifications.append({"language": language, "body": body, "status": status, "expression_id": expr.xmi_id,})
        operation_audit.append({
            "class": cls.qualified_name,
            "operation": operation.name,
            "operation_id": operation.xmi_id,
            "parameters": get_operation_parameters(operation),
            "specifications": specifications,
            "specification_count": len(expressions),
            "has_specification": bool(expressions),
        })
operation_audit_df = pd.DataFrame(operation_audit)

In [32]:
def get_descendant_classes(element, children_index):
    descendants = {}
    stack = list(children_index[element.key])
    while stack:
        current = stack.pop()
        if current.key in descendants:
            continue
        descendants[current.key] = current
        stack.extend(children_index[current.key])
    return list(descendants.values())

def find_operation_overrides(class_name, operation_name, children_index,):
    base = get_class(registry, class_name)
    results = []
    for descendant in get_descendant_classes(base, children_index):
        operations = [member for member in direct_members(descendant) if member.metaclass == "uml:Operation" and member.name == operation_name]
        for operation in operations:
            results.append({"class": descendant.qualified_name, "operation": operation,})
    return results

base_name = ("SysMLv1Tov2::Mappings::Initializers::KerMLInitializers" "::ToAnnotation_Init")
overrides = find_operation_overrides(base_name, "annotatingElement", children_index,)
for item in overrides:
    print(item["class"], item["operation"].xmi_id, item["operation"].attributes,)

SysMLv1Tov2::Mappings::UML4SysML::CommonStructure::CommentOwnership_Mapping Mappings-UML4SysML-CommonStructure-CommentOwnership_Mapping-annotatingElement_ {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-UML4SysML-CommonStructure-CommentOwnership_Mapping-annotatingElement_', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Operation', 'name': 'annotatingElement'}
SysMLv1Tov2::Mappings::UML4SysML::CommonStructure::CommentAnnotation_Mapping Mappings-UML4SysML-CommonStructure-CommentAnnotation_Mapping-annotatingElement_ {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-UML4SysML-CommonStructure-CommentAnnotation_Mapping-annotatingElement_', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Operation', 'name': 'annotatingElement'}
SysMLv1Tov2::Mappings::SysMLv1::Requirements::RefineAnnotation_Mapping Mappings-SysMLv1-Requirements-RefineAnnotation_Mapping-annotatingElement_ {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-SysMLv1-Requirements-RefineAnnotation_Mapping-ann

In [33]:
for item in overrides:
    cls = get_class(registry, item["class"])
    operation = item["operation"]

    print(f"\n{'=' * 80}")
    print(cls.qualified_name)
    print("Operation:", operation.xmi_id)
    print("Attributes:", operation.attributes)

    print("\nReferences:")
    for ref in registry.get_references(operation.key):
        print({
            "element_tag": ref.element_tag,
            "attribute": ref.attribute,
            "value": ref.value,
            "target": (registry.elements[ref.target].qualified_name if ref.target else None),
            "status": ref.resolution_status,
        })

    print("\nParameters:")
    for parameter in direct_members(operation):
        if parameter.metaclass == "uml:Parameter":
            print(parameter.name, parameter.attributes)

            for ref in registry.get_references(parameter.key):
                print("  Reference:", ref.value, ref.resolution_status)

    print("\nSpecifications:")
    for descendant in get_containment_descendants(operation, registry):
        if descendant.metaclass == "uml:OpaqueExpression":
            print(descendant.attributes)


SysMLv1Tov2::Mappings::UML4SysML::CommonStructure::CommentOwnership_Mapping
Operation: Mappings-UML4SysML-CommonStructure-CommentOwnership_Mapping-annotatingElement_
Attributes: {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-UML4SysML-CommonStructure-CommentOwnership_Mapping-annotatingElement_', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Operation', 'name': 'annotatingElement'}

References:
{'element_tag': 'ownedRule', 'attribute': 'idref', 'value': 'Mappings-UML4SysML-CommonStructure-CommentOwnership_Mapping-annotatingElement_-bodyCondition', 'target': 'SysMLv1Tov2::Mappings::UML4SysML::CommonStructure::CommentOwnership_Mapping::annotatingElement::bodyCondition', 'status': 'resolved_local'}
{'element_tag': 'redefinedOperation', 'attribute': 'idref', 'value': 'Mappings-Initializers-KerMLInitializers-ToAnnotation_Init-annotatingElement_', 'target': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToAnnotation_Init::annotatingElement', 'status': 'resolved_local'}

In [34]:
def extract_redefinitions(registry, resource):
    redefinitions = []
    for operation in registry.get_elements(resource):
        if operation.metaclass != "uml:Operation":
            continue
        for ref in registry.get_references(operation.key):
            if ref.element_tag != "redefinedOperation":
                continue
            redefinitions.append({
                "operation": operation,
                "redefined_operation": (registry.elements.get(ref.target) if ref.resolved else None),
                "reference": ref,
                "resolution_status": ref.resolution_status,
            })
    return redefinitions

delegation_targets = [
    ("SysMLv1Tov2::Mappings::UML4SysML::CommonStructure", "Comment_Mapping",),
    ("SysMLv1Tov2::Mappings::SysMLv1::Requirements", "RefineMetadataUsage_Mapping",),
    ("SysMLv1Tov2::Mappings::SysMLv1::Requirements", "TraceMetadataUsage_Mapping",),
]



# Workflow:

- Deligated mapping inspect
- Mapping property extraction
- Operation redefinition 
- OCL extraction / dependencies
- Crosswalk Validation 
- Graph Reconstruction 

### Deligated mapping:

In [35]:
def inspect_property(prop):
    print(f"\nProperty: {prop.qualified_name}")
    print("XMI ID:", prop.xmi_id)
    print("Attributes:", prop.attributes)
    print("References:")
    for ref in registry.get_references(prop.key):
        target = (registry.elements[ref.target].qualified_name if ref.resolved else None)
        print({"element_tag": ref.element_tag, "value": ref.value, "target": target, "status": ref.resolution_status,})
    print("Multiplicity:")
    for child in direct_members(prop):
        if child.tag in {"lowerValue", "upperValue"}:
            print(child.tag, child.attributes)
            
property_ids = [
    "Mappings-UML4SysML-CommonStructure-Comment_Mapping-from",
    "Mappings-UML4SysML-CommonStructure-Comment_Mapping-to",
    "Mappings-SysMLv1-Requirements-RefineMetadataUsage_Mapping-from",
    "Mappings-SysMLv1-Requirements-TraceMetadataUsage_Mapping-from",
]


### Transformation:
UML Comment --> Comment_Mapping --> KerML Comment --> Annotation --> Comment_Annotation_Mapping
Abstraction --> MetadataUsage --> FeatureTyping + FeatureMembership

## Property Mapping and Validation:

In [36]:

import pandas as pd
# Identify mapping classes
mapping_base = get_class(registry, "SysMLv1Tov2::Mappings::Foundations::Mapping")
mapping_classes = []
for element in registry.get_elements("SysML_v1_to_v2"):
    if element.metaclass != "uml:Class":
        continue
    ancestor_keys = {ancestor.key for ancestor in get_ancestors(element, parents_index)}
    if mapping_base.key in ancestor_keys:
        mapping_classes.append(element)
print("Mapping classes identified:", len(mapping_classes))

# Extract property metadata
def get_property_metadata(prop, defining_class, mapping_class):
    refs = registry.get_references(prop.key)
    type_refs = [ref for ref in refs if ref.element_tag == "type"]
    redefined_refs = [ref for ref in refs if ref.element_tag == "redefinedProperty"]
    type_targets = [registry.elements.get(ref.target) if ref.resolved else None for ref in type_refs]
    redefined_targets = [registry.elements.get(ref.target) if ref.resolved else None for ref in redefined_refs]
    children = direct_members(prop)
    lower = next((child.attributes.get("value") for child in children if child.tag == "lowerValue"), None,)
    upper = next((child.attributes.get("value") for child in children if child.tag == "upperValue"), None,)
    return {
        "mapping_class": mapping_class.qualified_name,
        "mapping_class_xmi_id": mapping_class.xmi_id,
        "role": prop.name,
        "property": prop.qualified_name,
        "property_xmi_id": prop.xmi_id,
        "declared_by": defining_class.qualified_name,
        "is_direct": prop.parent == mapping_class.key,
        "property_type": [target.qualified_name if target else None for target in type_targets],
        "property_type_xmi_id": [target.xmi_id if target else None for target in type_targets],
        "type_reference_values": [ref.value for ref in type_refs],
        "type_resolution_status": [ref.resolution_status for ref in type_refs],
        "redefined_property": [target.qualified_name if target else None for target in redefined_targets],
        "redefined_property_xmi_id": [target.xmi_id if target else None for target in redefined_targets],
        "redefinition_resolution_status": [ref.resolution_status for ref in redefined_refs],
        "lower": lower,
        "upper": upper,
    }


# Resolve effective properties through inheritance

property_rows = []
audit_rows = []
for mapping_class in mapping_classes:
    lineage = [mapping_class] + get_ancestors(mapping_class, parents_index,)
    for role in ("from", "to"):

        # Gather declarations of this property throughout the mapping class's inheritance hierarchy.
        declarations = []
        for defining_class in lineage:
            for member in direct_members(defining_class):
                if(member.metaclass == "uml:Property" and member.name == role):
                    declarations.append((member, defining_class))
        candidate_keys = {prop.key for prop, _ in declarations}

        # Identify properties overridden by another declaration in this inheritance hierarchy.
        redefined_keys = set()
        for prop, _ in declarations:
            for ref in registry.get_references(prop.key):
                if (ref.element_tag == "redefinedProperty" and ref.resolved and ref.target in candidate_keys):
                    redefined_keys.add(ref.target)
        # Effective declarations are those not redefined by a morespecific declaration.
        effective = [(prop, defining_class) for prop, defining_class in declarations if prop.key not in redefined_keys]
        if len(effective) == 0:
            status = "missing"
        elif len(effective) == 1:
            status = "unique"
        else:
            status = "ambiguous"
        audit_rows.append({
            "mapping_class": mapping_class.qualified_name,
            "role": role,
            "status": status,
            "declaration_count": len(declarations),
            "effective_property_count": len(effective),
            "effective_properties": [prop.qualified_name for prop, _ in effective],
        })
        for prop, defining_class in effective:
            property_rows.append(get_property_metadata(prop, defining_class, mapping_class,))

# Create DataFrames
mapping_properties = pd.DataFrame(property_rows)
mapping_property_audit = pd.DataFrame(audit_rows)

# Summarize extraction
print("\nEffective property records:", len(mapping_properties))
print("\nProperty audit:")
print(mapping_property_audit["status"] .value_counts(dropna=False))

print("\nProperty records by role:")
print(mapping_properties["role"] .value_counts(dropna=False))

print("\nAmbiguous or missing properties:")
print(mapping_property_audit.loc[mapping_property_audit["status"] != "unique"].to_string(index=False))


Mapping classes identified: 772

Effective property records: 1544

Property audit:
status
unique    1544
Name: count, dtype: int64

Property records by role:
role
from    772
to      772
Name: count, dtype: int64

Ambiguous or missing properties:
Empty DataFrame
Columns: [mapping_class, role, status, declaration_count, effective_property_count, effective_properties]
Index: []


In [37]:
# Saving mapping CSVs
mapping_properties.to_csv("C:/Users/Drew/Documents/MyFiles/repo/sysml/data/mapping_properties.csv", index=False)
mapping_property_audit.to_csv("C:/Users/Drew/Documents/MyFiles/repo/sysml/data/mapping_property_audit.csv", index=False)


## Crosswalk:

In [38]:
# Pivot effective properties into one row per mapping class.
property_crosswalk = (mapping_properties .pivot(
        index=["mapping_class", "mapping_class_xmi_id"],
        columns="role",
        values=["property", "property_xmi_id", "declared_by", "property_type",],
    )
)
property_crosswalk.columns = [f"{field}_{role}" for field, role in property_crosswalk.columns]
property_crosswalk = property_crosswalk.reset_index()
# Convenient source/target labels.
property_crosswalk["source_type"] = property_crosswalk["property_type_from"].str[0]
property_crosswalk["target_type"] = property_crosswalk["property_type_to"].str[0]
property_crosswalk.head()

,mapping_class,mapping_class_xmi_id,property_from,property_to,property_xmi_id_from,property_xmi_id_to,declared_by_from,declared_by_to,property_type_from,property_type_to,source_type,target_type
0,SysMLv1Tov2::Mappings::CommonMappings::CommonF...,Mappings-CommonMappings-CommonFeatureReference...,SysMLv1Tov2::Mappings::CommonMappings::CommonF...,SysMLv1Tov2::Mappings::Initializers::KerMLInit...,Mappings-CommonMappings-CommonFeatureReference...,Mappings-Initializers-KerMLInitializers-ToFeat...,SysMLv1Tov2::Mappings::CommonMappings::CommonF...,SysMLv1Tov2::Mappings::Initializers::KerMLInit...,[UML::CommonStructure::TypedElement],[KerML::Kernel::Expressions::FeatureReferenceE...,UML::CommonStructure::TypedElement,KerML::Kernel::Expressions::FeatureReferenceEx...
1,SysMLv1Tov2::Mappings::CommonMappings::CommonM...,Mappings-CommonMappings-CommonMembership_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonM...,SysMLv1Tov2::Mappings::Initializers::KerMLInit...,Mappings-CommonMappings-CommonMembership_Mappi...,Mappings-Initializers-KerMLInitializers-ToMemb...,SysMLv1Tov2::Mappings::CommonMappings::CommonM...,SysMLv1Tov2::Mappings::Initializers::KerMLInit...,[UML::CommonStructure::TypedElement],[KerML::Root::Namespaces::Membership],UML::CommonStructure::TypedElement,KerML::Root::Namespaces::Membership
2,SysMLv1Tov2::Mappings::CommonMappings::CommonP...,Mappings-CommonMappings-CommonParameterReferen...,SysMLv1Tov2::Mappings::CommonMappings::CommonP...,SysMLv1Tov2::Mappings::Initializers::KerMLInit...,Mappings-CommonMappings-CommonParameterReferen...,Mappings-Initializers-KerMLInitializers-ToFeat...,SysMLv1Tov2::Mappings::CommonMappings::CommonP...,SysMLv1Tov2::Mappings::Initializers::KerMLInit...,[UML::CommonStructure::Element],[KerML::Core::Features::FeatureTyping],UML::CommonStructure::Element,KerML::Core::Features::FeatureTyping
3,SysMLv1Tov2::Mappings::CommonMappings::CommonP...,Mappings-CommonMappings-CommonParameterReferen...,SysMLv1Tov2::Mappings::CommonMappings::CommonP...,SysMLv1Tov2::Mappings::Initializers::KerMLInit...,Mappings-CommonMappings-CommonParameterReferen...,Mappings-Initializers-KerMLInitializers-ToPara...,SysMLv1Tov2::Mappings::CommonMappings::CommonP...,SysMLv1Tov2::Mappings::Initializers::KerMLInit...,[UML::CommonStructure::Element],[KerML::Kernel::Behaviors::ParameterMembership],UML::CommonStructure::Element,KerML::Kernel::Behaviors::ParameterMembership
4,SysMLv1Tov2::Mappings::CommonMappings::CommonP...,Mappings-CommonMappings-CommonParameterReferen...,SysMLv1Tov2::Mappings::CommonMappings::CommonP...,SysMLv1Tov2::Mappings::Initializers::SystemIni...,Mappings-CommonMappings-CommonParameterReferen...,Mappings-Initializers-SystemInitializers-ToRef...,SysMLv1Tov2::Mappings::CommonMappings::CommonP...,SysMLv1Tov2::Mappings::Initializers::SystemIni...,[UML::CommonStructure::Element],[SysML::Systems::DefinitionAndUsage::Reference...,UML::CommonStructure::Element,SysML::Systems::DefinitionAndUsage::ReferenceU...


In [39]:
operation_rows = []

for mapping_class in mapping_classes:
    for operation in direct_members(mapping_class):
        if operation.metaclass != "uml:Operation":
            continue

        refs = registry.get_references(operation.key)

        redefined = [ref for ref in refs if ref.element_tag == "redefinedOperation"]

        specifications = [
            element
            for element in registry.get_elements("SysML_v1_to_v2")
            if element.parent == operation.key
            and element.metaclass == "uml:OpaqueExpression"
        ]

        # Include nested OpaqueExpressions as well.
        descendants = []
        queue = list(direct_members(operation))

        while queue:
            child = queue.pop()
            descendants.append(child)
            queue.extend(direct_members(child))

        specifications = [element for element in descendants if element.metaclass == "uml:OpaqueExpression"]

        operation_rows.append({
            "mapping_class": mapping_class.qualified_name,
            "operation": operation.qualified_name,
            "operation_name": operation.name,
            "operation_xmi_id": operation.xmi_id,
            "redefined_operations": [registry.elements[ref.target].qualified_name if ref.resolved else None for ref in redefined],
            "ocl": [element.attributes.get("body") for element in specifications],
        })

mapping_operations = pd.DataFrame(operation_rows)

print("Direct mapping operations:", len(mapping_operations))
mapping_operations.head()

Direct mapping operations: 946


,mapping_class,operation,operation_name,operation_xmi_id,redefined_operations,ocl
0,SysMLv1Tov2::Mappings::CommonMappings::CommonR...,SysMLv1Tov2::Mappings::CommonMappings::CommonR...,direction,Mappings-CommonMappings-CommonReferenceUsageIn...,[SysMLv1Tov2::Mappings::Initializers::KerMLIni...,[result = KerML::FeatureDirectionKind::_'in']
1,SysMLv1Tov2::Mappings::CommonMappings::CommonR...,SysMLv1Tov2::Mappings::CommonMappings::CommonR...,declaredName,Mappings-CommonMappings-CommonReferenceUsageIn...,[SysMLv1Tov2::Mappings::Initializers::KerMLIni...,[result = from.name]
2,SysMLv1Tov2::Mappings::CommonMappings::CommonR...,SysMLv1Tov2::Mappings::CommonMappings::CommonR...,ownedMemberParameter,Mappings-CommonMappings-CommonReturnParameterR...,[SysMLv1Tov2::Mappings::Initializers::KerMLIni...,[result = if not from.oclIsKindOf(UML::TypedEl...
3,SysMLv1Tov2::Mappings::CommonMappings::CommonR...,SysMLv1Tov2::Mappings::CommonMappings::CommonR...,type,Mappings-CommonMappings-CommonReturnParameterR...,[SysMLv1Tov2::Mappings::Initializers::KerMLIni...,[result = if from.oclIsKindOf(UML::TypedElemen...
4,SysMLv1Tov2::Mappings::CommonMappings::CommonR...,SysMLv1Tov2::Mappings::CommonMappings::CommonR...,ownedRelationship,Mappings-CommonMappings-CommonReturnParameterR...,[SysMLv1Tov2::Mappings::Initializers::KerMLIni...,[result = if from.oclIsKindOf(UML::TypedElemen...


In [40]:
def classify_mapping_operation(row):
    name = row["operation_name"]
    redefined = row["redefined_operations"]

    if name == "filter":
        return "mapping_filter"

    if name in {"getMapped", "getMappedColl"}:
        return "mapping_dispatch"

    if name in {"ownedRelationship", "ownedMemberFeature",
                "ownedMemberElement", "ownedMemberParameter"}:
        return "relationship_construction"

    if redefined:
        return "property_population_candidate"

    return "other"


mapping_operations["operation_category"] = (mapping_operations.apply(classify_mapping_operation, axis=1,))

print(mapping_operations["operation_category"] .value_counts())

operation_category
relationship_construction        444
property_population_candidate    360
mapping_filter                    96
other                             46
Name: count, dtype: int64


In [41]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

mapping_operations[
    mapping_operations["operation_category"]
    == "property_population_candidate"
][
    ["mapping_class", "operation_name", "redefined_operations", "ocl",]
].head(20).to_string(index=False)

"                                                                                            mapping_class   operation_name                                                                            redefined_operations                                                                                                                                                                                                                                                                           ocl\n                             SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInUntyped_Mapping        direction             [SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init::direction]                                                                                                                                                                                                                                 [result = KerML::FeatureDirectionKind::_'in']\n                   

In [42]:

# Index initializer classes and their directly declared properties.
initializer_base = get_class(registry, "SysMLv1Tov2::Mappings::Foundations::Initializer")
initializer_classes = []
for element in registry.get_elements("SysML_v1_to_v2"):
    if element.metaclass != "uml:Class":
        continue
    ancestor_keys = {ancestor.key for ancestor in get_ancestors(element, parents_index)}
    if initializer_base.key in ancestor_keys:
        initializer_classes.append(element)

initializer_property_rows = []
for cls in initializer_classes:
    for prop in direct_members(cls):
        if prop.metaclass != "uml:Property":
            continue
        initializer_property_rows.append(get_property_metadata(prop, cls, cls))

initializer_properties = pd.DataFrame(initializer_property_rows)

print("Initializer classes:", len(initializer_classes))
print("Initializer properties:", len(initializer_properties))
initializer_properties[["mapping_class", "role", "property_type", "lower", "upper",]].head(20)

Initializer classes: 908
Initializer properties: 1006


,mapping_class,role,property_type,lower,upper
0,SysMLv1Tov2::Mappings::Foundations::Mapping,from,[UML::CommonStructure::Element],1,1
1,SysMLv1Tov2::Mappings::Factories::StringParameterFeatureValue_Factory,string,[PrimitiveTypes::String],1,1
2,SysMLv1Tov2::Mappings::Factories::DirectedReferenceUsage_Factory,featureDirectionKind,[KerML::Core::Types::FeatureDirectionKind],1,1
3,SysMLv1Tov2::Mappings::Factories::FeatureTyping_Factory,type,[UML::CommonStructure::NamedElement],1,1
4,SysMLv1Tov2::Mappings::Factories::LiteralNull_Factory,to,[KerML::Kernel::Expressions::NullExpression],1,1
5,SysMLv1Tov2::Mappings::Factories::MultiplicityUpperBoundMembership_Factory,multiplicityElement,[SysMLv1Tov2::Mappings::Factories::MultiplicityElement_Factory],1,1
6,SysMLv1Tov2::Mappings::Factories::MultiplicityUpperBoundMembership_Factory,upperBound,[SysMLv1Tov2::Mappings::Factories::UpperBound_Factory],1,1
7,SysMLv1Tov2::Mappings::Factories::UpperBound_Factory,to,[KerML::Kernel::Expressions::LiteralInteger],1,1
8,SysMLv1Tov2::Mappings::Factories::UpperBound_Factory,multiplicityUpperBoundMembership,[SysMLv1Tov2::Mappings::Factories::MultiplicityUpperBoundMembership_Factory],1,1
9,SysMLv1Tov2::Mappings::Factories::FlowEndParameterMembership_Factory,end,[UML::CommonStructure::NamedElement],1,1


In [43]:

# Extract the initializeroperation referenced by each property-population candidate.
property_operation_rows = []

for _, row in mapping_operations[mapping_operations["operation_category"] == "property_population_candidate"].iterrows():

    for initializer_operation in row["redefined_operations"]:
        if initializer_operation is None:
            continue

        initializer_class, operation_name = (initializer_operation.rsplit("::", 1))

        property_operation_rows.append({
            "mapping_class": row["mapping_class"],
            "operation": row["operation"],
            "operation_name": row["operation_name"],
            "initializer_operation": initializer_operation,
            "initializer_class": initializer_class,
            "target_property_name": operation_name,
            "ocl": row["ocl"],
        })

property_operation_crosswalk = pd.DataFrame(property_operation_rows)
property_operation_crosswalk.head().to_string()

"                                                                                     mapping_class                                                                                              operation operation_name                                                                     initializer_operation                                                             initializer_class target_property_name                                                                                                                                                                                                                                                                            ocl\n0                     SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInUntyped_Mapping                SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInUntyped_Mapping::direction      direction         SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init::direction        SysMLv1

In [44]:
# Map each mapping class to its effective target type.
target_type_by_mapping = (property_crosswalk.set_index("mapping_class")["target_type"].to_dict())
target_property_rows = []
for _, operation_row in property_operation_crosswalk.iterrows():
    mapping_class = operation_row["mapping_class"]
    operation_name = operation_row["operation_name"]
    target_type = target_type_by_mapping.get(mapping_class)
    if not target_type:
        continue
    try:
        target_class = get_class(registry, target_type)
    except ValueError:
        target_property_rows.append({
            "mapping_class": mapping_class,
            "operation": operation_row["operation"],
            "operation_name": operation_name,
            "target_type": target_type,
            "target_property": None,
            "declared_by": None,
            "match_status": "target_class_not_found",
        })
        continue
    hierarchy = [target_class] + get_ancestors(target_class, parents_index,)
    matches = []
    for defining_class in hierarchy:
        for member in direct_members(defining_class):
            if (member.metaclass == "uml:Property" and member.name == operation_name):
                matches.append((member, defining_class))
    if not matches:
        target_property_rows.append({
            "mapping_class": mapping_class,
            "operation": operation_row["operation"],
            "operation_name": operation_name,
            "target_type": target_type,
            "target_property": None,
            "declared_by": None,
            "match_status": "no_name_match",
        })
    for prop, defining_class in matches:
        target_property_rows.append({
            "mapping_class": mapping_class,
            "operation": operation_row["operation"],
            "operation_name": operation_name,
            "initializer_operation": operation_row["initializer_operation"],
            "target_type": target_type,
            "target_property": prop.qualified_name,
            "target_property_xmi_id": prop.xmi_id,
            "declared_by": defining_class.qualified_name,
            "match_status": ("unique_name_match" if len(matches) == 1 else "multiple_name_matches"),
        })


target_property_matches = pd.DataFrame(target_property_rows)
print(target_property_matches["match_status"].value_counts())

match_status
no_name_match    361
Name: count, dtype: int64


In [45]:
target_resources = ["KerML", "SysML_v2",]
target_inheritance_edges = []
target_unresolved = {}
for resource in target_resources:
    edges, unresolved = extract_inheritance(registry, resource)
    target_inheritance_edges.extend(edges)
    target_unresolved[resource] = len(unresolved)
    print(resource, "inheritance relationships:", len(edges), "unresolved:", len(unresolved),)

target_parents_index, target_children_index = (build_inheritance_index(target_inheritance_edges))



KerML inheritance relationships: 89 unresolved: 0
SysML_v2 inheritance relationships: 121 unresolved: 0


In [46]:
def get_class_in_resource(registry, resource, qualified_name):
    matches = [element for element in registry.get_elements(resource) if element.metaclass == "uml:Class" and element.qualified_name == qualified_name]
    if len(matches) != 1:
        raise ValueError(f"Expected one class for {qualified_name} in {resource}; " f"found {len(matches)}")
    return matches[0]

def get_inherited_properties(target_class, registry, parents_index):
    hierarchy = [target_class, *get_ancestors(target_class, parents_index),]
    properties = []
    for cls in hierarchy:
        for member in registry.get_elements(cls.resource):
            if (member.parent == cls.key and member.metaclass == "uml:Property"):
                properties.append({
                    "property": member,
                    "property_name": member.name,
                    "property_qualified_name": member.qualified_name,
                    "declared_by": cls.qualified_name,
                    "declaring_class": cls,
                })
    return properties

property_operation_crosswalk = property_operation_crosswalk.merge(
    property_crosswalk[["mapping_class", "target_type"]].drop_duplicates(), on="mapping_class", how="left", validate="many_to_one",
    )

target_property_matches = []
for _, row in property_operation_crosswalk.iterrows():
    target_type = row["target_type"]
    resource = "KerML" if target_type.startswith("KerML::") else "SysML_v2"
    target_class = get_class_in_resource(registry, resource, target_type,)
    properties = get_inherited_properties(target_class, registry, target_parents_index,)
    matches = [prop for prop in properties if prop["property_name"] == row["operation_name"]]
    target_property_matches.append({
        **row.to_dict(),
        "target_property_matches": [
            {"qualified_name": prop["property_qualified_name"], "declared_by": prop["declared_by"], "xmi_id": prop["property"].xmi_id,} 
            for prop in matches],
        "match_count": len(matches),
        "match_status": ("unique_match" if len(matches) == 1 else "multiple_matches" if len(matches) > 1 else "no_match"),
    })

target_property_matches = pd.DataFrame(target_property_matches)

In [47]:
examples = [
    ("KerML::Core::Features::Feature", "StructuralFeature_Mapping",),
    ("KerML::Root::Namespaces::MembershipImport", "ElementImport_Mapping",),
]

for target_type, mapping_name in examples:
    target_class = get_class_in_resource(registry, "KerML", target_type)
    print(f"\n{'=' * 80}")
    print(target_type)
    properties = get_inherited_properties(target_class, registry, target_parents_index)
    print("\nTarget properties:")
    for prop in properties:
        print(prop["property_name"], "<-", prop["declared_by"],)
    mapping_class = get_class(
        registry,
        next(row["mapping_class"] for _, row in property_operation_crosswalk.iterrows() if row["mapping_class"].endswith(mapping_name))
    )
    print("\nMapping operations:")
    for member in direct_members(mapping_class):
        if member.metaclass == "uml:Operation":
            print(member.name)


KerML::Core::Features::Feature

Target properties:
owningType <- KerML::Core::Features::Feature
isUnique <- KerML::Core::Features::Feature
isOrdered <- KerML::Core::Features::Feature
type <- KerML::Core::Features::Feature
ownedRedefinition <- KerML::Core::Features::Feature
ownedSubsetting <- KerML::Core::Features::Feature
owningFeatureMembership <- KerML::Core::Features::Feature
isComposite <- KerML::Core::Features::Feature
isEnd <- KerML::Core::Features::Feature
endOwningType <- KerML::Core::Features::Feature
ownedTyping <- KerML::Core::Features::Feature
featuringType <- KerML::Core::Features::Feature
ownedTypeFeaturing <- KerML::Core::Features::Feature
isDerived <- KerML::Core::Features::Feature
chainingFeature <- KerML::Core::Features::Feature
ownedFeatureInverting <- KerML::Core::Features::Feature
ownedFeatureChaining <- KerML::Core::Features::Feature
isPortion <- KerML::Core::Features::Feature
isVariable <- KerML::Core::Features::Feature
isConstant <- KerML::Core::Features::Featu

In [48]:
mapping_names = ["StructuralFeature_Mapping", "ElementImport_Mapping",]

selected = mapping_operations[mapping_operations["mapping_class"].str.endswith(tuple(mapping_names))]

display(
    selected[["mapping_class", "operation_name", "redefined_operations", "operation_category", "ocl",]].sort_values(
        ["mapping_class", "operation_name"]
    )
)

,mapping_class,operation_name,redefined_operations,operation_category,ocl
319,SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeature_Mapping,isAbstract,[SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToType_Init::isAbstract],property_population_candidate,[result = false]
321,SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeature_Mapping,isOrdered,[SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init::isOrdered],property_population_candidate,[result = from.isOrdered]
323,SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeature_Mapping,isReadOnly,[SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init::isReadOnly],property_population_candidate,[result = from.isReadOnly]
320,SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeature_Mapping,isUnique,[SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init::isUnique],property_population_candidate,[result = from.isUnique]
322,SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeature_Mapping,ownedRelationship,"[SysMLv1Tov2::Mappings::UML4SysML::CommonStructure::ElementMain_Mapping::ownedRelationship, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init::ownedRelationship]",relationship_construction,"[result = let typing: KerML::FeatureTyping = StructuralFeatureToFeatureTyping_Mapping.getMapped(from) in if typing.oclIsUndefined() then Set{MultiplicityMembership_Mapping.getMapped(from)} else Set{MultiplicityMembership_Mapping.getMapped(from), typing} endif]"
609,SysMLv1Tov2::Mappings::UML4SysML::Packages::ElementImport_Mapping,filter,[SysMLv1Tov2::Mappings::Foundations::Mapping::filter],mapping_filter,[result = if src.oclIsKindOf(UML::ElementImport) then Helper.hasMainMapping(src.oclAsType(UML::ElementImport).importedElement) else false endif]
612,SysMLv1Tov2::Mappings::UML4SysML::Packages::ElementImport_Mapping,importedMemberName,[SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToImport_Init::importedMemberName],property_population_candidate,[result = from.alias]
611,SysMLv1Tov2::Mappings::UML4SysML::Packages::ElementImport_Mapping,importedMembership,[SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembershipImport_Init::importedMembership],property_population_candidate,[result = ElementOwningMembership_Mapping.getMapped(from.importedElement)]
610,SysMLv1Tov2::Mappings::UML4SysML::Packages::ElementImport_Mapping,visibility,[SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToImport_Init::visibility],property_population_candidate,[result = Helper.getKerMLVisibilityKind(from.visibility)]


In [49]:
def classify_ocl_operation(row):
    name = row["operation_name"]
    ocl = " ".join(row["ocl"]) if isinstance(row["ocl"], list) else str(row["ocl"])

    if name == "filter":
        return "mapping_filter"

    if name in {"ownedRelationship", "ownedMemberFeature", "ownedMemberElement", "ownedMemberParameter",}:
        return "relationship_construction"

    if "getMapped(" in ocl or "getMappedColl(" in ocl:
        return "delegated_mapping"

    if "Helper." in ocl:
        return "transformed_value"

    if "result = from." in ocl:
        return "direct_value_transfer"

    if "result = false" in ocl or "result = true" in ocl:
        return "constant_value"

    return "other"

## Mapping Dependencies:

In [50]:
import re
import pandas as pd

MAPPING_CALL_PATTERN = re.compile(
    r"(?P<callee>(?:[A-Za-z_]\w*::)*[A-Za-z_]\w*)"
    r"\s*\.\s*"
    r"(?P<method>getMappedColl|getMapped)"
    r"\s*\("
)

def normalize_ocl(value):
    if isinstance(value, (list, tuple)):
        return "\n".join(str(x) for x in value if x is not None)
    if value is None:
        return ""
    return str(value)

def extract_mapping_calls(ocl):
    text = normalize_ocl(ocl)
    calls = []
    for match in MAPPING_CALL_PATTERN.finditer(text):
        open_pos = match.end() - 1
        depth = 0
        close_pos = None
        for i in range(open_pos, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close_pos = i
                    break
        if close_pos is None:
            continue
        calls.append({
            "callee": match.group("callee"),
            "method": match.group("method"),
            "arguments": text[open_pos + 1:close_pos].strip(),
            "call_expression": text[match.start():close_pos + 1],
        })
    return calls

def resolve_mapping_class(name):
    exact = [cls for cls in known_mapping_classes if cls == name]
    if exact:
        return exact, "resolved"
    matches = [cls for cls in known_mapping_classes if cls.endswith(f"::{name}")]
    if len(matches) == 1:
        return matches, "resolved"
    if len(matches) > 1:
        return matches, "ambiguous"
    return [], "unresolved"

known_mapping_classes = (mapping_operations["mapping_class"] .dropna() .unique() .tolist())

mapping_call_records = []
for _, row in mapping_operations.iterrows():
    calls = extract_mapping_calls(row["ocl"])
    for call in calls:
        candidates, status = resolve_mapping_class(call["callee"])
        mapping_call_records.append({
            "source_mapping": row["mapping_class"],
            "source_operation": row["operation"],
            "source_operation_name": row["operation_name"],
            "target_mapping_name": call["callee"],
            "target_mapping_candidates": candidates,
            "target_mapping": (candidates[0] if status == "resolved" else None),
            "method": call["method"],
            "arguments": call["arguments"],
            "call_expression": call["call_expression"],
            "resolution_status": status,
        })

mapping_calls = pd.DataFrame(mapping_call_records)
print(mapping_calls["resolution_status"].value_counts())


resolution_status
resolved      782
unresolved     47
Name: count, dtype: int64


In [51]:
mapping_base = get_class_in_resource(registry, "SysML_v1_to_v2", "SysMLv1Tov2::Mappings::Foundations::Mapping",)
mapping_class_records = [
    element
    for element in registry.get_elements("SysML_v1_to_v2")
    if element.metaclass == "uml:Class"
    and (
        element.key == mapping_base.key
        or any(ancestor.key == mapping_base.key for ancestor in get_ancestors(element, parents_index))
    )
]
mapping_class_names = [element.qualified_name for element in mapping_class_records]



In [52]:
test_names = ["StructuralFeatureToFeatureTyping_Mapping", "Property_Mapping", "ElementOwningMembership_Mapping",]
for name in test_names:
    matches = [cls for cls in mapping_class_names if cls.endswith(f"::{name}")]
    print(name, matches)

StructuralFeatureToFeatureTyping_Mapping ['SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureToFeatureTyping_Mapping']
Property_Mapping []
ElementOwningMembership_Mapping ['SysMLv1Tov2::Mappings::UML4SysML::CommonStructure::ElementOwningMembership_Mapping']


In [53]:
def resolve_mapping_class(name):
    exact = [cls for cls in mapping_class_names if cls == name]
    if exact:
        return exact, "resolved"
    matches = [cls for cls in mapping_class_names if cls.endswith(f"::{name}")]
    if len(matches) == 1:
        return matches, "resolved"
    if len(matches) > 1:
        return matches, "ambiguous"
    return [], "unresolved"

resolved_records = mapping_calls["target_mapping_name"].apply(resolve_mapping_class)
mapping_calls["target_mapping_candidates"] = resolved_records.apply(lambda result: result[0])
mapping_calls["resolution_status"] = resolved_records.apply(lambda result: result[1])
mapping_calls["target_mapping"] = mapping_calls["target_mapping_candidates"].apply(lambda candidates: candidates[0] if len(candidates) == 1 else None)
print(mapping_calls["resolution_status"].value_counts())

resolution_status
resolved      801
unresolved     28
Name: count, dtype: int64


In [54]:
unresolved_calls = mapping_calls.loc[
    mapping_calls["resolution_status"] == "unresolved"
].copy()
display(
    unresolved_calls["target_mapping_name"]
    .value_counts()
    .rename_axis("target_mapping_name")
    .reset_index(name="occurrences")
)

,target_mapping_name,occurrences
0,LiteralSpecificationTyping_Mapping,3
1,PropertyMembership_Mapping,3
2,InitialStateMembership_Mapping,2
3,Property_Mapping,2
4,PinFeatureTyping_Mapping,2
5,AssociationToFeatureTyping_Mapping,2
6,AllocationTargetReferenceUsageRedefinition_Mapping,1
7,TransitionTimeTriggerCalculationUsage_Mapping,1
8,ObjectFlowItemFeature_Mapping,1
9,ObjectFlowItemFeatureUntyped_Mapping,1


In [55]:
def find_classes_by_name(registry, name, resource="SysML_v1_to_v2"):
    return [
        element
        for element in registry.get_elements(resource)
        if element.metaclass == "uml:Class"
        and element.name == name
    ]

for name in sorted(unresolved_calls["target_mapping_name"].unique()):
    matches = find_classes_by_name(registry, name)

    print(f"\n{name}: {len(matches)} match(es)")
    for match in matches:
        print(" ", match.qualified_name)


AllocationSourceReferenceUsageRedefinition_Mapping: 1 match(es)
  SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationSourceReferenceUsageRedefinition_Mapping

AllocationTargetFeatureMembership_Mapping: 1 match(es)
  SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationTargetFeatureMembership_Mapping

AllocationTargetReferenceUsageRedefinition_Mapping: 1 match(es)
  SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationTargetReferenceUsageRedefinition_Mapping

AllocationUsageTargetEndFeatureMembership_Mapping: 1 match(es)
  SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetEndFeatureMembership_Mapping

AssociationToFeatureTyping_Mapping: 0 match(es)

CommonElementImport_Mapping: 0 match(es)

DefaultValueOpaqueExpression_Mapping: 0 match(es)

InitialStateMembership_Mapping: 0 match(es)

InstanceSpecificationToGeneralization_Mapping: 0 match(es)

LiteralSpecificationTyping_Mapping: 0 match(es)

MessageMembership_Mapping: 1 match(es)
  SysMLv1Tov2::Mappings::UML4Sys

In [56]:
def find_named_elements(registry, name):
    return [element for element in registry.get_elements() if element.name == name]


unresolved_name_audit = []

for name in sorted(unresolved_calls["target_mapping_name"].unique()):
    matches = find_named_elements(registry, name)

    for match in matches:
        unresolved_name_audit.append({
            "requested_name": name,
            "found_name": match.name,
            "resource": match.resource,
            "metaclass": match.metaclass,
            "qualified_name": match.qualified_name,
            "xmi_id": match.xmi_id,
        })

unresolved_name_audit = pd.DataFrame(unresolved_name_audit)

display(unresolved_name_audit)

,requested_name,found_name,resource,metaclass,qualified_name,xmi_id
0,AllocationSourceReferenceUsageRedefinition_Mapping,AllocationSourceReferenceUsageRedefinition_Mapping,SysML_v1_to_v2,uml:Class,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationSourceReferenceUsageRedefinition_Mapping,Mappings-SysMLv1-Allocations-AllocationSourceReferenceUsageRedefinition_Mapping
1,AllocationTargetFeatureMembership_Mapping,AllocationTargetFeatureMembership_Mapping,SysML_v1_to_v2,uml:Class,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationTargetFeatureMembership_Mapping,Mappings-SysMLv1-Allocations-AllocationTargetFeatureMembership_Mapping
2,AllocationTargetReferenceUsageRedefinition_Mapping,AllocationTargetReferenceUsageRedefinition_Mapping,SysML_v1_to_v2,uml:Class,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationTargetReferenceUsageRedefinition_Mapping,Mappings-SysMLv1-Allocations-AllocationTargetReferenceUsageRedefinition_Mapping
3,AllocationUsageTargetEndFeatureMembership_Mapping,AllocationUsageTargetEndFeatureMembership_Mapping,SysML_v1_to_v2,uml:Class,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetEndFeatureMembership_Mapping,Mappings-SysMLv1-Allocations-AllocationUsageTargetEndFeatureMembership_Mapping
4,MessageMembership_Mapping,MessageMembership_Mapping,SysML_v1_to_v2,uml:Class,SysMLv1Tov2::Mappings::UML4SysML::Interactions::MessageMembership_Mapping,Mappings-UML4SysML-Interactions-MessageMembership_Mapping
5,ObjectFlowItemFeatureUntyped_Mapping,ObjectFlowItemFeatureUntyped_Mapping,SysML_v1_to_v2,uml:Class,SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeatureUntyped_Mapping,Mappings-UML4SysML-Activities-ObjectFlowItemFeatureUntyped_Mapping
6,ObjectFlowItemFeature_Mapping,ObjectFlowItemFeature_Mapping,SysML_v1_to_v2,uml:Class,SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeature_Mapping,Mappings-UML4SysML-Activities-ObjectFlowItemFeature_Mapping


In [57]:
found_but_unresolved = [
    "AllocationSourceReferenceUsageRedefinition_Mapping",
    "AllocationTargetFeatureMembership_Mapping",
    "AllocationTargetReferenceUsageRedefinition_Mapping",
    "AllocationUsageTargetEndFeatureMembership_Mapping",
    "MessageMembership_Mapping",
    "ObjectFlowItemFeatureUntyped_Mapping",
    "ObjectFlowItemFeature_Mapping",
]

for name in found_but_unresolved:
    matches = find_named_elements(registry, name)

    for cls in matches:
        ancestors = get_ancestors(cls, parents_index)

        print(f"\n{cls.qualified_name}")
        print("Metaclass:", cls.metaclass)
        print("Is in mapping_class_names:", cls.qualified_name in mapping_class_names)
        print("Mapping ancestor:", any(
            ancestor.key == mapping_base.key
            for ancestor in ancestors
        ))
        print("Parents:", [a.qualified_name for a in parents_index[cls.key]])


SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationSourceReferenceUsageRedefinition_Mapping
Metaclass: uml:Class
Is in mapping_class_names: False
Mapping ancestor: False
Parents: ['SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRedefinition_Init']

SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationTargetFeatureMembership_Mapping
Metaclass: uml:Class
Is in mapping_class_names: False
Mapping ancestor: False
Parents: ['SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init']

SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationTargetReferenceUsageRedefinition_Mapping
Metaclass: uml:Class
Is in mapping_class_names: False
Mapping ancestor: False
Parents: ['SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRedefinition_Init']

SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetEndFeatureMembership_Mapping
Metaclass: uml:Class
Is in mapping_class_names: False
Mapping ancestor: False
Parents: ['SysMLv1Tov2::Mappings::Init

In [58]:
mapping_class_records = [
    element
    for element in registry.get_elements("SysML_v1_to_v2")
    if element.metaclass == "uml:Class"
    and (
        element.key == mapping_base.key
        or any(
            ancestor.key == mapping_base.key
            for ancestor in get_ancestors(element, parents_index)
        )
        or (element.name or "").endswith("_Mapping")
    )
]

mapping_class_names = [
    element.qualified_name
    for element in mapping_class_records
]

mapping_class_audit = []

for cls in mapping_class_records:
    has_mapping_ancestor = (
        cls.key == mapping_base.key
        or any(
            ancestor.key == mapping_base.key
            for ancestor in get_ancestors(cls, parents_index)
        )
    )

    mapping_class_audit.append({
        "mapping_class": cls.qualified_name,
        "xmi_id": cls.xmi_id,
        "identified_by": (
            "mapping_inheritance"
            if has_mapping_ancestor
            else "mapping_name_suffix"
        ),
        "direct_parents": [
            parent.qualified_name
            for parent in parents_index[cls.key]
        ],
    })

mapping_class_audit = pd.DataFrame(mapping_class_audit)

print(mapping_class_audit["identified_by"].value_counts())

identified_by
mapping_inheritance    773
mapping_name_suffix     13
Name: count, dtype: int64


In [59]:
resolved_records = mapping_calls["target_mapping_name"].apply(resolve_mapping_class)
mapping_calls["target_mapping_candidates"] = resolved_records.apply(lambda result: result[0])
mapping_calls["resolution_status"] = resolved_records.apply(lambda result: result[1])
mapping_calls["target_mapping"] = mapping_calls["target_mapping_candidates"].apply(lambda candidates: candidates[0] if len(candidates) == 1 else None)
print(mapping_calls["resolution_status"].value_counts())

resolution_status
resolved      808
unresolved     21
Name: count, dtype: int64


In [60]:
unresolved_calls = mapping_calls.loc[
    mapping_calls["resolution_status"] == "unresolved"
].copy()

display(
    unresolved_calls[
        [
            "source_mapping",
            "source_operation_name",
            "target_mapping_name",
            "call_expression",
        ]
    ].sort_values(
        ["target_mapping_name", "source_mapping"]
    )
)

display(
    unresolved_calls["target_mapping_name"]
    .value_counts()
    .rename_axis("target_mapping_name")
    .reset_index(name="occurrences")
)

,source_mapping,source_operation_name,target_mapping_name,call_expression
702,SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefined_Mapping,ownedRelationship,AssociationToFeatureTyping_Mapping,AssociationToFeatureTyping_Mapping.getMapped(from)
371,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsage_Mapping,ownedRelationship,AssociationToFeatureTyping_Mapping,AssociationToFeatureTyping_Mapping.getMapped(from)
193,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping,ownedRelationship,CommonElementImport_Mapping,CommonElementImport_Mapping.getMapped(e)
356,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping,ownedRelationship,DefaultValueOpaqueExpression_Mapping,DefaultValueOpaqueExpression_Mapping.getMapped(from.defaultValue)
218,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::Region_Mapping,ownedRelationship,InitialStateMembership_Mapping,InitialStateMembership_Mapping.getMapped(e)
228,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::StateDefinition_Mapping,ownedRelationship,InitialStateMembership_Mapping,InitialStateMembership_Mapping.getMapped(e)
285,SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping,ownedFeatureMembership,InstanceSpecificationToGeneralization_Mapping,"InstanceSpecificationToGeneralization_Mapping.getMapped(from, c)"
311,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationCommon_Mapping,ownedRelationship,LiteralSpecificationTyping_Mapping,LiteralSpecificationTyping_Mapping.getMapped(from)
306,SysMLv1Tov2::Mappings::UML4SysML::Values::TimeExpression_Mapping,ownedRelationship,LiteralSpecificationTyping_Mapping,LiteralSpecificationTyping_Mapping.getMapped(from)
314,SysMLv1Tov2::Mappings::UML4SysML::Values::ValueSpecification_Mapping,ownedRelationship,LiteralSpecificationTyping_Mapping,LiteralSpecificationTyping_Mapping.getMapped(from)


,target_mapping_name,occurrences
0,LiteralSpecificationTyping_Mapping,3
1,PropertyMembership_Mapping,3
2,InitialStateMembership_Mapping,2
3,Property_Mapping,2
4,AssociationToFeatureTyping_Mapping,2
5,PinFeatureTyping_Mapping,2
6,CommonElementImport_Mapping,1
7,InstanceSpecificationToGeneralization_Mapping,1
8,DefaultValueOpaqueExpression_Mapping,1
9,TransitionChangeTriggerConstraintUsage_Mapping,1


In [61]:
resolved_calls = mapping_calls.loc[mapping_calls["resolution_status"] == "resolved"].copy()
dependency_edges = (
    resolved_calls
    .groupby(["source_mapping", "target_mapping"], as_index=False)
    .agg(
        call_count=("call_expression", "size"),
        source_operations=("source_operation_name", lambda x: sorted(set(x))),
        methods=("method", lambda x: sorted(set(x))),
        call_expressions=("call_expression", list),
    )
)
dependency_edges= dependency_edges.sort_values(["source_mapping", "target_mapping"]).reset_index(drop=True)
display(dependency_edges.head(20))

,source_mapping,target_mapping,call_count,source_operations,methods,call_expressions
0,SysMLv1Tov2::Mappings::CommonMappings::CommonFeatureReferenceExpression_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,1,[ownedRelationship],[getMapped],[CommonMembership_Mapping.getMapped(from)]
1,SysMLv1Tov2::Mappings::CommonMappings::CommonFeatureReferenceExpression_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterFeatureMembership_Mapping,1,[ownedRelationship],[getMapped],[CommonReturnParameterFeatureMembership_Mapping.getMapped(from)]
2,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInMembership_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInUntyped_Mapping,1,[ownedMemberParameter],[getMapped],[CommonParameterReferenceUsageInUntyped_Mapping.getMapped(from)]
3,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInMembership_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageIn_Mapping,2,[ownedMemberParameter],[getMapped],"[CommonParameterReferenceUsageIn_Mapping.getMapped(from), CommonParameterReferenceUsageIn_Mapping.getMapped(from)]"
4,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageIn_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInFeatureTyping_Mapping,1,[ownedRelationship],[getMapped],[CommonParameterReferenceUsageInFeatureTyping_Mapping.getMapped(from)]
5,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInFeatureMembership_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInUntyped_Mapping,1,[ownedMemberFeature],[getMapped],[CommonReferenceUsageInUntyped_Mapping.getMapped(from)]
6,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInFeatureMembership_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageIn_Mapping,1,[ownedMemberFeature],[getMapped],[CommonReferenceUsageIn_Mapping.getMapped(from)]
7,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageIn_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInFeatureTyping_Mapping,1,[ownedRelationship],[getMapped],[CommonReferenceUsageInFeatureTyping_Mapping.getMapped(from)]
8,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterFeatureMembership_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterFeatureUntyped_Mapping,3,[ownedMemberParameter],[getMapped],"[CommonReturnParameterFeatureUntyped_Mapping.getMapped(from), CommonReturnParameterFeatureUntyped_Mapping.getMapped(from), CommonReturnParameterFeatureUntyped_Mapping.getMapped(from)]"
9,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterFeature_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterFeatureTyping_Mapping,1,[ownedRelationship],[getMapped],[CommonReturnParameterFeatureTyping_Mapping.getMapped(from)]


In [62]:
assert dependency_edges["call_count"].sum() == len(resolved_calls)

print("Resolved call occurrences:", len(resolved_calls))
print("Unique dependency edges:", len(dependency_edges))
print("Mappings calling other mappings:", dependency_edges["source_mapping"].nunique())
print("Mappings being called:", dependency_edges["target_mapping"].nunique())

Resolved call occurrences: 808
Unique dependency edges: 777
Mappings calling other mappings: 512
Mappings being called: 565


In [63]:
dependency_edges_all = (
    mapping_calls
    .groupby(["source_mapping", "target_mapping_name", "resolution_status",], as_index=False, dropna=False,)
    .agg(
        target_mapping=("target_mapping", "first"),
        call_count=("call_expression", "size"),
        source_operations=("source_operation_name", lambda x: sorted(set(x)),),
        methods=("method", lambda x: sorted(set(x))),
        call_expressions=("call_expression", list),
    )
)
dependency_edges_all = dependency_edges_all.rename(columns={"target_mapping_name": "target_mapping_label",})
dependency_edges_all = dependency_edges_all.sort_values(["resolution_status", "source_mapping", "target_mapping_label"]).reset_index(drop=True)

assert dependency_edges_all["call_count"].sum() == len(mapping_calls)

print(dependency_edges_all.groupby("resolution_status")["call_count"] .sum())
print("Total dependency pairs:", len(dependency_edges_all))

resolution_status
resolved      808
unresolved     21
Name: call_count, dtype: int64
Total dependency pairs: 797


In [64]:
from pathlib import Path

xmi_text = Path("C:/Users/Drew/Documents/MyFiles/repo/sysml/lang_files/processed/sysml_1_to_2.xmi").read_text(encoding="utf-8")

unresolved_name_audit = []

for name in sorted(unresolved_calls["target_mapping_name"].unique()):
    unresolved_name_audit.append({
        "name": name,
        "in_raw_xmi": name in xmi_text,
        "registry_matches": len(find_named_elements(registry, name)),
    })

display(pd.DataFrame(unresolved_name_audit))

,name,in_raw_xmi,registry_matches
0,AssociationToFeatureTyping_Mapping,True,0
1,CommonElementImport_Mapping,True,0
2,DefaultValueOpaqueExpression_Mapping,True,0
3,InitialStateMembership_Mapping,True,0
4,InstanceSpecificationToGeneralization_Mapping,True,0
5,LiteralSpecificationTyping_Mapping,True,0
6,PinFeatureTyping_Mapping,True,0
7,PropertyDefaultValue_Mapping,True,0
8,PropertyMembership_Mapping,True,0
9,Property_Mapping,True,0


In [65]:
import re

def find_xmi_occurrences(text, term, context=250):
    matches = []
    for match in re.finditer(re.escape(term), text):
        start = max(0, match.start() - context)
        end = min(len(text), match.end() + context)
        matches.append({"position": match.start(), "context": text[start:end],})
    return matches

for name in sorted(unresolved_calls["target_mapping_name"].unique()):
    occurrences = find_xmi_occurrences(xmi_text, name)
    print(f"\n{'=' * 80}")
    print(f"{name}: {len(occurrences)} occurrence(s)")
    for occurrence in occurrences[:3]:
        print(f"\nPosition: {occurrence['position']}")
        print(occurrence["context"])


AssociationToFeatureTyping_Mapping: 2 occurrence(s)

Position: 1768649
ion" xmi:type="uml:Constraint" name="bodyCondition">
<specification xmi:id="Mappings-UML4SysML-StructuredClassifiers-AssociationMetadataUsage_Mapping-ownedRelationship_-bodyCondition-specification0" xmi:type="uml:OpaqueExpression" body="result = Set{AssociationToFeatureTyping_Mapping.getMapped(from), AssociationMetadataUsageFeatureMembership_Mapping.getMapped(from)}" language="OCL2.0" name=""> </specification>
</bodyCondition>
<ownedParameter xmi:id="Mappings-UML4SysML-StructuredClassifiers-AssociationMetadataUsage_Mapping-owned

Position: 3228204
nstraint" name="bodyCondition">
<specification xmi:id="Mappings-UML4SysML-SimpleClassifiers-AttributeRedefined_Mapping-ownedRelationship_-bodyCondition-specification0" xmi:type="uml:OpaqueExpression" body="result = let typing: KerML::FeatureTyping = AssociationToFeatureTyping_Mapping.getMapped(from) in let subsetting: Set(KerML::Subsetting) = from.subsettedProperty -&gt;c

In [66]:
import re

property_calls = re.findall(r"(?<![\w])Property_Mapping\s*\.\s*getMapped(?:Coll)?\s*\(", xmi_text,)
print("Exact Property_Mapping calls:", len(property_calls))

def audit_mapping_name(name, text):
    # Exact mapping invocation
    call_pattern = re.compile(rf"(?<![\w]){re.escape(name)}" rf"\s*\.\s*getMapped(?:Coll)?\s*\(")
    # Exact occurrence in a UML class declaration
    class_pattern = re.compile(rf'xmi:type="uml:Class"[^>]*\bname="{re.escape(name)}"' rf'|name="{re.escape(name)}"[^>]*xmi:type="uml:Class"')
    return {
        "name": name,
        "exact_mapping_calls": len(call_pattern.findall(text)),
        "class_declaration_matches": len(class_pattern.findall(text)),
        "registry_matches": len(find_named_elements(registry, name)),
    }

mapping_name_audit = pd.DataFrame([
    audit_mapping_name(name, xmi_text)
    for name in sorted(unresolved_calls["target_mapping_name"].unique())
])

display(mapping_name_audit)

Exact Property_Mapping calls: 2


,name,exact_mapping_calls,class_declaration_matches,registry_matches
0,AssociationToFeatureTyping_Mapping,2,0,0
1,CommonElementImport_Mapping,1,0,0
2,DefaultValueOpaqueExpression_Mapping,1,0,0
3,InitialStateMembership_Mapping,3,0,0
4,InstanceSpecificationToGeneralization_Mapping,1,0,0
5,LiteralSpecificationTyping_Mapping,3,0,0
6,PinFeatureTyping_Mapping,2,0,0
7,PropertyDefaultValue_Mapping,1,0,0
8,PropertyMembership_Mapping,3,0,0
9,Property_Mapping,2,0,0


In [67]:
raw_call_counts = (
    mapping_name_audit[["name", "exact_mapping_calls"]]
    .rename(columns={"name": "target_mapping_name", "exact_mapping_calls": "raw_call_count",})
)

extracted_call_counts = (
    mapping_calls.loc[
        mapping_calls["resolution_status"] == "unresolved"
    ]
    .groupby("target_mapping_name")
    .size()
    .rename("extracted_call_count")
    .reset_index()
)

call_count_audit = raw_call_counts.merge(
    extracted_call_counts,
    on="target_mapping_name",
    how="outer",
).fillna(0)

call_count_audit["difference"] = (
    call_count_audit["raw_call_count"]
    - call_count_audit["extracted_call_count"]
)

display(
    call_count_audit.sort_values(
        "difference", ascending=False
    )
)

,target_mapping_name,raw_call_count,extracted_call_count,difference
3,InitialStateMembership_Mapping,3,2,1
0,AssociationToFeatureTyping_Mapping,2,2,0
1,CommonElementImport_Mapping,1,1,0
2,DefaultValueOpaqueExpression_Mapping,1,1,0
4,InstanceSpecificationToGeneralization_Mapping,1,1,0
5,LiteralSpecificationTyping_Mapping,3,3,0
6,PinFeatureTyping_Mapping,2,2,0
7,PropertyDefaultValue_Mapping,1,1,0
8,PropertyMembership_Mapping,3,3,0
9,Property_Mapping,2,2,0


In [68]:
# Show all three raw occurrences with their surrounding OCL text.
for occurrence in find_xmi_occurrences(xmi_text, "InitialStateMembership_Mapping", context=500,):
    print("\n", "=" * 80)
    print(occurrence["context"])


mi:id="Mappings-Helper-stateOwnedRelationship_Element-bodyCondition-specification0" xmi:type="uml:OpaqueExpression" body="result = let initialState : Set(UML::Element) = from.ownedElement-&gt;select(e | e.oclIsKindOf(UML::Pseudostate) and e.oclAsType(UML::Pseudostate).kind = UML::PseudostateKind::initial) in let toElementOMS : Set(UML::Element) = from.ownedElement - initialState in toElementOMS-&gt;collect(e | ElementOwningMembership_Mapping.getMapped(e)) -&gt;union(initialState-&gt;collect(e | InitialStateMembership_Mapping.getMapped(e)))" language="OCL2.0" name=""> </specification>
</bodyCondition>

<ownedParameter xmi:id="Mappings-Helper-stateOwnedRelationship_Element-src" xmi:type="uml:Parameter" name="src">
<lowerValue xmi:id="Mappings-Helper-stateOwnedRelationship_Element-src-lowerValue" xmi:type="uml:LiteralInteger" name="" value="1"/>
<type href="https://www.omg.org/spec/UML/20161101/UML.xmi#Element"/>
<upperValue xmi:id="Mappings-Helper-stateOwnedRelationship_Element-src-uppe

In [69]:
mapping_calls.loc[mapping_calls["target_mapping_name"] == "InitialStateMembership_Mapping", ["source_mapping", "source_operation_name", "call_expression",],]

,source_mapping,source_operation_name,call_expression
218,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::Region_Mapping,ownedRelationship,InitialStateMembership_Mapping.getMapped(e)
228,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::StateDefinition_Mapping,ownedRelationship,InitialStateMembership_Mapping.getMapped(e)


In [70]:
helper_operation = next(e for e in registry.get_elements("SysML_v1_to_v2") if e.metaclass == "uml:Operation" and e.qualified_name.endswith("::Helper::stateOwnedRelationship"))
print(helper_operation.qualified_name)

SysMLv1Tov2::Mappings::Helper::stateOwnedRelationship


In [71]:
initializer_properties.to_csv("C:/Users/Drew/Documents/MyFiles/repo/sysml/data/initializer_properties.csv", index=False)
property_operation_crosswalk.to_csv("C:/Users/Drew/Documents/MyFiles/repo/sysml/data/property_operation_crosswalk.csv", index=False)


# Getting a Solid Dependency Map:

- OCL: defining pre/post conditions
- Mappings calls to expected behavior
- Property references
- Conditonal expr
- Implimentation classifications 

In [72]:
ocl_operations = []
for operation in registry.get_elements("SysML_v1_to_v2"):
    if operation.metaclass != "uml:Operation":
        continue
    owner = registry.elements.get(operation.parent)
    if owner is None:
        continue
    # Retrieve OCL expressions nested under this operation.
    specifications = [
        element
        for element in registry.get_elements("SysML_v1_to_v2")
        if element.tag == "specification"
        and any(
            ancestor.key == operation.key
            for ancestor in get_ancestors(element, parents_index)
        )
    ]
    ocl_bodies = []
    for specification in specifications:
        ocl_bodies.extend(
            element
            for element in registry.get_elements("SysML_v1_to_v2")
            if element.metaclass == "uml:OpaqueExpression"
            and element.parent == specification.key
        )
    ocl_operations.append({
        "operation_id": operation.key,
        "operation_xmi_id": operation.xmi_id,
        "operation": operation.qualified_name,
        "operation_name": operation.name,
        "owner_id": owner.key,
        "owner": owner.qualified_name,
        "owner_metaclass": owner.metaclass,
        "ocl_bodies": [
            body for body in ocl_bodies
        ],
        "ocl_body_count": len(ocl_bodies),
    })
ocl_operations = pd.DataFrame(ocl_operations)

In [73]:
print("Total operations:", len(ocl_operations))
print("\nOperations by owner metaclass:")
display(ocl_operations["owner_metaclass"] .value_counts(dropna=False))
print("\nOperations by owner package:")
display(ocl_operations.assign(package=ocl_operations["owner"].str.rsplit("::", n=1).str[0])["package"] .value_counts() .head(20))
print("\nOCL body counts:")
display(ocl_operations["ocl_body_count"].value_counts())

Total operations: 1157

Operations by owner metaclass:


owner_metaclass
uml:Class    1157
Name: count, dtype: int64


Operations by owner package:


package
SysMLv1Tov2::Mappings::Factories                                       87
SysMLv1Tov2::Mappings::UML4SysML::Activities                           81
SysMLv1Tov2::Mappings::UML4SysML::Classification                       77
SysMLv1Tov2::Mappings::SysMLv1::ModelElements                          70
SysMLv1Tov2::Mappings::UML4SysML::CommonBehavior                       69
SysMLv1Tov2::Mappings::Initializers::KerMLInitializers                 67
SysMLv1Tov2::Mappings::SysMLv1::Requirements                           57
SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers                51
SysMLv1Tov2::Mappings::UML4SysML::Actions::ObjectActions               49
SysMLv1Tov2::Mappings::UML4SysML::Packages                             45
SysMLv1Tov2::Mappings::UML4SysML::StateMachines                        40
SysMLv1Tov2::Mappings::UML4SysML::Actions::InvocationActions           39
SysMLv1Tov2::Mappings::UML4SysML::Values                               36
SysMLv1Tov2::Mappings::UML4Sys


OCL body counts:


ocl_body_count
0    1157
Name: count, dtype: int64

In [74]:
from collections import defaultdict

operation_ocl_bodies = defaultdict(list)

for element in registry.get_elements("SysML_v1_to_v2"):

    if element.metaclass != "uml:OpaqueExpression":
        continue

    current = element

    while current.parent is not None:
        parent = registry.elements.get(current.parent)

        if parent is None:
            break

        if parent.metaclass == "uml:Operation":
            operation_ocl_bodies[parent.key].append(element)
            break

        current = parent

ocl_operations["ocl_bodies"] = (ocl_operations["operation_id"] .map(operation_ocl_bodies) .apply(lambda x: x if isinstance(x, list) else []))
ocl_operations["ocl_body_count"] = (ocl_operations["ocl_bodies"].str.len())

print("Total operations:", len(ocl_operations))

print("\nOCL body counts:")
display(ocl_operations["ocl_body_count"].value_counts())
print("\nOperations with OCL bodies:")
display(ocl_operations.loc[ocl_operations["ocl_body_count"] > 0, ["operation", "ocl_body_count"],].head(20))

Total operations: 1157

OCL body counts:


ocl_body_count
1    1101
0      56
Name: count, dtype: int64


Operations with OCL bodies:


,operation,ocl_body_count
0,SysMLv1Tov2::Mappings::Foundations::Mapping::filter,1
1,SysMLv1Tov2::Mappings::Foundations::Mapping::getMappedColl,1
2,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped,1
3,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped,1
5,SysMLv1Tov2::Mappings::Factories::ReturnParameterFeature_Factory::direction,1
7,SysMLv1Tov2::Mappings::Factories::StringParameterFeatureValue_Factory::value,1
9,SysMLv1Tov2::Mappings::Factories::AssignmentActionUsageParameterMembership_Factory::ownedMemberParameter,1
11,SysMLv1Tov2::Mappings::Factories::DirectedReferenceUsage_Factory::direction,1
13,SysMLv1Tov2::Mappings::Factories::FeatureTyping_Factory::type,1
15,SysMLv1Tov2::Mappings::Factories::LiteralNull_Factory::ownedRelationship,1


In [75]:
sample_operation = ocl_operations.loc[ocl_operations["ocl_body_count"] > 0].iloc[0]

sample_expression = sample_operation["ocl_bodies"][0]

print("Expression record:")
print(vars(sample_expression))

print("\nContainment ancestry:")
current = sample_expression
while current is not None:
    print(current.metaclass, current.tag, current.name, current.xmi_id,)
    current = (registry.elements.get(current.parent) if current.parent is not None else None)

Expression record:
{'resource': 'SysML_v1_to_v2', 'xmi_id': 'Mappings-Foundations-Mapping-filter_Element-bodyCondition-specification0', 'tag': 'specification', 'metaclass': 'uml:OpaqueExpression', 'name': '', 'attributes': {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Foundations-Mapping-filter_Element-bodyCondition-specification0', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:OpaqueExpression', 'body': 'result = true', 'language': 'OCL2.0', 'name': ''}, 'parent': ('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-filter_Element-bodyCondition', 1), 'children': [], 'element_category': None, 'qualified_name': 'SysMLv1Tov2::Mappings::Foundations::Mapping::filter::bodyCondition', 'namespace': 'SysMLv1Tov2::Mappings::Foundations::Mapping::filter::bodyCondition', 'registry_key': ('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-filter_Element-bodyCondition-specification0', 1)}

Containment ancestry:
uml:OpaqueExpression specification  Mappings-Foundations-Mapping-filter_Element-b

### Adjustment:

In [76]:
ocl_specifications = []
for operation in registry.get_elements("SysML_v1_to_v2"):
    if operation.metaclass != "uml:Operation":
        continue
    for constraint in registry.get_elements("SysML_v1_to_v2"):
        if (constraint.metaclass != "uml:Constraint" or constraint.parent != operation.key):
            continue
        expressions = [
            element
            for element in registry.get_elements("SysML_v1_to_v2")
            if (element.metaclass == "uml:OpaqueExpression" and element.parent == constraint.key)
        ]

        if not expressions:
            ocl_specifications.append({
                "operation_id": operation.key,
                "operation": operation.qualified_name,
                "constraint_id": constraint.key,
                "constraint_name": constraint.name,
                "expression_id": None,
                "language": None,
                "body": None,
            })
            continue
        for expression in expressions:
            ocl_specifications.append({
                "operation_id": operation.key,
                "operation": operation.qualified_name,
                "constraint_id": constraint.key,
                "constraint_name": constraint.name,
                "expression_id": expression.key,
                "language": expression.attributes.get("language"),
                "body": expression.attributes.get("body"),
            })

ocl_specifications = pd.DataFrame(ocl_specifications)

def classify_specification(row):
    operation = row["operation"]
    constraint = row["constraint_name"]
    language = row["language"]
    body = row["body"]

    if constraint == "postcondition":
        return "framework_contract"

    if language == "English":
        if pd.isna(body) or not str(body).strip():
            return "incomplete_english_specification"
        return "english_specification"

    if language == "OCL2.0" and constraint == "bodyCondition":
        return "ocl_body_condition"

    return "other_specification"


ocl_specifications["specification_category"] = (ocl_specifications.apply(classify_specification, axis=1,))

In [77]:
print("Constraint records:", len(ocl_specifications))

print("\nConstraint names:")
display(ocl_specifications["constraint_name"].value_counts(dropna=False))

print("\nExpression languages:")
display(ocl_specifications["language"].value_counts(dropna=False))

print("\nExpressions per operation:")
display(
    ocl_specifications.groupby("operation")["expression_id"]
    .count()
    .value_counts()
)

Constraint records: 1101

Constraint names:


constraint_name
bodyCondition    1098
postcondition       3
Name: count, dtype: int64


Expression languages:


language
OCL2.0     1087
English      14
Name: count, dtype: int64


Expressions per operation:


expression_id
1    1099
2       1
Name: count, dtype: int64

In [78]:
ocl_specifications.loc[
    ocl_specifications["operation"].str.endswith("::filter"),
    ["operation", "constraint_name", "language", "body"],
].head(10)

,operation,constraint_name,language,body
0,SysMLv1Tov2::Mappings::Foundations::Mapping::filter,bodyCondition,OCL2.0,result = true
86,SysMLv1Tov2::Mappings::SysMLv1::Activities::RateOwningMembership_Mapping::filter,bodyCondition,OCL2.0,"result = Helper.hasStereotypeApplied(src, 'SysML::Activities::Rate') or Helper.hasStereotypeApplied(src, 'SysML::Activities::Continuous') or Helper.hasStereotypeApplied(src, 'SysML::Activities::Discrete')"
88,SysMLv1Tov2::Mappings::SysMLv1::Activities::ProbabilityMetadataUsage_Mapping::filter,bodyCondition,OCL2.0,"result = Helper.hasStereotypeApplied(src, 'SysML::Activities::Probability')"
90,SysMLv1Tov2::Mappings::SysMLv1::Activities::ProbabilityOwningMembership_Mapping::filter,bodyCondition,OCL2.0,"result = Helper.hasStereotypeApplied(src, 'SysML::Activities::Probability')"
92,SysMLv1Tov2::Mappings::SysMLv1::Activities::ProbabilityMetadataUsageFeatureTyping_Mapping::filter,bodyCondition,OCL2.0,"result = Helper.hasStereotypeApplied(src, 'SysML::Activities::Probability')"
94,SysMLv1Tov2::Mappings::SysMLv1::Activities::RateMetadataUsageContinuousReferenceUsageRedefinition_Mapping::filter,bodyCondition,OCL2.0,"result = Helper.hasStereotypeApplied(src, 'SysML::Activities::Continuous')"
96,SysMLv1Tov2::Mappings::SysMLv1::Activities::ProbabilityMetadataUsageReferenceUsage_Mapping::filter,bodyCondition,OCL2.0,"result = Helper.hasStereotypeApplied(src, 'SysML::Activities::Probability')"
98,SysMLv1Tov2::Mappings::SysMLv1::Activities::RateMetadataUsage_Mapping::filter,bodyCondition,OCL2.0,"result = Helper.hasStereotypeApplied(src, 'SysML::Activities::Rate') or Helper.hasStereotypeApplied(src, 'SysML::Activities::Continuous') or Helper.hasStereotypeApplied(src, 'SysML::Activities::Discrete')"
100,SysMLv1Tov2::Mappings::SysMLv1::Activities::RateMetadataUsageFeatureTyping_Mapping::filter,bodyCondition,OCL2.0,"result = Helper.hasStereotypeApplied(src, 'SysML::Activities::Rate') or Helper.hasStereotypeApplied(src, 'SysML::Activities::Continuous') or Helper.hasStereotypeApplied(src, 'SysML::Activities::Discrete')"
102,SysMLv1Tov2::Mappings::SysMLv1::Activities::RateMetadataUsageContinuousReferenceUsage_Mapping::filter,bodyCondition,OCL2.0,"result = Helper.hasStereotypeApplied(src, 'SysML::Activities::Continuous')"


In [79]:
# Three postconditions
postconditions = ocl_specifications.loc[
    ocl_specifications["constraint_name"] == "postcondition"
].copy()

display(
    postconditions[
        ["operation", "constraint_name", "language", "body"]
    ].to_string(index=False)
)

# Fourteen English expressions
english_expressions = ocl_specifications.loc[
    ocl_specifications["language"] == "English"
].copy()

display(
    english_expressions[
        ["operation", "constraint_name", "language", "body"]
    ].to_string(index=False)
)

constraints_per_operation = (
    ocl_specifications
    .groupby("operation")
    .size()
)

multiple_constraints = constraints_per_operation[
    constraints_per_operation > 1
]

display(
    ocl_specifications.loc[
        ocl_specifications["operation"].isin(
            multiple_constraints.index
        ),
        ["operation", "constraint_name", "language", "body"],
    ]
)

"                                                 operation constraint_name language                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  body\nSysMLv1Tov2::Mappings::Foundations::Mapping::getMappedColl   postcondition   OCL2.0                                                                                                                                                                                                                                                                                                                                                         

'                                                                                                     operation constraint_name language                                                 body\n                  SysMLv1Tov2::Mappings::SysMLv1::Ports&Flows::PortConjugation_Mapping::originalPortDefinition   bodyCondition  English                                        result = from\n                          SysMLv1Tov2::Mappings::SysMLv1::Ports&Flows::PortConjugation_Mapping::conjugatedType   bodyCondition  English                 result = conjugatedPortDefinition.to\n                       SysMLv1Tov2::Mappings::UML4SysML::Classification::Generalization_Mapping::subclassifier   bodyCondition  English result = Classifier_Mapping.getMapped(from.specific)\n                                SysMLv1Tov2::Mappings::UML4SysML::UseCases::IncludeFeatureTyping_Mapping::type   bodyCondition  English                               result = from.addition\n   SysMLv1Tov2::Mappings::UML4SysML::Actions::Obj

,operation,constraint_name,language,body
2,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped,postcondition,OCL2.0,self.filter(fromVar) and self.to.allFeatures()->selectByKind(UML::Property)->reject(isDerived) ->forAll(p | let ops: Operation = self.allFeatures() ->selectByKind(UML::Operation)->any(o | o.name = p.name) in p = ops()) and result = self.to
3,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped,postcondition,OCL2.0,self.filter(fromVar) and self.to.allFeatures()->selectByKind(UML::Property)->reject(isDerived) ->forAll(p | let ops: Operation = self.allFeatures() ->selectByKind(UML::Operation)->any(o | o.name = p.name) in if ops.ownedParameter ->select(p | p.direction = UML::ParameterDirectionKind::_'in') ->size()=1 then p = ops(qual) else if ops.ownedParameter ->select(p | p.direction = UML::ParameterDirectionKind::_'in') ->size()=0 then p = ops() else invalid endif endif) and result = self.to


In [80]:
display(
    ocl_specifications["specification_category"]
    .value_counts(dropna=False)
)

specification_category
ocl_body_condition                  1084
english_specification                 12
framework_contract                     3
incomplete_english_specification       2
Name: count, dtype: int64

In [81]:
ocl_mapping_calls = []

for _, row in ocl_specifications.iterrows():

    if pd.isna(row["body"]) or not str(row["body"]).strip():
        continue

    calls = extract_mapping_calls(row["body"])

    for call in calls:
        ocl_mapping_calls.append({
            "operation_id": row["operation_id"],
            "operation": row["operation"],
            "constraint_id": row["constraint_id"],
            "constraint_name": row["constraint_name"],
            "language": row["language"],
            "specification_category": row["specification_category"],
            **call,
        })

ocl_mapping_calls = pd.DataFrame(ocl_mapping_calls)
print("Mapping-call occurrences:", len(ocl_mapping_calls))
display(ocl_mapping_calls.groupby(["method", "language"]).size().rename("count").reset_index())
display(ocl_mapping_calls.loc[ocl_mapping_calls["operation"].str.contains("Helper::", na=False), ["operation", "callee", "method", "arguments"],].head(20))

Mapping-call occurrences: 855


,method,language,count
0,getMapped,English,1
1,getMapped,OCL2.0,852
2,getMappedColl,OCL2.0,2


,operation,callee,method,arguments
19,SysMLv1Tov2::Mappings::Helper::actionOwnedRelationship,ElementOwningMembership_Mapping,getMapped,e
20,SysMLv1Tov2::Mappings::Helper::actionOwnedRelationship,ElementFeatureMembership_Mapping,getMapped,e
21,SysMLv1Tov2::Mappings::Helper::stateOwnedRelationship,ElementOwningMembership_Mapping,getMapped,e
22,SysMLv1Tov2::Mappings::Helper::stateOwnedRelationship,InitialStateMembership_Mapping,getMapped,e
23,SysMLv1Tov2::Mappings::Helper::packageOwnedRelationship,InformationFlow_Mapping,getMapped,"i, r"
24,SysMLv1Tov2::Mappings::Helper::packageOwnedRelationship,InformationFlow_Mapping,getMapped,"i, r"
25,SysMLv1Tov2::Mappings::Helper::packageOwnedRelationship,CommentOwnership_Mapping,getMapped,c
26,SysMLv1Tov2::Mappings::Helper::packageOwnedRelationship,ElementOwningMembership_Mapping,getMapped,e
27,SysMLv1Tov2::Mappings::Helper::packageOwnedRelationship,PackageImport_Mapping,getMapped,i
28,SysMLv1Tov2::Mappings::Helper::packageOwnedRelationship,ElementOwningMembership_Mapping,getMapped,e


In [82]:
def resolve_mapping_class(name):
    # Exact qualified-name match
    exact = [cls for cls in mapping_class_records if cls.qualified_name == name]
    if exact:
        return exact, "resolved_exact"
    # Match an unqualified or partially qualified name
    matches = [cls for cls in mapping_class_records if cls.qualified_name.endswith(f"::{name}")]
    if len(matches) == 1:
        return matches, "resolved_suffix"
    if len(matches) > 1:
        return matches, "ambiguous"
    return [], "unresolved"

resolved_calls = []
for _, row in ocl_mapping_calls.iterrows():
    candidates, status = resolve_mapping_class(row["callee"])
    resolved_calls.append({
        **row.to_dict(),
        "target_class": (candidates[0].qualified_name if len(candidates) == 1 else None),
        "target_class_id": (candidates[0].key if len(candidates) == 1 else None),
        "resolution_status": status,
        "candidate_classes": [cls.qualified_name for cls in candidates],
    })
ocl_mapping_calls_resolved = pd.DataFrame(resolved_calls)


print("Resolution status:")
display(ocl_mapping_calls_resolved["resolution_status"] .value_counts())

print("\nUnresolved calls:")
display(ocl_mapping_calls_resolved.loc[ocl_mapping_calls_resolved["resolution_status"] == "unresolved", ["operation", "callee", "method", "arguments"],])

print("\nAmbiguous calls:")
display(ocl_mapping_calls_resolved.loc[ocl_mapping_calls_resolved["resolution_status"] == "ambiguous", ["operation", "callee", "candidate_classes"],]
)

Resolution status:


resolution_status
resolved_suffix    830
unresolved          25
Name: count, dtype: int64


Unresolved calls:


,operation,callee,method,arguments
0,SysMLv1Tov2::Mappings::Foundations::Mapping::getMappedColl,self,getMapped,e
22,SysMLv1Tov2::Mappings::Helper::stateOwnedRelationship,InitialStateMembership_Mapping,getMapped,e
214,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping::ownedRelationship,CommonElementImport_Mapping,getMapped,e
239,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::Region_Mapping::ownedRelationship,InitialStateMembership_Mapping,getMapped,e
249,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::StateDefinition_Mapping::ownedRelationship,InitialStateMembership_Mapping,getMapped,e
285,SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertySubsetting_Mapping::subsettingFeature,Property_Mapping,getMapped,from
286,SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertySubsetting_Mapping::subsettedFeature,Property_Mapping,getMapped,subsettedProperty
290,SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping::ownedMemberFeature,LiteralUnlimitedToInteger_Mapping,getMapped,from.upperValue
291,SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping::ownedMemberFeature,LiteralUnlimitedToUnbounded_Mapping,getMapped,from.upperValue
309,SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping::ownedFeatureMembership,InstanceSpecificationToGeneralization_Mapping,getMapped,"from, c"



Ambiguous calls:


,operation,callee,candidate_classes


In [83]:
def classify_mapping_call_receiver(row):
    if row["callee"] == "self":
        return "self_dispatch"

    return "mapping_class_reference"


ocl_mapping_calls_resolved["receiver_kind"] = (
    ocl_mapping_calls_resolved.apply(
        classify_mapping_call_receiver,
        axis=1,
    )
)

display(
    ocl_mapping_calls_resolved
    .groupby(["receiver_kind", "resolution_status"])
    .size()
    .rename("count")
    .reset_index()
)

,receiver_kind,resolution_status,count
0,mapping_class_reference,resolved_suffix,830
1,mapping_class_reference,unresolved,24
2,self_dispatch,unresolved,1


In [84]:
unresolved_mapping_calls = (
    ocl_mapping_calls_resolved.loc[
        (ocl_mapping_calls_resolved["receiver_kind"]
         == "mapping_class_reference")
        & (ocl_mapping_calls_resolved["resolution_status"]
           == "unresolved")
    ]
)

unresolved_mapping_targets = (
    unresolved_mapping_calls
    .groupby("callee")
    .agg(
        call_count=("callee", "size"),
        calling_operations=("operation", list),
    )
    .reset_index()
    .sort_values("callee")
)

display(unresolved_mapping_targets)

,callee,call_count,calling_operations
0,AssociationToFeatureTyping_Mapping,2,"[SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsage_Mapping::ownedRelationship, SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefined_Mapping::ownedRelationship]"
1,CommonElementImport_Mapping,1,[SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping::ownedRelationship]
2,DefaultValueOpaqueExpression_Mapping,1,[SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping::ownedRelationship]
3,InitialStateMembership_Mapping,3,"[SysMLv1Tov2::Mappings::Helper::stateOwnedRelationship, SysMLv1Tov2::Mappings::UML4SysML::StateMachines::Region_Mapping::ownedRelationship, SysMLv1Tov2::Mappings::UML4SysML::StateMachines::StateDefinition_Mapping::ownedRelationship]"
4,InstanceSpecificationToGeneralization_Mapping,1,[SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping::ownedFeatureMembership]
5,LiteralSpecificationTyping_Mapping,3,"[SysMLv1Tov2::Mappings::UML4SysML::Values::TimeExpression_Mapping::ownedRelationship, SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationCommon_Mapping::ownedRelationship, SysMLv1Tov2::Mappings::UML4SysML::Values::ValueSpecification_Mapping::ownedRelationship]"
6,LiteralUnlimitedToInteger_Mapping,1,[SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping::ownedMemberFeature]
7,LiteralUnlimitedToUnbounded_Mapping,1,[SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping::ownedMemberFeature]
8,PinFeatureTyping_Mapping,2,"[SysMLv1Tov2::Mappings::UML4SysML::Actions::Actions::ValuePin_Mapping::ownedRelationship, SysMLv1Tov2::Mappings::UML4SysML::Actions::ObjectActions::COAPin_Mapping::ownedRelationship]"
9,PropertyDefaultValue_Mapping,1,[SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefined_Mapping::ownedRelationship]


In [85]:
unresolved

[]

In [86]:
unresolved_names = unresolved_mapping_targets["callee"].drop_duplicates().tolist()

class_name_audit = []

all_classes = [element for element in registry.get_elements("SysML_v1_to_v2") if element.metaclass == "uml:Class"]

for name in unresolved_names:

    matches = [cls for cls in all_classes if cls.name == name or (cls.qualified_name or "").endswith(f"::{name}")]

    class_name_audit.append({
        "callee": name,
        "class_matches": len(matches),
        "qualified_matches": [cls.qualified_name for cls in matches],
        "xmi_ids": [cls.xmi_id for cls in matches],
    })

class_name_audit = pd.DataFrame(class_name_audit)

display(class_name_audit)

,callee,class_matches,qualified_matches,xmi_ids
0,AssociationToFeatureTyping_Mapping,0,[],[]
1,CommonElementImport_Mapping,0,[],[]
2,DefaultValueOpaqueExpression_Mapping,0,[],[]
3,InitialStateMembership_Mapping,0,[],[]
4,InstanceSpecificationToGeneralization_Mapping,0,[],[]
5,LiteralSpecificationTyping_Mapping,0,[],[]
6,LiteralUnlimitedToInteger_Mapping,0,[],[]
7,LiteralUnlimitedToUnbounded_Mapping,0,[],[]
8,PinFeatureTyping_Mapping,0,[],[]
9,PropertyDefaultValue_Mapping,0,[],[]


### Resolution: 

All missing mapping elements are confirmed to be meta-metamodel artifacts and not issues in the translation and relationship process

# Building Normalized Dependencies Table:

In [87]:
def add_mapping_resolution(df):
    df = df.copy()

    df["reference_type"] = df["callee"].apply(
        lambda name: "self_dispatch" if name == "self"
        else "mapping_class_reference"
    )

    results = df["callee"].apply(
        lambda name: ([], "not_applicable")
        if name == "self"
        else resolve_mapping_class(name)
    )

    df["candidate_classes"] = results.apply(
        lambda result: [cls.qualified_name for cls in result[0]]
    )

    df["resolution_status"] = results.apply(
        lambda result: result[1]
    )

    df["target_class"] = results.apply(
        lambda result: (
            result[0][0].qualified_name
            if result[1].startswith("resolved") else None
        )
    )

    df["target_class_id"] = results.apply(
        lambda result: (
            result[0][0].xmi_id
            if result[1].startswith("resolved") else None
        )
    )

    return df


ocl_mapping_calls = add_mapping_resolution(ocl_mapping_calls)

In [88]:
display(
    ocl_mapping_calls.groupby(
        ["reference_type", "resolution_status"],
        dropna=False
    ).size().rename("count").reset_index()
)

,reference_type,resolution_status,count
0,mapping_class_reference,resolved_suffix,830
1,mapping_class_reference,unresolved,24
2,self_dispatch,not_applicable,1


In [89]:
dependency_edges = ocl_mapping_calls.copy().reset_index(drop=True)
dependency_edges.insert(0, "call_id", [f"CALL_{i:05d}" for i in range(len(dependency_edges))])
dependency_edges["source_class"] = (dependency_edges["operation"] .str.rsplit("::", n=1) .str[0])
dependency_edges["dependency_type"] = (
    dependency_edges["reference_type"].map({"mapping_class_reference": "mapping_dependency", "self_dispatch": "self_dispatch",})
)
dependency_edges["dependency_status"] = (
    dependency_edges["resolution_status"].map({"resolved_suffix": "resolved", "unresolved": "unresolved_mapping_target", "not_applicable": "not_applicable",})
)

unresolved_mapping_targets = (
    dependency_edges.loc[dependency_edges["dependency_status"].eq("unresolved_mapping_target")]
    .groupby("callee")
    .agg(
        call_count=("call_id", "count"),
        calling_operations=("operation", lambda x: sorted(set(x))),
        calling_classes=("source_class", lambda x: sorted(set(x))),
    )
    .reset_index()
    .rename(columns={"callee": "target_name"})
    .sort_values("target_name")
    .reset_index(drop=True)
)

assert len(dependency_edges) == 855
assert dependency_edges["call_id"].is_unique
assert (
    dependency_edges["dependency_status"]
    .value_counts()
    .to_dict()
    == {"resolved": 830, "unresolved_mapping_target": 24, "not_applicable": 1,}
)
assert len(unresolved_mapping_targets) == 15
assert unresolved_mapping_targets["call_count"].sum() == 24

display(unresolved_mapping_targets)

,target_name,call_count,calling_operations,calling_classes
0,AssociationToFeatureTyping_Mapping,2,"[SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefined_Mapping::ownedRelationship, SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsage_Mapping::ownedRelationship]","[SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefined_Mapping, SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsage_Mapping]"
1,CommonElementImport_Mapping,1,[SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping::ownedRelationship],[SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping]
2,DefaultValueOpaqueExpression_Mapping,1,[SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping::ownedRelationship],[SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping]
3,InitialStateMembership_Mapping,3,"[SysMLv1Tov2::Mappings::Helper::stateOwnedRelationship, SysMLv1Tov2::Mappings::UML4SysML::StateMachines::Region_Mapping::ownedRelationship, SysMLv1Tov2::Mappings::UML4SysML::StateMachines::StateDefinition_Mapping::ownedRelationship]","[SysMLv1Tov2::Mappings::Helper, SysMLv1Tov2::Mappings::UML4SysML::StateMachines::Region_Mapping, SysMLv1Tov2::Mappings::UML4SysML::StateMachines::StateDefinition_Mapping]"
4,InstanceSpecificationToGeneralization_Mapping,1,[SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping::ownedFeatureMembership],[SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping]
5,LiteralSpecificationTyping_Mapping,3,"[SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationCommon_Mapping::ownedRelationship, SysMLv1Tov2::Mappings::UML4SysML::Values::TimeExpression_Mapping::ownedRelationship, SysMLv1Tov2::Mappings::UML4SysML::Values::ValueSpecification_Mapping::ownedRelationship]","[SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationCommon_Mapping, SysMLv1Tov2::Mappings::UML4SysML::Values::TimeExpression_Mapping, SysMLv1Tov2::Mappings::UML4SysML::Values::ValueSpecification_Mapping]"
6,LiteralUnlimitedToInteger_Mapping,1,[SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping::ownedMemberFeature],[SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping]
7,LiteralUnlimitedToUnbounded_Mapping,1,[SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping::ownedMemberFeature],[SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping]
8,PinFeatureTyping_Mapping,2,"[SysMLv1Tov2::Mappings::UML4SysML::Actions::Actions::ValuePin_Mapping::ownedRelationship, SysMLv1Tov2::Mappings::UML4SysML::Actions::ObjectActions::COAPin_Mapping::ownedRelationship]","[SysMLv1Tov2::Mappings::UML4SysML::Actions::Actions::ValuePin_Mapping, SysMLv1Tov2::Mappings::UML4SysML::Actions::ObjectActions::COAPin_Mapping]"
9,PropertyDefaultValue_Mapping,1,[SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefined_Mapping::ownedRelationship],[SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefined_Mapping]


In [90]:
def classify_source_owner(qualified_name):
    matches = [e for e in registry.get_elements("SysML_v1_to_v2") if e.metaclass == "uml:Class" and e.qualified_name == qualified_name]

    if len(matches) != 1:
        return "unresolved_source_owner"

    owner = matches[0]

    if owner.name == "Helper":
        return "helper"

    if owner.name.endswith("_Mapping"):
        return "mapping_class"

    if owner.name.endswith("_Init"):
        return "initializer"

    if owner.name.endswith("_Factory"):
        return "factory"

    return "other_class"

dependency_edges["source_owner_type"] = (dependency_edges["source_class"].apply(classify_source_owner))
display(dependency_edges.groupby("source_owner_type", dropna=False).size().rename("call_count").reset_index())

,source_owner_type,call_count
0,factory,1
1,helper,12
2,mapping_class,841
3,other_class,1


In [91]:
display(
    dependency_edges.loc[
        dependency_edges["dependency_status"].eq("unresolved_mapping_target"),
        ["source_class", "source_owner_type", "operation", "callee", "method", "arguments", "call_expression",],
    ].sort_values(["source_owner_type", "callee", "operation"])
)

,source_class,source_owner_type,operation,callee,method,arguments,call_expression
22,SysMLv1Tov2::Mappings::Helper,helper,SysMLv1Tov2::Mappings::Helper::stateOwnedRelationship,InitialStateMembership_Mapping,getMapped,e,InitialStateMembership_Mapping.getMapped(e)
726,SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefined_Mapping,mapping_class,SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefined_Mapping::ownedRelationship,AssociationToFeatureTyping_Mapping,getMapped,from,AssociationToFeatureTyping_Mapping.getMapped(from)
395,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsage_Mapping,mapping_class,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsage_Mapping::ownedRelationship,AssociationToFeatureTyping_Mapping,getMapped,from,AssociationToFeatureTyping_Mapping.getMapped(from)
214,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping,mapping_class,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping::ownedRelationship,CommonElementImport_Mapping,getMapped,e,CommonElementImport_Mapping.getMapped(e)
380,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping,mapping_class,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping::ownedRelationship,DefaultValueOpaqueExpression_Mapping,getMapped,from.defaultValue,DefaultValueOpaqueExpression_Mapping.getMapped(from.defaultValue)
239,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::Region_Mapping,mapping_class,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::Region_Mapping::ownedRelationship,InitialStateMembership_Mapping,getMapped,e,InitialStateMembership_Mapping.getMapped(e)
249,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::StateDefinition_Mapping,mapping_class,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::StateDefinition_Mapping::ownedRelationship,InitialStateMembership_Mapping,getMapped,e,InitialStateMembership_Mapping.getMapped(e)
309,SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping,mapping_class,SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping::ownedFeatureMembership,InstanceSpecificationToGeneralization_Mapping,getMapped,"from, c","InstanceSpecificationToGeneralization_Mapping.getMapped(from, c)"
335,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationCommon_Mapping,mapping_class,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationCommon_Mapping::ownedRelationship,LiteralSpecificationTyping_Mapping,getMapped,from,LiteralSpecificationTyping_Mapping.getMapped(from)
330,SysMLv1Tov2::Mappings::UML4SysML::Values::TimeExpression_Mapping,mapping_class,SysMLv1Tov2::Mappings::UML4SysML::Values::TimeExpression_Mapping::ownedRelationship,LiteralSpecificationTyping_Mapping,getMapped,from,LiteralSpecificationTyping_Mapping.getMapped(from)


In [92]:
mapping_dependencies = dependency_edges.loc[dependency_edges["source_owner_type"].eq("mapping_class")].copy()
helper_dependencies = dependency_edges.loc[dependency_edges["source_owner_type"].eq("helper")].copy()

print("Mapping dependencies:",
      len(mapping_dependencies))

print("Helper dependencies:",
      len(helper_dependencies))

print("\nUnresolved mapping dependencies:",
      mapping_dependencies["dependency_status"]
      .eq("unresolved_mapping_target").sum())

print("\nUnresolved helper dependencies:",
      helper_dependencies["dependency_status"]
      .eq("unresolved_mapping_target").sum())

Mapping dependencies: 841
Helper dependencies: 12

Unresolved mapping dependencies: 23

Unresolved helper dependencies: 1


In [93]:
mapping_dependency_edges = mapping_dependencies[
    ["call_id", "source_class", "operation", "operation_id", "constraint_id", 
     "language", "specification_category", "callee", "target_class", "target_class_id", 
     "arguments", "call_expression", "dependency_status",]
].copy()

mapping_dependency_edges = mapping_dependency_edges.rename(
    columns={"source_class": "source_node", "callee": "target_name",}
)

## After Re-validation:

In [94]:
import pandas as pd

# Build mapping crosswalk from the property crosswalk
mapping_table = property_crosswalk.copy()
mapping_table = mapping_table.rename(
    columns={
        "mapping_class": "mapping_name",
        "mapping_class_xmi_id": "mapping_xmi_id",
        "property_from": "source_property",
        "property_xmi_id_from": "source_property_xmi_id",
        "declared_by_from": "source_property_declared_by",
        "property_type_from": "source_property_type",
        "property_to": "target_property",
        "property_xmi_id_to": "target_property_xmi_id",
        "declared_by_to": "target_property_declared_by",
        "property_type_to": "target_property_type",
    }
)

# Add inheritance ancestry
def get_mapping_ancestors(mapping_name):
    cls = get_class(registry, mapping_name)
    return [ancestor.qualified_name for ancestor in get_ancestors(cls, parents_index)]

mapping_table["ancestors"] = (mapping_table["mapping_name"].apply(get_mapping_ancestors))

# Summarize mapping operations
operation_summary = (
    mapping_operations
    .groupby("mapping_class")
    .agg(
        operation_count=("operation", "count"),
        operations=("operation_name", lambda x: sorted(set(x))),
        operation_categories=("operation_category", lambda x: sorted(set(x))),)
    .reset_index()
    .rename(columns={"mapping_class": "mapping_name"})
)
# Summarize mapping dependencies
dependency_summary = (
    dependency_edges.loc[dependency_edges["source_owner_type"].eq("mapping_class")]
    .groupby("source_class")
    .agg(
        dependency_count=("call_id", "count"),
        resolved_dependency_count=("dependency_status", lambda x: x.eq("resolved").sum()),
        unresolved_dependency_count=("dependency_status", lambda x: x.eq("unresolved_mapping_target").sum()),
        dependencies=("callee", lambda x: sorted(set(x))),
    )
    .reset_index()
    .rename(columns={"source_class": "mapping_name"})
)
# Merge operation and dependency summaries
mapping_table = mapping_table.merge(operation_summary, on="mapping_name", how="left", validate="one_to_one",)
mapping_table = mapping_table.merge(dependency_summary, on="mapping_name", how="left", validate="one_to_one",)
count_columns = ["operation_count", "dependency_count", "resolved_dependency_count", "unresolved_dependency_count",]
mapping_table[count_columns] = (mapping_table[count_columns] .fillna(0) .astype(int))
# Represent missing operation/dependency collections as empty lists.
list_columns = ["operations", "operation_categories", "dependencies",]
for column in list_columns:
    mapping_table[column] = mapping_table[column].apply(lambda x: x if isinstance(x, list) else [])

# Remove duplicate columns and validate identities
mapping_table = mapping_table.loc[:, ~mapping_table.columns.duplicated()].copy()
assert mapping_table["mapping_name"].is_unique
assert mapping_table["mapping_name"].notna().all()

# Identify mapping classes missing from the crosswalk
crosswalk_names = set(mapping_table["mapping_name"])
mapping_class_inventory = pd.DataFrame([
    {
        "mapping_name": cls.qualified_name,
        "xmi_id": cls.xmi_id,
        "registry_key": cls.key,
        "identified_by": (
            "mapping_inheritance" if cls.key == mapping_base.key 
            or any(ancestor.key == mapping_base.key 
                   for ancestor in get_ancestors(cls, parents_index)) 
                   else "mapping_name_suffix"),
    }
    for cls in mapping_class_records
])
mapping_class_inventory["has_crosswalk"] = (mapping_class_inventory["mapping_name"].isin(crosswalk_names))
missing_crosswalk_df = mapping_class_inventory.loc[~mapping_class_inventory["has_crosswalk"]].copy()
# Record crosswalk coverage in the complete inventory.
mapping_class_inventory["has_crosswalk"] = (mapping_class_inventory["mapping_name"].isin(crosswalk_names))
# Final validation
print("Mapping crosswalk rows:", len(mapping_table))
print("Unique crosswalk mappings:", mapping_table["mapping_name"].nunique())
print("\nComplete mapping-class inventory:", len(mapping_class_inventory))

print("Classes missing crosswalk:", len(missing_crosswalk_df))
print("\nDependency summary:")
print(mapping_table[["dependency_count", "resolved_dependency_count", "unresolved_dependency_count",]].sum())

assert mapping_table["mapping_name"].is_unique
assert mapping_class_inventory["mapping_name"].is_unique
assert (len(mapping_table) + len(missing_crosswalk_df) == len(mapping_class_inventory))

display(mapping_table.head())
display(missing_crosswalk_df)

Mapping crosswalk rows: 772
Unique crosswalk mappings: 772

Complete mapping-class inventory: 786
Classes missing crosswalk: 14

Dependency summary:
dependency_count               829
resolved_dependency_count      808
unresolved_dependency_count     21
dtype: int64


,mapping_name,mapping_xmi_id,source_property,target_property,source_property_xmi_id,target_property_xmi_id,source_property_declared_by,target_property_declared_by,source_property_type,target_property_type,source_type,target_type,ancestors,operation_count,operations,operation_categories,dependency_count,resolved_dependency_count,unresolved_dependency_count,dependencies
0,SysMLv1Tov2::Mappings::CommonMappings::CommonFeatureReferenceExpression_Mapping,Mappings-CommonMappings-CommonFeatureReferenceExpression_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonFeatureReferenceExpression_Mapping::from,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureReferenceExpression_Init::to,Mappings-CommonMappings-CommonFeatureReferenceExpression_Mapping-from,Mappings-Initializers-KerMLInitializers-ToFeatureReferenceExpression_Init-to,SysMLv1Tov2::Mappings::CommonMappings::CommonFeatureReferenceExpression_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureReferenceExpression_Init,[UML::CommonStructure::TypedElement],[KerML::Kernel::Expressions::FeatureReferenceExpression],UML::CommonStructure::TypedElement,KerML::Kernel::Expressions::FeatureReferenceExpression,"[SysMLv1Tov2::Mappings::Foundations::Mapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureReferenceExpression_Init, SysMLv1Tov2::Mappings::Foundations::Initializer, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToExpression_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToStep_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToType_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToNamespace_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init]",1,[ownedRelationship],[relationship_construction],2,2,0,"[CommonMembership_Mapping, CommonReturnParameterFeatureMembership_Mapping]"
1,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,Mappings-CommonMappings-CommonMembership_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping::from,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init::to,Mappings-CommonMappings-CommonMembership_Mapping-from,Mappings-Initializers-KerMLInitializers-ToMembership_Init-to,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init,[UML::CommonStructure::TypedElement],[KerML::Root::Namespaces::Membership],UML::CommonStructure::TypedElement,KerML::Root::Namespaces::Membership,"[SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init, SysMLv1Tov2::Mappings::Foundations::Mapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init, SysMLv1Tov2::Mappings::Foundations::Initializer, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init]",1,[memberElement],[property_population_candidate],0,0,0,[]
2,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInFeatureTyping_Mapping,Mappings-CommonMappings-CommonParameterReferenceUsageInFeatureTyping_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInFeatureTyping_Mapping::from,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init::to,Mappings-CommonMappings-CommonParameterReferenceUsageInFeatureTyping_Mapping-from,Mappings-Initializers-KerMLInitializers-ToFeatureTyping_Init-to,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInFeatureTyping_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init,[UML::CommonStructure::Element],[KerML::Core::Features::FeatureTyping],UML::CommonStructure::Element,KerML::Core::Features::FeatureTyping,"[SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init, SysMLv1Tov2::Mappings::Foundations::Mapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToSpecialization_Init, 

,mapping_name,xmi_id,registry_key,identified_by,has_crosswalk
1,SysMLv1Tov2::Mappings::Foundations::Mapping,Mappings-Foundations-Mapping,"(SysML_v1_to_v2, Mappings-Foundations-Mapping, 1)",mapping_inheritance,False
92,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetFeature_Mapping,Mappings-SysMLv1-Allocations-AllocationUsageTargetFeature_Mapping,"(SysML_v1_to_v2, Mappings-SysMLv1-Allocations-AllocationUsageTargetFeature_Mapping, 1)",mapping_name_suffix,False
93,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationTargetReferenceUsageRedefinition_Mapping,Mappings-SysMLv1-Allocations-AllocationTargetReferenceUsageRedefinition_Mapping,"(SysML_v1_to_v2, Mappings-SysMLv1-Allocations-AllocationTargetReferenceUsageRedefinition_Mapping, 1)",mapping_name_suffix,False
96,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetFeatureSubsetting_Mapping,Mappings-SysMLv1-Allocations-AllocationUsageTargetFeatureSubsetting_Mapping,"(SysML_v1_to_v2, Mappings-SysMLv1-Allocations-AllocationUsageTargetFeatureSubsetting_Mapping, 1)",mapping_name_suffix,False
99,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetFeatureSubsettingFeature_Mapping,Mappings-SysMLv1-Allocations-AllocationUsageTargetFeatureSubsettingFeature_Mapping,"(SysML_v1_to_v2, Mappings-SysMLv1-Allocations-AllocationUsageTargetFeatureSubsettingFeature_Mapping, 1)",mapping_name_suffix,False
101,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetFeatureChaining_Mapping,Mappings-SysMLv1-Allocations-AllocationUsageTargetFeatureChaining_Mapping,"(SysML_v1_to_v2, Mappings-SysMLv1-Allocations-AllocationUsageTargetFeatureChaining_Mapping, 1)",mapping_name_suffix,False
102,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationSourceReferenceUsageRedefinition_Mapping,Mappings-SysMLv1-Allocations-AllocationSourceReferenceUsageRedefinition_Mapping,"(SysML_v1_to_v2, Mappings-SysMLv1-Allocations-AllocationSourceReferenceUsageRedefinition_Mapping, 1)",mapping_name_suffix,False
103,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationTargetFeatureMembership_Mapping,Mappings-SysMLv1-Allocations-AllocationTargetFeatureMembership_Mapping,"(SysML_v1_to_v2, Mappings-SysMLv1-Allocations-AllocationTargetFeatureMembership_Mapping, 1)",mapping_name_suffix,False
107,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetEndFeatureMembership_Mapping,Mappings-SysMLv1-Allocations-AllocationUsageTargetEndFeatureMembership_Mapping,"(SysML_v1_to_v2, Mappings-SysMLv1-Allocations-AllocationUsageTargetEndFeatureMembership_Mapping, 1)",mapping_name_suffix,False
240,SysMLv1Tov2::Mappings::UML4SysML::Classification::LowerBoundValueFeatureMembership_Mapping,Mappings-UML4SysML-Classification-LowerBoundValueFeatureMembership_Mapping,"(SysML_v1_to_v2, Mappings-UML4SysML-Classification-LowerBoundValueFeatureMembership_Mapping, 1)",mapping_name_suffix,False


In [95]:
# Dependencies originating from the 14 excluded classes
excluded_names = set(missing_crosswalk_df["mapping_name"])
excluded_dependencies = dependency_edges.loc[dependency_edges["source_class"].isin(excluded_names)].copy()
print("Dependency calls from excluded classes:", len(excluded_dependencies))
display(excluded_dependencies.groupby(["source_class", "dependency_status"], dropna=False).size().rename("call_count").reset_index())

Dependency calls from excluded classes: 13


,source_class,dependency_status,call_count
0,SysMLv1Tov2::Mappings::Foundations::Mapping,not_applicable,1
1,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationTargetFeatureMembership_Mapping,resolved,1
2,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetEndFeatureMembership_Mapping,resolved,1
3,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetFeatureChaining_Mapping,resolved,1
4,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetFeatureSubsettingFeature_Mapping,resolved,2
5,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetFeatureSubsetting_Mapping,resolved,1
6,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetFeature_Mapping,resolved,1
7,SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeature_Mapping,resolved,1
8,SysMLv1Tov2::Mappings::UML4SysML::Classification::LowerBoundValueFeatureMembership_Mapping,resolved,1
9,SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping,unresolved_mapping_target,2


In [96]:
supplemental_classes = missing_crosswalk_df.loc[missing_crosswalk_df["identified_by"].eq("mapping_name_suffix")]
supplemental_property_audit = []
for mapping_name in supplemental_classes["mapping_name"]:
    cls = get_class(registry, mapping_name)
    hierarchy = [cls] + get_ancestors(cls, parents_index)
    for owner in hierarchy:
        for prop in direct_members(owner):
            if (prop.metaclass == "uml:Property" and prop.name in {"from", "to"}):
                redefinitions = [ref for ref in registry.get_references(prop.key) if ref.element_tag == "redefinedProperty"]
                supplemental_property_audit.append({
                    "mapping_name": mapping_name,
                    "declared_by": owner.qualified_name,
                    "property": prop.name,
                    "property_xmi_id": prop.xmi_id,
                    "redefined_property_ids": [ref.value for ref in redefinitions],
                })

supplemental_property_audit = pd.DataFrame(supplemental_property_audit)

display(supplemental_property_audit.sort_values(["mapping_name", "property", "declared_by"]))

,mapping_name,declared_by,property,property_xmi_id,redefined_property_ids
31,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationSourceReferenceUsageRedefinition_Mapping,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationSourceReferenceUsageRedefinition_Mapping,from,Mappings-SysMLv1-Allocations-AllocationSourceReferenceUsageRedefinition_Mapping-from,[Mappings-Foundations-Mapping-from]
37,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationSourceReferenceUsageRedefinition_Mapping,SysMLv1Tov2::Mappings::Foundations::Initializer,to,Mappings-Foundations-Initializer-to,[]
36,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationSourceReferenceUsageRedefinition_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init,to,Mappings-Initializers-KerMLInitializers-ToElement_Init-to,[Mappings-Foundations-Initializer-to]
32,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationSourceReferenceUsageRedefinition_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRedefinition_Init,to,Mappings-Initializers-KerMLInitializers-ToRedefinition_Init-to,[Mappings-Initializers-KerMLInitializers-ToSubsetting_Init-to]
35,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationSourceReferenceUsageRedefinition_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init,to,Mappings-Initializers-KerMLInitializers-ToRelationship_Init-to,[Mappings-Initializers-KerMLInitializers-ToElement_Init-to]
...,...,...,...,...,...
86,SysMLv1Tov2::Mappings::UML4SysML::Interactions::MessageMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init,to,Mappings-Initializers-KerMLInitializers-ToFeatureMembership_Init-to,"[Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-to, Mappings-Initializers-KerMLInitializers-ToOwningMembership_Init-to]"
90,SysMLv1Tov2::Mappings::UML4SysML::Interactions::MessageMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init,to,Mappings-Initializers-KerMLInitializers-ToMembership_Init-to,[Mappings-Initializers-KerMLInitializers-ToRelationship_Init-to]
88,SysMLv1Tov2::Mappings::UML4SysML::Interactions::MessageMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToOwningMembership_Init,to,Mappings-Initializers-KerMLInitializers-ToOwningMembership_Init-to,[Mappings-Initializers-KerMLInitializers-ToMembership_Init-to]
89,SysMLv1Tov2::Mappings::UML4SysML::Interactions::MessageMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init,to,Mappings-Initializers-KerMLInitializers-ToRelationship_Init-to,[Mappings-Initializers-KerMLInitializers-ToElement_Init-to]


In [97]:
mapping_table['unresolved_dependency_count'].unique()

array([0, 1, 2])

# SysML1 --> SysML2 Translation:

In [98]:
operation_columns = ["mapping_class", "operation", "operation_name", "operation_xmi_id", "redefined_operations", "operation_category",]
mapping_operation_base = (mapping_operations[operation_columns] .rename(columns={"mapping_class": "mapping_name"}) .copy())
mapping_operation_specs = mapping_operation_base.merge(
    ocl_specifications[["operation", "constraint_id", "constraint_name", "expression_id", "language", "body", "specification_category",]],
    on="operation",
    how="left",
    validate="one_to_many",
)
mapping_operation_table = mapping_operation_specs.merge(
    mapping_table[["mapping_name", "mapping_xmi_id", "source_type", "target_type",]],
    on="mapping_name",
    how="left",
    validate="many_to_one",
)
mapping_operation_table = mapping_operation_table.sort_values(["mapping_name", "operation_name"]).reset_index(drop=True)

In [99]:
print("Mapping operation rows:", len(mapping_operation_table))
print("Unique operations:", mapping_operation_table["operation_xmi_id"].nunique())
print("Operations without specifications:", mapping_operation_table["body"].isna().sum())
print("Operations without crosswalk:", mapping_operation_table["source_type"].isna().sum())

Mapping operation rows: 946
Unique operations: 946
Operations without specifications: 0
Operations without crosswalk: 0


## Test:

In [100]:
def get_effective_operations(mapping_name):
    try:
        cls = get_class(registry, mapping_name)
    except ValueError as exc:
        print(f"Skipping {mapping_name}: {exc}")
        return []
    hierarchy = [cls] + get_ancestors(cls, parents_index)
    operations = []
    for owner in hierarchy:
        for operation in get_containment_children(owner, registry):
            if operation.metaclass != "uml:Operation":
                continue
            operations.append({
                "mapping_name": mapping_name,
                "declared_by": owner.qualified_name,
                "operation": operation.qualified_name,
                "operation_name": operation.name,
                "operation_xmi_id": operation.xmi_id,
                "is_direct": owner.key == cls.key,
            })
    return operations

test_mappings = [
    "SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping",
    "SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeature_Mapping",
]

# Find an additional mapping with a Comment source type, if present.
comment_candidates = mapping_table.loc[
    mapping_table["source_type"].astype(str).str.contains(
        "Comment", case=False, na=False
    ),
    "mapping_name",
].tolist()

if comment_candidates:
    test_mappings.append(comment_candidates[0])

# Verify every name before extraction.
for name in test_mappings:
    print(name, "->", len([
        e for e in registry.get_elements("SysML_v1_to_v2")
        if e.qualified_name == name and e.metaclass == "uml:Class"
    ]))

SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping -> 1
SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeature_Mapping -> 1
SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ConcernDocumentation_Mapping -> 1


In [101]:
test_operations = pd.DataFrame([
    operation
    for mapping_name in test_mappings
    for operation in get_effective_operations(mapping_name)
])


In [102]:
display(
    test_operations.groupby(
        ["mapping_name", "is_direct"]
    ).size().unstack(fill_value=0)
)
display(
    test_operations.loc[
        ~test_operations["is_direct"]
    ].groupby(
        ["mapping_name", "declared_by"]
    )["operation_name"]
    .agg(list)
    .reset_index()
)
duplicates = test_operations.loc[
    test_operations.duplicated(
        ["mapping_name", "operation_xmi_id"],
        keep=False
    )
].sort_values(["mapping_name", "operation_xmi_id"])

display(duplicates)
display(
    test_operations[
        [
            "mapping_name",
            "operation_name",
            "declared_by",
            "is_direct",
        ]
    ].sort_values(
        ["mapping_name", "is_direct", "operation_name"],
        ascending=[True, False, True],
    )
)


is_direct,False,True
mapping_name,,
SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,17,1
SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ConcernDocumentation_Mapping,12,1
SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeature_Mapping,19,5


,mapping_name,declared_by,operation_name
0,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,SysMLv1Tov2::Mappings::Foundations::Mapping,"[filter, getMappedColl, getMapped, getMapped]"
1,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init,"[elementId, ownedRelationship, aliasId, shortName, declaredName]"
2,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init,"[memberName, visibility, memberElement, memberShortName, membershipOwningNamespace]"
3,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init,"[ownedRelatedElement, target, source]"
4,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ConcernDocumentation_Mapping,SysMLv1Tov2::Mappings::Foundations::Mapping,"[filter, getMappedColl, getMapped, getMapped]"
5,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ConcernDocumentation_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToAnnotatingElement_Init,[annotation]
6,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ConcernDocumentation_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToComment_Init,"[body, locale]"
7,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ConcernDocumentation_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init,"[elementId, ownedRelationship, aliasId, shortName, declaredName]"
8,SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeature_Mapping,SysMLv1Tov2::Mappings::Foundations::Mapping,"[filter, getMappedColl, getMapped, getMapped]"
9,SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeature_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init,"[elementId, ownedRelationship, aliasId, shortName, declaredName]"


,mapping_name,declared_by,operation,operation_name,operation_xmi_id,is_direct


,mapping_name,operation_name,declared_by,is_direct
0,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,memberElement,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,True
15,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,aliasId,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init,False
17,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,declaredName,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init,False
13,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,elementId,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init,False
6,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,filter,SysMLv1Tov2::Mappings::Foundations::Mapping,False
8,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,getMapped,SysMLv1Tov2::Mappings::Foundations::Mapping,False
9,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,getMapped,SysMLv1Tov2::Mappings::Foundations::Mapping,False
7,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,getMappedColl,SysMLv1Tov2::Mappings::Foundations::Mapping,False
3,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,memberElement,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init,False
1,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,memberName,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init,False


## Scaled Implimentation:

In [103]:
def get_effective_operations(mapping_name):
    cls = get_class(registry, mapping_name)
    hierarchy = [cls] + get_ancestors(cls, parents_index)
    operations = []
    for owner in hierarchy:
        for operation in direct_members(owner):
            if operation.metaclass != "uml:Operation":
                continue
            redefinitions = []
            for ref in registry.get_references(operation.key):
                if ref.element_tag != "redefinedOperation":
                    continue
                target = (registry.elements.get(ref.target) if ref.resolved else None)
                redefinitions.append({
                    "operation": (target.qualified_name if target else None),
                    "operation_xmi_id": (target.xmi_id if target else None),
                    "resolution_status": ref.resolution_status,
                })
            operations.append({
                "mapping_name": mapping_name,
                "declared_by": owner.qualified_name,
                "operation": operation.qualified_name,
                "operation_name": operation.name,
                "operation_xmi_id": operation.xmi_id,
                "is_direct": owner.key == cls.key,
                "redefined_operations": redefinitions,
            })
    return operations

mapping_operation_declarations = pd.DataFrame([operation for mapping_name in mapping_table["mapping_name"] for operation in get_effective_operations(mapping_name)])
print("Declaration rows:", len(mapping_operation_declarations))
print("Unique mapping-operation pairs:", mapping_operation_declarations[["mapping_name", "operation_xmi_id"]].drop_duplicates().shape[0])
print("\nDirect vs inherited:")
display(mapping_operation_declarations.groupby("is_direct").size())
print("\nRedefined operation declarations:")
display(
    mapping_operation_declarations.loc[
        mapping_operation_declarations["redefined_operations"].map(bool),
        ["mapping_name", "operation_name", "declared_by", "redefined_operations",],
    ].head(15)
)

Declaration rows: 17051
Unique mapping-operation pairs: 17051

Direct vs inherited:


is_direct
False    16105
True       946
dtype: int64


Redefined operation declarations:


,mapping_name,operation_name,declared_by,redefined_operations
0,SysMLv1Tov2::Mappings::CommonMappings::CommonFeatureReferenceExpression_Mapping,ownedRelationship,SysMLv1Tov2::Mappings::CommonMappings::CommonFeatureReferenceExpression_Mapping,"[{'operation': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init::ownedRelationship', 'operation_xmi_id': 'Mappings-Initializers-KerMLInitializers-ToElement_Init-ownedRelationship_', 'resolution_status': 'resolved_local'}]"
20,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,memberElement,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,"[{'operation': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init::memberElement', 'operation_xmi_id': 'Mappings-Initializers-KerMLInitializers-ToMembership_Init-memberElement_', 'resolution_status': 'resolved_local'}]"
23,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,memberElement,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init,"[{'operation': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init::target', 'operation_xmi_id': 'Mappings-Initializers-KerMLInitializers-ToRelationship_Init-target_', 'resolution_status': 'resolved_local'}]"
25,SysMLv1Tov2::Mappings::CommonMappings::CommonMembership_Mapping,membershipOwningNamespace,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init,"[{'operation': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init::source', 'operation_xmi_id': 'Mappings-Initializers-KerMLInitializers-ToRelationship_Init-source_', 'resolution_status': 'resolved_local'}]"
38,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInFeatureTyping_Mapping,type,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInFeatureTyping_Mapping,"[{'operation': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init::type', 'operation_xmi_id': 'Mappings-Initializers-KerMLInitializers-ToFeatureTyping_Init-type_', 'resolution_status': 'resolved_local'}]"
39,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInFeatureTyping_Mapping,type,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init,"[{'operation': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToSpecialization_Init::general', 'operation_xmi_id': 'Mappings-Initializers-KerMLInitializers-ToSpecialization_Init-general_', 'resolution_status': 'resolved_local'}]"
40,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInFeatureTyping_Mapping,typedFeature,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init,"[{'operation': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToSpecialization_Init::specific', 'operation_xmi_id': 'Mappings-Initializers-KerMLInitializers-ToSpecialization_Init-specific_', 'resolution_status': 'resolved_local'}]"
45,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInFeatureTyping_Mapping,specific,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToSpecialization_Init,"[{'operation': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init::source', 'operation_xmi_id': 'Mappings-Initializers-KerMLInitializers-ToRelationship_Init-source_', 'resolution_status': 'resolved_local'}]"
46,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInFeatureTyping_Mapping,general,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToSpecialization_Init,"[{'operation': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init::target', 'operation_xmi_id': 'Mappings-Initializers-KerMLInitializers-ToRelationship_Init-target_', 'resolution_status': 'resolved_local'}]"
55,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInMembership_Mapping,ownedMemberParameter,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInMembership_Mapping,"[{'operation': 'SysMLv

### Verification / Debugging

In [104]:
print("Declaration columns:")
print(mapping_operation_declarations.columns.tolist())

print("\nSpecification columns:")
print(ocl_specifications.columns.tolist())

print("\nSpecification operation ID uniqueness:")
print(ocl_specifications["operation_id"].nunique(), "unique IDs across", len(ocl_specifications), "specification rows")

print("\nOperation ID overlap:")
declaration_ids = set(mapping_operation_declarations["operation_xmi_id"].dropna())
specification_ids = set(ocl_specifications["operation_id"].dropna())

print("Declaration IDs:", len(declaration_ids))
print("Specification IDs:", len(specification_ids))
print("Matching IDs:", len(declaration_ids & specification_ids))
print("Unmatched declaration IDs:", len(declaration_ids - specification_ids))

Declaration columns:
['mapping_name', 'declared_by', 'operation', 'operation_name', 'operation_xmi_id', 'is_direct', 'redefined_operations']

Specification columns:
['operation_id', 'operation', 'constraint_id', 'constraint_name', 'expression_id', 'language', 'body', 'specification_category']

Specification operation ID uniqueness:
1101 unique IDs across 1101 specification rows

Operation ID overlap:
Declaration IDs: 1024
Specification IDs: 1101
Matching IDs: 0
Unmatched declaration IDs: 1024


In [105]:
print("Declaration operation IDs:")
display(mapping_operation_declarations[["operation", "operation_xmi_id"]].drop_duplicates("operation_xmi_id").head(10))
print("\nSpecification operation IDs:")
display(ocl_specifications[["operation", "operation_id"]].drop_duplicates("operation_id").head(10))

Declaration operation IDs:


,operation,operation_xmi_id
0,SysMLv1Tov2::Mappings::CommonMappings::CommonFeatureReferenceExpression_Mapping::ownedRelationship,Mappings-CommonMappings-CommonFeatureReferenceExpression_Mapping-ownedRelationship_
1,SysMLv1Tov2::Mappings::Foundations::Mapping::filter,Mappings-Foundations-Mapping-filter_Element
2,SysMLv1Tov2::Mappings::Foundations::Mapping::getMappedColl,Mappings-Foundations-Mapping-getMappedColl_Element
3,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped,Mappings-Foundations-Mapping-getMapped_Element
4,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped,Mappings-Foundations-Mapping-getMapped_Element_Element
5,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init::isUnique,Mappings-Initializers-KerMLInitializers-ToFeature_Init-isUnique_
6,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init::isOrdered,Mappings-Initializers-KerMLInitializers-ToFeature_Init-isOrdered_
7,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init::isComposite,Mappings-Initializers-KerMLInitializers-ToFeature_Init-isComposite_
8,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init::isPortion,Mappings-Initializers-KerMLInitializers-ToFeature_Init-isPortion_
9,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init::isDerived,Mappings-Initializers-KerMLInitializers-ToFeature_Init-isDerived_



Specification operation IDs:


,operation,operation_id
0,SysMLv1Tov2::Mappings::Foundations::Mapping::filter,"(SysML_v1_to_v2, Mappings-Foundations-Mapping-filter_Element, 1)"
1,SysMLv1Tov2::Mappings::Foundations::Mapping::getMappedColl,"(SysML_v1_to_v2, Mappings-Foundations-Mapping-getMappedColl_Element, 1)"
2,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped,"(SysML_v1_to_v2, Mappings-Foundations-Mapping-getMapped_Element, 1)"
3,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped,"(SysML_v1_to_v2, Mappings-Foundations-Mapping-getMapped_Element_Element, 1)"
4,SysMLv1Tov2::Mappings::Factories::ReturnParameterFeature_Factory::direction,"(SysML_v1_to_v2, Mappings-Factories-ReturnParameterFeature_Factory-direction_, 1)"
5,SysMLv1Tov2::Mappings::Factories::StringParameterFeatureValue_Factory::value,"(SysML_v1_to_v2, Mappings-Factories-StringParameterFeatureValue_Factory-value_, 1)"
6,SysMLv1Tov2::Mappings::Factories::AssignmentActionUsageParameterMembership_Factory::ownedMemberParameter,"(SysML_v1_to_v2, Mappings-Factories-AssignmentActionUsageParameterMembership_Factory-ownedMemberParameter_, 1)"
7,SysMLv1Tov2::Mappings::Factories::DirectedReferenceUsage_Factory::direction,"(SysML_v1_to_v2, Mappings-Factories-DirectedReferenceUsage_Factory-direction_, 1)"
8,SysMLv1Tov2::Mappings::Factories::FeatureTyping_Factory::type,"(SysML_v1_to_v2, Mappings-Factories-FeatureTyping_Factory-type_, 1)"
9,SysMLv1Tov2::Mappings::Factories::LiteralNull_Factory::ownedRelationship,"(SysML_v1_to_v2, Mappings-Factories-LiteralNull_Factory-ownedRelationship_, 1)"


In [106]:
for _, row in ocl_specifications.head(5).iterrows():
    print({"operation_id": row["operation_id"], "operation": row["operation"],})


for _, row in mapping_operation_declarations[
    ["operation", "operation_xmi_id"]
].drop_duplicates("operation_xmi_id").head(5).iterrows():

    print({"operation": row["operation"], "operation_xmi_id": row["operation_xmi_id"],})

{'operation_id': ('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-filter_Element', 1), 'operation': 'SysMLv1Tov2::Mappings::Foundations::Mapping::filter'}
{'operation_id': ('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-getMappedColl_Element', 1), 'operation': 'SysMLv1Tov2::Mappings::Foundations::Mapping::getMappedColl'}
{'operation_id': ('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-getMapped_Element', 1), 'operation': 'SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped'}
{'operation_id': ('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-getMapped_Element_Element', 1), 'operation': 'SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped'}
{'operation_id': ('SysML_v1_to_v2', 'Mappings-Factories-ReturnParameterFeature_Factory-direction_', 1), 'operation': 'SysMLv1Tov2::Mappings::Factories::ReturnParameterFeature_Factory::direction'}
{'operation': 'SysMLv1Tov2::Mappings::CommonMappings::CommonFeatureReferenceExpression_Mapping::ownedRelationship', 'operation_xmi_id': 'Mappings-Comm

#### Debugging Fixes:

In [107]:
# Normalize specification ID
ocl_specifications = ocl_specifications.copy()
ocl_specifications["operation_resource"] = (ocl_specifications["operation_id"].str[0])
ocl_specifications["operation_xmi_id"] = (ocl_specifications["operation_id"].str[1])
ocl_specifications["operation_occurrence"] = (ocl_specifications["operation_id"].str[2])

# Validate Normalized
print("Matching XMI IDs:", len(set(mapping_operation_declarations["operation_xmi_id"]) & set(ocl_specifications["operation_xmi_id"])))
print("\nDuplicate specification XMI IDs:")
display(
    ocl_specifications.loc[ocl_specifications["operation_xmi_id"].duplicated(keep=False), ["operation", "operation_xmi_id", "operation_occurrence"],]
)
print("\nSpecification resources:")
display(ocl_specifications["operation_resource"].value_counts())

Matching XMI IDs: 1022

Duplicate specification XMI IDs:


,operation,operation_xmi_id,operation_occurrence



Specification resources:


operation_resource
SysML_v1_to_v2    1101
Name: count, dtype: int64

In [108]:
# Determine if unmatched declarations are from helpers/factories/initializers/other
declaration_ids = set(mapping_operation_declarations["operation_xmi_id"].dropna())
specification_ids = set(ocl_specifications["operation_xmi_id"].dropna())
unmatched_declaration_ids = declaration_ids - specification_ids

display(
    mapping_operation_declarations.loc[
        mapping_operation_declarations["operation_xmi_id"].isin(
            unmatched_declaration_ids
        ),
        ["mapping_name", "declared_by", "operation", "operation_xmi_id", "is_direct",],
    ].drop_duplicates()
)

,mapping_name,declared_by,operation,operation_xmi_id,is_direct
64,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featureOfType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_,False
65,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featuringType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featuringType_,False
131,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInFeatureMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featureOfType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_,False
132,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInFeatureMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featuringType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featuringType_,False
220,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterFeatureMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featureOfType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_,False
...,...,...,...,...,...
16708,SysMLv1Tov2::Mappings::UML4SysML::Values::OpaqueExpressionFeatureFeatureMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featuringType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featuringType_,False
16855,SysMLv1Tov2::Mappings::UML4SysML::Values::OpaqueExpressionParameterMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featureOfType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_,False
16856,SysMLv1Tov2::Mappings::UML4SysML::Values::OpaqueExpressionParameterMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featuringType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featuringType_,False
16900,SysMLv1Tov2::Mappings::UML4SysML::Values::OpaqueExpressionReferenceUsageReturnParameterMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featureOfType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_,False


In [109]:
mapping_operation_specifications = (
    mapping_operation_declarations.merge(
        ocl_specifications[
            ["operation_xmi_id", "operation_id", "constraint_id", "constraint_name", "expression_id", "language", "body", "specification_category",]
        ],
        on="operation_xmi_id",
        how="left",
        validate="many_to_one",
        indicator=True,
    )
)

print("Rows:", len(mapping_operation_specifications))

print("\nSpecification match status:")
display(
    mapping_operation_specifications["_merge"].value_counts()
)

print("\nDeclarations without specifications:")
display(
    mapping_operation_specifications.loc[
        mapping_operation_specifications["_merge"].eq("left_only"),
        ["mapping_name", "declared_by", "operation", "operation_xmi_id"],
    ].drop_duplicates()
)

Rows: 17051

Specification match status:


_merge
both          16769
left_only       282
right_only        0
Name: count, dtype: int64


Declarations without specifications:


,mapping_name,declared_by,operation,operation_xmi_id
64,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featureOfType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_
65,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featuringType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featuringType_
131,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInFeatureMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featureOfType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_
132,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInFeatureMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featuringType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featuringType_
220,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterFeatureMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featureOfType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_
...,...,...,...,...
16708,SysMLv1Tov2::Mappings::UML4SysML::Values::OpaqueExpressionFeatureFeatureMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featuringType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featuringType_
16855,SysMLv1Tov2::Mappings::UML4SysML::Values::OpaqueExpressionParameterMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featureOfType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_
16856,SysMLv1Tov2::Mappings::UML4SysML::Values::OpaqueExpressionParameterMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featuringType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featuringType_
16900,SysMLv1Tov2::Mappings::UML4SysML::Values::OpaqueExpressionReferenceUsageReturnParameterMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featureOfType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_


In [110]:
unmatched_operations = (
    mapping_operation_declarations.loc[
        mapping_operation_declarations["operation_xmi_id"].isin(
            unmatched_declaration_ids
        ),
        ["operation", "operation_xmi_id"],
    ]
    .drop_duplicates()
)

for _, row in unmatched_operations.iterrows():
    operation = registry.get_element(
        "SysML_v1_to_v2",
        row["operation_xmi_id"],
    )

    print(f"\n{row['operation']}")
    print("Record attributes:")
    print(vars(operation))

    print("\nDirect children:")
    for child in direct_members(operation):
        print(
            child.metaclass,
            child.tag,
            child.name,
            child.xmi_id,
        )


SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featureOfType
Record attributes:
{'resource': 'SysML_v1_to_v2', 'xmi_id': 'Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_', 'tag': 'ownedOperation', 'metaclass': 'uml:Operation', 'name': 'featureOfType', 'attributes': {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Operation', 'isAbstract': 'true', 'name': 'featureOfType'}, 'parent': ('SysML_v1_to_v2', 'Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init', 1), 'children': [('SysML_v1_to_v2', 'Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_-result', 1)], 'element_category': None, 'qualified_name': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featureOfType', 'namespace': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeat

In [111]:
descendants = [
    cls
    for cls in registry.get_elements("SysML_v1_to_v2")
    if cls.metaclass == "uml:Class"
    and any(
        ancestor.key == initializer.key
        for ancestor in get_ancestors(cls, parents_index)
    )
]

display(pd.DataFrame([
    {
        "class": cls.qualified_name,
        "is_abstract": cls.attributes.get(
            "isAbstract", "false"
        ).lower() == "true",
    }
    for cls in descendants
]).groupby("is_abstract").size().rename("class_count"))

is_abstract
False    801
True     107
Name: class_count, dtype: int64

In [112]:
descendant_operations = []

for cls in descendants:
    for op in direct_members(cls):
        if op.metaclass != "uml:Operation":
            continue

        descendant_operations.append({
            "class": cls.qualified_name,
            "class_is_abstract": (
                cls.attributes.get("isAbstract", "false").lower()
                == "true"
            ),
            "operation": op.qualified_name,
            "operation_name": op.name,
            "operation_xmi_id": op.xmi_id,
            "operation_is_abstract": (
                op.attributes.get("isAbstract", "false").lower()
                == "true"
            ),
        })

descendant_operations = pd.DataFrame(descendant_operations)

display(
    descendant_operations.loc[
        descendant_operations["operation_name"].isin(
            ["featureOfType", "featuringType"]
        )
    ].sort_values(["operation_name", "class"])
)

print(
    "Descendants declaring operations:",
    descendant_operations["class"].nunique()
)

print(
    "Total direct operation declarations:",
    len(descendant_operations)
)

,class,class_is_abstract,operation,operation_name,operation_xmi_id,operation_is_abstract
1100,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,True,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featureOfType,featureOfType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_,True
1101,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,True,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featuringType,featuringType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featuringType_,True


Descendants declaring operations: 802
Total direct operation declarations: 1128


In [113]:
initializer_operation_metadata = pd.DataFrame([
    {
        "operation": (
            "SysMLv1Tov2::Mappings::Initializers::KerMLInitializers"
            "::ToTypeFeaturing_Init::featureOfType"
        ),
        "is_abstract": True,
        "return_type": "KerML::Core::Features::Feature",
        "has_ocl_specification": False,
        "formal_redefinition_count": 0,
    },
    {
        "operation": (
            "SysMLv1Tov2::Mappings::Initializers::KerMLInitializers"
            "::ToTypeFeaturing_Init::featuringType"
        ),
        "is_abstract": True,
        "return_type": "KerML::Core::Types::Type",
        "has_ocl_specification": False,
        "formal_redefinition_count": 0,
    },
])


target_names = [
    "KerML::Core::Features::Feature",
    "KerML::Core::Types::Type",
    "KerML::Core::Features::TypeFeaturing",
]

for name in target_names:
    matches = [
        e for e in registry.get_elements("KerML")
        if e.qualified_name == name
        and e.metaclass == "uml:Class"
    ]

    print(f"\n{name}: {len(matches)} matches")

    if matches:
        cls = matches[0]
        for member in direct_members(cls):
            print(member.metaclass, member.tag, member.name, member.xmi_id,)


KerML::Core::Features::Feature: 1 matches

KerML::Core::Types::Type: 1 matches

KerML::Core::Features::TypeFeaturing: 1 matches


In [114]:
feature = get_class_in_resource(
    registry, "KerML", "KerML::Core::Features::Feature"
)

relevant_names = {"featuringType", "ownedTypeFeaturing"}

for prop in registry.get_elements("KerML"):
    if prop.parent != feature.key or prop.name not in relevant_names:
        continue

    print(f"\n{prop.name}")
    print("Attributes:", prop.attributes)

    for ref in registry.get_references(prop.key):
        print({
            "element_tag": ref.element_tag,
            "value": ref.value,
            "target": (
                registry.elements[ref.target].qualified_name
                if ref.resolved else None
            ),
            "status": ref.resolution_status,
        })


featuringType
Attributes: {'{http://www.omg.org/spec/XMI/20161101}id': 'Core-Features-Feature-featuringType', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Property', 'isDerived': 'true', 'isOrdered': 'true', 'name': 'featuringType'}
{'element_tag': 'association', 'value': 'Core-Features-A_featuringType_featureOfType', 'target': 'KerML::Core::Features::A_featuringType_featureOfType', 'status': 'resolved_local'}
{'element_tag': 'type', 'value': 'Core-Types-Type', 'target': 'KerML::Core::Types::Type', 'status': 'resolved_local'}

ownedTypeFeaturing
Attributes: {'{http://www.omg.org/spec/XMI/20161101}id': 'Core-Features-Feature-ownedTypeFeaturing', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Property', 'isDerived': 'true', 'isOrdered': 'true', 'name': 'ownedTypeFeaturing'}
{'element_tag': 'association', 'value': 'Core-Features-A_ownedTypeFeaturing_owningFeatureOfType', 'target': 'KerML::Core::Features::A_ownedTypeFeaturing_owningFeatureOfType', 'status': 'resolved_local'}
{'e

In [115]:
association_ids = [
    "Core-Features-A_featuringType_featureOfType",
    "Core-Features-A_ownedTypeFeaturing_owningFeatureOfType",
    "Core-Features-A_featureOfType_typeFeaturing",
]

for xmi_id in association_ids:
    association_key = registry.elements_by_xmi_id[("KerML", xmi_id)][0]
    association = registry.elements[association_key]
    print(f"\n{association.qualified_name}")

    for ref in registry.get_references(association.key):
        if ref.element_tag != "memberEnd" or not ref.resolved:
            continue
        end = registry.elements[ref.target]

        print({
            "end": end.qualified_name,
            "attributes": end.attributes,
            "references": [
                {
                    "element_tag": r.element_tag,
                    "value": r.value,
                    "target": (registry.elements[r.target].qualified_name if r.resolved else None),
                }
                for r in registry.get_references(end.key)
            ],
        })


KerML::Core::Features::A_featuringType_featureOfType
{'end': 'KerML::Core::Features::Feature::featuringType', 'attributes': {'{http://www.omg.org/spec/XMI/20161101}id': 'Core-Features-Feature-featuringType', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Property', 'isDerived': 'true', 'isOrdered': 'true', 'name': 'featuringType'}, 'references': [{'element_tag': 'association', 'value': 'Core-Features-A_featuringType_featureOfType', 'target': 'KerML::Core::Features::A_featuringType_featureOfType'}, {'element_tag': 'type', 'value': 'Core-Types-Type', 'target': 'KerML::Core::Types::Type'}]}
{'end': 'KerML::Core::Features::A_featuringType_featureOfType::featureOfType', 'attributes': {'{http://www.omg.org/spec/XMI/20161101}id': 'Core-Features-A_featuringType_featureOfType-featureOfType', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Property', 'isDerived': 'true', 'name': 'featureOfType'}, 'references': [{'element_tag': 'association', 'value': 'Core-Features-A_featuringType_featur

In [116]:
base = get_class_in_resource(
    registry,
    "SysML_v1_to_v2",
    "SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init",
)

for child in children_index.get(base.key, []):
    print(f"\n{'=' * 70}")
    print("Subclass:", child.qualified_name)
    print("Metaclass:", child.metaclass)
    print("Attributes:", child.attributes)

    operations = [
        element
        for element in registry.get_elements("SysML_v1_to_v2")
        if element.parent == child.key
        and element.metaclass == "uml:Operation"
    ]

    print("Direct operations:")
    for operation in operations:
        print({
            "name": operation.name,
            "attributes": operation.attributes,
            "redefinitions": [
                {
                    "target": (
                        registry.elements[ref.target].qualified_name
                        if ref.resolved else ref.value
                    ),
                    "status": ref.resolution_status,
                }
                for ref in registry.get_references(operation.key)
                if ref.element_tag == "redefinedOperation"
            ],
        })


Subclass: SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init
Metaclass: uml:Class
Attributes: {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Initializers-KerMLInitializers-ToFeatureMembership_Init', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Class', 'isAbstract': 'true', 'name': 'ToFeatureMembership_Init'}
Direct operations:
{'name': 'ownedMemberFeature', 'attributes': {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Initializers-KerMLInitializers-ToFeatureMembership_Init-ownedMemberFeature_', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Operation', 'name': 'ownedMemberFeature'}, 'redefinitions': [{'target': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToOwningMembership_Init::ownedMemberElement', 'status': 'resolved_local'}]}
{'name': 'ownedRelatedElement', 'attributes': {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Initializers-KerMLInitializers-ToFeatureMembership_Init-ownedRelatedElement_', '{http://www

In [117]:
child = get_class_in_resource(registry, "SysML_v1_to_v2", "SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init",)

print("Class:", child.qualified_name)

print("\nDirect parents:")
for parent in parents_index.get(child.key, []):
    print(parent.qualified_name)

print("\nDirect properties:")
for element in registry.get_elements("SysML_v1_to_v2"):
    if element.parent == child.key and element.metaclass == "uml:Property":
        print({
            "name": element.name,
            "attributes": element.attributes,
            "references": [
                {
                    "reference": ref.element_tag,
                    "target": (
                        registry.elements[ref.target].qualified_name
                        if ref.resolved else ref.value
                    ),
                }
                for ref in registry.get_references(element.key)
            ],
        })

Class: SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init

Direct parents:
SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init
SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToOwningMembership_Init

Direct properties:
{'name': 'to', 'attributes': {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Initializers-KerMLInitializers-ToFeatureMembership_Init-to', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Property', 'name': 'to'}, 'references': [{'reference': 'association', 'target': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::A_ToFeatureMembership_Init_to'}, {'reference': 'redefinedProperty', 'target': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::to'}, {'reference': 'redefinedProperty', 'target': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToOwningMembership_Init::to'}, {'reference': 'type', 'target': 'KerML::Core::Types::FeatureMembership'}]}


In [118]:
ancestors = get_ancestors(child, parents_index)

hierarchy = [child] + [
    registry.elements[ancestor.key]
    for ancestor in ancestors
]

print("Inheritance hierarchy:")
for element in hierarchy:
    print(element.qualified_name)

print("\nOperation declarations:")
for owner in hierarchy:
    operations = [
        element
        for element in registry.get_elements("SysML_v1_to_v2")
        if element.parent == owner.key
        and element.metaclass == "uml:Operation"
    ]

    for operation in operations:
        redefinitions = [
            registry.elements[ref.target].qualified_name
            if ref.resolved else ref.value
            for ref in registry.get_references(operation.key)
            if ref.element_tag == "redefinedOperation"
        ]

        print({
            "declared_by": owner.qualified_name,
            "operation": operation.name,
            "abstract": operation.attributes.get("isAbstract"),
            "redefines": redefinitions,
        })

Inheritance hierarchy:
SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init
SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init
SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToOwningMembership_Init
SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init
SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init
SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init
SysMLv1Tov2::Mappings::Foundations::Initializer

Operation declarations:
{'declared_by': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init', 'operation': 'ownedMemberFeature', 'abstract': None, 'redefines': ['SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToOwningMembership_Init::ownedMemberElement']}
{'declared_by': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init', 'operation': 'ownedRelatedElement', 'abstract': None, 'redefines': ['SysML

In [119]:
base = child  # ToFeatureMembership_Init

descendants = get_ancestors(base, children_index)

concrete_descendants = [
    element for element in descendants
    if element.metaclass == "uml:Class"
    and element.attributes.get("isAbstract") != "true"
]

print("Total descendants:", len(descendants))
print("Concrete descendants:", len(concrete_descendants))

for cls in concrete_descendants:
    operations = [
        element
        for element in registry.get_elements("SysML_v1_to_v2")
        if element.parent == cls.key
        and element.metaclass == "uml:Operation"
    ]

    print({
        "class": cls.qualified_name,
        "direct_operations": [op.name for op in operations],
    })

Total descendants: 169
Concrete descendants: 158
{'class': 'SysMLv1Tov2::Mappings::Factories::MultiplicityUpperBoundMembership_Factory', 'direct_operations': ['create']}
{'class': 'SysMLv1Tov2::Mappings::Factories::FlowItemFeatureMembership_Factory', 'direct_operations': ['create', 'ownedMemberFeature']}
{'class': 'SysMLv1Tov2::Mappings::Factories::MultiplicityMembership_Factory', 'direct_operations': ['create', 'ownedMemberFeature']}
{'class': 'SysMLv1Tov2::Mappings::Factories::FlowUsageFeatureMembership_Factory', 'direct_operations': ['create', 'ownedMemberFeature']}
{'class': 'SysMLv1Tov2::Mappings::Factories::AssignmentActionUsageFeatureMembership2_Factory', 'direct_operations': ['create', 'ownedMemberFeature']}
{'class': 'SysMLv1Tov2::Mappings::Factories::AssignmentActionUsageFeatureMembership3_Factory', 'direct_operations': ['create', 'ownedMemberFeature']}
{'class': 'SysMLv1Tov2::Mappings::Factories::MultiplicityLowerBoundMembership_Factory', 'direct_operations': ['create', 'own

In [120]:
from collections import Counter, defaultdict

operation_counts = Counter()
override_classes = defaultdict(list)
direct_operation_records = []

abstract_endpoint_operations = {
    "featureOfType",
    "featuringType",
}

for cls in concrete_descendants:
    operations = [
        element
        for element in registry.get_elements("SysML_v1_to_v2")
        if element.parent == cls.key
        and element.metaclass == "uml:Operation"
    ]

    for operation in operations:
        operation_counts[operation.name] += 1

        direct_operation_records.append({
            "class": cls.qualified_name,
            "operation": operation.name,
            "abstract": operation.attributes.get("isAbstract"),
        })

        if operation.name in abstract_endpoint_operations:
            override_classes[operation.name].append(
                cls.qualified_name
            )

print("Direct operation declaration counts:")
for name, count in operation_counts.most_common():
    print(f"{name}: {count}")

print("\nDirect declarations of abstract endpoint operations:")
for name in sorted(abstract_endpoint_operations):
    print(name, override_classes[name])

print("\nConcrete descendants:", len(concrete_descendants))

Direct operation declaration counts:
ownedMemberFeature: 112
ownedMemberParameter: 39
create: 16
filter: 7
memberFeature: 7
visibility: 6
kind: 6
isReadOnly: 1
memberName: 1
memberParameter: 1
ownedRelatedElement: 1

Direct declarations of abstract endpoint operations:
featureOfType []
featuringType []

Concrete descendants: 158


In [121]:
for cls in concrete_descendants:
    if "::Factories::" not in cls.qualified_name:
        continue

    operations = [
        element
        for element in registry.get_elements("SysML_v1_to_v2")
        if element.parent == cls.key
        and element.metaclass == "uml:Operation"
        and element.name == "create"
    ]

    for operation in operations:
        specs = ocl_specifications[
            ocl_specifications["operation_id"] == operation.key
        ]

        print(f"\n{'=' * 80}")
        print("Factory:", cls.qualified_name)

        for _, spec in specs.iterrows():
            print({
                "language": spec["language"],
                "body": spec["body"],
            })


Factory: SysMLv1Tov2::Mappings::Factories::MultiplicityUpperBoundMembership_Factory

Factory: SysMLv1Tov2::Mappings::Factories::FlowItemFeatureMembership_Factory

Factory: SysMLv1Tov2::Mappings::Factories::MultiplicityMembership_Factory

Factory: SysMLv1Tov2::Mappings::Factories::FlowUsageFeatureMembership_Factory

Factory: SysMLv1Tov2::Mappings::Factories::AssignmentActionUsageFeatureMembership2_Factory

Factory: SysMLv1Tov2::Mappings::Factories::AssignmentActionUsageFeatureMembership3_Factory

Factory: SysMLv1Tov2::Mappings::Factories::MultiplicityLowerBoundMembership_Factory

Factory: SysMLv1Tov2::Mappings::Factories::AssignmentActionUsageParameterMembership_Factory

Factory: SysMLv1Tov2::Mappings::Factories::FlowEndParameterMembership_Factory

Factory: SysMLv1Tov2::Mappings::Factories::StringParameterMembership_Factory

Factory: SysMLv1Tov2::Mappings::Factories::DirectedReferenceUsageParameterMembership_Factory

Factory: SysMLv1Tov2::Mappings::Factories::ParameterMembership_Factor

In [122]:
direct_ops = []

for cls in concrete_descendants:
    for operation in registry.get_elements("SysML_v1_to_v2"):
        if (
            operation.parent == cls.key
            and operation.metaclass == "uml:Operation"
        ):
            direct_ops.append({
                "class": cls.qualified_name,
                "operation": operation,
                "operation_name": operation.name,
            })

owned_member_feature = [
    record for record in direct_ops
    if record["operation_name"] == "ownedMemberFeature"
]

print("ownedMemberFeature declarations:", len(owned_member_feature))

for record in owned_member_feature[:5]:
    operation = record["operation"]

    specs = ocl_specifications[
        ocl_specifications["operation_id"] == operation.key
    ]

    print(f"\n{'=' * 80}")
    print("Class:", record["class"])
    print("Operation:", operation.name)

    for _, spec in specs.iterrows():
        print({
            "language": spec["language"],
            "body": spec["body"],
        })

ownedMemberFeature declarations: 112

Class: SysMLv1Tov2::Mappings::Factories::FlowItemFeatureMembership_Factory
Operation: ownedMemberFeature
{'language': 'OCL2.0', 'body': 'result = FlowItem_Factory.create(item)'}

Class: SysMLv1Tov2::Mappings::Factories::MultiplicityMembership_Factory
Operation: ownedMemberFeature
{'language': 'OCL2.0', 'body': "result = if upperValue = 1 then if lowerValue = 0 then Helper.getMultiplicityRangeByName('zeroOrOne') else if lowerValue = 1 then Helper.getMultiplicityRangeByName('exactlyOne') else MultiplicityElement_Factory.create(lowerValue, upperValue) endif endif else if upperValue = -1 then if lowerValue = 0 then Helper.getMultiplicityRangeByName('zeroToMany') else if lowerValue = 1 then Helper.getMultiplicityRangeByName('oneToMany') else MultiplicityElement_Factory.create(lowerValue, upperValue) endif endif else MultiplicityElement_Factory.create(lowerValue, upperValue) endif endif"}

Class: SysMLv1Tov2::Mappings::Factories::FlowUsageFeatureMembersh

In [123]:
non_factory_owned_member_feature = [
    record for record in owned_member_feature
    if "::Mappings::Factories::" not in record["class"]
]

print(
    "Non-factory declarations:",
    len(non_factory_owned_member_feature),
)

for record in non_factory_owned_member_feature[:10]:
    operation = record["operation"]

    specs = ocl_specifications[
        ocl_specifications["operation_id"] == operation.key
    ]

    print(f"\n{'=' * 80}")
    print("Class:", record["class"])

    for _, spec in specs.iterrows():
        print({
            "language": spec["language"],
            "body": spec["body"],
        })

Non-factory declarations: 105

Class: SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInFeatureMembership_Mapping
{'language': 'OCL2.0', 'body': 'result = if from.type.oclIsUndefined() then CommonReferenceUsageInUntyped_Mapping.getMapped(from) else CommonReferenceUsageIn_Mapping.getMapped(from) endif'}

Class: SysMLv1Tov2::Mappings::SysMLv1::Activities::RateMetadataUsageContinuousFeatureMembership_Mapping
{'language': 'OCL2.0', 'body': 'result = RateMetadataUsageContinuousReferenceUsage_Mapping.getMapped(from)'}

Class: SysMLv1Tov2::Mappings::SysMLv1::Activities::RateMetadataUsageDiscreteFeatureMembership_Mapping
{'language': 'OCL2.0', 'body': 'result = RateMetadataUsageDiscreteReferenceUsage_Mapping.getMapped(from)'}

Class: SysMLv1Tov2::Mappings::SysMLv1::Activities::ProbabilityMetadataUsageFeatureMembership_Mapping
{'language': 'OCL2.0', 'body': 'result = ProbabilityMetadataUsageReferenceUsage_Mapping.getMapped(from)'}

Class: SysMLv1Tov2::Mappings::SysMLv1::Requirements:

In [124]:
delegation_records = []

for record in non_factory_owned_member_feature:
    operation = record["operation"]

    specs = ocl_specifications[
        ocl_specifications["operation_id"] == operation.key
    ]

    for _, spec in specs.iterrows():
        body = spec["body"]

        callees = re.findall(
            r"\b([A-Za-z_]\w*)\.getMapped\s*\(",
            body,
        )

        for callee in callees:
            resolved_records, resolution_status = (
                resolve_mapping_class(callee)
            )

            resolved_class = (
                resolved_records[0].qualified_name
                if len(resolved_records) == 1
                else None
            )

            delegation_records.append({
                "caller_class": record["class"],
                "callee_name": callee,
                "resolved_class": resolved_class,
                "resolution_status": resolution_status,
                "body": body,
            })

delegation_table = pd.DataFrame(delegation_records)

print("Delegation calls:", len(delegation_table))
print("Unique callees:", delegation_table["callee_name"].nunique())
print(
    "Unresolved or ambiguous:",
    delegation_table["resolved_class"].isna().sum(),
)

display(
    delegation_table[
        [
            "caller_class",
            "callee_name",
            "resolved_class",
            "resolution_status",
        ]
    ].drop_duplicates()
)

Delegation calls: 100
Unique callees: 99
Unresolved or ambiguous: 2


,caller_class,callee_name,resolved_class,resolution_status
0,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInFeatureMembership_Mapping,CommonReferenceUsageInUntyped_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInUntyped_Mapping,resolved_suffix
1,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInFeatureMembership_Mapping,CommonReferenceUsageIn_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageIn_Mapping,resolved_suffix
2,SysMLv1Tov2::Mappings::SysMLv1::Activities::RateMetadataUsageContinuousFeatureMembership_Mapping,RateMetadataUsageContinuousReferenceUsage_Mapping,SysMLv1Tov2::Mappings::SysMLv1::Activities::RateMetadataUsageContinuousReferenceUsage_Mapping,resolved_suffix
3,SysMLv1Tov2::Mappings::SysMLv1::Activities::RateMetadataUsageDiscreteFeatureMembership_Mapping,RateMetadataUsageDiscreteReferenceUsage_Mapping,SysMLv1Tov2::Mappings::SysMLv1::Activities::RateMetadataUsageDiscreteReferenceUsage_Mapping,resolved_suffix
4,SysMLv1Tov2::Mappings::SysMLv1::Activities::ProbabilityMetadataUsageFeatureMembership_Mapping,ProbabilityMetadataUsageReferenceUsage_Mapping,SysMLv1Tov2::Mappings::SysMLv1::Activities::ProbabilityMetadataUsageReferenceUsage_Mapping,resolved_suffix
...,...,...,...,...
95,SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowGuardSuccessionTargetEndFeatureMembership_Mapping,ObjectFlowGuardSuccessionTargetEndFeature_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowGuardSuccessionTargetEndFeature_Mapping,resolved_suffix
96,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::InitialStateSubactionMembership_Mapping,InitialState_Mapping,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::InitialState_Mapping,resolved_suffix
97,SysMLv1Tov2::Mappings::UML4SysML::UseCases::UseCaseObjectiveMembership_Mapping,UseCaseObjectiveRequirementUsage_Mapping,SysMLv1Tov2::Mappings::UML4SysML::UseCases::UseCaseObjectiveRequirementUsage_Mapping,resolved_suffix
98,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEndMembership_Mapping,OwnedEnd_Mapping,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping,resolved_suffix


In [125]:
unresolved_delegations = delegation_table[
    delegation_table["resolved_class"].isna()
]

display(
    unresolved_delegations[
        ["caller_class", "callee_name", "resolution_status", "body",]
    ].to_string(index=False)
)

'                                                                              caller_class                         callee_name resolution_status                                                                                                                                                                     body\nSysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping   LiteralUnlimitedToInteger_Mapping        unresolved result = if from.upper <> -1 then LiteralUnlimitedToInteger_Mapping.getMapped(from.upperValue) else LiteralUnlimitedToUnbounded_Mapping.getMapped(from.upperValue) endif\nSysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping LiteralUnlimitedToUnbounded_Mapping        unresolved result = if from.upper <> -1 then LiteralUnlimitedToInteger_Mapping.getMapped(from.upperValue) else LiteralUnlimitedToUnbounded_Mapping.getMapped(from.upperValue) endif'

In [126]:
callee_crosswalk = delegation_table.merge(
    mapping_table[
        ["mapping_name", "source_type", "target_type"]
    ],
    left_on="resolved_class",
    right_on="mapping_name",
    how="left",
    validate="many_to_one",
)

display(
    callee_crosswalk[
        [
            "caller_class",
            "callee_name",
            "source_type",
            "target_type",
        ]
    ].drop_duplicates()
)

print("Delegation calls:", len(callee_crosswalk))

print("\nMissing crosswalk entries:")
print(callee_crosswalk["source_type"].isna().sum())

print("\nDistinct source types:")
print(callee_crosswalk["source_type"].nunique())

print("\nDistinct target types:")
print(callee_crosswalk["target_type"].nunique())

display(
    callee_crosswalk
    .groupby(
        ["source_type", "target_type"],
        dropna=False
    )
    .size()
    .rename("call_count")
    .reset_index()
    .sort_values("call_count", ascending=False)
)

,caller_class,callee_name,source_type,target_type
0,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInFeatureMembership_Mapping,CommonReferenceUsageInUntyped_Mapping,UML::CommonStructure::TypedElement,SysML::Systems::DefinitionAndUsage::ReferenceUsage
1,SysMLv1Tov2::Mappings::CommonMappings::CommonReferenceUsageInFeatureMembership_Mapping,CommonReferenceUsageIn_Mapping,UML::CommonStructure::TypedElement,SysML::Systems::DefinitionAndUsage::ReferenceUsage
2,SysMLv1Tov2::Mappings::SysMLv1::Activities::RateMetadataUsageContinuousFeatureMembership_Mapping,RateMetadataUsageContinuousReferenceUsage_Mapping,UML::CommonStructure::Element,SysML::Systems::DefinitionAndUsage::ReferenceUsage
3,SysMLv1Tov2::Mappings::SysMLv1::Activities::RateMetadataUsageDiscreteFeatureMembership_Mapping,RateMetadataUsageDiscreteReferenceUsage_Mapping,UML::CommonStructure::Element,SysML::Systems::DefinitionAndUsage::ReferenceUsage
4,SysMLv1Tov2::Mappings::SysMLv1::Activities::ProbabilityMetadataUsageFeatureMembership_Mapping,ProbabilityMetadataUsageReferenceUsage_Mapping,UML::CommonStructure::Element,SysML::Systems::DefinitionAndUsage::ReferenceUsage
...,...,...,...,...
95,SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowGuardSuccessionTargetEndFeatureMembership_Mapping,ObjectFlowGuardSuccessionTargetEndFeature_Mapping,UML::Activities::ObjectFlow,KerML::Core::Features::Feature
96,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::InitialStateSubactionMembership_Mapping,InitialState_Mapping,UML::StateMachines::Pseudostate,SysML::Systems::Actions::ActionUsage
97,SysMLv1Tov2::Mappings::UML4SysML::UseCases::UseCaseObjectiveMembership_Mapping,UseCaseObjectiveRequirementUsage_Mapping,UML::UseCases::UseCase,SysML::Systems::Requirements::RequirementUsage
98,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEndMembership_Mapping,OwnedEnd_Mapping,UML::Classification::Property,KerML::Core::Features::Feature


Delegation calls: 100

Missing crosswalk entries:
5

Distinct source types:
42

Distinct target types:
26


,source_type,target_type,call_count
71,NaN,NaN,5
37,UML::CommonBehavior::Trigger,SysML::Systems::DefinitionAndUsage::ReferenceUsage,4
31,UML::Classification::Property,KerML::Core::Features::Feature,4
39,UML::CommonStructure::Abstraction,SysML::Systems::DefinitionAndUsage::ReferenceUsage,3
47,UML::CommonStructure::Element,SysML::Systems::DefinitionAndUsage::ReferenceUsage,3
...,...,...,...
30,UML::Classification::ParameterSet,SysML::Systems::DefinitionAndUsage::ReferenceUsage,1
32,UML::Classification::Property,SysML::Systems::DefinitionAndUsage::ReferenceUsage,1
33,UML::CommonBehavior::Trigger,KerML::Kernel::Expressions::InvocationExpression,1
34,UML::CommonBehavior::Trigger,SysML::Systems::Calculations::CalculationUsage,1


In [127]:
missing_crosswalk = callee_crosswalk[callee_crosswalk["source_type"].isna() | callee_crosswalk["target_type"].isna()]
display(missing_crosswalk[["caller_class", "callee_name", "resolved_class", "resolution_status", "source_type", "target_type", "body",]].drop_duplicates())

target_summary = (
    callee_crosswalk
    .groupby("target_type", dropna=False)
    .agg(
        call_count=("callee_name", "size"),
        distinct_mappings=("callee_name", "nunique"),
        distinct_callers=("caller_class", "nunique"),
        source_types=("source_type", "nunique"),
    )
    .reset_index()
    .sort_values(["call_count", "distinct_mappings"], ascending=False,)
)

display(target_summary)

,caller_class,callee_name,resolved_class,resolution_status,source_type,target_type,body
30,SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping,LiteralUnlimitedToInteger_Mapping,None,unresolved,NaN,NaN,result = if from.upper <> -1 then LiteralUnlimitedToInteger_Mapping.getMapped(from.upperValue) else LiteralUnlimitedToUnbounded_Mapping.getMapped(from.upperValue) endif
31,SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping,LiteralUnlimitedToUnbounded_Mapping,None,unresolved,NaN,NaN,result = if from.upper <> -1 then LiteralUnlimitedToInteger_Mapping.getMapped(from.upperValue) else LiteralUnlimitedToUnbounded_Mapping.getMapped(from.upperValue) endif
70,SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeatureMembership_Mapping,ObjectFlowItemFeatureUntyped_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeatureUntyped_Mapping,resolved_suffix,NaN,NaN,result = if from.source.type.oclIsUndefined() then ObjectFlowItemFeatureUntyped_Mapping.getMapped(from.source) else ObjectFlowItemFeature_Mapping.getMapped(from.source) endif
71,SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeatureMembership_Mapping,ObjectFlowItemFeature_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeature_Mapping,resolved_suffix,NaN,NaN,result = if from.source.type.oclIsUndefined() then ObjectFlowItemFeatureUntyped_Mapping.getMapped(from.source) else ObjectFlowItemFeature_Mapping.getMapped(from.source) endif
82,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetEndFeatureMembership_Mapping,AllocationUsageTargetFeature_Mapping,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetFeature_Mapping,resolved_suffix,NaN,NaN,result = AllocationUsageTargetFeature_Mapping.getMapped(from)


,target_type,call_count,distinct_mappings,distinct_callers,source_types
17,SysML::Systems::DefinitionAndUsage::ReferenceUsage,38,38,37,24
0,KerML::Core::Features::Feature,23,23,23,16
26,NaN,5,5,3,0
9,SysML::Systems::Actions::ActionUsage,4,4,4,4
3,KerML::Kernel::Expressions::LiteralInteger,3,3,3,2
1,KerML::Kernel::Expressions::FeatureChainExpression,3,2,3,2
14,SysML::Systems::Connections::BindingConnectorAsUsage,2,2,2,1
16,SysML::Systems::Constraints::ConstraintUsage,2,2,2,2
21,SysML::Systems::Requirements::RequirementUsage,2,2,2,2
2,KerML::Kernel::Expressions::InvocationExpression,1,1,1,1


In [128]:
feature_class = get_class_in_resource(registry, "KerML", "KerML::Core::Features::Feature",)
feature_descendants = get_ancestors(feature_class, target_children_index,)
feature_family = {feature_class.qualified_name, *(element.qualified_name for element in feature_descendants),}
callee_crosswalk["target_is_feature_or_subtype"] = (callee_crosswalk["target_type"].isin(feature_family))
print(callee_crosswalk["target_is_feature_or_subtype"] .value_counts(dropna=False))
display(callee_crosswalk[["callee_name", "target_type", "target_is_feature_or_subtype",]].drop_duplicates())

target_is_feature_or_subtype
True     93
False     7
Name: count, dtype: int64


,callee_name,target_type,target_is_feature_or_subtype
0,CommonReferenceUsageInUntyped_Mapping,SysML::Systems::DefinitionAndUsage::ReferenceUsage,True
1,CommonReferenceUsageIn_Mapping,SysML::Systems::DefinitionAndUsage::ReferenceUsage,True
2,RateMetadataUsageContinuousReferenceUsage_Mapping,SysML::Systems::DefinitionAndUsage::ReferenceUsage,True
3,RateMetadataUsageDiscreteReferenceUsage_Mapping,SysML::Systems::DefinitionAndUsage::ReferenceUsage,True
4,ProbabilityMetadataUsageReferenceUsage_Mapping,SysML::Systems::DefinitionAndUsage::ReferenceUsage,True
...,...,...,...
95,ObjectFlowGuardSuccessionTargetEndFeature_Mapping,KerML::Core::Features::Feature,True
96,InitialState_Mapping,SysML::Systems::Actions::ActionUsage,True
97,UseCaseObjectiveRequirementUsage_Mapping,SysML::Systems::Requirements::RequirementUsage,True
98,OwnedEnd_Mapping,KerML::Core::Features::Feature,True


In [129]:
known_non_feature_targets = callee_crosswalk[callee_crosswalk["target_type"].notna() & ~callee_crosswalk["target_is_feature_or_subtype"]]

print("Known non-Feature targets:")
display(known_non_feature_targets[["callee_name", "source_type", "target_type"]].drop_duplicates())

print("\nMissing crosswalks:")
display(callee_crosswalk[
    callee_crosswalk["source_type"].isna() | callee_crosswalk["target_type"].isna()
    ][["callee_name", "resolved_class", "resolution_status"]].drop_duplicates())

Known non-Feature targets:


,callee_name,source_type,target_type
32,NamedElementMain_Mapping,UML::CommonStructure::NamedElement,KerML::Root::Elements::Element
55,RSFAReferenceUsageFeatureValue_Mapping,UML::Actions::ReadStructuralFeatureAction,KerML::Kernel::FeatureValues::FeatureValue



Missing crosswalks:


,callee_name,resolved_class,resolution_status
30,LiteralUnlimitedToInteger_Mapping,None,unresolved
31,LiteralUnlimitedToUnbounded_Mapping,None,unresolved
70,ObjectFlowItemFeatureUntyped_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeatureUntyped_Mapping,resolved_suffix
71,ObjectFlowItemFeature_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeature_Mapping,resolved_suffix
82,AllocationUsageTargetFeature_Mapping,SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetFeature_Mapping,resolved_suffix


In [130]:
missing_mapping_names = ["ObjectFlowItemFeatureUntyped_Mapping", "ObjectFlowItemFeature_Mapping", "AllocationUsageTargetFeature_Mapping",]

for name in missing_mapping_names:
    rows = mapping_table[mapping_table["mapping_name"] == name]

    print(f"\n{name}")
    print("Crosswalk rows:", len(rows))

    if not rows.empty:
        display(rows)


ObjectFlowItemFeatureUntyped_Mapping
Crosswalk rows: 0

ObjectFlowItemFeature_Mapping
Crosswalk rows: 0

AllocationUsageTargetFeature_Mapping
Crosswalk rows: 0


In [131]:
candidate_names = ["ObjectFlowItemFeatureUntyped_Mapping", "ObjectFlowItemFeature_Mapping", "AllocationUsageTargetFeature_Mapping",]

base_mapping_name = ("SysMLv1Tov2::Mappings::Foundations::Mapping")

for name in candidate_names:
    records, status = resolve_mapping_class(name)

    print(f"\n{'=' * 80}")
    print(name, "|", status)

    if len(records) != 1:
        print("Expected one class; found:", len(records))
        continue

    cls = records[0]
    ancestors = get_ancestors(cls, parents_index)

    formally_inherits_mapping = any(
        ancestor.qualified_name == base_mapping_name
        for ancestor in ancestors
    )

    print("Qualified name:", cls.qualified_name)
    print("Abstract:", cls.attributes.get("isAbstract"))
    print("Formally inherits Mapping:", formally_inherits_mapping)

    print("Direct from/to properties:")
    for element in registry.get_elements("SysML_v1_to_v2"):
        if (
            element.parent == cls.key
            and element.metaclass == "uml:Property"
            and element.name in {"from", "to"}
        ):
            print(
                element.name,
                element.xmi_id,
            )

    print("Ancestors:")
    for ancestor in ancestors:
        print(" ", ancestor.qualified_name)


ObjectFlowItemFeatureUntyped_Mapping | resolved_suffix
Qualified name: SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeatureUntyped_Mapping
Abstract: None
Formally inherits Mapping: False
Direct from/to properties:
from Mappings-UML4SysML-Activities-ObjectFlowItemFeatureUntyped_Mapping-from
to Mappings-UML4SysML-Activities-ObjectFlowItemFeatureUntyped_Mapping-to
Ancestors:
  SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init
  SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToType_Init
  SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToNamespace_Init
  SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init
  SysMLv1Tov2::Mappings::Foundations::Initializer

ObjectFlowItemFeature_Mapping | resolved_suffix
Qualified name: SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeature_Mapping
Abstract: None
Formally inherits Mapping: False
Direct from/to properties:
Ancestors:
  SysMLv1Tov2::Mappings::UML4SysML::Activ

In [132]:
candidate_names = [
    "ObjectFlowItemFeatureUntyped_Mapping",
    "ObjectFlowItemFeature_Mapping",
    "AllocationUsageTargetFeature_Mapping",
]

for name in candidate_names:
    records, status = resolve_mapping_class(name)

    if len(records) != 1:
        continue

    cls = records[0]
    hierarchy = [cls] + get_ancestors(cls, parents_index)

    print(f"\n{'=' * 80}")
    print(cls.qualified_name)

    for property_name in ("from", "to"):
        declarations = []

        for owner in hierarchy:
            for element in registry.get_elements("SysML_v1_to_v2"):
                if (
                    element.parent == owner.key
                    and element.metaclass == "uml:Property"
                    and element.name == property_name
                ):
                    declarations.append({
                        "property": property_name,
                        "declared_by": owner.qualified_name,
                        "property_id": element.xmi_id,
                        "type": [
                            registry.elements[ref.target].qualified_name
                            for ref in registry.get_references(element.key)
                            if ref.attribute == "type"
                            and ref.resolved
                            and ref.target in registry.elements
                        ],
                    })

        print(property_name, declarations)


SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeatureUntyped_Mapping
from [{'property': 'from', 'declared_by': 'SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeatureUntyped_Mapping', 'property_id': 'Mappings-UML4SysML-Activities-ObjectFlowItemFeatureUntyped_Mapping-from', 'type': []}]
to [{'property': 'to', 'declared_by': 'SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeatureUntyped_Mapping', 'property_id': 'Mappings-UML4SysML-Activities-ObjectFlowItemFeatureUntyped_Mapping-to', 'type': []}, {'property': 'to', 'declared_by': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init', 'property_id': 'Mappings-Initializers-KerMLInitializers-ToFeature_Init-to', 'type': []}, {'property': 'to', 'declared_by': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToType_Init', 'property_id': 'Mappings-Initializers-KerMLInitializers-ToType_Init-to', 'type': []}, {'property': 'to', 'declared_by': 'SysMLv1Tov2::Mappings::Initializers::

In [133]:
cls = records[0]  # ObjectFlowItemFeatureUntyped_Mapping
prop = next(
    element
    for element in registry.get_elements("SysML_v1_to_v2")
    if element.parent == cls.key
    and element.metaclass == "uml:Property"
    and element.name == "from"
)

print("Attributes:")
for key, value in prop.attributes.items():
    print(key, "=", value)

print("\nReferences:")
for ref in registry.get_references(prop.key):
    print({
        "element_tag": ref.element_tag,
        "attribute": ref.attribute,
        "value": ref.value,
        "target": ref.target,
        "resolved": ref.resolved,
        "resolution_status": ref.resolution_status,
    })



for property_name in ("from", "to"):
    print(f"\n{property_name} declarations:")

    for owner in hierarchy:
        for element in registry.get_elements("SysML_v1_to_v2"):
            if (
                element.parent == owner.key
                and element.metaclass == "uml:Property"
                and element.name == property_name
            ):
                print("\n", owner.qualified_name)

                for ref in registry.get_references(element.key):
                    if ref.attribute == "redefinedProperty":
                        print({
                            "value": ref.value,
                            "target": ref.target,
                            "resolved": ref.resolved,
                        })

Attributes:
{http://www.omg.org/spec/XMI/20161101}id = Mappings-SysMLv1-Allocations-AllocationUsageTargetFeature_Mapping-from
{http://www.omg.org/spec/XMI/20161101}type = uml:Property
name = from

References:
{'element_tag': 'redefinedProperty', 'attribute': 'idref', 'value': 'Mappings-Foundations-Mapping-from', 'target': ('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-from', 1), 'resolved': True, 'resolution_status': 'resolved_local'}
{'element_tag': 'type', 'attribute': 'href', 'value': 'https://www.omg.org/spec/UML/20161101/UML.xmi#NamedElement', 'target': ('UML_Metamodel', 'NamedElement', 1), 'resolved': True, 'resolution_status': 'resolved_external'}

from declarations:

 SysMLv1Tov2::Mappings::SysMLv1::Allocations::AllocationUsageTargetFeature_Mapping

to declarations:

 SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init

 SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToType_Init

 SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToNamespace_

In [134]:
def get_property_references(prop, registry, element_tag):
    results = []

    for ref in registry.get_references(prop.key):
        if ref.element_tag != element_tag:
            continue

        target = (registry.elements.get(ref.target) if ref.resolved else None)

        results.append({
            "value": ref.value,
            "target": target.qualified_name if target else None,
            "resolved": ref.resolved,
            "resolution_status": ref.resolution_status,
        })

    return results

for name in candidate_names:
    records, status = resolve_mapping_class(name)

    if len(records) != 1:
        continue

    cls = records[0]
    hierarchy = [cls] + get_ancestors(cls, parents_index)

    print(f"\n{'=' * 80}")
    print(cls.qualified_name)

    for property_name in ("from", "to"):
        declarations = []

        for owner in hierarchy:
            for prop in registry.get_elements("SysML_v1_to_v2"):
                if (
                    prop.parent == owner.key
                    and prop.metaclass == "uml:Property"
                    and prop.name == property_name
                ):
                    declarations.append({
                        "declared_by": owner.qualified_name,
                        "property_id": prop.xmi_id,
                        "type": get_property_references(
                            prop, registry, "type"
                        ),
                        "redefines": get_property_references(
                            prop, registry, "redefinedProperty"
                        ),
                    })

        print(f"\n{property_name}:")
        for declaration in declarations:
            print(declaration)


SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeatureUntyped_Mapping

from:
{'declared_by': 'SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeatureUntyped_Mapping', 'property_id': 'Mappings-UML4SysML-Activities-ObjectFlowItemFeatureUntyped_Mapping-from', 'type': [{'value': 'https://www.omg.org/spec/UML/20161101/UML.xmi#ObjectNode', 'target': 'UML::Activities::ObjectNode', 'resolved': True, 'resolution_status': 'resolved_external'}], 'redefines': [{'value': 'Mappings-Foundations-Mapping-from', 'target': 'SysMLv1Tov2::Mappings::Foundations::Mapping::from', 'resolved': True, 'resolution_status': 'resolved_local'}]}

to:
{'declared_by': 'SysMLv1Tov2::Mappings::UML4SysML::Activities::ObjectFlowItemFeatureUntyped_Mapping', 'property_id': 'Mappings-UML4SysML-Activities-ObjectFlowItemFeatureUntyped_Mapping-to', 'type': [{'value': 'https://www.omg.org/spec/KerML/20250201/KerML.xmi#Kernel-Interactions-PayloadFeature', 'target': 'KerML::Kernel::Interactions::PayloadFea

### Resolution:

In [135]:
terms = ("LiteralUnlimited", "UnlimitedNatural", "Unbounded", "upperValue",)

for resource in ("SysML_v1_to_v2", "UML_Metamodel", "UML_PrimitiveTypes", "KerML", "SysML_v2",):
    print(f"\n{'=' * 70}\n{resource}")

    matches = []

    for element in registry.get_elements(resource):
        name = element.name or ""
        attributes = str(element.attributes)

        if any(term in name or term in attributes for term in terms):
            matches.append(element)

    for element in matches:
        print(element.metaclass, element.qualified_name, element.xmi_id,)


SysML_v1_to_v2
uml:LiteralUnlimitedNatural SysMLv1Tov2::Mappings::Foundations::Initializer::inputs Mappings-Foundations-Initializer-inputs-upperValue
uml:LiteralUnlimitedNatural SysMLv1Tov2::Mappings::Foundations::Initializer::to Mappings-Foundations-Initializer-to-upperValue
uml:LiteralUnlimitedNatural SysMLv1Tov2::Mappings::Foundations::Mapping::from Mappings-Foundations-Mapping-from-upperValue
uml:LiteralUnlimitedNatural SysMLv1Tov2::Mappings::Foundations::Mapping::filter::result Mappings-Foundations-Mapping-filter_Element-result-upperValue
uml:LiteralUnlimitedNatural SysMLv1Tov2::Mappings::Foundations::Mapping::filter::src Mappings-Foundations-Mapping-filter_Element-src-upperValue
uml:LiteralUnlimitedNatural SysMLv1Tov2::Mappings::Foundations::Mapping::getMappedColl::fromColl Mappings-Foundations-Mapping-getMappedColl_Element-fromColl-upperValue
uml:LiteralUnlimitedNatural SysMLv1Tov2::Mappings::Foundations::Mapping::getMappedColl::result Mappings-Foundations-Mapping-getMappedColl

In [136]:
print(ocl_mapping_calls.columns.tolist())
print(ocl_mapping_calls.head(2).to_string())

['operation_id', 'operation', 'constraint_id', 'constraint_name', 'language', 'specification_category', 'callee', 'method', 'arguments', 'call_expression', 'reference_type', 'candidate_classes', 'resolution_status', 'target_class', 'target_class_id']
                                                                                            operation_id                                                                                        operation                                                                                                        constraint_id constraint_name language specification_category                      callee     method             arguments                                             call_expression           reference_type                                                                 candidate_classes resolution_status                                                                    target_class                                                 target_c

In [137]:
literal_names = {
    "LiteralUnlimitedToInteger_Mapping",
    "LiteralUnlimitedToUnbounded_Mapping",
}

literal_calls = ocl_mapping_calls.loc[
    ocl_mapping_calls["callee"].isin(literal_names), 
    ["operation", "constraint_name", "language", "callee", "method", "arguments", 
     "call_expression", "candidate_classes", "target_class", "resolution_status", "reference_type",],
     ].copy()

print(literal_calls.to_string(index=False))

                                                                                                     operation constraint_name language                              callee    method       arguments                                                call_expression candidate_classes target_class resolution_status          reference_type
SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping::ownedMemberFeature   bodyCondition   OCL2.0   LiteralUnlimitedToInteger_Mapping getMapped from.upperValue   LiteralUnlimitedToInteger_Mapping.getMapped(from.upperValue)                []         None        unresolved mapping_class_reference
SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping::ownedMemberFeature   bodyCondition   OCL2.0 LiteralUnlimitedToUnbounded_Mapping getMapped from.upperValue LiteralUnlimitedToUnbounded_Mapping.getMapped(from.upperValue)                []         None        unresolved mapping_class_referenc

In [138]:
operation_names = literal_calls["operation"].unique()

literal_specs = ocl_specifications.loc[ocl_specifications["operation"].isin(operation_names)].copy()

print(
    literal_specs[
        ["operation", "constraint_name", "language", "body"]
    ].to_string(index=False)
)

                                                                                                     operation constraint_name language                                                                                                                                                                     body
SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping::ownedMemberFeature   bodyCondition   OCL2.0 result = if from.upper <> -1 then LiteralUnlimitedToInteger_Mapping.getMapped(from.upperValue) else LiteralUnlimitedToUnbounded_Mapping.getMapped(from.upperValue) endif


In [139]:
source_names = {"MultiplicityElement", "LiteralUnlimitedNatural", "LiteralInteger", "ValueSpecification",}

for element in registry.get_elements("UML_Metamodel"):
    if element.name in source_names:
        print(
            element.metaclass,
            element.qualified_name,
            element.xmi_id,
        )

uml:Class UML::Values::LiteralInteger LiteralInteger
uml:Class UML::Values::LiteralUnlimitedNatural LiteralUnlimitedNatural
uml:Class UML::Values::ValueSpecification ValueSpecification
uml:Class UML::CommonStructure::MultiplicityElement MultiplicityElement


In [140]:
multiplicity = get_class_in_resource(registry, "UML_Metamodel", "UML::CommonStructure::MultiplicityElement",)

for prop in registry.get_elements("UML_Metamodel"):
    if (prop.parent == multiplicity.key and prop.metaclass == "uml:Property" and prop.name in {"upper", "upperValue"}):
        print("\nProperty:", prop.name)
        print("Attributes:", prop.attributes)

        for ref in registry.get_references(prop.key):
            print({
                "element_tag": ref.element_tag,
                "value": ref.value,
                "target": (registry.elements[ref.target].qualified_name if ref.resolved and ref.target in registry.elements else None),
                "resolved": ref.resolved,
            })


Property: upper
Attributes: {'{http://www.omg.org/spec/XMI/20131001}type': 'uml:Property', '{http://www.omg.org/spec/XMI/20131001}id': 'MultiplicityElement-upper', 'name': 'upper', 'isDerived': 'true'}
{'element_tag': 'type', 'value': 'http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi#UnlimitedNatural', 'target': 'PrimitiveTypes::UnlimitedNatural', 'resolved': True}

Property: upperValue
Attributes: {'{http://www.omg.org/spec/XMI/20131001}type': 'uml:Property', '{http://www.omg.org/spec/XMI/20131001}id': 'MultiplicityElement-upperValue', 'name': 'upperValue', 'aggregation': 'composite', 'type': 'ValueSpecification', 'association': 'A_upperValue_owningUpper'}
{'element_tag': 'subsettedProperty', 'value': 'Element-ownedElement', 'target': 'UML::CommonStructure::Element::ownedElement', 'resolved': True}


In [141]:
target_names = {"LiteralInteger", "LiteralInfinity", "LiteralUnlimitedNatural", "Unbounded", "Multiplicity", "MultiplicityRange",}

for resource in ("KerML", "SysML_v2"):
    print(f"\n{'=' * 70}\n{resource}")

    for element in registry.get_elements(resource):
        if element.name in target_names:
            print(element.metaclass, element.qualified_name, element.xmi_id,)


KerML
uml:Class KerML::Kernel::Expressions::LiteralInteger Kernel-Expressions-LiteralInteger
uml:Class KerML::Kernel::Expressions::LiteralInfinity Kernel-Expressions-LiteralInfinity
uml:Class KerML::Kernel::Multiplicities::MultiplicityRange Kernel-Multiplicities-MultiplicityRange
uml:Class KerML::Core::Types::Multiplicity Core-Types-Multiplicity

SysML_v2


In [142]:
mapping_name = ("SysMLv1Tov2::Mappings::UML4SysML::Classification::" "UpperBoundValueFeatureMembership_Mapping")

records, status = resolve_mapping_class(mapping_name)

print("Resolution:", status)

if len(records) == 1:
    cls = records[0]

    print("Class:", cls.qualified_name)

    for prop in registry.get_elements("SysML_v1_to_v2"):
        if (
            prop.parent == cls.key
            and prop.metaclass == "uml:Property"
            and prop.name in {"from", "to"}
        ):
            print(
                prop.name,
                get_property_references(prop, registry, "type"),
            )

Resolution: resolved_exact
Class: SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping
from [{'value': 'https://www.omg.org/spec/UML/20161101/UML.xmi#MultiplicityElement', 'target': 'UML::CommonStructure::MultiplicityElement', 'resolved': True, 'resolution_status': 'resolved_external'}]


In [143]:
def inspect_class_structure(registry, resource, qualified_name):
    cls = get_class_in_resource(registry, resource, qualified_name)

    print(f"\n{'=' * 80}")
    print(cls.qualified_name)

    print("\nGeneralizations:")
    for ref in registry.get_references(cls.key):
        if ref.element_tag == "general":
            target = (
                registry.elements.get(ref.target)
                if ref.resolved else None
            )
            print(target.qualified_name if target else ref.value)

    print("\nDirect properties:")
    for element in registry.get_elements(resource):
        if (
            element.parent == cls.key
            and element.metaclass == "uml:Property"
        ):
            print({
                "name": element.name,
                "id": element.xmi_id,
                "attributes": element.attributes,
                "type": get_property_references(
                    element, registry, "type"
                ),
            })


inspect_class_structure(
    registry, "UML_Metamodel",
    "UML::Values::LiteralUnlimitedNatural",
)

inspect_class_structure(
    registry, "UML_Metamodel",
    "UML::Values::LiteralInteger",
)

inspect_class_structure(
    registry, "KerML",
    "KerML::Kernel::Expressions::LiteralInteger",
)

inspect_class_structure(
    registry, "KerML",
    "KerML::Kernel::Expressions::LiteralInfinity",
)


UML::Values::LiteralUnlimitedNatural

Generalizations:

Direct properties:
{'name': 'value', 'id': 'LiteralUnlimitedNatural-value', 'attributes': {'{http://www.omg.org/spec/XMI/20131001}type': 'uml:Property', '{http://www.omg.org/spec/XMI/20131001}id': 'LiteralUnlimitedNatural-value', 'name': 'value'}, 'type': [{'value': 'http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi#UnlimitedNatural', 'target': 'PrimitiveTypes::UnlimitedNatural', 'resolved': True, 'resolution_status': 'resolved_external'}]}

UML::Values::LiteralInteger

Generalizations:

Direct properties:
{'name': 'value', 'id': 'LiteralInteger-value', 'attributes': {'{http://www.omg.org/spec/XMI/20131001}type': 'uml:Property', '{http://www.omg.org/spec/XMI/20131001}id': 'LiteralInteger-value', 'name': 'value'}, 'type': [{'value': 'http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi#Integer', 'target': 'PrimitiveTypes::Integer', 'resolved': True, 'resolution_status': 'resolved_external'}]}

KerML::Kernel::Expressions::

In [144]:
mapping_name = (
    "SysMLv1Tov2::Mappings::UML4SysML::Classification::"
    "UpperBoundValueFeatureMembership_Mapping"
)

records, status = resolve_mapping_class(mapping_name)
assert len(records) == 1

cls = records[0]
hierarchy = [cls] + get_ancestors(cls, parents_index)

print("Class:", cls.qualified_name)
print("\nTarget declarations:")

for owner in hierarchy:
    for prop in registry.get_elements("SysML_v1_to_v2"):
        if (
            prop.parent == owner.key
            and prop.metaclass == "uml:Property"
            and prop.name == "to"
        ):
            print({
                "declared_by": owner.qualified_name,
                "type": get_property_references(
                    prop, registry, "type"
                ),
                "redefines": get_property_references(
                    prop, registry, "redefinedProperty"
                ),
            })

Class: SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping

Target declarations:
{'declared_by': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init', 'type': [{'value': 'https://www.omg.org/spec/KerML/20250201/KerML.xmi#Core-Types-FeatureMembership', 'target': 'KerML::Core::Types::FeatureMembership', 'resolved': True, 'resolution_status': 'resolved_external'}], 'redefines': [{'value': 'Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-to', 'target': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::to', 'resolved': True, 'resolution_status': 'resolved_local'}, {'value': 'Mappings-Initializers-KerMLInitializers-ToOwningMembership_Init-to', 'target': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToOwningMembership_Init::to', 'resolved': True, 'resolution_status': 'resolved_local'}]}
{'declared_by': 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturin

In [145]:
for qualified_name in ["KerML::Kernel::Expressions::LiteralInteger", "KerML::Kernel::Expressions::LiteralInfinity",]:
    cls= get_class_in_resource(registry, "KerML", qualified_name)

    ancestors = get_ancestors(cls, target_parents_index)

    print(f"\n{qualified_name}")
    print("Ancestors:")

    for ancestor in ancestors:
        print(" ", ancestor.qualified_name)

    print("Inherited properties:")

    hierarchy = [cls] + ancestors

    for owner in hierarchy:
        for prop in registry.get_elements("KerML"):
            if (prop.parent == owner.key and prop.metaclass == "uml:Property"):
                print(owner.qualified_name, prop.name, get_property_references(prop, registry, "type"),)


literal_infinity = get_class_in_resource(registry, "KerML", "KerML::Kernel::Expressions::LiteralInfinity",)
for resource in ("KerML", "SysML_v2", "SysML_v1_to_v2"):
    print(f"\n{'=' * 70}\n{resource}")
    for ref in registry.get_references():
        if (ref.resolved and ref.target == literal_infinity.key and ref.element_tag == "type"):
            source = registry.elements.get(ref.source)
            if source is not None:
                print(source.qualified_name, source.metaclass, source.name,)


KerML::Kernel::Expressions::LiteralInteger
Ancestors:
  KerML::Kernel::Expressions::LiteralExpression
  KerML::Kernel::Functions::Expression
  KerML::Kernel::Behaviors::Step
  KerML::Core::Features::Feature
  KerML::Core::Types::Type
  KerML::Root::Namespaces::Namespace
  KerML::Root::Elements::Element
Inherited properties:
KerML::Kernel::Expressions::LiteralInteger value [{'value': 'https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi#Integer', 'target': 'PrimitiveTypes::Integer', 'resolved': True, 'resolution_status': 'resolved_external'}]
KerML::Kernel::Functions::Expression function [{'value': 'Kernel-Functions-Function', 'target': 'KerML::Kernel::Functions::Function', 'resolved': True, 'resolution_status': 'resolved_local'}]
KerML::Kernel::Functions::Expression result [{'value': 'Core-Features-Feature', 'target': 'KerML::Core::Features::Feature', 'resolved': True, 'resolution_status': 'resolved_local'}]
KerML::Kernel::Functions::Expression isModelLevelEvaluable [{'value': 'ht

In [146]:
def resolve_reference_target(ref):
    if not ref.resolved or ref.target is None:
        return None

    return registry.elements.get(ref.target)


def reference_target_name(ref):
    target = resolve_reference_target(ref)
    return target.qualified_name if target is not None else None

search_terms = ("LiteralUnlimited", "LiteralInteger", "LiteralInfinity",)

literal_classes = [
    e for e in registry.get_elements("SysML_v1_to_v2")
    if e.metaclass == "uml:Class"
    and any(term.lower() in (e.name or "").lower()
            for term in search_terms)
]

for cls in literal_classes:
    print("\n" + "=" * 90)
    print(cls.qualified_name)
    print("XMI ID:", cls.xmi_id)
    print("Abstract:", cls.attributes.get("isAbstract"))

    hierarchy = [cls] + get_ancestors(cls, parents_index)

    for property_name in ("from", "to"):
        print(f"\n{property_name} declarations:")

        for owner in hierarchy:
            for prop in registry.get_elements("SysML_v1_to_v2"):
                if (
                    prop.parent == owner.key
                    and prop.metaclass == "uml:Property"
                    and prop.name == property_name
                ):
                    refs = registry.get_references(source=prop.key)
                    print({
                        "declared_by": owner.qualified_name,
                        "property_id": prop.xmi_id,
                        "redefines": [
                            reference_target_name(ref)
                            for ref in refs
                            if ref.element_tag == "redefinedProperty"
                        ],

                        "type": [
                            reference_target_name(ref)
                            for ref in refs
                            if ref.element_tag == "type"
                        ],
                    })


candidate_names = {cls.qualified_name for cls in literal_classes}

for cls in literal_classes:
    print("\n" + "=" * 90)
    print(cls.qualified_name)

    hierarchy = [cls] + get_ancestors(cls, parents_index)

    for owner in hierarchy:
        operations = [
            e for e in registry.get_elements("SysML_v1_to_v2")
            if e.parent == owner.key
            and e.metaclass == "uml:Operation"
        ]

        for op in operations:
            specs = ocl_specifications[
                ocl_specifications["operation_id"].apply(
                    lambda x: x[1] == op.xmi_id
                )
            ]

            print("\nOperation:", owner.qualified_name + "::" + op.name)
            print("Abstract:", op.attributes.get("isAbstract"))

            if specs.empty:
                print("No extracted specification")
            else:
                for _, spec in specs.iterrows():
                    print("Language:", spec["language"])
                    print("Body:", spec["body"])


SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralUnlimitedInteger_Mapping
XMI ID: Mappings-UML4SysML-Values-LiteralUnlimitedInteger_Mapping
Abstract: None

from declarations:
{'declared_by': 'SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralUnlimitedInteger_Mapping', 'property_id': 'Mappings-UML4SysML-Values-LiteralUnlimitedInteger_Mapping-from', 'redefines': ['SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationCommon_Mapping::from'], 'type': ['UML::Values::LiteralUnlimitedNatural']}
{'declared_by': 'SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationCommon_Mapping', 'property_id': 'Mappings-UML4SysML-Values-LiteralSpecificationCommon_Mapping-from', 'redefines': ['SysMLv1Tov2::Mappings::UML4SysML::Values::ValueSpecification_Mapping::from'], 'type': ['UML::Values::LiteralSpecification']}
{'declared_by': 'SysMLv1Tov2::Mappings::UML4SysML::Values::ValueSpecification_Mapping', 'property_id': 'Mappings-UML4SysML-Values-ValueSpecification_Mapping-from', 'redefines': ['

In [147]:

ref = next(
    ref for ref in refs
    if ref.element_tag == "redefinedProperty"
)

print(ref.target)
print(type(ref.target))
print(registry.elements.get(ref.target))



('SysML_v1_to_v2', 'Mappings-Initializers-KerMLInitializers-ToElement_Init-to', 1)
<class 'tuple'>
ElementRecord(resource='SysML_v1_to_v2', xmi_id='Mappings-Initializers-KerMLInitializers-ToElement_Init-to', tag='ownedAttribute', metaclass='uml:Property', name='to', attributes={'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Initializers-KerMLInitializers-ToElement_Init-to', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Property', 'name': 'to'}, parent=('SysML_v1_to_v2', 'Mappings-Initializers-KerMLInitializers-ToElement_Init', 1), children=[('SysML_v1_to_v2', 'Mappings-Initializers-KerMLInitializers-ToElement_Init-to-lowerValue', 1), ('SysML_v1_to_v2', 'Mappings-Initializers-KerMLInitializers-ToElement_Init-to-upperValue', 1)], registry_key=('SysML_v1_to_v2', 'Mappings-Initializers-KerMLInitializers-ToElement_Init-to', 1), element_category=None, qualified_name='SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init::to', namespace='SysMLv1Tov2::Mappings::Ini

#### Literal Mapping Root Cause Diff:

Finite conversion
- The actual class declares from as UML LiteralUnlimitedNatural.
- Its to property is KerML LiteralInteger.
- Its value() operation returns from.value.
- The target's value property is an integer.

Unlimited conversion
- LiteralUnlimitedUnbounded_Mapping inherits the source declaration from LiteralUnlimitedInteger_Mapping.
- It redefines the target property to KerML LiteralInfinity.
- Its filter() operation explicitly requires from.value = -1.
- LiteralInfinity has no direct value property, so the inherited value() operation is not needed to populate a target value property.

OCL call identity: LiteralUnlimitedToInteger_Mapping and LiteralUnlimitedToUnbounded_Mapping.
Semantically corresponding class identities: LiteralUnlimitedInteger_Mapping and LiteralUnlimitedUnbounded_Mapping.

In [148]:
literal_aliases = {
    "LiteralUnlimitedToInteger_Mapping":
        "SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralUnlimitedInteger_Mapping",

    "LiteralUnlimitedToUnbounded_Mapping":
        "SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralUnlimitedUnbounded_Mapping",
}

literal_calls = ocl_mapping_calls.loc[ocl_mapping_calls["callee"].isin(literal_aliases)].copy()
literal_calls[["operation", "callee", "method", "arguments", "call_expression", "resolution_status",]]

for _, row in literal_calls.iterrows():
    print("\nCaller:", row["operation"])
    print("Call:", row["call_expression"])
    print("Alias target:", literal_aliases[row["callee"]])

literal_calls["semantic_target_class"] = (literal_calls["callee"].map(literal_aliases))
literal_calls["semantic_resolution_status"] = ("resolved_semantic_alias")


Caller: SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping::ownedMemberFeature
Call: LiteralUnlimitedToInteger_Mapping.getMapped(from.upperValue)
Alias target: SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralUnlimitedInteger_Mapping

Caller: SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping::ownedMemberFeature
Call: LiteralUnlimitedToUnbounded_Mapping.getMapped(from.upperValue)
Alias target: SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralUnlimitedUnbounded_Mapping


In [149]:
print("dependency_edges:")
print(dependency_edges.columns.tolist())
print(dependency_edges.head())

print("\nocl_mapping_calls:")
print(ocl_mapping_calls.columns.tolist())
print(ocl_mapping_calls[["callee", "call_expression"]].head())

dependency_edges:
['call_id', 'operation_id', 'operation', 'constraint_id', 'constraint_name', 'language', 'specification_category', 'callee', 'method', 'arguments', 'call_expression', 'reference_type', 'candidate_classes', 'resolution_status', 'target_class', 'target_class_id', 'source_class', 'dependency_type', 'dependency_status', 'source_owner_type']
      call_id  \
0  CALL_00000   
1  CALL_00001   
2  CALL_00002   
3  CALL_00003   
4  CALL_00004   

                                                                                                               operation_id  \
0                                                   (SysML_v1_to_v2, Mappings-Foundations-Mapping-getMappedColl_Element, 1)   
1                     (SysML_v1_to_v2, Mappings-Factories-InformationFlowReferenceSubsetting_Factory-referencedFeature_, 1)   
2  (SysML_v1_to_v2, Mappings-CommonMappings-CommonReturnParameterReferenceUsageMembership_Mapping-ownedMemberParameter_, 1)   
3  (SysML_v1_to_v2, Mappings-Com

In [150]:
dependency_edges = dependency_edges.copy()

# Preserve the semantic alias separately from the original OCL callee.
dependency_edges["semantic_target_class"] = (
    dependency_edges["callee"].map(literal_aliases)
)

# Preserve the original resolution status unless an alias resolves the call.
alias_mask = dependency_edges["semantic_target_class"].notna()

dependency_edges["effective_target_class"] = (
    dependency_edges["target_class"].where(
        ~alias_mask,
        dependency_edges["semantic_target_class"],
    )
)

dependency_edges["effective_resolution_status"] = (
    dependency_edges["resolution_status"].where(
        ~alias_mask,
        "resolved_semantic_alias",
    )
)

In [151]:
literal_dependency_check = dependency_edges.loc[
    dependency_edges["semantic_target_class"].notna(),
    [
        "call_id",
        "source_class",
        "callee",
        "call_expression",
        "target_class",
        "semantic_target_class",
        "effective_target_class",  # Added
        "resolution_status",
        "effective_resolution_status",
    ],
]

print(literal_dependency_check.to_string(index=False))

assert len(literal_dependency_check) == 2
assert literal_dependency_check["call_id"].is_unique
assert literal_dependency_check["effective_target_class"].notna().all()

   call_id                                                                               source_class                              callee                                                call_expression target_class                                                       semantic_target_class                                                      effective_target_class resolution_status effective_resolution_status
CALL_00290 SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping   LiteralUnlimitedToInteger_Mapping   LiteralUnlimitedToInteger_Mapping.getMapped(from.upperValue)         None   SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralUnlimitedInteger_Mapping   SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralUnlimitedInteger_Mapping        unresolved     resolved_semantic_alias
CALL_00291 SysMLv1Tov2::Mappings::UML4SysML::Classification::UpperBoundValueFeatureMembership_Mapping LiteralUnlimitedToUnbounded_Mapping LiteralUnlimitedToUnbounded_Mapping.ge

In [152]:
print(
    dependency_edges.groupby(
        "effective_resolution_status",
        dropna=False,
    ).agg(
        calls=("call_id", "size"),
        unique_callers=("source_class", "nunique"),
        unique_targets=("effective_target_class", "nunique"),
    ).sort_values("calls", ascending=False)
)

print("\nUnresolved calls:")
print(
    dependency_edges.loc[
        dependency_edges["effective_resolution_status"].eq("unresolved"),
        ["call_id", "source_class", "callee", "call_expression"],
    ].to_string(index=False)
)

print("\nValidation:")
print({
    "total_calls": len(dependency_edges),
    "unique_call_ids": dependency_edges["call_id"].nunique(),
    "resolved_calls": dependency_edges["effective_target_class"].notna().sum(),
    "unresolved_calls": dependency_edges["effective_resolution_status"].eq("unresolved").sum(),
})

                             calls  unique_callers  unique_targets
effective_resolution_status                                       
resolved_suffix                830             523             576
unresolved                      22              20               0
resolved_semantic_alias          2               1               2
not_applicable                   1               1               0

Unresolved calls:
   call_id                                                                                   source_class                                         callee                                                   call_expression
CALL_00022                                                                  SysMLv1Tov2::Mappings::Helper                 InitialStateMembership_Mapping                       InitialStateMembership_Mapping.getMapped(e)
CALL_00214                            SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping                    CommonElementImp

In [153]:
unresolved_summary = (
    dependency_edges.loc[
        dependency_edges["effective_resolution_status"].eq("unresolved")
    ]
    .groupby("callee", dropna=False)
    .agg(
        call_count=("call_id", "size"),
        callers=("source_class", lambda x: sorted(set(x))),
        call_ids=("call_id", list),
    )
    .sort_values("call_count", ascending=False)
    .reset_index()
)

print(unresolved_summary.to_string(index=False))

unresolved_names = unresolved_summary["callee"].dropna().tolist()

unresolved_class_check = []

for callee in unresolved_names:
    records, status = resolve_mapping_class(callee)

    unresolved_class_check.append({
        "callee": callee,
        "resolution_status": status,
        "candidate_count": len(records),
        "candidate_classes": [
            record.qualified_name for record in records
        ],
    })

unresolved_class_check = pd.DataFrame(unresolved_class_check)

print(unresolved_class_check.to_string(index=False))

                                        callee  call_count                                                                                                                                                                                                                callers                             call_ids
                InitialStateMembership_Mapping           3                                             [SysMLv1Tov2::Mappings::Helper, SysMLv1Tov2::Mappings::UML4SysML::StateMachines::Region_Mapping, SysMLv1Tov2::Mappings::UML4SysML::StateMachines::StateDefinition_Mapping] [CALL_00022, CALL_00239, CALL_00249]
            LiteralSpecificationTyping_Mapping           3 [SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationCommon_Mapping, SysMLv1Tov2::Mappings::UML4SysML::Values::TimeExpression_Mapping, SysMLv1Tov2::Mappings::UML4SysML::Values::ValueSpecification_Mapping] [CALL_00330, CALL_00335, CALL_00338]
                    PropertyMembership_Mapping           3         

In [154]:
import difflib
import pandas as pd

all_transformation_classes = [e for e in registry.get_elements("SysML_v1_to_v2") if e.metaclass == "uml:Class"]
declared_class_names = {e.name: e.qualified_name for e in all_transformation_classes if e.name}
unresolved_candidates = []
for callee in unresolved_names:
    matches = difflib.get_close_matches(callee, declared_class_names.keys(), n=10, cutoff=0.45,)

    unresolved_candidates.append({
        "callee": callee,
        "similar_names": matches,
        "qualified_candidates": [declared_class_names[name] for name in matches],
    })

unresolved_candidate_table = pd.DataFrame(unresolved_candidates)
print(unresolved_candidate_table.to_string(index=False))


for callee in unresolved_names:
    matches = [e for e in registry.get_elements("SysML_v1_to_v2") if e.name == callee]
    print(f"\n{callee}: {len(matches)} exact-name matches")
    for e in matches:
        print(e.metaclass, e.qualified_name, e.xmi_id,)

                                        callee                                                                                                                                                                                                                                                                                                                                                                                                                                   similar_names                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

#### Difflib Candidate Mapping:

In [155]:
from difflib import SequenceMatcher

candidate_rows = []

for _, row in unresolved_candidate_table.iterrows():
    callee = row["callee"]

    for candidate in row["similar_names"]:
        qualified_name = declared_class_names[candidate]

        candidate_rows.append({
            "unresolved_callee": callee,
            "candidate_name": candidate,
            "candidate_class": qualified_name,
            "name_similarity": SequenceMatcher(
                None, callee, candidate
            ).ratio(),
        })

candidate_crosswalk = pd.DataFrame(candidate_rows)

candidate_crosswalk = candidate_crosswalk.merge(
    mapping_table[
        [
            "mapping_name",
            "source_type",
            "target_type",
            "operation_count",
            "operation_categories",
        ]
    ],
    left_on="candidate_class",
    right_on="mapping_name",
    how="left",
    validate="many_to_one",
    indicator=True,
)

candidate_crosswalk = candidate_crosswalk.sort_values(
    ["unresolved_callee", "name_similarity"],
    ascending=[True, False],
)

print(
    candidate_crosswalk[
        [
            "unresolved_callee",
            "candidate_name",
            "name_similarity",
            "source_type",
            "target_type",
            "operation_count",
            "_merge",
        ]
    ].to_string(index=False)
)

                             unresolved_callee                                            candidate_name  name_similarity                                   source_type                                            target_type  operation_count    _merge
            AssociationToFeatureTyping_Mapping                           AllocationFeatureTyping_Mapping         0.892308            UML::CommonStructure::NamedElement                   KerML::Core::Features::FeatureTyping              1.0      both
            AssociationToFeatureTyping_Mapping                            ReceptionFeatureTyping_Mapping         0.812500             UML::SimpleClassifiers::Reception                   KerML::Core::Features::FeatureTyping              2.0      both
            AssociationToFeatureTyping_Mapping             AssociationMetadataUsageFeatureTyping_Mapping         0.810127       UML::StructuredClassifiers::Association                   KerML::Core::Features::FeatureTyping              1.0      both


In [156]:
unresolved_context = dependency_edges.loc[
    dependency_edges["effective_resolution_status"].eq("unresolved"),
    [
        "call_id",
        "source_class",
        "operation",
        "callee",
        "arguments",
        "call_expression",
        "constraint_id",
    ],
].merge(
    ocl_specifications[
        ["constraint_id", "body"]
    ],
    on="constraint_id",
    how="left",
    validate="many_to_one",
)

for callee, group in unresolved_context.groupby("callee"):
    print(f"\n{'=' * 90}")
    print(callee)

    for _, row in group.iterrows():
        print(f"\nCall: {row['call_id']}")
        print(f"Caller: {row['operation']}")
        print(f"Expression: {row['call_expression']}")
        print(f"Body: {row['body']}")


AssociationToFeatureTyping_Mapping

Call: CALL_00395
Caller: SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsage_Mapping::ownedRelationship
Expression: AssociationToFeatureTyping_Mapping.getMapped(from)
Body: result = Set{AssociationToFeatureTyping_Mapping.getMapped(from), AssociationMetadataUsageFeatureMembership_Mapping.getMapped(from)}

Call: CALL_00726
Caller: SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefined_Mapping::ownedRelationship
Expression: AssociationToFeatureTyping_Mapping.getMapped(from)
Body: result = let typing: KerML::FeatureTyping = AssociationToFeatureTyping_Mapping.getMapped(from) in let subsetting: Set(KerML::Subsetting) = from.subsettedProperty ->collect(p | PropertySubsetting_Mapping.getMapped(from, p))->asSet() in let subsettingMultiplicityTyping: Set(KerML::Relationship) = subsetting ->union(Set{AttributeRedefinedRedefinition_Mapping.getMapped(from)})->union( if typing.oclIsUndefined() then Set{MultiplicityMe

In [157]:
candidate_names = [
    "InitialStateSubactionMembership_Mapping",
    "LiteralSpecificationFeatureTyping_Mapping",
    "ElementImport_Mapping",
    "DefaultValue_Mapping",
    "StereotypeMetadataDefinition_Mapping",
    "ChangeTriggerConstraintUsage_Mapping",
    "TimeTriggerCalculationUsage_Mapping",
    "PropertyCommon_Mapping",
]

candidate_classes = [
    e.qualified_name
    for e in all_transformation_classes
    if e.name in candidate_names
]

candidate_definitions = mapping_table.loc[
    mapping_table["mapping_name"].isin(candidate_classes),
    [
        "mapping_name",
        "source_type",
        "target_type",
        "operation_count",
        "operation_categories",
        "ancestors",
    ],
].copy()

print(candidate_definitions.to_string(index=False))

candidate_operation_specs = mapping_operation_specifications.loc[
    mapping_operation_specifications["mapping_name"].isin(candidate_classes),
    [
        "mapping_name",
        "operation_name",
        "declared_by",
        "is_direct",
        "language",
        "body",
    ],
].copy()

print(candidate_operation_specs.to_string(index=False))

ambiguous_callers = [
    "SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsage_Mapping",
    "SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefined_Mapping",
    "SysMLv1Tov2::Mappings::UML4SysML::Actions::Actions::ValuePin_Mapping",
    "SysMLv1Tov2::Mappings::UML4SysML::Actions::ObjectActions::COAPin_Mapping",
    "SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertySubsetting_Mapping",
    "SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping",
]

print(
    mapping_table.loc[
        mapping_table["mapping_name"].isin(ambiguous_callers),
        [
            "mapping_name",
            "source_type",
            "target_type",
            "ancestors",
        ],
    ].to_string(index=False)
)

                                                                            mapping_name                         source_type                                      target_type  operation_count                                       operation_categories                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [158]:
candidate_short_names = {
    "ChangeTriggerConstraintUsage_Mapping",
    "TimeTriggerCalculationUsage_Mapping",
    "ElementImport_Mapping",
    "InitialStateSubactionMembership_Mapping",
    "LiteralSpecificationFeatureTyping_Mapping",
    "StereotypeMetadataDefinition_Mapping",
    "DefaultValue_Mapping",
    "PropertyCommon_Mapping",
}

candidate_rows = mapping_table.loc[
    mapping_table["mapping_name"]
        .str.rsplit("::", n=1)
        .str[-1]
        .isin(candidate_short_names)
]

candidate_names = set(candidate_rows["mapping_name"])

candidate_ops = mapping_operation_specifications.loc[
    mapping_operation_specifications["mapping_name"].isin(candidate_names),
    [
        "mapping_name",
        "operation_name",
        "declared_by",
        "is_direct",
        "language",
        "body",
    ],
].sort_values(
    ["mapping_name", "is_direct", "operation_name"],
    ascending=[True, False, True],
)

display(candidate_rows[
    ["mapping_name", "source_type", "target_type", "operation_count"]
])

display(candidate_ops)

,mapping_name,source_type,target_type,operation_count
465,SysMLv1Tov2::Mappings::UML4SysML::Classification::DefaultValue_Mapping,UML::Classification::Property,KerML::Kernel::FeatureValues::FeatureValue,2
489,SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertyCommon_Mapping,UML::Classification::Property,KerML::Core::Features::Feature,4
507,SysMLv1Tov2::Mappings::UML4SysML::CommonBehavior::ChangeTriggerConstraintUsage_Mapping,UML::CommonBehavior::Trigger,SysML::Systems::Constraints::ConstraintUsage,1
536,SysMLv1Tov2::Mappings::UML4SysML::CommonBehavior::TimeTriggerCalculationUsage_Mapping,UML::CommonBehavior::Trigger,SysML::Systems::Calculations::CalculationUsage,2
604,SysMLv1Tov2::Mappings::UML4SysML::Packages::ElementImport_Mapping,UML::CommonStructure::ElementImport,KerML::Root::Namespaces::MembershipImport,4
628,SysMLv1Tov2::Mappings::UML4SysML::Packages::StereotypeMetadataDefinition_Mapping,UML::Packages::Stereotype,SysML::Systems::Metadata::MetadataDefinition,0
667,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::InitialStateSubactionMembership_Mapping,UML::StateMachines::Pseudostate,SysML::Systems::States::StateSubactionMembership,2
751,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationFeatureTyping_Mapping,UML::Values::LiteralSpecification,KerML::Core::Features::FeatureTyping,0


,mapping_name,operation_name,declared_by,is_direct,language,body
10355,SysMLv1Tov2::Mappings::UML4SysML::Classification::DefaultValue_Mapping,isDefault,SysMLv1Tov2::Mappings::UML4SysML::Classification::DefaultValue_Mapping,True,OCL2.0,result = true
10356,SysMLv1Tov2::Mappings::UML4SysML::Classification::DefaultValue_Mapping,value,SysMLv1Tov2::Mappings::UML4SysML::Classification::DefaultValue_Mapping,True,OCL2.0,result = from.defaultValue
10378,SysMLv1Tov2::Mappings::UML4SysML::Classification::DefaultValue_Mapping,aliasId,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init,False,OCL2.0,result = Set{}
10380,SysMLv1Tov2::Mappings::UML4SysML::Classification::DefaultValue_Mapping,declaredName,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init,False,OCL2.0,result = null
10376,SysMLv1Tov2::Mappings::UML4SysML::Classification::DefaultValue_Mapping,elementId,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init,False,OCL2.0,result = Helper.createUUID()
...,...,...,...,...,...,...
16594,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationFeatureTyping_Mapping,specific,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToSpecialization_Init,False,OCL2.0,
16597,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationFeatureTyping_Mapping,target,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init,False,OCL2.0,result = Set{}
16586,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationFeatureTyping_Mapping,type,SysMLv1Tov2::Mappings::UML4SysML::Classification::TypedElementFeatureTyping_Mapping,False,OCL2.0,result = if from.type.oclIsKindOf(UML::PrimitiveType) then Helper.getScalarValueType(from.type) else if from.type.oclIsKindOf(UML::Enumeration) then Helper.getEnumerationType(from.type) else Classifier_Mapping.getMapped(from.type) endif endif
16592,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationFeatureTyping_Mapping,type,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init,False,OCL2.0,result =


In [159]:
unresolved_callers = set(dependency_edges.loc[dependency_edges["dependency_status"] == "unresolved", "source_class",])

caller_rows = mapping_table.loc[
    mapping_table["mapping_name"].isin(unresolved_callers),
    ["mapping_name", "source_type", "target_type"],
].sort_values("mapping_name")

display(caller_rows)

,mapping_name,source_type,target_type


In [160]:
inspect_names = {
    "ChangeTriggerConstraintUsage_Mapping",
    "TimeTriggerCalculationUsage_Mapping",
    "ElementImport_Mapping",
    "InitialStateSubactionMembership_Mapping",
    "PropertyCommon_Mapping",
}

inspect_ops = mapping_operation_specifications.loc[
    mapping_operation_specifications["mapping_name"]
        .str.rsplit("::", n=1)
        .str[-1]
        .isin(inspect_names)
    & mapping_operation_specifications["is_direct"],
    [
        "mapping_name",
        "operation_name",
        "language",
        "body",
    ],
].sort_values(["mapping_name", "operation_name"])

for row in inspect_ops.itertuples(index=False):
    print(f"\n{row.mapping_name}::{row.operation_name}")
    print(f"Language: {row.language}")
    print(row.body)


SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertyCommon_Mapping::isComposite
Language: OCL2.0
result = from.isComposite

SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertyCommon_Mapping::isDerived
Language: OCL2.0
result = from.isDerived

SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertyCommon_Mapping::isEnd
Language: OCL2.0
result = if from.association.oclIsUndefined() then false else from.association.ownedEnd->includes(from) endif

SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertyCommon_Mapping::ownedRelationship
Language: OCL2.0
result = let typings: Set(KerML::FeatureTyping) = if from.type.oclIsUndefined() then Set{} else Set{StructuralFeatureToFeatureTyping_Mapping.getMapped(from)} endif in let subsettings: Set(KerML::Subsetting) = from.subsettedProperty ->collect(p | PropertySubsetting_Mapping.getMapped(from, p))->asSet() in let defaultValue: Set(KerML::OwningMembership) = if from.defaultValue.oclIsUndefined() then Set{} else Set{DefaultValue_M

In [161]:
caller_short_names = {"PropertySubsetting_Mapping", "AssociationMetadataUsage_Mapping", "AttributeRedefined_Mapping", "UseCase_Mapping", "Behavior_Mapping", "OpaqueBehavior_Mapping",}

caller_rows = mapping_table.loc[
    mapping_table["mapping_name"]
        .str.rsplit("::", n=1)
        .str[-1]
        .isin(caller_short_names),
    ["mapping_name", "source_type", "target_type"],
].sort_values("mapping_name")

caller_names = set(caller_rows["mapping_name"])

caller_ops = mapping_operation_specifications.loc[
    mapping_operation_specifications["mapping_name"].isin(caller_names)
    & mapping_operation_specifications["is_direct"],
    ["mapping_name", "operation_name", "language", "body",],
].sort_values(["mapping_name", "operation_name"])

display(caller_rows)

for row in caller_ops.itertuples(index=False):
    print(f"\n{row.mapping_name}::{row.operation_name}")
    print(f"Language: {row.language}")
    print(row.body)

,mapping_name,source_type,target_type
490,SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertySubsetting_Mapping,UML::Classification::Property,KerML::Core::Features::Subsetting
502,SysMLv1Tov2::Mappings::UML4SysML::CommonBehavior::Behavior_Mapping,UML::CommonBehavior::Behavior,KerML::Kernel::Behaviors::Behavior
531,SysMLv1Tov2::Mappings::UML4SysML::CommonBehavior::OpaqueBehavior_Mapping,UML::CommonBehavior::OpaqueBehavior,SysML::Systems::Actions::ActionDefinition
641,SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefined_Mapping,UML::Classification::Property,SysML::Systems::DefinitionAndUsage::ReferenceUsage
694,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsage_Mapping,UML::StructuredClassifiers::Association,SysML::Systems::Metadata::MetadataUsage
738,SysMLv1Tov2::Mappings::UML4SysML::UseCases::UseCase_Mapping,UML::UseCases::UseCase,SysML::Systems::UseCases::UseCaseDefinition



SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertySubsetting_Mapping::subsettedFeature
Language: OCL2.0
result=Property_Mapping.getMapped(subsettedProperty)

SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertySubsetting_Mapping::subsettingFeature
Language: OCL2.0
result=Property_Mapping.getMapped(from)

SysMLv1Tov2::Mappings::UML4SysML::CommonBehavior::Behavior_Mapping::ownedRelationship
Language: OCL2.0
result = let parameters: Set(UML::Element) = from.ownedElement->select(e | e.oclIsKindOf(UML::Parameter)) in let parameterSets: Set(UML::Element) = from.ownedElement->select(e | e.oclIsKindOf(UML::ParameterSet)) in let features: Set(UML::Element) = from.ownedElement->select(e | e.oclIsKindOf(UML::Property)) in let elementsOMS: Set(UML::Element) = (((from.ownedElement - parameters) parameterSets) - features) in elementsOMS->collect(e | ElementOwningMembership_Mapping.getMapped(e)) ->union(features->collect(e | PropertyMembership_Mapping.getMapped(e))) ->union(parameters-

In [162]:
property_membership_candidates = mapping_table.loc[
    mapping_table["source_type"].str.contains(
        r"UML::Classification::Property",
        na=False,
    )
    & mapping_table["target_type"].str.contains(
        r"Membership",
        na=False,
    ),
    [
        "mapping_name",
        "source_type",
        "target_type",
        "operation_count",
        "operation_categories",
        "ancestors",
    ],
].sort_values(["target_type", "mapping_name"])

display(property_membership_candidates)

,mapping_name,source_type,target_type,operation_count,operation_categories,ancestors
708,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::EndMembership_Mapping,UML::Classification::Property,KerML::Core::Features::EndFeatureMembership,0,[],"[SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping, SysMLv1Tov2::Mappings::Foundations::Mapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init, SysMLv1Tov2::Mappings::Foundations::Initializer, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToOwningMembership_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init]"
714,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::NonOwnedEndMembership_Mapping,UML::Classification::Property,KerML::Core::Features::EndFeatureMembership,2,"[mapping_filter, relationship_construction]","[SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::EndMembership_Mapping, SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping, SysMLv1Tov2::Mappings::Foundations::Mapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init, SysMLv1Tov2::Mappings::Foundations::Initializer, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToOwningMembership_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init]"
719,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEndMembership_Mapping,UML::Classification::Property,KerML::Core::Features::EndFeatureMembership,2,"[mapping_filter, relationship_construction]","[SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::EndMembership_Mapping, SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping, SysMLv1Tov2::Mappings::Foundations::Mapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init, SysMLv1Tov2::Mappings::Foundations::Initializer, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToOwningMembership_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init]"
696,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::ConnectionDefEndMembership_Mapping,UML::Classification::Property,KerML::Core::Types::FeatureMembership,1,[relationship_construction],"[SysMLv1Tov2::Mappings::Foundations::UniqueMapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init, SysMLv1Tov2::Mappings::Foundations::Mapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToOwningMembership_Init, SysMLv1Tov2::Mappings::Foundations::Initializer, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init]"
717,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::NonOwnedEndToSubsettedFeatureMembership_Mapping,UML::Classification::Property,KerML::Core::Types::FeatureMembership,2,"[mapping_filter, relationship_construction]","[SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init, SysMLv1Tov2::Mappings::Foundations::Mapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToType

In [163]:
association_typing_candidates = mapping_table.loc[
    mapping_table["target_type"].str.endswith(
        "FeatureTyping",
        na=False,
    )
    & mapping_table["source_type"].str.contains(
        r"Association|Property|StructuralFeature",
        na=False,
    ),
    [
        "mapping_name",
        "source_type",
        "target_type",
        "operation_count",
        "operation_categories",
        "ancestors",
    ],
].sort_values(["source_type", "mapping_name"])

display(association_typing_candidates)

,mapping_name,source_type,target_type,operation_count,operation_categories,ancestors
322,SysMLv1Tov2::Mappings::UML4SysML::Actions::StructuralFeatureActions::ASFVAFeatureTyping_Mapping,UML::Actions::AddStructuralFeatureValueAction,KerML::Core::Features::FeatureTyping,1,[property_population_candidate],"[SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init, SysMLv1Tov2::Mappings::Foundations::Mapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToSpecialization_Init, SysMLv1Tov2::Mappings::Foundations::Initializer, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init]"
324,SysMLv1Tov2::Mappings::UML4SysML::Actions::StructuralFeatureActions::ASFVAObjectReferenceUsageFeatureTyping_Mapping,UML::Actions::AddStructuralFeatureValueAction,KerML::Core::Features::FeatureTyping,1,[property_population_candidate],"[SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init, SysMLv1Tov2::Mappings::Foundations::Mapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToSpecialization_Init, SysMLv1Tov2::Mappings::Foundations::Initializer, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init]"
713,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::NonOwnedEndFeatureTyping_Mapping,UML::Classification::Property,KerML::Core::Features::FeatureTyping,0,[],"[SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureToFeatureTyping_Mapping, SysMLv1Tov2::Mappings::Foundations::UniqueMapping, SysMLv1Tov2::Mappings::UML4SysML::Classification::TypedElementFeatureTyping_Mapping, SysMLv1Tov2::Mappings::Foundations::Mapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init, SysMLv1Tov2::Mappings::Foundations::Initializer, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToSpecialization_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init]"
728,SysMLv1Tov2::Mappings::UML4SysML::UseCases::UseCaseActorFeatureTyping_Mapping,UML::Classification::Property,KerML::Core::Features::FeatureTyping,1,[property_population_candidate],"[SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init, SysMLv1Tov2::Mappings::Foundations::Mapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToSpecialization_Init, SysMLv1Tov2::Mappings::Foundations::Initializer, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init]"
499,SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureToFeatureTyping_Mapping,UML::Classification::StructuralFeature,KerML::Core::Features::FeatureTyping,0,[],"[SysMLv1Tov2::Mappings::UML4SysML::Classification::TypedElementFeatureTyping_Mapping, SysMLv1Tov2::Mappings::Foundations::Mapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init, SysMLv1Tov2::Mappings::Foundations::Initializer, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToSpecialization_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init]"
638,SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefinedFeatureTyping_Mapping,UML::Classification::StructuralFeature,KerML::Core::Features::FeatureTyping,0,[],"[SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureToFeatureTyping_Mapping, SysMLv1Tov2::Mappings::UML4SysML::Classification::TypedElementFeatureTyping_Mapping, SysMLv1Tov2::Mappings::Foundations::Mapping, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init, SysMLv1Tov2::Mappings::Foundations::Initializer, SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToSpecialization_Init, 

In [164]:
membership_candidates = mapping_table.loc[
    mapping_table["target_type"].str.endswith(("FeatureMembership", "EndFeatureMembership"), na=False,)
    & mapping_table["source_type"].str.contains(r"UML::.*(Property|StructuralFeature)", na=False,),
    ["mapping_name", "source_type", "target_type", "operation_count", "operation_categories", "ancestors",],
].sort_values(["source_type", "target_type", "mapping_name"])

display(membership_candidates.to_string(index=False))

typing_candidate_names = {
    "AssociationMetadataUsageFeatureTyping_Mapping", 
    "AttributeRedefinedFeatureTyping_Mapping", 
    "NonOwnedEndFeatureTyping_Mapping", 
    "StructuralFeatureToFeatureTyping_Mapping",
    }

typing_candidates = mapping_table.loc[mapping_table["mapping_name"] .str.rsplit("::", n=1) .str[-1] .isin(typing_candidate_names)]
display(typing_candidates[["mapping_name", "source_type", "target_type", "operation_count"]].to_string(index=False))

typing_names = set(typing_candidates["mapping_name"])

typing_ops = mapping_operation_specifications.loc[
    mapping_operation_specifications["mapping_name"].isin(typing_names)
    & mapping_operation_specifications["is_direct"],
    ["mapping_name", "operation_name", "language", "body"],
]

for row in typing_ops.itertuples(index=False):
    print(f"\n{row.mapping_name}::{row.operation_name}")
    print(row.body)


property_candidate_names = {"PropertyCommon_Mapping", "PropertySubsetting_Mapping", "StructuralFeatureMembership_Mapping",}
property_candidates = mapping_table.loc[mapping_table["mapping_name"] .str.rsplit("::", n=1) .str[-1] .isin(property_candidate_names)]
display(property_candidates[
    ["mapping_name", "source_type", "target_type", "operation_count"]
].to_string(index=False))

C:\Users\Drew\AppData\Local\Temp\ipykernel_10432\2047119384.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  & mapping_table["source_type"].str.contains(r"UML::.*(Property|StructuralFeature)", na=False,),


'                                                                                                                mapping_name                                   source_type                                 target_type  operation_count                                       operation_categories                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

'                                                                                          mapping_name                             source_type                          target_type  operation_count\n            SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureToFeatureTyping_Mapping  UML::Classification::StructuralFeature KerML::Core::Features::FeatureTyping                0\n          SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefinedFeatureTyping_Mapping  UML::Classification::StructuralFeature KerML::Core::Features::FeatureTyping                0\nSysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsageFeatureTyping_Mapping UML::StructuredClassifiers::Association KerML::Core::Features::FeatureTyping                1\n             SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::NonOwnedEndFeatureTyping_Mapping           UML::Classification::Property KerML::Core::Features::FeatureTyping                0'


SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsageFeatureTyping_Mapping::type
result = SYSML2::MetadataDefinition.allInstances() ->any(m | m.qualifiedName = 'SysMLv1Library::AssociationData')


'                                                                         mapping_name                            source_type                           target_type  operation_count\n             SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertyCommon_Mapping          UML::Classification::Property        KerML::Core::Features::Feature                4\n         SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertySubsetting_Mapping          UML::Classification::Property     KerML::Core::Features::Subsetting                2\nSysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping UML::Classification::StructuralFeature KerML::Core::Types::FeatureMembership                2'

In [165]:
inspect_short_names = {
    "StructuralFeatureMembership_Mapping",
    "EndMembership_Mapping",
    "OwnedEndMembership_Mapping",
    "NonOwnedEndMembership_Mapping",
    "ConnectionDefEndMembership_Mapping",
    "NonOwnedEndToSubsettedFeatureMembership_Mapping",
    "StructuralFeatureToFeatureTyping_Mapping",
    "AttributeRedefinedFeatureTyping_Mapping",
    "AssociationMetadataUsageFeatureTyping_Mapping",
    "PropertyCommon_Mapping",
}

inspect_rows = mapping_table.loc[
    mapping_table["mapping_name"]
        .str.rsplit("::", n=1)
        .str[-1]
        .isin(inspect_short_names)
]

display(inspect_rows[
    [
        "mapping_name",
        "source_type",
        "target_type",
        "source_property_declared_by",
        "target_property_declared_by",
    ]
].to_string(index=False))

'                                                                                            mapping_name                             source_type                                 target_type                                                                              source_property_declared_by                                                      target_property_declared_by\n                                SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertyCommon_Mapping           UML::Classification::Property              KerML::Core::Features::Feature                                 SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertyCommon_Mapping           SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init\n                   SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping  UML::Classification::StructuralFeature       KerML::Core::Types::FeatureMembership                    SysMLv1Tov2::Mappings::UML4SysML::Class

In [166]:
inspect_names = set(inspect_rows["mapping_name"])

inspect_declarations = mapping_operation_declarations.loc[
    mapping_operation_declarations["mapping_name"].isin(inspect_names),
    [
        "mapping_name",
        "operation_name",
        "declared_by",
        "is_direct",
        "redefined_operations",
    ],
].sort_values(["mapping_name", "operation_name", "is_direct"])

display(inspect_declarations.to_string(index=False))

"                                                                                            mapping_name            operation_name                                                                                              declared_by  is_direct                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     redefined_operations\n                                SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertyCommon_Mapping                   aliasId                                   SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init      False                

In [167]:
initializer_names = {
    "ToFeatureMembership_Init",
    "ToOwningMembership_Init",
    "ToMembership_Init",
    "ToFeatureTyping_Init",
    "ToSpecialization_Init",
}

initializer_records = [
    e for e in registry.get_elements("SysML_v1_to_v2")
    if e.qualified_name
    and e.qualified_name.rsplit("::", 1)[-1] in initializer_names
]

for cls in initializer_records:
    print(f"\n{cls.qualified_name}")

    members = [
        e for e in registry.get_elements("SysML_v1_to_v2")
        if e.parent == cls.key
        and e.tag in {"ownedAttribute", "ownedOperation"}
    ]

    for member in members:
        print(
            member.tag,
            member.name,
            member.xmi_id,
        )


SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init
ownedAttribute to Mappings-Initializers-KerMLInitializers-ToFeatureTyping_Init-to
ownedOperation type Mappings-Initializers-KerMLInitializers-ToFeatureTyping_Init-type_
ownedOperation typedFeature Mappings-Initializers-KerMLInitializers-ToFeatureTyping_Init-typedFeature_

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init
ownedAttribute to Mappings-Initializers-KerMLInitializers-ToMembership_Init-to
ownedOperation memberName Mappings-Initializers-KerMLInitializers-ToMembership_Init-memberName_
ownedOperation visibility Mappings-Initializers-KerMLInitializers-ToMembership_Init-visibility_
ownedOperation memberElement Mappings-Initializers-KerMLInitializers-ToMembership_Init-memberElement_
ownedOperation memberShortName Mappings-Initializers-KerMLInitializers-ToMembership_Init-memberShortName_
ownedOperation mem

In [168]:
membership_short_names = {
    "StructuralFeatureMembership_Mapping",
    "EndMembership_Mapping",
    "OwnedEndMembership_Mapping",
    "NonOwnedEndMembership_Mapping",
    "ConnectionDefEndMembership_Mapping",
    "NonOwnedEndToSubsettedFeatureMembership_Mapping",
}

membership_names = set(
    mapping_table.loc[
        mapping_table["mapping_name"]
            .str.rsplit("::", n=1)
            .str[-1]
            .isin(membership_short_names),
        "mapping_name",
    ]
)

membership_ops = mapping_operation_specifications.loc[
    mapping_operation_specifications["mapping_name"].isin(membership_names)
    & mapping_operation_specifications["is_direct"],
    ["mapping_name", "operation_name", "language", "body"],
]

for row in membership_ops.itertuples(index=False):
    print(f"\n{row.mapping_name}::{row.operation_name}")
    print(row.body)


SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping::ownedMemberFeature
result = NamedElementMain_Mapping.getMapped(from)

SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping::visibility
result = if (from.oclIsKindOf(UML::NamedElement)) then Helper.getKerMLVisibilityKind(from.oclAsType(UML::NamedElement).visibility) else KerML::VisibilityKind::public endif 

SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::ConnectionDefEndMembership_Mapping::ownedMemberFeature
result = ConnectionDefEnd_Mapping.getMapped(from)

SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::NonOwnedEndMembership_Mapping::ownedMemberFeature
result = NonOwnedEnd_Mapping.getMapped(from)

SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::NonOwnedEndMembership_Mapping::filter
result = src.oclIsKindOf(UML::Property) and not src.oclAsType(UML::Property).association.oclIsUndefined() and src.oclAsType(UML::Property).association.ownedEnd

In [169]:
candidate_short_names = {
    "StructuralFeatureMembership_Mapping",
    "PropertyCommon_Mapping",
    "AssociationMetadataUsageFeatureTyping_Mapping",
    "AttributeRedefinedFeatureTyping_Mapping",
}

candidate_names = set(
    mapping_table.loc[
        mapping_table["mapping_name"]
            .str.rsplit("::", n=1)
            .str[-1]
            .isin(candidate_short_names),
        "mapping_name",
    ]
)

relevant_operations = {
    "type",
    "typedFeature",
    "specific",
    "general",
    "ownedMemberFeature",
    "ownedRelatedElement",
    "visibility",
    "memberElement",
    "membershipOwningNamespace",
}

ops = mapping_operation_specifications.loc[
    mapping_operation_specifications["mapping_name"].isin(candidate_names)
    & mapping_operation_specifications["operation_name"].isin(
        relevant_operations
    ),
    [
        "mapping_name",
        "operation_name",
        "declared_by",
        "is_direct",
        "language",
        "body",
    ],
].sort_values(["mapping_name", "operation_name", "is_direct"])

for row in ops.itertuples(index=False):
    print(f"\n{row.mapping_name}::{row.operation_name}")
    print(f"Declared by: {row.declared_by}")
    print(f"Direct: {row.is_direct}")
    print(f"Language: {row.language}")
    print(row.body)


SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping::memberElement
Declared by: SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init
Direct: False
Language: OCL2.0
result = null

SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping::membershipOwningNamespace
Declared by: SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init
Direct: False
Language: OCL2.0
result = Set{}

SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping::ownedMemberFeature
Declared by: SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init
Direct: False
Language: OCL2.0
result = self.upperBound

SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping::ownedMemberFeature
Declared by: SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping
Direct: True
Language: OCL2.0
result = NamedElementMain_

In [170]:
initializer_names = {
    "ToFeatureMembership_Init",
    "ToMembership_Init",
    "ToFeatureTyping_Init",
}

initializer_classes = {
    e.qualified_name
    for e in registry.get_elements("SysML_v1_to_v2")
    if e.qualified_name
    and e.qualified_name.rsplit("::", 1)[-1] in initializer_names
}

initializer_ops = mapping_operation_specifications.loc[
    mapping_operation_specifications["declared_by"].isin(initializer_classes),
    [
        "mapping_name",
        "operation_name",
        "declared_by",
        "language",
        "body",
    ],
].drop_duplicates(subset=["declared_by", "operation_name"])

for row in initializer_ops.itertuples(index=False):
    print(f"\n{row.declared_by}::{row.operation_name}")
    print(row.body)


SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init::memberName
result = null

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init::visibility
result = KerML::VisibilityKind::public

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init::memberElement
result = null

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init::memberShortName
result = null

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToMembership_Init::membershipOwningNamespace
result = Set{}

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init::type
result = 

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init::typedFeature
result = 

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init::ownedMemberFeature
result = self.upperBound

SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureMembership_Init::ownedRelatedElement
result = Set{self.ow

In [171]:
short_names = {
    "StructuralFeatureMembership_Mapping",
    "AssociationMetadataUsageFeatureTyping_Mapping",
    "AttributeRedefinedFeatureTyping_Mapping",
}

candidate_names = set(
    mapping_table.loc[
        mapping_table["mapping_name"]
            .str.rsplit("::", n=1)
            .str[-1]
            .isin(short_names),
        "mapping_name",
    ]
)

candidate_ops = mapping_operation_specifications.loc[
    mapping_operation_specifications["mapping_name"].isin(candidate_names)
    & mapping_operation_specifications["is_direct"],
    [
        "mapping_name",
        "operation_name",
        "language",
        "body",
    ],
].sort_values(["mapping_name", "operation_name"])

for row in candidate_ops.itertuples(index=False):
    print(f"\n{row.mapping_name}::{row.operation_name}")
    print(row.body)


SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping::ownedMemberFeature
result = NamedElementMain_Mapping.getMapped(from)

SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping::visibility
result = if (from.oclIsKindOf(UML::NamedElement)) then Helper.getKerMLVisibilityKind(from.oclAsType(UML::NamedElement).visibility) else KerML::VisibilityKind::public endif 

SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsageFeatureTyping_Mapping::type
result = SYSML2::MetadataDefinition.allInstances() ->any(m | m.qualifiedName = 'SysMLv1Library::AssociationData')


In [172]:
association_name = next(
    name for name in candidate_names
    if name.endswith("AssociationMetadataUsageFeatureTyping_Mapping")
)

association_row = mapping_table.loc[
    mapping_table["mapping_name"] == association_name
].iloc[0]

print("Ancestors:")
for ancestor in association_row["ancestors"]:
    print(" ", ancestor)

print("\nDirect operations:")
print(
    candidate_ops.loc[
        candidate_ops["mapping_name"] == association_name
    ].to_string(index=False)
)

Ancestors:
  SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeatureTyping_Init
  SysMLv1Tov2::Mappings::Foundations::Mapping
  SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToSpecialization_Init
  SysMLv1Tov2::Mappings::Foundations::Initializer
  SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToRelationship_Init
  SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init

Direct operations:
                                                                                          mapping_name operation_name language                                                                                                              body
SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsageFeatureTyping_Mapping           type   OCL2.0 result = SYSML2::MetadataDefinition.allInstances() ->any(m | m.qualifiedName = 'SysMLv1Library::AssociationData')


### Final Validation and Refactor:

In [173]:
import pandas as pd
import time
print(time.time())

semantic_alias_rules = pd.DataFrame([
    {
        "caller_suffix": "PropertySubsetting_Mapping",
        "callee": "Property_Mapping",
        "semantic_target_class": ("SysMLv1Tov2::Mappings::UML4SysML::Classification" "::PropertyCommon_Mapping"),
        "semantic_alias_rule": "property_to_common_property",
        "semantic_alias_evidence": (
            "Both call arguments are UML Property; candidate source is "
            "UML Property and target is KerML Feature; target property "
            "is inherited from ToFeature_Init."
        ),
        "semantic_alias_confidence": "high",
    },
    {
        "caller_suffix": "Behavior_Mapping",
        "callee": "PropertyMembership_Mapping",
        "semantic_target_class": (
            "SysMLv1Tov2::Mappings::UML4SysML::Classification"
            "::StructuralFeatureMembership_Mapping"
        ),
        "semantic_alias_rule": "property_to_structural_feature_membership",
        "semantic_alias_evidence": (
            "Caller maps UML Properties into owned relationships; "
            "Property specializes StructuralFeature; candidate maps "
            "StructuralFeature to FeatureMembership and supplies "
            "ownedMemberFeature and visibility."
        ),
        "semantic_alias_confidence": "high",
    },
    {
        "caller_suffix": "OpaqueBehavior_Mapping",
        "callee": "PropertyMembership_Mapping",
        "semantic_target_class": (
            "SysMLv1Tov2::Mappings::UML4SysML::Classification"
            "::StructuralFeatureMembership_Mapping"
        ),
        "semantic_alias_rule": "property_to_structural_feature_membership",
        "semantic_alias_evidence": (
            "Caller maps UML Properties into owned relationships; "
            "Property specializes StructuralFeature; candidate maps "
            "StructuralFeature to FeatureMembership and supplies "
            "ownedMemberFeature and visibility."
        ),
        "semantic_alias_confidence": "high",
    },
    {
        "caller_suffix": "UseCase_Mapping",
        "callee": "PropertyMembership_Mapping",
        "semantic_target_class": (
            "SysMLv1Tov2::Mappings::UML4SysML::Classification"
            "::StructuralFeatureMembership_Mapping"
        ),
        "semantic_alias_rule": "property_to_structural_feature_membership",
        "semantic_alias_evidence": (
            "Caller maps owned UML Properties into memberships; "
            "the caller's property selection excludes association "
            "properties; candidate maps StructuralFeature to "
            "FeatureMembership."
        ),
        "semantic_alias_confidence": "high",
    },
    {
        "caller_suffix": "AssociationMetadataUsage_Mapping",
        "callee": "AssociationToFeatureTyping_Mapping",
        "semantic_target_class": (
            "SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers"
            "::AssociationMetadataUsageFeatureTyping_Mapping"
        ),
        "semantic_alias_rule": "association_metadata_feature_typing",
        "semantic_alias_evidence": (
            "Caller supplies UML Association; candidate source is "
            "UML Association and target is KerML FeatureTyping; "
            "direct type operation selects SysMLv1Library::AssociationData."
        ),
        "semantic_alias_confidence": "high",
    },
    {
        "caller_suffix": "AttributeRedefined_Mapping",
        "callee": "AssociationToFeatureTyping_Mapping",
        "semantic_target_class": (
            "SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers"
            "::AttributeRedefinedFeatureTyping_Mapping"
        ),
        "semantic_alias_rule": "redefined_attribute_feature_typing",
        "semantic_alias_evidence": (
            "Caller supplies UML StructuralFeature; candidate source "
            "is UML StructuralFeature and target is KerML FeatureTyping; "
            "candidate inherits StructuralFeatureToFeatureTyping_Mapping."
        ),
        "semantic_alias_confidence": "high",
    },
    {
    "caller_suffix": "ValuePin_Mapping",
    "callee": "PinFeatureTyping_Mapping",
    "semantic_target_class": (
        "SysMLv1Tov2::Mappings::UML4SysML::Actions::VariableActions"
        "::RVAReferenceUsageFeatureTyping_Mapping"
    ),
    "semantic_alias_rule": "pin_feature_typing",
    "semantic_alias_evidence": (
        "Caller maps UML ValuePin to ReferenceUsage and invokes "
        "PinFeatureTyping_Mapping. Candidate source is UML Pin and "
        "target is KerML FeatureTyping. It inherits the typed-element "
        "filter and type mapping, which maps the pin's UML type to "
        "the corresponding KerML typing relationship."
    ),
    "semantic_alias_confidence": "high",
},
{
    "caller_suffix": "COAPin_Mapping",
    "callee": "PinFeatureTyping_Mapping",
    "semantic_target_class": (
        "SysMLv1Tov2::Mappings::UML4SysML::Actions::VariableActions"
        "::RVAReferenceUsageFeatureTyping_Mapping"
    ),
    "semantic_alias_rule": "pin_feature_typing",
    "semantic_alias_evidence": (
        "Caller maps UML OutputPin to ReferenceUsage and invokes "
        "PinFeatureTyping_Mapping. Candidate source is UML Pin and "
        "target is KerML FeatureTyping. Its inherited typed-element "
        "mapping uses the pin's UML type to construct the typing "
        "relationship."
    ),
    "semantic_alias_confidence": "high",
},
{
    "caller_suffix": "LiteralSpecificationCommon_Mapping",
    "callee": "LiteralSpecificationTyping_Mapping",
    "semantic_target_class": (
        "SysMLv1Tov2::Mappings::UML4SysML::Values"
        "::LiteralSpecificationFeatureTyping_Mapping"
    ),
    "semantic_alias_rule": "literal_specification_feature_typing",
    "semantic_alias_evidence": (
        "Caller maps UML LiteralSpecification to KerML "
        "LiteralExpression and invokes LiteralSpecificationTyping_Mapping "
        "when the source has a defined type. Candidate maps UML "
        "LiteralSpecification to KerML FeatureTyping. It inherits the "
        "typed-element filter and type mapping, which maps the source "
        "element's UML type to the corresponding KerML typing relationship."
    ),
    "semantic_alias_confidence": "high",
},
{
    "caller_suffix": "TimeExpression_Mapping",
    "callee": "LiteralSpecificationTyping_Mapping",
    "semantic_target_class": (
        "SysMLv1Tov2::Mappings::UML4SysML::Values"
        "::LiteralSpecificationFeatureTyping_Mapping"
    ),
    "semantic_alias_rule": "literal_specification_feature_typing",
    "semantic_alias_evidence": (
        "Caller maps UML TimeExpression to KerML Expression and invokes "
        "LiteralSpecificationTyping_Mapping when the source has a "
        "defined type. Candidate maps UML LiteralSpecification to KerML "
        "FeatureTyping and inherits the typed-element filter and type "
        "mapping. The candidate's source type is compatible with the "
        "literal-specification typing operation, although the concrete "
        "implementation of its inherited typedFeature operation has not "
        "been independently verified."
    ),
    "semantic_alias_confidence": "high",
},
{
    "caller_suffix": "ValueSpecification_Mapping",
    "callee": "LiteralSpecificationTyping_Mapping",
    "semantic_target_class": (
        "SysMLv1Tov2::Mappings::UML4SysML::Values"
        "::LiteralSpecificationFeatureTyping_Mapping"
    ),
    "semantic_alias_rule": "literal_specification_feature_typing",
    "semantic_alias_evidence": (
        "Caller maps UML ValueSpecification to KerML Expression and "
        "invokes LiteralSpecificationTyping_Mapping when the source has "
        "a defined type. Candidate maps UML LiteralSpecification to "
        "KerML FeatureTyping and inherits the typed-element filter and "
        "type mapping, which maps the source element's UML type to the "
        "corresponding KerML typing relationship. The concrete "
        "implementation of its inherited typedFeature operation has not "
        "been independently verified."
    ),
    "semantic_alias_confidence": "high",
},
{
    "caller_suffix": "Helper",
    "callee": "InitialStateMembership_Mapping",
    "semantic_target_class": (
        "SysMLv1Tov2::Mappings::UML4SysML::StateMachines"
        "::InitialStateSubactionMembership_Mapping"
    ),
    "semantic_alias_rule": "initial_state_subaction_membership",
    "semantic_alias_evidence": (
        "The caller selects UML initial pseudostates and requests "
        "InitialStateMembership_Mapping.getMapped(e). The candidate "
        "inherits ToStateSubactionMembership_Init and "
        "ToFeatureMembership_Init, defines kind() as "
        "SysMLv2::SubactionKind::entry, and defines "
        "ownedMemberFeature() as InitialState_Mapping.getMapped(from)."
    ),
    "semantic_alias_confidence": "high",
},
{
    "caller_suffix": "Region_Mapping",
    "callee": "InitialStateMembership_Mapping",
    "semantic_target_class": (
        "SysMLv1Tov2::Mappings::UML4SysML::StateMachines"
        "::InitialStateSubactionMembership_Mapping"
    ),
    "semantic_alias_rule": "initial_state_subaction_membership",
    "semantic_alias_evidence": (
        "Region_Mapping::ownedRelationship selects UML initial "
        "pseudostates and requests InitialStateMembership_Mapping."
        " The candidate inherits ToStateSubactionMembership_Init "
        "and ToFeatureMembership_Init, defines kind() as "
        "SysMLv2::SubactionKind::entry, and maps its member feature "
        "through InitialState_Mapping.getMapped(from)."
    ),
    "semantic_alias_confidence": "high",
},
{
    "caller_suffix": "StateDefinition_Mapping",
    "callee": "InitialStateMembership_Mapping",
    "semantic_target_class": (
        "SysMLv1Tov2::Mappings::UML4SysML::StateMachines"
        "::InitialStateSubactionMembership_Mapping"
    ),
    "semantic_alias_rule": "initial_state_subaction_membership",
    "semantic_alias_evidence": (
        "StateDefinition_Mapping::ownedRelationship selects UML "
        "initial pseudostates and requests InitialStateMembership_Mapping."
        " The candidate inherits ToStateSubactionMembership_Init "
        "and ToFeatureMembership_Init, defines kind() as "
        "SysMLv2::SubactionKind::entry, and maps its member feature "
        "through InitialState_Mapping.getMapped(from)."
    ),
    "semantic_alias_confidence": "high",
},
])

placeholder_mappings = pd.DataFrame([
    {
        "call_id": "CALL_00214",
        "original_target": "CommonElementImport_Mapping",
        "placeholder_name": "UNRESOLVED_CommonElementImport",
        "caller": "ElementGroup_Mapping",
        "operation": "getMapped",
        "arguments": ["e"],
        "expected_target_role": "Import-like relationship",
        "resolution_status": "placeholder",
    },
    {
        "call_id": "CALL_00380",
        "original_target": "DefaultValueOpaqueExpression_Mapping",
        "placeholder_name": "UNRESOLVED_DefaultValueOpaqueExpression",
        "caller": "OwnedEnd_Mapping",
        "operation": "getMapped",
        "arguments": ["from.defaultValue"],
        "expected_target_role": "Relationship",
        "resolution_status": "placeholder",
    },
    {
        "call_id": "CALL_00309",
        "original_target": "InstanceSpecificationToGeneralization_Mapping",
        "placeholder_name": "UNRESOLVED_InstanceSpecificationToGeneralization",
        "caller": "InstanceSpecification_Mapping",
        "operation": "getMapped",
        "arguments": ["from", "c"],
        "expected_target_role": "Feature membership",
        "resolution_status": "placeholder",
    },
])

def apply_semantic_aliases(dependency_edges, semantic_alias_rules, mapping_table,):
    edges = dependency_edges.copy()
    required = {"call_id", "source_class", "callee", "target_class", "resolution_status",}
    missing = required - set(edges.columns)
    if missing:
        raise ValueError(f"dependency_edges is missing columns: {sorted(missing)}")
    if edges["call_id"].duplicated().any():
        raise ValueError("call_id must uniquely identify call occurrences.")
    # Preserve the original exact-resolution fields.
    original_target = edges["target_class"].copy()
    original_status = edges["resolution_status"].copy()
    edges["caller_suffix"] = (edges["source_class"].str.rsplit("::", n=1).str[-1])
    edges["callee_short"] = (edges["callee"].str.rsplit("::", n=1).str[-1])
    # Semantic-resolution fields.
    edges["semantic_target_class"] = pd.NA
    edges["semantic_alias_rule"] = pd.NA
    edges["semantic_alias_evidence"] = pd.NA
    edges["semantic_alias_confidence"] = pd.NA
    # Only unresolved exact references can receive a semantic alias.
    unresolved = (edges["target_class"].isna() | edges["resolution_status"].eq("unresolved"))
    for rule in semantic_alias_rules.itertuples(index=False):
        mask = (unresolved & edges["caller_suffix"].eq(rule.caller_suffix) & edges["callee_short"].eq(rule.callee))
        # Do not silently assign multiple rules to one call.
        already_assigned = edges["semantic_target_class"].notna()
        if (mask & already_assigned).any():
            raise ValueError(f"Overlapping semantic alias rules for {rule.callee} from {rule.caller_suffix}")
        edges.loc[mask, "semantic_target_class"] = (rule.semantic_target_class)
        edges.loc[mask, "semantic_alias_rule"] = (rule.semantic_alias_rule)
        edges.loc[mask, "semantic_alias_evidence"] = (rule.semantic_alias_evidence)
        edges.loc[mask, "semantic_alias_confidence"] = (rule.semantic_alias_confidence)

    # Effective resolution is a derived view. It does not overwrite the original XMI/OCL resolution.
    edges["effective_target_class"] = (edges["target_class"].combine_first(edges["semantic_target_class"]))
    edges["effective_resolution_status"] = (edges["resolution_status"].copy())
    semantic_resolved = edges["semantic_target_class"].notna()
    edges.loc[semantic_resolved, "effective_resolution_status",] = "resolved_semantic_alias"

    # Verify that the original exact-resolution data was preserved.
    pd.testing.assert_series_equal(edges["target_class"], original_target, check_names=False,)
    pd.testing.assert_series_equal(edges["resolution_status"], original_status, check_names=False,)

    # Every semantic target must exist in the mapping inventory.
    known_mappings = set(mapping_table["mapping_name"])
    assigned_targets = set(edges["semantic_target_class"].dropna())
    unknown_targets = assigned_targets - known_mappings
    if unknown_targets:
        raise ValueError(f"Semantic targets absent from mapping_table: {sorted(unknown_targets)}")

    return edges


1793570459.3017962


In [174]:
mapping_dependency_summary = (
    dependency_edges.groupby("source_class")
    .agg(
        dependency_count=("call_id", "size"),
        resolved_dependency_count=("effective_target_class", "count",),
        unresolved_dependency_count=("effective_target_class", lambda x: x.isna().sum(),),
        dependencies=("effective_target_class", lambda x: sorted(x.dropna().unique().tolist()),),
    )
    .reset_index()
    .rename(columns={"source_class": "mapping_name"})
)

In [175]:
mapping_analysis = mapping_table.merge(mapping_dependency_summary, on="mapping_name", how="left", validate="one_to_one",)

In [176]:
print(dependency_edges["effective_resolution_status"] .value_counts(dropna=False))

print(
    dependency_edges.groupby(["resolution_status", "effective_resolution_status"], dropna=False,)
    .size()
    .rename("count")
)

effective_resolution_status
resolved_suffix            830
unresolved                  22
resolved_semantic_alias      2
not_applicable               1
Name: count, dtype: int64
resolution_status  effective_resolution_status
not_applicable     not_applicable                   1
resolved_suffix    resolved_suffix                830
unresolved         resolved_semantic_alias          2
                   unresolved                      22
Name: count, dtype: int64


In [177]:
remaining_unresolved = dependency_edges.loc[dependency_edges["effective_target_class"].isna()].copy()

remaining_unresolved_summary = (
    remaining_unresolved.groupby(["source_class", "callee"], dropna=False,)
    .agg(call_count=("call_id", "size"), call_ids=("call_id", list), call_expressions=("call_expression", list),)
    .reset_index()
    .sort_values(["call_count", "source_class"], ascending=[False, True],)
)

print(remaining_unresolved_summary.to_string(index=False))

                                                                                  source_class                                         callee  call_count                 call_ids                                                                  call_expressions
                  SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertySubsetting_Mapping                               Property_Mapping           2 [CALL_00285, CALL_00286] [Property_Mapping.getMapped(from), Property_Mapping.getMapped(subsettedProperty)]
                                                   SysMLv1Tov2::Mappings::Foundations::Mapping                                           self           1             [CALL_00000]                                                               [self.getMapped(e)]
                                                                 SysMLv1Tov2::Mappings::Helper                 InitialStateMembership_Mapping           1             [CALL_00022]                                     [I

In [178]:
def mapping_name_by_suffix(mapping_table, suffix):
    matches = mapping_table.loc[mapping_table["mapping_name"].str.endswith(f"::{suffix}"), "mapping_name",].unique()

    if len(matches) != 1:
        raise ValueError(f"Expected one mapping ending in {suffix!r}; found {len(matches)}: {matches.tolist()}")
    return matches[0]


previously_validated_rules = [
    # Literal aliases: caller-independent.
    ("UpperBoundValueFeatureMembership_Mapping",
     "LiteralUnlimitedToInteger_Mapping",
     "LiteralUnlimitedInteger_Mapping"),

    ("UpperBoundValueFeatureMembership_Mapping",
     "LiteralUnlimitedToUnbounded_Mapping",
     "LiteralUnlimitedUnbounded_Mapping"),

    # Previously validated contextual aliases.
    ("ChangeTriggerExpressionFeatureTyping_Mapping",
     "TransitionChangeTriggerConstraintUsage_Mapping",
     "ChangeTriggerConstraintUsage_Mapping"),

    ("TimeTriggerExpressionFeatureTyping_Mapping",
     "TransitionTimeTriggerCalculationUsage_Mapping",
     "TimeTriggerCalculationUsage_Mapping"),

    ("AttributeRedefined_Mapping",
     "PropertyDefaultValue_Mapping",
     "DefaultValue_Mapping"),

    ("Region_Mapping",
     "InitialStateMembership_Mapping",
     "InitialStateSubactionMembership_Mapping"),

    ("StateDefinition_Mapping",
     "InitialStateMembership_Mapping",
     "InitialStateSubactionMembership_Mapping"),

    ("StereotypeOccurenceUsageFeatureTyping_Mapping",
     "StereotypeOccurenceDefinition_Mapping",
     "StereotypeMetadataDefinition_Mapping"),
]

previously_validated_rules = pd.DataFrame([
    {
        "caller_suffix": caller,
        "callee": callee,
        "semantic_target_class": mapping_name_by_suffix(mapping_table, target),
        "semantic_alias_rule": f"previously_validated_{target}",
        "semantic_alias_evidence": ("Previously investigated and accepted semantic mapping; " "original call identity is preserved."),
        "semantic_alias_confidence": "high",
    }
    for caller, callee, target in previously_validated_rules
])

In [179]:
# Preserve historical rules only for keys not already defined in the current semantic alias definition.
current_keys = set(zip(semantic_alias_rules["caller_suffix"], semantic_alias_rules["callee"],))
previously_validated_rules_filtered = previously_validated_rules.loc[
    ~previously_validated_rules.apply(
        lambda row: (row["caller_suffix"], row["callee"],) in current_keys,
        axis=1,
    )
].copy()
semantic_alias_rules = pd.concat([semantic_alias_rules, previously_validated_rules_filtered,], ignore_index=True,)

In [180]:
remaining_unresolved = dependency_edges.loc[
    dependency_edges["effective_target_class"].isna()
].copy()

remaining_unresolved_summary = (
    remaining_unresolved.groupby(
        ["source_class", "callee"],
        dropna=False,
    )
    .agg(
        call_count=("call_id", "size"),
        call_ids=("call_id", list),
        call_expressions=("call_expression", list),
    )
    .reset_index()
    .sort_values(
        ["callee", "source_class"],
    )
)

print(remaining_unresolved_summary.to_string(index=False))

                                                                                  source_class                                         callee  call_count                 call_ids                                                                  call_expressions
               SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefined_Mapping             AssociationToFeatureTyping_Mapping           1             [CALL_00726]                              [AssociationToFeatureTyping_Mapping.getMapped(from)]
     SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsage_Mapping             AssociationToFeatureTyping_Mapping           1             [CALL_00395]                              [AssociationToFeatureTyping_Mapping.getMapped(from)]
                           SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping                    CommonElementImport_Mapping           1             [CALL_00214]                                       

In [181]:
previous_alias_specs = [
    ("UpperBoundValueFeatureMembership_Mapping", "LiteralUnlimitedToInteger_Mapping", "LiteralUnlimitedInteger_Mapping",),
    ("UpperBoundValueFeatureMembership_Mapping", "LiteralUnlimitedToUnbounded_Mapping", "LiteralUnlimitedUnbounded_Mapping",),
    ("ChangeTriggerExpressionFeatureTyping_Mapping", "TransitionChangeTriggerConstraintUsage_Mapping", "ChangeTriggerConstraintUsage_Mapping",),
    ("TimeTriggerExpressionFeatureTyping_Mapping", "TransitionTimeTriggerCalculationUsage_Mapping", "TimeTriggerCalculationUsage_Mapping",),
    ("AttributeRedefined_Mapping", "PropertyDefaultValue_Mapping", "DefaultValue_Mapping",),
    ("StereotypeOccurenceUsageFeatureTyping_Mapping", "StereotypeOccurenceDefinition_Mapping", "StereotypeMetadataDefinition_Mapping",),
]

previous_alias_rules = pd.DataFrame([
    {
        "caller_suffix": caller,
        "callee": callee,
        "semantic_target_class": mapping_name_by_suffix(mapping_table, target),
        "semantic_alias_rule": f"previously_validated_{target}",
        "semantic_alias_evidence": ("Previously investigated and accepted semantic correspondence."),
        "semantic_alias_confidence": "high",
    }
    for caller, callee, target in previous_alias_specs
])

# Add only historical rules not already represented.
current_keys = set(zip(semantic_alias_rules["caller_suffix"], semantic_alias_rules["callee"],))
previous_alias_rules_filtered = previous_alias_rules.loc[~previous_alias_rules.apply(lambda row: (row["caller_suffix"], row["callee"],) in current_keys, axis=1,)].copy()
semantic_alias_rules = pd.concat([semantic_alias_rules, previous_alias_rules_filtered,], ignore_index=True,)

dependency_edges = apply_semantic_aliases(
    dependency_edges=dependency_edges, 
    semantic_alias_rules=semantic_alias_rules, 
    mapping_table=mapping_table,
    )


In [182]:
semantic_resolved = dependency_edges.loc[
    dependency_edges["semantic_target_class"].notna()
].copy()

alias_validation = (
    semantic_resolved.groupby(["caller_suffix", "callee_short", "semantic_target_class", "effective_resolution_status",], dropna=False,)
    .agg(call_count=("call_id", "size"), call_ids=("call_id", list),)
    .reset_index()
)
print(alias_validation.to_string(index=False))

                                caller_suffix                                   callee_short                                                                                  semantic_target_class effective_resolution_status  call_count                 call_ids
             AssociationMetadataUsage_Mapping             AssociationToFeatureTyping_Mapping SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsageFeatureTyping_Mapping     resolved_semantic_alias           1             [CALL_00395]
                   AttributeRedefined_Mapping             AssociationToFeatureTyping_Mapping           SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefinedFeatureTyping_Mapping     resolved_semantic_alias           1             [CALL_00726]
                   AttributeRedefined_Mapping                   PropertyDefaultValue_Mapping                                 SysMLv1Tov2::Mappings::UML4SysML::Classification::DefaultValue_Mapping     resolved_semantic

In [183]:
# 1. Inspect the caller's OCL specification.
caller_spec = mapping_operation_specifications.loc[
    mapping_operation_specifications["mapping_name"].str.endswith(
        "::ElementGroup_Mapping"
    )
    & mapping_operation_specifications["operation_name"].eq(
        "ownedRelationship"
    ),
    [
        "mapping_name",
        "operation_name",
        "language",
        "body",
    ],
]

print("CALLER SPECIFICATION")
print(caller_spec.to_string(index=False))
print("CALLER SPECIFICATION")
print(caller_spec.to_string(index=False))


# 2. Inspect the proposed target mapping's source/target crosswalk.
candidate = mapping_table.loc[
    mapping_table["mapping_name"].str.endswith(
        "::ElementImport_Mapping"
    )
]

print("\nCANDIDATE CROSSWALK")
print(candidate.to_string(index=False))


# 3. Inspect its direct and inherited operation declarations.
candidate_name = candidate.iloc[0]["mapping_name"]

candidate_declarations = mapping_operation_declarations.loc[
    mapping_operation_declarations["mapping_name"].eq(candidate_name),
    [
        "mapping_name",
        "declared_by",
        "operation_name",
        "operation_xmi_id",
        "is_direct",
        "redefined_operations",
    ],
].sort_values(
    ["is_direct", "declared_by", "operation_name"],
    ascending=[False, True, True],
)

print("\nCANDIDATE OPERATION DECLARATIONS")
print(candidate_declarations.to_string(index=False))


# 4. Inspect specifications for the candidate's declared operations.
candidate_specs = mapping_operation_specifications.loc[
    mapping_operation_specifications["mapping_name"].eq(candidate_name),
    [
        "declared_by",
        "operation_name",
        "is_direct",
        "language",
        "body",
    ],
]

print("\nCANDIDATE OPERATION SPECIFICATIONS")
print(candidate_specs.to_string(index=False))

CALLER SPECIFICATION
                                                       mapping_name    operation_name language                                                                                                                                                                                                                                                                                                                                           body
SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping ownedRelationship   OCL2.0 result = let elements : Set(KerML::Relationahip) = Helper.getTagValueAsElementColl(from, 'SysML::ModelElements::ElementGroup', 'member') ->collect(e | CommonElementImport_Mapping.getMapped(e)) in elements->including(ElementGroupMetadaMembership_Mapping.getMapped(from)) ->union(self.oclAsType(ElementMain_Mapping).ownedRelationship())
SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping ownedRelationship   OCL2.0                         

In [184]:
# Locate Helper.getTagValueAsElementColl
helper_operations = [e for e in registry.get_elements("SysML_v1_to_v2") if e.tag == "ownedOperation" and e.name == "getTagValueAsElementColl"]
for op in helper_operations:
    print(op.qualified_name, op.xmi_id)

# Inspect the ElementGroup stereotype and its member property.
element_group = [e for e in registry.get_elements("SysML_v1") if e.name == "ElementGroup"]

for e in element_group:
    print(e.tag, e.qualified_name, e.xmi_id)

    members = [
        child for child in registry.get_elements("SysML_v1")
        if child.parent == e.key
    ]

    for child in members:
        print(" ", child.tag, child.name, child.xmi_id, child.attributes,)

SysMLv1Tov2::Mappings::Helper::getTagValueAsElementColl Mappings-Helper-getTagValueAsElementColl_Element_String_String
packagedElement SysML::ModelElements::ElementGroup SysML.ElementGroup
  ownedAttribute base_Comment SysML.ElementGroup.base_Comment {'{http://www.omg.org/spec/XMI/20131001}id': 'SysML.ElementGroup.base_Comment', '{http://www.omg.org/spec/XMI/20131001}type': 'uml:Property', 'name': 'base_Comment'}
  ownedAttribute criterion SysML.ElementGroup.criterion {'{http://www.omg.org/spec/XMI/20131001}id': 'SysML.ElementGroup.criterion', '{http://www.omg.org/spec/XMI/20131001}type': 'uml:Property', 'isDerived': 'true', 'name': 'criterion'}
  ownedAttribute member SysML.ElementGroup.member {'{http://www.omg.org/spec/XMI/20131001}id': 'SysML.ElementGroup.member', '{http://www.omg.org/spec/XMI/20131001}type': 'uml:Property', 'isDerived': 'true', 'name': 'member'}
  ownedAttribute name SysML.ElementGroup.name {'{http://www.omg.org/spec/XMI/20131001}id': 'SysML.ElementGroup.name', '{h

In [185]:
member_op = next(e for e in registry.get_elements("SysML_v1") if e.xmi_id == "SysML.ElementGroup.member_")
print("MEMBER OPERATION")
print(member_op.qualified_name, member_op.xmi_id)
# Find all contained specifications and constraints.
member_descendants = [e for e in registry.get_elements("SysML_v1") if e.parent == member_op.key]
for e in member_descendants:
    print(e.tag, e.name, e.xmi_id, e.attributes,)

helper_op = next(e for e in registry.get_elements("SysML_v1_to_v2") if e.xmi_id == "Mappings-Helper-getTagValueAsElementColl_Element_String_String")
print("HELPER OPERATION")
print(helper_op.qualified_name)
for e in registry.get_elements("SysMLv1_to_v2"):
    if e.parent == helper_op.key:
        print(e.tag, e.name, e.xmi_id, e.attributes)

MEMBER OPERATION
SysML::ModelElements::ElementGroup::member SysML.ElementGroup.member_
ownedParameter result SysML.ElementGroup.member_.result {'{http://www.omg.org/spec/XMI/20131001}id': 'SysML.ElementGroup.member_.result', '{http://www.omg.org/spec/XMI/20131001}type': 'uml:Parameter', 'direction': '_return', 'effect': 'create', 'name': 'result'}
HELPER OPERATION
SysMLv1Tov2::Mappings::Helper::getTagValueAsElementColl


In [186]:
result_param = next(
    e for e in registry.get_elements("SysML_v1")
    if e.xmi_id == "SysML.ElementGroup.member_.result"
)

print("RETURN PARAMETER")
print(result_param.attributes)

print("\nREFERENCES")
for ref in registry.get_references(source=result_param.key):
    print(
        ref.attribute,
        ref.value,
        ref.target,
        ref.resolution_status,
    )

RETURN PARAMETER
{'{http://www.omg.org/spec/XMI/20131001}id': 'SysML.ElementGroup.member_.result', '{http://www.omg.org/spec/XMI/20131001}type': 'uml:Parameter', 'direction': '_return', 'effect': 'create', 'name': 'result'}

REFERENCES
href http://www.omg.org/spec/UML/20131001/UML.xmi#Element ('UML_Metamodel', 'Element', 1) resolved_external


In [187]:
helper_matches = [
    e for e in registry.get_elements("SysML_v1_to_v2")
    if e.xmi_id ==
    "Mappings-Helper-getTagValueAsElementColl_Element_String_String"
]

print("HELPER MATCHES:", len(helper_matches))

for helper_op in helper_matches:
    print(helper_op.qualified_name, helper_op.xmi_id)

    print("\nCONTAINED ELEMENTS")
    for e in registry.get_elements("SysML_v1_to_v2"):
        if e.parent == helper_op.key:
            print(e.tag, e.name, e.xmi_id, e.attributes)

    print("\nREFERENCES")
    for ref in registry.get_references(source=helper_op.key):
        print(
            ref.attribute,
            ref.value,
            ref.target,
            ref.resolution_status,
        )

HELPER MATCHES: 1
SysMLv1Tov2::Mappings::Helper::getTagValueAsElementColl Mappings-Helper-getTagValueAsElementColl_Element_String_String

CONTAINED ELEMENTS
ownedParameter result Mappings-Helper-getTagValueAsElementColl_Element_String_String-result {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Helper-getTagValueAsElementColl_Element_String_String-result', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Parameter', 'isUnique': 'false', 'name': 'result'}
ownedParameter element Mappings-Helper-getTagValueAsElementColl_Element_String_String-element {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Helper-getTagValueAsElementColl_Element_String_String-element', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Parameter', 'name': 'element'}
ownedParameter stereotypeName Mappings-Helper-getTagValueAsElementColl_Element_String_String-stereotypeName {'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Helper-getTagValueAsElementColl_Element_String_String-stereotypeName', '{

In [188]:
inspect_names = [
    "RVAReferenceUsageFeatureTyping_Mapping",
    "REAFeatureValueOperatorExpressionFeatureTyping_Mapping",
    "COAInvocationExpessionFeatureTyping_Mapping",
    "ValuePin_Mapping",
    "COAPin_Mapping",
    "ElementGroup_Mapping",
]

def inspect_mapping_details(mapping_suffix):
    matches = mapping_table.loc[
        mapping_table["mapping_name"].str.endswith(
            f"::{mapping_suffix}"
        )
    ]

    print(f"\n{'=' * 100}")
    print(mapping_suffix)

    if matches.empty:
        print("No mapping_table crosswalk found.")
        return

    for _, row in matches.iterrows():
        mapping_name = row["mapping_name"]

        print("\nCROSSWALK")
        print("Mapping:", mapping_name)
        print("Source:", row["source_type"])
        print("Target:", row["target_type"])
        print("Ancestors:", row["ancestors"])

        # Direct operation declarations and their specifications
        direct = mapping_operation_specifications.loc[
            mapping_operation_specifications["mapping_name"].eq(
                mapping_name
            ),
            [
                "operation_name",
                "language",
                "body",
            ],
        ]

        print("\nDIRECT OPERATION SPECIFICATIONS")
        print(direct.to_string(index=False))

        # Direct and inherited operation declarations
        declarations = mapping_operation_declarations.loc[
            mapping_operation_declarations["mapping_name"].eq(
                mapping_name
            )
        ].copy()

        print("\nRELEVANT INHERITED DECLARATIONS")
        relevant = declarations.loc[
            declarations["operation_name"].isin(
                ["filter", "type", "typedFeature",
                 "ownedRelationship", "source", "target"]
            ),
            [
                "declared_by",
                "operation_name",
                "operation_xmi_id",
                "is_direct",
                "redefined_operations",
            ],
        ]

        print(relevant.to_string(index=False))

        # Inspect the UML operation parameters in the raw registry.
        print("\nOPERATION PARAMETERS")

        for _, declaration in relevant.iterrows():
            op_id = declaration["operation_xmi_id"]

            operations = [
                e for e in registry.get_elements("SysML_v1_to_v2")
                if e.xmi_id == op_id
                and e.tag == "uml:Operation"
            ]

            for op in operations:
                parameters = [
                    e for e in registry.get_elements("SysML_v1_to_v2")
                    if e.parent == op.key
                    and e.tag == "ownedParameter"
                ]

                print(
                    declaration["declared_by"],
                    declaration["operation_name"],
                )

                for parameter in parameters:
                    print("  ", parameter.attributes)


# Pin mappings and candidate relationships
for name in [
    "ValuePin_Mapping",
    "COAPin_Mapping",
    "RVAReferenceUsageFeatureTyping_Mapping",
    "REAFeatureValueOperatorExpressionFeatureTyping_Mapping",
    "COAInvocationExpessionFeatureTyping_Mapping",
]:
    inspect_mapping_details(name)

# ElementGroup mapping
inspect_mapping_details("ElementGroup_Mapping")


ValuePin_Mapping

CROSSWALK
Mapping: SysMLv1Tov2::Mappings::UML4SysML::Actions::Actions::ValuePin_Mapping
Source: UML::Actions::ValuePin
Target: SysML::Systems::DefinitionAndUsage::ReferenceUsage
Ancestors: ['SysMLv1Tov2::Mappings::UML4SysML::Actions::Actions::Pin_Mapping', 'SysMLv1Tov2::Mappings::Initializers::SystemInitializers::ToReferenceUsage_Init', 'SysMLv1Tov2::Mappings::UML4SysML::CommonStructure::NamedElementMain_Mapping', 'SysMLv1Tov2::Mappings::Initializers::SystemInitializers::ToUsage_Init', 'SysMLv1Tov2::Mappings::UML4SysML::CommonStructure::ElementMain_Mapping', 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToFeature_Init', 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToElement_Init', 'SysMLv1Tov2::Mappings::Foundations::MainMapping', 'SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToType_Init', 'SysMLv1Tov2::Mappings::Foundations::Initializer', 'SysMLv1Tov2::Mappings::Foundations::UniqueMapping', 'SysMLv1Tov2::Mappings::Initializers::KerM

In [189]:
from urllib.parse import urldefrag, unquote

class ReferenceResolver:
    """ 
    Between the language files, common references are used. This class traverses the self-made references within the file
    to create the comprehensive existing map based on KerML and the SysML1-SysML2 conversion file. 
    """
    def __init__(self, registry):
        self.registry = registry

    def resolve(self, reference):
        source_resource = reference.source[0]
        value = reference.value
        target = None
        resource = None
        fragment = None
        if reference.attribute == "idref":
            resource = source_resource
            fragment = value
        elif reference.attribute == "href":
            uri, fragment = urldefrag(value)
            fragment = unquote(fragment)

            if not fragment:
                return None

            if not uri:
                resource = source_resource
            else:
                resource = self.registry.uri_aliases.get(uri)
        else:
            return None
        if resource is None:
            return None
        key = (resource, fragment)
        target = self.registry.elements.get(key)
        if target is not None:
            reference.target = key
            reference.resolved = True
        return target

    def resolve_all(self):
        for reference in self.registry.references:
            self.resolve(reference)
        total = len(self.registry.references)
        resolved = sum(reference.resolved for reference in self.registry.references)
        return {"total": total, "resolved": resolved, "unresolved": total - resolved, "resolution_rate": (resolved / total if total else 1.0)}

In [190]:
RESOURCE_MANIFEST = {
    "UML_Profile": {"path": "processed/uml.xmi", "uri": None, "dependencies": [],},
    "SysML_v1": {"path": "processed/sysml_1.xmi", "uri": None, "dependencies": ["UML_20131001", "UML_PrimitiveTypes_20131001", "UML_StandardProfile_20131001",],},
    "KerML": {"path": "processed/kerml.xmi", "uri": None, "dependencies": ["UML_PrimitiveTypes_20161101",],},
    "SysML_v2": {"path": "processed/sysml_2.xmi", "uri": "https://www.omg.org/spec/SysML/20250201/SysML.xmi", "dependencies": ["KerML", "UML_PrimitiveTypes_20161101",],},
    "SysML_v1_to_v2": {"path": "processed/sysml_v1_to_v2.xmi", "uri": None, "dependencies": ["UML_20161101", "UML_PrimitiveTypes_20161101", "SysML_v2",],},
}

registry.uri_aliases = {
    # UML 2.5.1 / 2016-11-01
    "http://www.omg.org/spec/UML/20161101/UML.xmi": "UML_20161101",
    "https://www.omg.org/spec/UML/20161101/UML.xmi": "UML_20161101",
    "https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi": "UML_PrimitiveTypes_20161101",
    # UML 2013-10-01
    "http://www.omg.org/spec/UML/20131001/UML.xmi": "UML_20131001",
    "http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi": "UML_PrimitiveTypes_20131001",
    "http://www.omg.org/spec/UML/20131001/StandardProfile.xmi": "UML_StandardProfile_20131001",
    # KerML 2025-02-01
    "https://www.omg.org/spec/KerML/20250201/KerML.xmi": "KerML",
}

In [191]:

registry.resolve_references()

def inspect_unresolved_references(registry):
    for resource in registry.resources:
        unresolved = [ref for ref in registry.references if ref.source[0] == resource and not ref.resolved]
        print(f"\n{resource}")
        print("Unresolved:", len(unresolved))
        print("Reference attributes:", Counter(ref.attribute for ref in unresolved))
        print("Reference tags:", Counter(ref.element_tag for ref in unresolved))
        print("Examples:")
        for ref in unresolved[:10]:
            print(f" {ref.source} | " f"{ref.element_tag} | " f"{ref.attribute} | " f"{ref.value}")

inspect_unresolved_references(registry)


UML_Metamodel
Unresolved: 206
Reference attributes: Counter({'href': 200, 'idref': 6})
Reference tags: Counter({'type': 200, 'constrainedElement': 4, 'precondition': 2})
Examples:
 ('UML_Metamodel', 'Activity-isReadOnly', 1) | type | href | http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi#Boolean
 ('UML_Metamodel', 'Activity-isSingleExecution', 1) | type | href | http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi#Boolean
 ('UML_Metamodel', 'ActivityEdge-isConsistentWith-result', 1) | type | href | http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi#Boolean
 ('UML_Metamodel', 'ActivityNode-isConsistentWith-result', 1) | type | href | http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi#Boolean
 ('UML_Metamodel', 'ActivityPartition-isDimension', 1) | type | href | http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi#Boolean
 ('UML_Metamodel', 'ActivityPartition-isExternal', 1) | type | href | http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi#Boolean
 ('UML_Me

# Current Status:

## Completed:
A. Metamodel registry and XMI extraction
B. Mapping-class inventory and source/target crosswalk
C. Inheritance and operation extraction
D. OCL and English specifications
E. Mapping dependency extraction

## Progressing:
A. Semantic aliases
B. Remaining unresolved dependencies

## Transformation Approach:
MetaModel Registry --> Mapping class inventory and translation layer --> Inheritance/Redefinition/Specification/Dependencies --> Unified Representation 



In [192]:
audit_inputs = {
    "mapping_class_records": mapping_class_records,
    "mapping_table": mapping_table,
    "mapping_operation_declarations": mapping_operation_declarations,
    "ocl_specifications": ocl_specifications,
    "dependency_edges": dependency_edges,
    "semantic_alias_rules": semantic_alias_rules,
}

for name, obj in audit_inputs.items():
    print(f"\n{'=' * 80}")
    print(f"{name}: {type(obj).__name__}")
    print(f"{'=' * 80}")

    if isinstance(obj, pd.DataFrame):
        print("Shape:", obj.shape)
        print("Columns:", obj.columns.tolist())
        print("\nDtypes:")
        print(obj.dtypes.to_string())
        print("\nSample:")
        print(obj.head(2))

    elif isinstance(obj, list):
        print("Length:", len(obj))

        if obj:
            print("First element type:", type(obj[0]).__name__)
            print("First element:")
            display(obj[0])

    else:
        print("Value:", obj)


mapping_class_records: list
Length: 786
First element type: ElementRecord
First element:


ElementRecord(resource='SysML_v1_to_v2', xmi_id='Mappings-Foundations-MainMapping', tag='packagedElement', metaclass='uml:Class', name='MainMapping', attributes={'{http://www.omg.org/spec/XMI/20161101}id': 'Mappings-Foundations-MainMapping', '{http://www.omg.org/spec/XMI/20161101}type': 'uml:Class', 'isAbstract': 'true', 'name': 'MainMapping'}, parent=('SysML_v1_to_v2', 'Mappings-Foundations', 1), children=[('SysML_v1_to_v2', 'Mappings-Foundations-MainMapping-_generalization.Mappings-Foundations-UniqueMapping', 1)], registry_key=('SysML_v1_to_v2', 'Mappings-Foundations-MainMapping', 1), element_category=None, qualified_name='SysMLv1Tov2::Mappings::Foundations::MainMapping', namespace='SysMLv1Tov2::Mappings::Foundations')


mapping_table: DataFrame
Shape: (772, 20)
Columns: ['mapping_name', 'mapping_xmi_id', 'source_property', 'target_property', 'source_property_xmi_id', 'target_property_xmi_id', 'source_property_declared_by', 'target_property_declared_by', 'source_property_type', 'target_property_type', 'source_type', 'target_type', 'ancestors', 'operation_count', 'operations', 'operation_categories', 'dependency_count', 'resolved_dependency_count', 'unresolved_dependency_count', 'dependencies']

Dtypes:
mapping_name                   object
mapping_xmi_id                 object
source_property                object
target_property                object
source_property_xmi_id         object
target_property_xmi_id         object
source_property_declared_by    object
target_property_declared_by    object
source_property_type           object
target_property_type           object
source_type                    object
target_type                    object
ancestors                      object
operation_coun

In [193]:
print("\nRegistry elements:", len(registry.elements))
print("Inheritance edges:", len(inheritance_edges))

print("\nRegistry resources:")
print(
    pd.Series([
        element.key[0]
        for element in registry.elements.values()
    ]).value_counts().to_string()
)


Registry elements: 25837
Inheritance edges: 1588

Registry resources:
SysML_v1_to_v2            14689
UML_Metamodel              5201
SysML_v2                   2463
KerML                      2294
SysML_v1                    682
UML_DiagramInterchange      319
UML_StandardProfile         177
UML_PrimitiveTypes           12


In [194]:
import pandas as pd


# 1. Validate operation identity uniqueness
assert mapping_operation_declarations["operation_xmi_id"].notna().all()
assert ocl_specifications["operation_xmi_id"].notna().all()
duplicate_spec_ids = (
    ocl_specifications.loc[ocl_specifications["operation_xmi_id"].duplicated(keep=False), ["operation_xmi_id", "operation", "constraint_id"],]
    .sort_values("operation_xmi_id")
)
print("Duplicate specification operation IDs:", len(duplicate_spec_ids))
if not duplicate_spec_ids.empty:
    display(duplicate_spec_ids)

# 2. Join inherited/direct operation declarations to specs
operation_specification_audit = mapping_operation_declarations.merge(
    ocl_specifications[["operation_xmi_id", "operation_id", "constraint_id", "constraint_name", "expression_id", "language", "body", "specification_category",]],
    on="operation_xmi_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

operation_specification_audit["has_specification"] = (operation_specification_audit["_merge"].eq("both"))

print("\nOperation specification join:")
print(operation_specification_audit["_merge"].value_counts())

print("Unique declared operations:", mapping_operation_declarations["operation_xmi_id"].nunique(),)
print("Unique operations with specifications:", ocl_specifications["operation_xmi_id"].nunique(),)


# 3. Normalize formal operation redefinitions
redefinition_rows = []
for row in mapping_operation_declarations.itertuples(index=False):
    for ref in row.redefined_operations or []:
        redefinition_rows.append({
            "mapping_name": row.mapping_name,
            "declared_by": row.declared_by,
            "operation": row.operation,
            "operation_xmi_id": row.operation_xmi_id,
            "redefined_operation": ref.get("operation"),
            "redefined_operation_xmi_id": ref.get("operation_xmi_id"),
            "resolution_status": ref.get("resolution_status"),
        })
operation_redefinitions = pd.DataFrame(redefinition_rows)
print("\nFormal operation redefinitions:", len(operation_redefinitions))
if not operation_redefinitions.empty:
    print(operation_redefinitions["resolution_status"] .value_counts(dropna=False))


# 4. Audit semantic alias rule coverage
alias_rule_audit = semantic_alias_rules.copy()
alias_rule_audit["matched_call_count"] = alias_rule_audit.apply(
    lambda rule: (dependency_edges["caller_suffix"].eq(rule["caller_suffix"]) & dependency_edges["callee_short"].eq(rule["callee"])).sum(),
    axis=1,
)

alias_rule_audit["matched_unresolved_count"] = alias_rule_audit.apply(
    lambda rule: (
        dependency_edges["caller_suffix"].eq(rule["caller_suffix"])
        & dependency_edges["callee_short"].eq(rule["callee"])
        & (dependency_edges["target_class"].isna() | dependency_edges["resolution_status"].eq("unresolved"))
    ).sum(),
    axis=1,
)

print("\nSemantic alias rule coverage:")
print(alias_rule_audit[
    ["caller_suffix", "callee", "semantic_target_class", "semantic_alias_confidence", "matched_call_count", "matched_unresolved_count",]
    ].to_string(index=False))

# 5. Current dependency resolution status
print("\nOriginal dependency resolution:")
print(dependency_edges["resolution_status"].value_counts(dropna=False))
print("\nEffective dependency resolution:")
print(dependency_edges["effective_resolution_status"] .value_counts(dropna=False))
print("\nEffective dependency target coverage:")
print("Calls with effective target:", dependency_edges["effective_target_class"].notna().sum(),)



# 6. Identify unresolved calls for candidate discovery
unresolved_dependency_calls = dependency_edges.loc[dependency_edges["effective_resolution_status"].eq("unresolved")].copy()
print("\nUnresolved dependency calls:", len(unresolved_dependency_calls))
if not unresolved_dependency_calls.empty:
    unresolved_summary = (
        unresolved_dependency_calls
        .groupby(["source_class", "callee", "resolution_status"], dropna=False,)
        .agg(call_count=("call_id", "size"), call_ids=("call_id", list), operations=("operation", lambda x: sorted(set(x))),)
        .reset_index()
        .sort_values(["call_count", "source_class"], ascending=[False, True])
    )
    display(unresolved_summary)


# 7. Per-mapping operation coverage
mapping_operation_audit = (
    operation_specification_audit
    .groupby("mapping_name")
    .agg(
        declaration_count=("operation_xmi_id", "size"),
        direct_declaration_count=("is_direct", "sum"),
        inherited_declaration_count=("is_direct", lambda x: (~x).sum(),),
        declarations_with_specs=("has_specification", "sum"),
        unique_operation_count=("operation_xmi_id", "nunique"),
    )
    .reset_index()
)

mapping_operation_audit["declarations_without_specs"] = (mapping_operation_audit["declaration_count"] - mapping_operation_audit["declarations_with_specs"])

print("\nMapping operation audit:")
print("Mappings represented:", len(mapping_operation_audit))
print("Declarations without specifications:", mapping_operation_audit["declarations_without_specs"].sum(),)
display(mapping_operation_audit.sort_values(["declarations_without_specs", "declaration_count"], ascending=[False, False],).head(20))

Duplicate specification operation IDs: 0

Operation specification join:
_merge
both          16769
left_only       282
right_only        0
Name: count, dtype: int64
Unique declared operations: 1024
Unique operations with specifications: 1101

Formal operation redefinitions: 4373
resolution_status
resolved_local    4373
Name: count, dtype: int64

Semantic alias rule coverage:
                                caller_suffix                                         callee                                                                                  semantic_target_class semantic_alias_confidence  matched_call_count  matched_unresolved_count
                   PropertySubsetting_Mapping                               Property_Mapping                               SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertyCommon_Mapping                      high                   2                         2
                             Behavior_Mapping                     PropertyMembership_M

,source_class,callee,resolution_status,call_count,call_ids,operations
0,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping,CommonElementImport_Mapping,unresolved,1,[CALL_00214],[SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping::ownedRelationship]
1,SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping,InstanceSpecificationToGeneralization_Mapping,unresolved,1,[CALL_00309],[SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping::ownedFeatureMembership]
2,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping,DefaultValueOpaqueExpression_Mapping,unresolved,1,[CALL_00380],[SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping::ownedRelationship]



Mapping operation audit:
Mappings represented: 772
Declarations without specifications: 282


,mapping_name,declaration_count,direct_declaration_count,inherited_declaration_count,declarations_with_specs,unique_operation_count,declarations_without_specs
630,SysMLv1Tov2::Mappings::UML4SysML::Packages::StereotypeOccurenceUsageInfinityReturnParameterMembership_Mapping,29,3,26,27,29,2
10,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterFeatureMembership_Mapping,27,1,26,25,27,2
15,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageMembership_Mapping,27,1,26,25,27,2
242,SysMLv1Tov2::Mappings::UML4SysML::Actions::InvocationActions::COAOutputPinParameterMembership_Mapping,27,2,25,25,27,2
303,SysMLv1Tov2::Mappings::UML4SysML::Actions::ObjectActions::RICOAFeatureValueOperatorParameterMembership_Mapping,27,2,25,25,27,2
337,SysMLv1Tov2::Mappings::UML4SysML::Actions::StructuralFeatureActions::ASFVATargetParameterMembership_Mapping,27,2,25,25,27,2
503,SysMLv1Tov2::Mappings::UML4SysML::CommonBehavior::ChangeEventReturnParameterMembership_Mapping,27,1,26,25,27,2
525,SysMLv1Tov2::Mappings::UML4SysML::CommonBehavior::ChangeTriggerReturnParameterMembership_Mapping,27,1,26,25,27,2
553,SysMLv1Tov2::Mappings::UML4SysML::CommonBehavior::TimeTriggerReturnParameterMembership_Mapping,27,1,26,25,27,2
639,SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefinedMembership_Mapping,27,2,25,25,27,2


In [195]:
# ============================================================
# STAGE 2: UNRESOLVED DEPENDENCY REVIEW
# ============================================================

# 1. Individual unresolved calls
unresolved_calls = dependency_edges.loc[
    dependency_edges["effective_resolution_status"].eq("unresolved")
].copy()

unresolved_detail = unresolved_calls[
    [
        "call_id",
        "source_class",
        "operation",
        "callee",
        "method",
        "arguments",
        "call_expression",
        "candidate_classes",
        "resolution_status",
        "dependency_type",
    ]
].sort_values(["source_class", "callee", "call_id"])

print("Unresolved call occurrences:", len(unresolved_detail))
display(unresolved_detail)

# 2. Group calls by caller and callee
unresolved_groups = (
    unresolved_calls
    .groupby(
        ["source_class", "callee", "resolution_status"],
        dropna=False,
    )
    .agg(
        call_count=("call_id", "size"),
        call_ids=("call_id", list),
        operations=("operation", lambda x: sorted(set(x))),
        arguments=("arguments", lambda x: list(x)),
        candidate_classes=(
            "candidate_classes",
            lambda x: list(x),
        ),
    )
    .reset_index()
    .sort_values(["callee", "source_class"])
)

print("\nUnresolved caller/callee groups:", len(unresolved_groups))
display(unresolved_groups)

# 3. Alias rules that did not match unresolved calls
unmatched_alias_rules = alias_rule_audit.loc[
    alias_rule_audit["matched_unresolved_count"].eq(0)
].copy()

print("\nAlias rules without an unresolved call match:",
      len(unmatched_alias_rules))

display(
    unmatched_alias_rules[
        [
            "caller_suffix",
            "callee",
            "semantic_target_class",
            "semantic_alias_confidence",
            "matched_call_count",
            "matched_unresolved_count",
        ]
    ]
)

# 4. Check the operation declarations without specifications
missing_specifications = operation_specification_audit.loc[
    ~operation_specification_audit["has_specification"]
].copy()

print("Missing-specification declaration rows:", len(missing_specifications))
print(
    "Unique missing-specification operation IDs:",
    missing_specifications["operation_xmi_id"].nunique()
)

missing_specification_operations = (
    missing_specifications[
        [
            "mapping_name",
            "declared_by",
            "operation",
            "operation_name",
            "operation_xmi_id",
            "is_direct",
        ]
    ]
    .drop_duplicates("operation_xmi_id")
    .sort_values(["declared_by", "operation_name"])
    .drop(columns="operation_name")
)

display(missing_specification_operations)

Unresolved call occurrences: 3


,call_id,source_class,operation,callee,method,arguments,call_expression,candidate_classes,resolution_status,dependency_type
214,CALL_00214,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping::ownedRelationship,CommonElementImport_Mapping,getMapped,e,CommonElementImport_Mapping.getMapped(e),[],unresolved,mapping_dependency
309,CALL_00309,SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping::ownedFeatureMembership,InstanceSpecificationToGeneralization_Mapping,getMapped,"from, c","InstanceSpecificationToGeneralization_Mapping.getMapped(from, c)",[],unresolved,mapping_dependency
380,CALL_00380,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping::ownedRelationship,DefaultValueOpaqueExpression_Mapping,getMapped,from.defaultValue,DefaultValueOpaqueExpression_Mapping.getMapped(from.defaultValue),[],unresolved,mapping_dependency



Unresolved caller/callee groups: 3


,source_class,callee,resolution_status,call_count,call_ids,operations,arguments,candidate_classes
0,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping,CommonElementImport_Mapping,unresolved,1,[CALL_00214],[SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping::ownedRelationship],[e],[[]]
2,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping,DefaultValueOpaqueExpression_Mapping,unresolved,1,[CALL_00380],[SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping::ownedRelationship],[from.defaultValue],[[]]
1,SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping,InstanceSpecificationToGeneralization_Mapping,unresolved,1,[CALL_00309],[SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping::ownedFeatureMembership],"[from, c]",[[]]



Alias rules without an unresolved call match: 0


,caller_suffix,callee,semantic_target_class,semantic_alias_confidence,matched_call_count,matched_unresolved_count


Missing-specification declaration rows: 282
Unique missing-specification operation IDs: 2


,mapping_name,declared_by,operation,operation_xmi_id,is_direct
64,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featureOfType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featureOfType_,False
65,SysMLv1Tov2::Mappings::CommonMappings::CommonParameterReferenceUsageInMembership_Mapping,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init,SysMLv1Tov2::Mappings::Initializers::KerMLInitializers::ToTypeFeaturing_Init::featuringType,Mappings-Initializers-KerMLInitializers-ToTypeFeaturing_Init-featuringType_,False


In [196]:
# STAGE 3: EFFECTIVE RESOLUTION AND ALIAS DIAGNOSTIC

# First, inspect the actual columns available.
print("Dependency columns:")
print(unresolved_calls.columns.tolist())

print("\nAlias-rule columns:")
print(semantic_alias_rules.columns.tolist())

# Keep only the fields needed from the alias-rule table and rename them.
alias_rules_for_join = semantic_alias_rules[
    [
        "caller_suffix",
        "callee",
        "semantic_target_class",
        "semantic_alias_rule",
        "semantic_alias_confidence",
    ]
].rename(
    columns={
        "callee": "rule_callee",
        "semantic_target_class": "rule_target_class",
        "semantic_alias_rule": "rule_description",
        "semantic_alias_confidence": "rule_confidence",
    }
)

unresolved_alias_matches = unresolved_calls.merge(
    alias_rules_for_join,
    left_on=["caller_suffix", "callee_short"],
    right_on=["caller_suffix", "rule_callee"],
    how="left",
    indicator=True,
)

print("Alias-rule matches among currently unresolved calls:")
display(
    unresolved_alias_matches[
        [
            "call_id",
            "caller_suffix",
            "callee_short",
            "rule_target_class",
            "rule_description",
            "rule_confidence",
            "_merge",
        ]
    ].sort_values(["callee_short", "caller_suffix"])
)

print("Unresolved calls grouped by callee:")
display(
    unresolved_alias_matches.groupby(
        ["callee_short", "_merge"], dropna=False
    )
    .agg(
        call_count=("call_id", "size"),
        callers=("caller_suffix", lambda x: sorted(set(x))),
        call_ids=("call_id", list),
    )
    .reset_index()
    .sort_values(["callee_short", "_merge"])
)

Dependency columns:
['call_id', 'operation_id', 'operation', 'constraint_id', 'constraint_name', 'language', 'specification_category', 'callee', 'method', 'arguments', 'call_expression', 'reference_type', 'candidate_classes', 'resolution_status', 'target_class', 'target_class_id', 'source_class', 'dependency_type', 'dependency_status', 'source_owner_type', 'semantic_target_class', 'effective_target_class', 'effective_resolution_status', 'caller_suffix', 'callee_short', 'semantic_alias_rule', 'semantic_alias_evidence', 'semantic_alias_confidence']

Alias-rule columns:
['caller_suffix', 'callee', 'semantic_target_class', 'semantic_alias_rule', 'semantic_alias_evidence', 'semantic_alias_confidence']
Alias-rule matches among currently unresolved calls:


,call_id,caller_suffix,callee_short,rule_target_class,rule_description,rule_confidence,_merge
0,CALL_00214,ElementGroup_Mapping,CommonElementImport_Mapping,NaN,NaN,NaN,left_only
2,CALL_00380,OwnedEnd_Mapping,DefaultValueOpaqueExpression_Mapping,NaN,NaN,NaN,left_only
1,CALL_00309,InstanceSpecification_Mapping,InstanceSpecificationToGeneralization_Mapping,NaN,NaN,NaN,left_only


Unresolved calls grouped by callee:


C:\Users\Drew\AppData\Local\Temp\ipykernel_10432\520469716.py:53: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  unresolved_alias_matches.groupby(


,callee_short,_merge,call_count,callers,call_ids
0,CommonElementImport_Mapping,left_only,1,[ElementGroup_Mapping],[CALL_00214]
1,CommonElementImport_Mapping,right_only,0,NaN,NaN
2,CommonElementImport_Mapping,both,0,NaN,NaN
3,DefaultValueOpaqueExpression_Mapping,left_only,1,[OwnedEnd_Mapping],[CALL_00380]
4,DefaultValueOpaqueExpression_Mapping,right_only,0,NaN,NaN
5,DefaultValueOpaqueExpression_Mapping,both,0,NaN,NaN
6,InstanceSpecificationToGeneralization_Mapping,left_only,1,[InstanceSpecification_Mapping],[CALL_00309]
7,InstanceSpecificationToGeneralization_Mapping,right_only,0,NaN,NaN
8,InstanceSpecificationToGeneralization_Mapping,both,0,NaN,NaN


In [197]:
print("unresolved_calls shape:", unresolved_calls.shape)
print("semantic_alias_rules shape:", semantic_alias_rules.shape)
print("unresolved_alias_matches shape:", unresolved_alias_matches.shape)

print("\nUnresolved caller/callee keys:")
print(
    unresolved_calls[["caller_suffix", "callee_short"]]
    .drop_duplicates()
    .to_string(index=False)
)

print("\nAlias-rule caller/callee keys:")
print(
    semantic_alias_rules[["caller_suffix", "callee"]]
    .drop_duplicates()
    .to_string(index=False)
)

print("\nUnresolved calls (first 20):")
print(
    unresolved_calls[
        ["call_id", "caller_suffix", "callee_short",
         "effective_resolution_status"]
    ].to_string(index=False)
)

print("\nMerge indicator counts:")
print(unresolved_alias_matches["_merge"].value_counts(dropna=False))

unresolved_calls shape: (3, 28)
semantic_alias_rules shape: (20, 6)
unresolved_alias_matches shape: (3, 33)

Unresolved caller/callee keys:
                caller_suffix                                  callee_short
         ElementGroup_Mapping                   CommonElementImport_Mapping
InstanceSpecification_Mapping InstanceSpecificationToGeneralization_Mapping
             OwnedEnd_Mapping          DefaultValueOpaqueExpression_Mapping

Alias-rule caller/callee keys:
                                caller_suffix                                         callee
                   PropertySubsetting_Mapping                               Property_Mapping
                             Behavior_Mapping                     PropertyMembership_Mapping
                       OpaqueBehavior_Mapping                     PropertyMembership_Mapping
                              UseCase_Mapping                     PropertyMembership_Mapping
             AssociationMetadataUsage_Mapping             A

In [198]:
# STAGE 3A: CHECK ALIAS KEY UNIQUENESS AND MATCH QUALITY

# 1. Check whether the alias table has duplicate caller/callee keys.
duplicate_alias_keys = (
    semantic_alias_rules.groupby(
        ["caller_suffix", "callee"],
        observed=True,
        dropna=False,
    )
    .size()
    .reset_index(name="rule_count")
    .query("rule_count > 1")
)

print("Duplicate alias keys:", len(duplicate_alias_keys))
display(duplicate_alias_keys)

# 2. Count unique calls matched, rather than merged rows.
matched_call_ids = unresolved_alias_matches.loc[
    unresolved_alias_matches["_merge"].eq("both"), "call_id"
].unique()

unmatched_call_ids = unresolved_alias_matches.loc[
    unresolved_alias_matches["_merge"].eq("left_only"), "call_id"
].unique()

print("Unique unresolved calls:", unresolved_calls["call_id"].nunique())
print("Unique calls with an alias match:", len(matched_call_ids))
print("Unique calls without an alias match:", len(unmatched_call_ids))

# 3. Show every matching rule alongside the call's current resolution.
matched_alias_detail = unresolved_alias_matches.loc[
    unresolved_alias_matches["_merge"].eq("both"),
    [
        "call_id",
        "caller_suffix",
        "callee_short",
        "rule_target_class",
        "rule_description",
        "rule_confidence",
        "semantic_target_class",
        "effective_target_class",
        "effective_resolution_status",
    ],
].copy()

display(
    matched_alias_detail.sort_values(
        ["callee_short", "caller_suffix", "rule_target_class"]
    )
)

# 4. Compare the rule's proposed target with the call's applied target.
print("Matched calls with no effective target:")
display(
    matched_alias_detail.loc[
        matched_alias_detail["effective_target_class"].isna()
    ]
)

Duplicate alias keys: 0


,caller_suffix,callee,rule_count


Unique unresolved calls: 3
Unique calls with an alias match: 0
Unique calls without an alias match: 3


,call_id,caller_suffix,callee_short,rule_target_class,rule_description,rule_confidence,semantic_target_class,effective_target_class,effective_resolution_status


Matched calls with no effective target:


,call_id,caller_suffix,callee_short,rule_target_class,rule_description,rule_confidence,semantic_target_class,effective_target_class,effective_resolution_status


In [199]:
# STAGE 3B: AUDIT ALIAS RULE DUPLICATION AND APPLICATION

# 1. Inspect all records associated with duplicated keys.
duplicate_rule_detail = semantic_alias_rules.merge(
    duplicate_alias_keys[["caller_suffix", "callee"]],
    on=["caller_suffix", "callee"],
    how="inner",
)

print("Duplicated alias-rule records:")
display(
    duplicate_rule_detail.sort_values(
        ["caller_suffix", "callee", "semantic_target_class"]
    )
)

# 2. Determine whether the duplicated rows are exact duplicates.
print(
    "Exact duplicate rows:",
    semantic_alias_rules.duplicated().sum()
)

print(
    "Unique caller/callee/target combinations:",
    semantic_alias_rules[
        ["caller_suffix", "callee", "semantic_target_class"]
    ].drop_duplicates().shape[0],
)

# 3. Inspect the alias fields currently stored in dependency_edges.
alias_application_audit = dependency_edges[
    [
        "call_id",
        "caller_suffix",
        "callee_short",
        "semantic_target_class",
        "semantic_alias_rule",
        "semantic_alias_evidence",
        "semantic_alias_confidence",
        "effective_target_class",
        "effective_resolution_status",
    ]
].copy()

print("Dependency alias application counts:")
display(
    alias_application_audit.groupby(
        ["effective_resolution_status", "semantic_alias_confidence"],
        dropna=False,
        observed=True,
    )
    .size()
    .reset_index(name="count")
)

# 4. Compare matching alias rules with dependency records.
alias_rule_lookup = semantic_alias_rules[
    [
        "caller_suffix",
        "callee",
        "semantic_target_class",
        "semantic_alias_rule",
    ]
].drop_duplicates()

alias_application_audit = alias_application_audit.merge(
    alias_rule_lookup,
    left_on=["caller_suffix", "callee_short"],
    right_on=["caller_suffix", "callee"],
    how="left",
    suffixes=("_applied", "_rule"),
)

# Matching rule exists, but no alias was applied.
unapplied_aliases = alias_application_audit.loc[
    alias_application_audit["semantic_target_class_rule"].notna()
    & alias_application_audit["semantic_target_class_applied"].isna()
]

print("Matching rules without an applied alias:")
display(
    unapplied_aliases.sort_values(
        ["caller_suffix", "callee_short"]
    )
)

# Applied alias differs from the rule's target.
target_mismatches = alias_application_audit.loc[
    alias_application_audit["semantic_target_class_rule"].notna()
    & alias_application_audit["semantic_target_class_applied"].notna()
    & (
        alias_application_audit["semantic_target_class_applied"]
        != alias_application_audit["semantic_target_class_rule"]
    )
]

print("Applied aliases with target mismatches:")
display(
    target_mismatches.sort_values(
        ["caller_suffix", "callee_short"]
    )
)

Duplicated alias-rule records:


,caller_suffix,callee,semantic_target_class,semantic_alias_rule,semantic_alias_evidence,semantic_alias_confidence


Exact duplicate rows: 0
Unique caller/callee/target combinations: 20
Dependency alias application counts:


,effective_resolution_status,semantic_alias_confidence,count
0,not_applicable,NaN,1
1,resolved_semantic_alias,high,21
2,resolved_suffix,NaN,830
3,unresolved,NaN,3


Matching rules without an applied alias:


,call_id,caller_suffix,callee_short,semantic_target_class_applied,semantic_alias_rule_applied,semantic_alias_evidence,semantic_alias_confidence,effective_target_class,effective_resolution_status,callee,semantic_target_class_rule,semantic_alias_rule_rule


Applied aliases with target mismatches:


,call_id,caller_suffix,callee_short,semantic_target_class_applied,semantic_alias_rule_applied,semantic_alias_evidence,semantic_alias_confidence,effective_target_class,effective_resolution_status,callee,semantic_target_class_rule,semantic_alias_rule_rule


In [200]:
import inspect

print("Function signature:")
print(inspect.signature(apply_semantic_aliases))

print("\nFunction implementation:")
print(inspect.getsource(apply_semantic_aliases))

Function signature:
(dependency_edges, semantic_alias_rules, mapping_table)

Function implementation:
def apply_semantic_aliases(dependency_edges, semantic_alias_rules, mapping_table,):
    edges = dependency_edges.copy()
    required = {"call_id", "source_class", "callee", "target_class", "resolution_status",}
    missing = required - set(edges.columns)
    if missing:
        raise ValueError(f"dependency_edges is missing columns: {sorted(missing)}")
    if edges["call_id"].duplicated().any():
        raise ValueError("call_id must uniquely identify call occurrences.")
    # Preserve the original exact-resolution fields.
    original_target = edges["target_class"].copy()
    original_status = edges["resolution_status"].copy()
    edges["caller_suffix"] = (edges["source_class"].str.rsplit("::", n=1).str[-1])
    edges["callee_short"] = (edges["callee"].str.rsplit("::", n=1).str[-1])
    # Semantic-resolution fields.
    edges["semantic_target_class"] = pd.NA
    edges["semantic_alias_

In [201]:
# STAGE 3C: CONSOLIDATE AND REAPPLY SEMANTIC ALIASES

# 1. Preserve the currently applied semantic aliases for regression testing.
previously_resolved_aliases = (
    dependency_edges.loc[
        dependency_edges["effective_resolution_status"].eq(
            "resolved_semantic_alias"
        ),
        ["call_id", "effective_target_class"],
    ]
    .copy()
    .set_index("call_id")
    .sort_index()
)

# 2. Verify each caller/callee key has only one semantic target.
target_counts = (
    semantic_alias_rules.groupby(
        ["caller_suffix", "callee"],
        observed=True,
    )["semantic_target_class"]
    .nunique()
)

conflicting_keys = target_counts[target_counts > 1]

if not conflicting_keys.empty:
    raise ValueError(
        "Some caller/callee keys have conflicting semantic targets:\n"
        f"{conflicting_keys}"
    )

# 3. Consolidate repeated rules while preserving all distinct evidence.
def combine_unique_evidence(values):
    unique_values = list(dict.fromkeys(
        str(value) for value in values if pd.notna(value)
    ))
    return " | ".join(unique_values)

semantic_alias_rules_clean = (
    semantic_alias_rules.groupby(
        ["caller_suffix", "callee"],
        as_index=False,
        observed=True,
    )
    .agg(
        semantic_target_class=("semantic_target_class", "first"),
        semantic_alias_rule=("semantic_alias_rule", "first"),
        semantic_alias_evidence=(
            "semantic_alias_evidence",
            combine_unique_evidence,
        ),
        semantic_alias_confidence=("semantic_alias_confidence", "first"),
    )
)

# 4. Verify that consolidation did not combine conflicting metadata.
metadata_counts = (
    semantic_alias_rules.groupby(
        ["caller_suffix", "callee"],
        observed=True,
    )
    .agg(
        target_count=("semantic_target_class", "nunique"),
        rule_count=("semantic_alias_rule", "nunique"),
        confidence_count=("semantic_alias_confidence", "nunique"),
    )
)

if (
    metadata_counts[["target_count", "rule_count", "confidence_count"]]
    .gt(1)
    .any()
    .any()
):
    raise ValueError(
        "Conflicting alias metadata exists. Inspect metadata_counts."
    )

print("Original alias rules:", len(semantic_alias_rules))
print("Consolidated alias rules:", len(semantic_alias_rules_clean))
print("Evidence statements preserved through consolidation.")

# 5. Apply the consolidated rules.
dependency_edges_updated = apply_semantic_aliases(
    dependency_edges,
    semantic_alias_rules_clean,
    mapping_table,
)

# 6. Confirm previously resolved aliases remain unchanged.
updated_previous_aliases = (
    dependency_edges_updated.loc[
        dependency_edges_updated["call_id"].isin(
            previously_resolved_aliases.index
        ),
        ["call_id", "effective_target_class"],
    ]
    .copy()
    .set_index("call_id")
    .sort_index()
)

pd.testing.assert_frame_equal(
    previously_resolved_aliases,
    updated_previous_aliases,
)

print("Previously resolved aliases preserved:",
      len(previously_resolved_aliases))

# 7. Validate final resolution counts.
print("\nEffective resolution counts:")
display(
    dependency_edges_updated["effective_resolution_status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="count")
)

print("\nSemantic alias assignments:")
display(
    dependency_edges_updated.loc[
        dependency_edges_updated["semantic_target_class"].notna(),
        [
            "call_id",
            "caller_suffix",
            "callee_short",
            "semantic_target_class",
            "semantic_alias_confidence",
        ],
    ].sort_values(["caller_suffix", "callee_short"])
)

# 8. Inspect the remaining unresolved calls.
remaining_unresolved = dependency_edges_updated.loc[
    dependency_edges_updated["effective_resolution_status"].eq("unresolved")
].copy()

print("\nRemaining unresolved calls:", len(remaining_unresolved))
display(
    remaining_unresolved[
        [
            "call_id",
            "source_class",
            "callee",
            "method",
            "arguments",
            "call_expression",
        ]
    ].sort_values(["callee", "source_class"])
)

# 9. Commit only after all checks have passed.
dependency_edges = dependency_edges_updated
semantic_alias_rules = semantic_alias_rules_clean

Original alias rules: 20
Consolidated alias rules: 20
Evidence statements preserved through consolidation.
Previously resolved aliases preserved: 21

Effective resolution counts:


,status,count
0,resolved_suffix,830
1,resolved_semantic_alias,21
2,unresolved,3
3,not_applicable,1



Semantic alias assignments:


,call_id,caller_suffix,callee_short,semantic_target_class,semantic_alias_confidence
395,CALL_00395,AssociationMetadataUsage_Mapping,AssociationToFeatureTyping_Mapping,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::AssociationMetadataUsageFeatureTyping_Mapping,high
726,CALL_00726,AttributeRedefined_Mapping,AssociationToFeatureTyping_Mapping,SysMLv1Tov2::Mappings::UML4SysML::SimpleClassifiers::AttributeRedefinedFeatureTyping_Mapping,high
731,CALL_00731,AttributeRedefined_Mapping,PropertyDefaultValue_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Classification::DefaultValue_Mapping,high
441,CALL_00441,Behavior_Mapping,PropertyMembership_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping,high
713,CALL_00713,COAPin_Mapping,PinFeatureTyping_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Actions::VariableActions::RVAReferenceUsageFeatureTyping_Mapping,high
493,CALL_00493,ChangeTriggerExpressionFeatureTyping_Mapping,TransitionChangeTriggerConstraintUsage_Mapping,SysMLv1Tov2::Mappings::UML4SysML::CommonBehavior::ChangeTriggerConstraintUsage_Mapping,high
22,CALL_00022,Helper,InitialStateMembership_Mapping,SysMLv1Tov2::Mappings::UML4SysML::StateMachines::InitialStateSubactionMembership_Mapping,high
335,CALL_00335,LiteralSpecificationCommon_Mapping,LiteralSpecificationTyping_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationFeatureTyping_Mapping,high
497,CALL_00497,OpaqueBehavior_Mapping,PropertyMembership_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Classification::StructuralFeatureMembership_Mapping,high
285,CALL_00285,PropertySubsetting_Mapping,Property_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Classification::PropertyCommon_Mapping,high



Remaining unresolved calls: 3


,call_id,source_class,callee,method,arguments,call_expression
214,CALL_00214,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping,CommonElementImport_Mapping,getMapped,e,CommonElementImport_Mapping.getMapped(e)
380,CALL_00380,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping,DefaultValueOpaqueExpression_Mapping,getMapped,from.defaultValue,DefaultValueOpaqueExpression_Mapping.getMapped(from.defaultValue)
309,CALL_00309,SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping,InstanceSpecificationToGeneralization_Mapping,getMapped,"from, c","InstanceSpecificationToGeneralization_Mapping.getMapped(from, c)"


# NB Fixes:

In [202]:
# Inspect the registry's existing reference-resolution interface.

print("Registry reference-related methods:")
print([
    name for name in dir(registry)
    if "resolv" in name.lower() or "reference" in name.lower()
])

print("\nRegistry element key examples:")
for key in list(registry.elements.keys())[:5]:
    print(key)

print("\nElement lookup method:")
import inspect

for method_name in [
    "get_elements_by_xmi_id",
    "get_element",
    "resolve_references",
]:
    method = getattr(registry, method_name, None)

    if method is not None:
        print(f"\n{method_name}{inspect.signature(method)}")

Registry reference-related methods:
['_resolve_href', '_resolve_idref', '_set_reference_result', 'get_references', 'reference_summary', 'references', 'resolve_references']

Registry element key examples:
('UML_Metamodel', '_0', 1)
('UML_Metamodel', '_packageImport.0', 1)
('UML_Metamodel', '_packageImport.1', 1)
('UML_Metamodel', '_packageImport.2', 1)
('UML_Metamodel', '_packageImport.3', 1)

Element lookup method:

get_elements_by_xmi_id(resource, xmi_id)

get_element(resource, xmi_id, occurrence=None)

resolve_references()


In [203]:
print("Columns:")
print(semantic_alias_rules.columns.tolist())

print("\nData types:")
print(semantic_alias_rules.dtypes)

Columns:
['caller_suffix', 'callee', 'semantic_target_class', 'semantic_alias_rule', 'semantic_alias_evidence', 'semantic_alias_confidence']

Data types:
caller_suffix                object
callee                       object
semantic_target_class        object
semantic_alias_rule          object
semantic_alias_evidence      object
semantic_alias_confidence    object
dtype: object


In [204]:
# 1. Check duplicate caller/callee combinations
rule_key = ["caller_suffix", "callee"]

duplicate_rules = semantic_alias_rules[
    semantic_alias_rules.duplicated(rule_key, keep=False)
].sort_values(rule_key)

print("Duplicate rule keys:", len(duplicate_rules))
display(duplicate_rules)

# 2. Check whether any caller/callee pair maps to multiple targets
target_counts = (
    semantic_alias_rules
    .groupby(rule_key)["semantic_target_class"]
    .nunique()
)

conflicting_targets = target_counts[target_counts > 1]

print("\nConflicting target assignments:", len(conflicting_targets))
display(conflicting_targets)

# 3. Check missing metadata
metadata_columns = [
    "semantic_target_class",
    "semantic_alias_rule",
    "semantic_alias_evidence",
    "semantic_alias_confidence",
]

missing_metadata = semantic_alias_rules[
    semantic_alias_rules[metadata_columns].isna().any(axis=1)
]

print("\nRules with missing metadata:", len(missing_metadata))
display(missing_metadata)

# 4. Inspect confidence categories
print("\nConfidence values:")
display(semantic_alias_rules["semantic_alias_confidence"].value_counts(dropna=False))



Duplicate rule keys: 0


,caller_suffix,callee,semantic_target_class,semantic_alias_rule,semantic_alias_evidence,semantic_alias_confidence



Conflicting target assignments: 0


Series([], Name: semantic_target_class, dtype: int64)


Rules with missing metadata: 0


,caller_suffix,callee,semantic_target_class,semantic_alias_rule,semantic_alias_evidence,semantic_alias_confidence



Confidence values:


semantic_alias_confidence
high    20
Name: count, dtype: int64

In [205]:
# Downgrade aliases whose inherited typedFeature implementation has not been independently verified.
unverified_aliases = (semantic_alias_rules["semantic_alias_evidence"] .str.contains("has not been independently verified", case=False, na=False,))
semantic_alias_rules.loc[unverified_aliases, "semantic_alias_confidence",] = "medium"
print("Updated confidence counts:")
display(semantic_alias_rules["semantic_alias_confidence"] .value_counts(dropna=False))
print("\nDowngraded rules:")
display(semantic_alias_rules.loc[unverified_aliases, ["caller_suffix", "callee", "semantic_target_class", "semantic_alias_confidence", "semantic_alias_evidence",],])


Updated confidence counts:


semantic_alias_confidence
high      18
medium     2
Name: count, dtype: int64


Downgraded rules:


,caller_suffix,callee,semantic_target_class,semantic_alias_confidence,semantic_alias_evidence
13,TimeExpression_Mapping,LiteralSpecificationTyping_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationFeatureTyping_Mapping,medium,"Caller maps UML TimeExpression to KerML Expression and invokes LiteralSpecificationTyping_Mapping when the source has a defined type. Candidate maps UML LiteralSpecification to KerML FeatureTyping and inherits the typed-element filter and type mapping. The candidate's source type is compatible with the literal-specification typing operation, although the concrete implementation of its inherited typedFeature operation has not been independently verified."
19,ValueSpecification_Mapping,LiteralSpecificationTyping_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationFeatureTyping_Mapping,medium,"Caller maps UML ValueSpecification to KerML Expression and invokes LiteralSpecificationTyping_Mapping when the source has a defined type. Candidate maps UML LiteralSpecification to KerML FeatureTyping and inherits the typed-element filter and type mapping, which maps the source element's UML type to the corresponding KerML typing relationship. The concrete implementation of its inherited typedFeature operation has not been independently verified."


In [206]:
import inspect

print(inspect.signature(apply_semantic_aliases))

(dependency_edges, semantic_alias_rules, mapping_table)


In [207]:
# Reapply semantic aliases using the updated confidence metadata.
dependency_edges_updated = apply_semantic_aliases(
    dependency_edges,
    semantic_alias_rules,
    mapping_table,
)

# Confirm that original call identities and resolution statuses are preserved.
assert dependency_edges_updated["call_id"].is_unique
assert dependency_edges_updated["call_id"].equals(dependency_edges["call_id"])

# Inspect the updated confidence assignments.
display(
    dependency_edges_updated.loc[
        dependency_edges_updated["semantic_alias_confidence"].eq("medium"),
        [
            "call_id",
            "source_class",
            "callee",
            "semantic_target_class",
            "semantic_alias_confidence",
        ],
    ]
)

,call_id,source_class,callee,semantic_target_class,semantic_alias_confidence
330,CALL_00330,SysMLv1Tov2::Mappings::UML4SysML::Values::TimeExpression_Mapping,LiteralSpecificationTyping_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationFeatureTyping_Mapping,medium
338,CALL_00338,SysMLv1Tov2::Mappings::UML4SysML::Values::ValueSpecification_Mapping,LiteralSpecificationTyping_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationFeatureTyping_Mapping,medium


In [208]:
dependency_edges = dependency_edges_updated.copy()

In [209]:
# Compare placeholder records against the dependency inventory.

print("Placeholder records:", len(placeholder_mappings))
print("Dependency calls:", len(dependency_edges))

# Confirm required columns.
print("\nPlaceholder columns:")
print(placeholder_mappings.columns.tolist())

print("\nDependency columns:")
print(dependency_edges.columns.tolist())

# Identify unresolved calls.
unresolved_calls = dependency_edges[
    dependency_edges["resolution_status"].str.contains(
        "unresolved",
        case=False,
        na=False,
    )
].copy()

print("\nUnresolved dependency calls:", len(unresolved_calls))

# Compare placeholder call IDs with unresolved call IDs.
placeholder_ids = set(placeholder_mappings["call_id"])
unresolved_ids = set(unresolved_calls["call_id"])

print("\nPlaceholder IDs with no unresolved dependency:")
print(sorted(placeholder_ids - unresolved_ids))

print("\nUnresolved dependency IDs without placeholders:")
print(sorted(unresolved_ids - placeholder_ids))

Placeholder records: 3
Dependency calls: 855

Placeholder columns:
['call_id', 'original_target', 'placeholder_name', 'caller', 'operation', 'arguments', 'expected_target_role', 'resolution_status']

Dependency columns:
['call_id', 'operation_id', 'operation', 'constraint_id', 'constraint_name', 'language', 'specification_category', 'callee', 'method', 'arguments', 'call_expression', 'reference_type', 'candidate_classes', 'resolution_status', 'target_class', 'target_class_id', 'source_class', 'dependency_type', 'dependency_status', 'source_owner_type', 'semantic_target_class', 'effective_target_class', 'effective_resolution_status', 'caller_suffix', 'callee_short', 'semantic_alias_rule', 'semantic_alias_evidence', 'semantic_alias_confidence']

Unresolved dependency calls: 24

Placeholder IDs with no unresolved dependency:
[]

Unresolved dependency IDs without placeholders:
['CALL_00022', 'CALL_00239', 'CALL_00249', 'CALL_00285', 'CALL_00286', 'CALL_00290', 'CALL_00291', 'CALL_00330', '

In [210]:
# Identify calls that remain unresolved after semantic alias application.

unresolved_calls = dependency_edges[
    dependency_edges["effective_resolution_status"].eq("unresolved")
].copy()

placeholder_ids = set(placeholder_mappings["call_id"])
unresolved_ids = set(unresolved_calls["call_id"])

print("Original unresolved calls:",
      dependency_edges["resolution_status"].eq("unresolved").sum())

print("Effectively unresolved calls:", len(unresolved_calls))
print("Placeholder records:", len(placeholder_mappings))

print("\nPlaceholder IDs without unresolved dependencies:")
print(sorted(placeholder_ids - unresolved_ids))

print("\nUnresolved dependencies without placeholders:")
print(sorted(unresolved_ids - placeholder_ids))

print("\nRemaining unresolved calls:")
display(
    unresolved_calls[
        [
            "call_id",
            "source_class",
            "callee",
            "call_expression",
            "effective_resolution_status",
        ]
    ].sort_values("call_id")
)

Original unresolved calls: 24
Effectively unresolved calls: 3
Placeholder records: 3

Placeholder IDs without unresolved dependencies:
[]

Unresolved dependencies without placeholders:
[]

Remaining unresolved calls:


,call_id,source_class,callee,call_expression,effective_resolution_status
214,CALL_00214,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping,CommonElementImport_Mapping,CommonElementImport_Mapping.getMapped(e),unresolved
309,CALL_00309,SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping,InstanceSpecificationToGeneralization_Mapping,"InstanceSpecificationToGeneralization_Mapping.getMapped(from, c)",unresolved
380,CALL_00380,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping,DefaultValueOpaqueExpression_Mapping,DefaultValueOpaqueExpression_Mapping.getMapped(from.defaultValue),unresolved


In [211]:
assert placeholder_ids == unresolved_ids, (
    f"Placeholder mismatch: "
    f"missing={unresolved_ids - placeholder_ids}, "
    f"orphaned={placeholder_ids - unresolved_ids}"
)

assert placeholder_mappings["call_id"].is_unique

print("Placeholder validation passed.")

Placeholder validation passed.


In [212]:
# Final dependency-resolution summary

print("Dependency table shape:", dependency_edges.shape)
print("Unique call IDs:", dependency_edges["call_id"].nunique())

print("\nOriginal resolution status:")
display(
    dependency_edges["resolution_status"]
    .value_counts(dropna=False)
)

print("\nEffective resolution status:")
display(
    dependency_edges["effective_resolution_status"]
    .value_counts(dropna=False)
)

print("\nSemantic alias confidence:")
display(
    dependency_edges["semantic_alias_confidence"]
    .value_counts(dropna=False)
)

print("\nEffective target availability:")
display(
    dependency_edges["effective_target_class"]
    .isna()
    .value_counts()
    .rename(index={True: "Missing target", False: "Target assigned"})
)

# Check that every effectively unresolved call has a placeholder.
effective_unresolved_ids = set(
    dependency_edges.loc[
        dependency_edges["effective_resolution_status"].eq("unresolved"),
        "call_id",
    ]
)

placeholder_ids = set(placeholder_mappings["call_id"])

assert effective_unresolved_ids == placeholder_ids
assert dependency_edges["call_id"].is_unique

print("\nFinal dependency validation passed.")

Dependency table shape: (855, 28)
Unique call IDs: 855

Original resolution status:


resolution_status
resolved_suffix    830
unresolved          24
not_applicable       1
Name: count, dtype: int64


Effective resolution status:


effective_resolution_status
resolved_suffix            830
resolved_semantic_alias     21
unresolved                   3
not_applicable               1
Name: count, dtype: int64


Semantic alias confidence:


semantic_alias_confidence
<NA>      834
high       19
medium      2
Name: count, dtype: int64


Effective target availability:


effective_target_class
Target assigned    851
Missing target       4
Name: count, dtype: int64


Final dependency validation passed.


In [217]:
audit_inputs["dependency_edges"] = dependency_edges.copy()
audit_inputs["semantic_alias_rules"] = semantic_alias_rules.copy()
audit_inputs["placeholder_mappings"] = placeholder_mappings.copy()
# Inspect audit input types and sizes.
for name, value in audit_inputs.items():
    if isinstance(value, pd.DataFrame):
        print(f"{name}: DataFrame {value.shape}")
    elif isinstance(value, (list, tuple, set)):
        print(f"{name}: {type(value).__name__}, {len(value)} items")
    else:
        print(f"{name}: {type(value).__name__}")

# Verify the three refreshed entries.
assert audit_inputs["dependency_edges"].equals(dependency_edges)
assert audit_inputs["semantic_alias_rules"].equals(semantic_alias_rules)
assert audit_inputs["placeholder_mappings"].equals(placeholder_mappings)

print("\nAudit inputs refreshed successfully.")

mapping_class_records: list, 786 items
mapping_table: DataFrame (772, 20)
mapping_operation_declarations: DataFrame (17051, 7)
ocl_specifications: DataFrame (1101, 11)
dependency_edges: DataFrame (855, 28)
semantic_alias_rules: DataFrame (20, 6)
placeholder_mappings: DataFrame (3, 8)

Audit inputs refreshed successfully.


In [ ]:
history = get_ipython().history_manager.input_hist_raw

for cell_number, cell in enumerate(history):
    if "get_descendants" in cell or "get_containment_descendants" in cell:
        matching_lines = [
            line.strip()
            for line in cell.splitlines()
            if "get_descendants" in line
            or "get_containment_descendants" in line
        ]

        if matching_lines:
            print(f"\nCell {cell_number}:")
            for line in matching_lines:
                print(" ", line)

In [ ]:
history = get_ipython().history_manager.input_hist_raw

for cell_number in [22, 24, 26, 30]:
    print(f"\n{'=' * 25} CELL {cell_number} {'=' * 25}")
    print(history[cell_number])

In [219]:
# 1. Verify audit inputs are synchronized.
audit_checks = {
    "dependency_edges": audit_inputs["dependency_edges"].equals(dependency_edges),
    "semantic_alias_rules": audit_inputs["semantic_alias_rules"].equals(semantic_alias_rules),
    "placeholder_mappings": audit_inputs["placeholder_mappings"].equals(placeholder_mappings),
}

print("AUDIT INPUT SYNCHRONIZATION")
for name, passed in audit_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")


# 2. Compare unresolved effective dependencies with placeholders.
unresolved_ids = set(
    dependency_edges.loc[
        dependency_edges["effective_resolution_status"] == "unresolved",
        "call_id"
    ]
)

placeholder_ids = set(placeholder_mappings["call_id"])

print("\nUNRESOLVED DEPENDENCY CHECK")
print("Unresolved dependency count:", len(unresolved_ids))
print("Placeholder count:", len(placeholder_ids))
print("Unresolved dependencies without placeholders:",
      sorted(unresolved_ids - placeholder_ids))
print("Placeholders without unresolved dependencies:",
      sorted(placeholder_ids - unresolved_ids))


# 3. Check for duplicate placeholder and dependency identifiers.
print("\nIDENTIFIER CHECK")
print("Duplicate placeholder call IDs:",
      placeholder_mappings["call_id"].duplicated().sum())
print("Duplicate dependency call IDs:",
      dependency_edges["call_id"].duplicated().sum())

AUDIT INPUT SYNCHRONIZATION
dependency_edges: PASS
semantic_alias_rules: PASS
placeholder_mappings: PASS

UNRESOLVED DEPENDENCY CHECK
Unresolved dependency count: 3
Placeholder count: 3
Unresolved dependencies without placeholders: []
Placeholders without unresolved dependencies: []

IDENTIFIER CHECK
Duplicate placeholder call IDs: 0
Duplicate dependency call IDs: 0


In [222]:
print("SEMANTIC ALIAS RULE CHECKS")

# 1. Check duplicate rules and caller/callee combinations.
print(
    "Duplicate semantic alias rules:",
    semantic_alias_rules["semantic_alias_rule"].duplicated().sum()
)

print(
    "Duplicate caller/callee pairs:",
    semantic_alias_rules.duplicated(
        subset=["caller_suffix", "callee"]
    ).sum()
)

# 2. Check required metadata.
required_columns = [
    "caller_suffix",
    "callee",
    "semantic_target_class",
    "semantic_alias_rule",
    "semantic_alias_evidence",
    "semantic_alias_confidence",
]

print("\nMissing values in required fields:")
print(semantic_alias_rules[required_columns].isna().sum())

print("\nEmpty string counts:")
for column in required_columns:
    print(
        f"{column}:",
        semantic_alias_rules[column]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )

# 3. Check confidence categories.
print("\nConfidence distribution:")
print(
    semantic_alias_rules["semantic_alias_confidence"]
    .value_counts(dropna=False)
)

# 4. Check whether every rule is used by dependency edges.
used_rules = set(
    dependency_edges["semantic_alias_rule"].dropna()
)

defined_rules = set(
    semantic_alias_rules["semantic_alias_rule"].dropna()
)

print("\nRule usage:")
print("Defined rules:", len(defined_rules))
print("Used rules:", len(used_rules))
print("Unused rules:", sorted(defined_rules - used_rules))
print("Undefined rules referenced by edges:", sorted(used_rules - defined_rules))

SEMANTIC ALIAS RULE CHECKS
Duplicate semantic alias rules: 7
Duplicate caller/callee pairs: 0

Missing values in required fields:
caller_suffix                0
callee                       0
semantic_target_class        0
semantic_alias_rule          0
semantic_alias_evidence      0
semantic_alias_confidence    0
dtype: int64

Empty string counts:
caller_suffix: 0
callee: 0
semantic_target_class: 0
semantic_alias_rule: 0
semantic_alias_evidence: 0
semantic_alias_confidence: 0

Confidence distribution:
semantic_alias_confidence
high      18
medium     2
Name: count, dtype: int64

Rule usage:
Defined rules: 13
Used rules: 13
Unused rules: []
Undefined rules referenced by edges: []


In [223]:
# Group records by semantic alias rule.
rule_summary = (
    semantic_alias_rules
    .groupby("semantic_alias_rule")
    .agg(
        record_count=("semantic_alias_rule", "size"),
        caller_count=("caller_suffix", "nunique"),
        callee_count=("callee", "nunique"),
        target_count=("semantic_target_class", "nunique"),
        confidence_count=("semantic_alias_confidence", "nunique"),
    )
    .sort_values("record_count", ascending=False)
)

display(rule_summary)

# Identify rules with inconsistent target classes or confidence levels.
inconsistent_rules = rule_summary.loc[
    (rule_summary["target_count"] > 1)
    | (rule_summary["confidence_count"] > 1)
]

print("\nRules with inconsistent targets or confidence:")
display(inconsistent_rules)

,record_count,caller_count,callee_count,target_count,confidence_count
semantic_alias_rule,,,,,
initial_state_subaction_membership,3,3,1,1,1
literal_specification_feature_typing,3,3,1,1,2
property_to_structural_feature_membership,3,3,1,1,1
pin_feature_typing,2,2,1,1,1
association_metadata_feature_typing,1,1,1,1,1
previously_validated_ChangeTriggerConstraintUsage_Mapping,1,1,1,1,1
previously_validated_DefaultValue_Mapping,1,1,1,1,1
previously_validated_LiteralUnlimitedInteger_Mapping,1,1,1,1,1
previously_validated_LiteralUnlimitedUnbounded_Mapping,1,1,1,1,1



Rules with inconsistent targets or confidence:


,record_count,caller_count,callee_count,target_count,confidence_count
semantic_alias_rule,,,,,
literal_specification_feature_typing,3,3,1,1,2


In [224]:
display(
    semantic_alias_rules.loc[
        semantic_alias_rules["semantic_alias_rule"]
        == "literal_specification_feature_typing",
        [
            "caller_suffix",
            "callee",
            "semantic_target_class",
            "semantic_alias_confidence",
            "semantic_alias_evidence",
        ],
    ].reset_index(drop=True)
)

,caller_suffix,callee,semantic_target_class,semantic_alias_confidence,semantic_alias_evidence
0,LiteralSpecificationCommon_Mapping,LiteralSpecificationTyping_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationFeatureTyping_Mapping,high,"Caller maps UML LiteralSpecification to KerML LiteralExpression and invokes LiteralSpecificationTyping_Mapping when the source has a defined type. Candidate maps UML LiteralSpecification to KerML FeatureTyping. It inherits the typed-element filter and type mapping, which maps the source element's UML type to the corresponding KerML typing relationship."
1,TimeExpression_Mapping,LiteralSpecificationTyping_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationFeatureTyping_Mapping,medium,"Caller maps UML TimeExpression to KerML Expression and invokes LiteralSpecificationTyping_Mapping when the source has a defined type. Candidate maps UML LiteralSpecification to KerML FeatureTyping and inherits the typed-element filter and type mapping. The candidate's source type is compatible with the literal-specification typing operation, although the concrete implementation of its inherited typedFeature operation has not been independently verified."
2,ValueSpecification_Mapping,LiteralSpecificationTyping_Mapping,SysMLv1Tov2::Mappings::UML4SysML::Values::LiteralSpecificationFeatureTyping_Mapping,medium,"Caller maps UML ValueSpecification to KerML Expression and invokes LiteralSpecificationTyping_Mapping when the source has a defined type. Candidate maps UML LiteralSpecification to KerML FeatureTyping and inherits the typed-element filter and type mapping, which maps the source element's UML type to the corresponding KerML typing relationship. The concrete implementation of its inherited typedFeature operation has not been independently verified."


# IMPLEMENTATION:

In [225]:
# Existing DataFrame inputs.
for name in [
    "mapping_table",
    "dependency_edges",
    "semantic_alias_rules",
    "placeholder_mappings",
]:
    df = globals()[name]
    print(f"\n{name}: {df.shape}")
    print(df.columns.tolist())

# Registry reference structure.
print("\nRegistry references:")
print("Type:", type(registry.references).__name__)
print("Count:", len(registry.references))

if registry.references:
    first_ref = next(iter(registry.references.values())) \
        if isinstance(registry.references, dict) \
        else registry.references[0]

    print("Record type:", type(first_ref).__name__)
    print("Attributes:", vars(first_ref))


mapping_table: (772, 20)
['mapping_name', 'mapping_xmi_id', 'source_property', 'target_property', 'source_property_xmi_id', 'target_property_xmi_id', 'source_property_declared_by', 'target_property_declared_by', 'source_property_type', 'target_property_type', 'source_type', 'target_type', 'ancestors', 'operation_count', 'operations', 'operation_categories', 'dependency_count', 'resolved_dependency_count', 'unresolved_dependency_count', 'dependencies']

dependency_edges: (855, 28)
['call_id', 'operation_id', 'operation', 'constraint_id', 'constraint_name', 'language', 'specification_category', 'callee', 'method', 'arguments', 'call_expression', 'reference_type', 'candidate_classes', 'resolution_status', 'target_class', 'target_class_id', 'source_class', 'dependency_type', 'dependency_status', 'source_owner_type', 'semantic_target_class', 'effective_target_class', 'effective_resolution_status', 'caller_suffix', 'callee_short', 'semantic_alias_rule', 'semantic_alias_evidence', 'semantic_

In [226]:
# Inspect one ordinary XMI reference.
ref = registry.references[0]

print("XMI REFERENCE")
print(vars(ref))

print("\nSOURCE ELEMENT")
source = registry.elements.get(ref.source)
print(vars(source) if source else "Not found")

print("\nTARGET ELEMENT")
target = registry.elements.get(ref.target) if ref.target else None
print(vars(target) if target else "Not found")


# Inspect representative mapping dependencies.
print("\nMAPPING DEPENDENCY")
display(
    dependency_edges[
        [
            "call_id",
            "operation_id",
            "source_class",
            "target_class",
            "target_class_id",
            "dependency_type",
            "resolution_status",
            "effective_target_class",
            "effective_resolution_status",
        ]
    ].head(5)
)


# Inspect unresolved dependencies.
print("\nUNRESOLVED DEPENDENCIES")
display(
    dependency_edges.loc[
        dependency_edges["effective_resolution_status"] == "unresolved",
        [
            "call_id",
            "source_class",
            "callee",
            "effective_target_class",
            "effective_resolution_status",
        ],
    ]
)

XMI REFERENCE
{'source': ('UML_Metamodel', 'Activity-maximum_one_parameter_node', 1), 'element_tag': 'constrainedElement', 'attribute': 'idref', 'value': 'Activity', 'target': ('UML_Metamodel', 'Activity', 1), 'resolved': True, 'resolution_status': 'resolved_local'}

SOURCE ELEMENT
{'resource': 'UML_Metamodel', 'xmi_id': 'Activity-maximum_one_parameter_node', 'tag': 'ownedRule', 'metaclass': 'uml:Constraint', 'name': 'maximum_one_parameter_node', 'attributes': {'{http://www.omg.org/spec/XMI/20131001}type': 'uml:Constraint', '{http://www.omg.org/spec/XMI/20131001}id': 'Activity-maximum_one_parameter_node', 'name': 'maximum_one_parameter_node'}, 'parent': ('UML_Metamodel', 'Activity', 1), 'children': [('UML_Metamodel', 'Activity-maximum_one_parameter_node-_specification', 1)], 'element_category': None, 'qualified_name': 'UML::Activities::Activity::maximum_one_parameter_node', 'namespace': 'UML::Activities::Activity', 'registry_key': ('UML_Metamodel', 'Activity-maximum_one_parameter_node'

,call_id,operation_id,source_class,target_class,target_class_id,dependency_type,resolution_status,effective_target_class,effective_resolution_status
0,CALL_00000,"(SysML_v1_to_v2, Mappings-Foundations-Mapping-getMappedColl_Element, 1)",SysMLv1Tov2::Mappings::Foundations::Mapping,None,None,self_dispatch,not_applicable,<NA>,not_applicable
1,CALL_00001,"(SysML_v1_to_v2, Mappings-Factories-InformationFlowReferenceSubsetting_Factory-referencedFeature_, 1)",SysMLv1Tov2::Mappings::Factories::InformationFlowReferenceSubsetting_Factory,SysMLv1Tov2::Mappings::UML4SysML::InformationFlows::InformationFlowEnd_Mapping,Mappings-UML4SysML-InformationFlows-InformationFlowEnd_Mapping,mapping_dependency,resolved_suffix,SysMLv1Tov2::Mappings::UML4SysML::InformationFlows::InformationFlowEnd_Mapping,resolved_suffix
2,CALL_00002,"(SysML_v1_to_v2, Mappings-CommonMappings-CommonReturnParameterReferenceUsageMembership_Mapping-ownedMemberParameter_, 1)",SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageMembership_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageUntyped_Mapping,Mappings-CommonMappings-CommonReturnParameterReferenceUsageUntyped_Mapping,mapping_dependency,resolved_suffix,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageUntyped_Mapping,resolved_suffix
3,CALL_00003,"(SysML_v1_to_v2, Mappings-CommonMappings-CommonReturnParameterReferenceUsageMembership_Mapping-ownedMemberParameter_, 1)",SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageMembership_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageUntyped_Mapping,Mappings-CommonMappings-CommonReturnParameterReferenceUsageUntyped_Mapping,mapping_dependency,resolved_suffix,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageUntyped_Mapping,resolved_suffix
4,CALL_00004,"(SysML_v1_to_v2, Mappings-CommonMappings-CommonReturnParameterReferenceUsageMembership_Mapping-ownedMemberParameter_, 1)",SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageMembership_Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageUntyped_Mapping,Mappings-CommonMappings-CommonReturnParameterReferenceUsageUntyped_Mapping,mapping_dependency,resolved_suffix,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageUntyped_Mapping,resolved_suffix



UNRESOLVED DEPENDENCIES


,call_id,source_class,callee,effective_target_class,effective_resolution_status
214,CALL_00214,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping,CommonElementImport_Mapping,<NA>,unresolved
309,CALL_00309,SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping,InstanceSpecificationToGeneralization_Mapping,<NA>,unresolved
380,CALL_00380,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping,DefaultValueOpaqueExpression_Mapping,<NA>,unresolved


In [227]:
# Inspect a resolved mapping dependency.
row = dependency_edges.loc[
    dependency_edges["effective_resolution_status"] == "resolved_suffix"
].iloc[0]

print("RESOLVED DEPENDENCY")
print("Source class:", row["source_class"])
print("Target class:", row["effective_target_class"])
print("Target XMI ID:", row["target_class_id"])

print("\nSource registry matches:")
for element in registry.get_elements_by_xmi_id(
    "SysML_v1_to_v2",
    row["operation_id"][1].split("-")[-1]
):
    print(element.qualified_name)

print("\nTarget registry matches:")
for element in registry.get_elements_by_xmi_id(
    "SysML_v1_to_v2",
    row["target_class_id"]
):
    print(element.qualified_name)


# Inspect unresolved dependency records.
print("\nUNRESOLVED DEPENDENCIES")
display(
    dependency_edges.loc[
        dependency_edges["effective_resolution_status"] == "unresolved",
        [
            "call_id",
            "source_class",
            "callee",
            "effective_target_class",
            "semantic_target_class",
        ],
    ]
)

RESOLVED DEPENDENCY
Source class: SysMLv1Tov2::Mappings::Factories::InformationFlowReferenceSubsetting_Factory
Target class: SysMLv1Tov2::Mappings::UML4SysML::InformationFlows::InformationFlowEnd_Mapping
Target XMI ID: Mappings-UML4SysML-InformationFlows-InformationFlowEnd_Mapping

Source registry matches:

Target registry matches:
SysMLv1Tov2::Mappings::UML4SysML::InformationFlows::InformationFlowEnd_Mapping

UNRESOLVED DEPENDENCIES


,call_id,source_class,callee,effective_target_class,semantic_target_class
214,CALL_00214,SysMLv1Tov2::Mappings::SysMLv1::ModelElements::ElementGroup_Mapping,CommonElementImport_Mapping,<NA>,<NA>
309,CALL_00309,SysMLv1Tov2::Mappings::UML4SysML::Classification::InstanceSpecification_Mapping,InstanceSpecificationToGeneralization_Mapping,<NA>,<NA>
380,CALL_00380,SysMLv1Tov2::Mappings::UML4SysML::StructuredClassifiers::OwnedEnd_Mapping,DefaultValueOpaqueExpression_Mapping,<NA>,<NA>


In [228]:
# Build a qualified-name index for the SysML v1-to-v2 resource.
mapping_elements = registry.get_elements("SysML_v1_to_v2")

elements_by_qname = {
    element.qualified_name: element
    for element in mapping_elements
    if element.qualified_name is not None
}

# Check source and effective target lookups.
dependency_endpoint_checks = []

for _, row in dependency_edges.iterrows():
    source = elements_by_qname.get(row["source_class"])

    target_name = row["effective_target_class"]
    target = elements_by_qname.get(target_name) if pd.notna(target_name) else None

    dependency_endpoint_checks.append({
        "call_id": row["call_id"],
        "source_found": source is not None,
        "target_found": target is not None,
        "effective_resolution_status": row["effective_resolution_status"],
        "source_xmi_id": source.xmi_id if source else None,
        "target_xmi_id": target.xmi_id if target else None,
    })

endpoint_check_df = pd.DataFrame(dependency_endpoint_checks)

print("SOURCE LOOKUP")
print(endpoint_check_df["source_found"].value_counts(dropna=False))

print("\nTARGET LOOKUP")
print(endpoint_check_df.groupby(
    "effective_resolution_status"
)["target_found"].value_counts(dropna=False))

print("\nMissing source elements:")
display(endpoint_check_df.loc[~endpoint_check_df["source_found"]].head(10))

print("\nMissing targets for resolved dependencies:")
display(
    endpoint_check_df.loc[
        endpoint_check_df["effective_resolution_status"].isin(
            ["resolved_suffix", "resolved_semantic_alias"]
        )
        & ~endpoint_check_df["target_found"]
    ]
)

SOURCE LOOKUP
source_found
True    855
Name: count, dtype: int64

TARGET LOOKUP
effective_resolution_status  target_found
not_applicable               False             1
resolved_semantic_alias      True             21
resolved_suffix              True            830
unresolved                   False             3
Name: count, dtype: int64

Missing source elements:


,call_id,source_found,target_found,effective_resolution_status,source_xmi_id,target_xmi_id



Missing targets for resolved dependencies:


,call_id,source_found,target_found,effective_resolution_status,source_xmi_id,target_xmi_id


In [229]:
# Index placeholders by dependency call ID.
placeholders_by_call = (
    placeholder_mappings.set_index("call_id").to_dict("index")
)

dependency_records = []

for _, row in dependency_edges.iterrows():

    source = elements_by_qname[row["source_class"]]

    status = row["effective_resolution_status"]
    target_name = row["effective_target_class"]

    target = (elements_by_qname.get(target_name) if pd.notna(target_name) else None)

    placeholder = placeholders_by_call.get(row["call_id"], {})

    # Preserve the placeholder name when no target element is resolved.
    if target is not None:
        final_target_name = target.qualified_name
    else:
        final_target_name = placeholder.get("placeholder_name")

    dependency_records.append({
        # Unified relationship identity
        "relationship_id": row["call_id"],
        "relationship_type": row["dependency_type"],

        # Source endpoint
        "source_resource": source.resource,
        "source_element_id": source.xmi_id,
        "source_element": source.qualified_name,

        # Target endpoint
        "target_resource": target.resource if target else None,
        "target_element_id": target.xmi_id if target else None,
        "target_element": final_target_name,

        # Resolution information
        "resolution_method": status,
        "resolution_status": row["resolution_status"],
        "effective_resolution_status": status,

        # Provenance
        "provenance": row["call_id"],
        "operation_id": row["operation_id"],
        "operation": row["operation"],
        "constraint_id": row["constraint_id"],
        "constraint_name": row["constraint_name"],
        "call_expression": row["call_expression"],

        # Semantic alias evidence
        "semantic_alias_rule": row["semantic_alias_rule"],
        "evidence": row["semantic_alias_evidence"],
        "confidence": row["semantic_alias_confidence"],

        # Unresolved placeholder metadata
        "expected_target_role": placeholder.get("expected_target_role"),
        "original_target": placeholder.get("original_target"),
    })


dependency_relationships = pd.DataFrame(dependency_records)

print("Dependency relationship table:", dependency_relationships.shape)
print("\nResolution distribution:")
print(
    dependency_relationships["effective_resolution_status"]
    .value_counts(dropna=False)
)

display(dependency_relationships.head())

Dependency relationship table: (855, 22)

Resolution distribution:
effective_resolution_status
resolved_suffix            830
resolved_semantic_alias     21
unresolved                   3
not_applicable               1
Name: count, dtype: int64


,relationship_id,relationship_type,source_resource,source_element_id,source_element,target_resource,target_element_id,target_element,resolution_method,resolution_status,effective_resolution_status,provenance,operation_id,operation,constraint_id,constraint_name,call_expression,semantic_alias_rule,evidence,confidence,expected_target_role,original_target
0,CALL_00000,self_dispatch,SysML_v1_to_v2,Mappings-Foundations-Mapping-_generalization.Mappings-Foundations-Initializer,SysMLv1Tov2::Mappings::Foundations::Mapping,None,None,None,not_applicable,not_applicable,not_applicable,CALL_00000,"(SysML_v1_to_v2, Mappings-Foundations-Mapping-getMappedColl_Element, 1)",SysMLv1Tov2::Mappings::Foundations::Mapping::getMappedColl,"(SysML_v1_to_v2, Mappings-Foundations-Mapping-getMappedColl_Element-postcondition, 1)",postcondition,self.getMapped(e),<NA>,<NA>,<NA>,None,None
1,CALL_00001,mapping_dependency,SysML_v1_to_v2,Mappings-Factories-InformationFlowReferenceSubsetting_Factory-_generalization.Mappings-Initializers-KerMLInitializers-ToReferenceSubsetting_Init,SysMLv1Tov2::Mappings::Factories::InformationFlowReferenceSubsetting_Factory,SysML_v1_to_v2,Mappings-UML4SysML-InformationFlows-InformationFlowEnd_Mapping-_generalization.Mappings-Initializers-KerMLInitializers-ToFeature_Init,SysMLv1Tov2::Mappings::UML4SysML::InformationFlows::InformationFlowEnd_Mapping,resolved_suffix,resolved_suffix,resolved_suffix,CALL_00001,"(SysML_v1_to_v2, Mappings-Factories-InformationFlowReferenceSubsetting_Factory-referencedFeature_, 1)",SysMLv1Tov2::Mappings::Factories::InformationFlowReferenceSubsetting_Factory::referencedFeature,"(SysML_v1_to_v2, Mappings-Factories-InformationFlowReferenceSubsetting_Factory-referencedFeature_-bodyCondition, 1)",bodyCondition,"InformationFlowEnd_Mapping.getMapped(informationFlow, end)",<NA>,<NA>,<NA>,None,None
2,CALL_00002,mapping_dependency,SysML_v1_to_v2,Mappings-CommonMappings-CommonReturnParameterReferenceUsageMembership_Mapping-_generalization.Mappings-Foundations-Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageMembership_Mapping,SysML_v1_to_v2,Mappings-CommonMappings-CommonReturnParameterReferenceUsageUntyped_Mapping-_generalization.Mappings-Initializers-SystemInitializers-ToReferenceUsage_Init,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageUntyped_Mapping,resolved_suffix,resolved_suffix,resolved_suffix,CALL_00002,"(SysML_v1_to_v2, Mappings-CommonMappings-CommonReturnParameterReferenceUsageMembership_Mapping-ownedMemberParameter_, 1)",SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageMembership_Mapping::ownedMemberParameter,"(SysML_v1_to_v2, Mappings-CommonMappings-CommonReturnParameterReferenceUsageMembership_Mapping-ownedMemberParameter_-bodyCondition, 1)",bodyCondition,CommonReturnParameterReferenceUsageUntyped_Mapping.getMapped(from),<NA>,<NA>,<NA>,None,None
3,CALL_00003,mapping_dependency,SysML_v1_to_v2,Mappings-CommonMappings-CommonReturnParameterReferenceUsageMembership_Mapping-_generalization.Mappings-Foundations-Mapping,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageMembership_Mapping,SysML_v1_to_v2,Mappings-CommonMappings-CommonReturnParameterReferenceUsageUntyped_Mapping-_generalization.Mappings-Initializers-SystemInitializers-ToReferenceUsage_Init,SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageUntyped_Mapping,resolved_suffix,resolved_suffix,resolved_suffix,CALL_00003,"(SysML_v1_to_v2, Mappings-CommonMappings-CommonReturnParameterReferenceUsageMembership_Mapping-ownedMemberParameter_, 1)",SysMLv1Tov2::Mappings::CommonMappings::CommonReturnParameterReferenceUsageMembership_Mapping::ownedMemberParameter,"(SysML_v1_to_v2, Mappings-CommonMappings-CommonReturnParameterReferenceUsageMembership_Mapping-ownedMemberParameter_-bodyCondition, 1)",bodyCondition,CommonReturnParameterReferenceUsageUntyped_Mapping.getMapped(from),<NA>,<NA>,<NA>,None,None
4,CALL_00004,mapping_dependency,Sys

In [230]:
xmi_records = []

for i, ref in enumerate(registry.references):

    source = registry.elements.get(ref.source)
    target = registry.elements.get(ref.target) if ref.target else None

    xmi_records.append({
        # Unified identity and relationship
        "relationship_id": f"XMIREF_{i:05d}",
        "relationship_type": ref.element_tag,

        # Source endpoint
        "source_resource": source.resource if source else ref.source[0],
        "source_element_id": source.xmi_id if source else ref.source[1],
        "source_element": source.qualified_name if source else None,

        # Target endpoint
        "target_resource": target.resource if target else None,
        "target_element_id": target.xmi_id if target else None,
        "target_element": target.qualified_name if target else None,

        # Resolution
        "resolution_method": ref.attribute,
        "resolution_status": ref.resolution_status,
        "effective_resolution_status": ref.resolution_status,

        # Provenance
        "provenance": f"{ref.source}|{ref.element_tag}|{ref.attribute}",
        "operation_id": None,
        "operation": None,
        "constraint_id": None,
        "constraint_name": None,
        "call_expression": None,

        # Semantic alias metadata
        "semantic_alias_rule": None,
        "evidence": None,
        "confidence": None,

        # Placeholder metadata
        "expected_target_role": None,
        "original_target": None,

        # XMI-specific reference details
        "reference_element_tag": ref.element_tag,
        "reference_attribute": ref.attribute,
        "reference_value": ref.value,
    })

xmi_relationships = pd.DataFrame(xmi_records)

print("XMI relationship table:", xmi_relationships.shape)

print("\nResolution distribution:")
print(
    xmi_relationships["effective_resolution_status"]
    .value_counts(dropna=False)
)

print("\nMissing endpoint counts:")
print(
    xmi_relationships[
        ["source_element", "target_element"]
    ].isna().sum()
)

display(xmi_relationships.head())

XMI relationship table: (18918, 25)

Resolution distribution:
effective_resolution_status
resolved_local                    15796
missing_external_target            1732
resolved_external                  1212
external_resource_unregistered      172
ambiguous_target                      6
Name: count, dtype: int64

Missing endpoint counts:
source_element       0
target_element    1910
dtype: int64


,relationship_id,relationship_type,source_resource,source_element_id,source_element,target_resource,target_element_id,target_element,resolution_method,resolution_status,effective_resolution_status,provenance,operation_id,operation,constraint_id,constraint_name,call_expression,semantic_alias_rule,evidence,confidence,expected_target_role,original_target,reference_element_tag,reference_attribute,reference_value
0,XMIREF_00000,constrainedElement,UML_Metamodel,Activity-maximum_one_parameter_node,UML::Activities::Activity::maximum_one_parameter_node,UML_Metamodel,Activity,UML::Activities::Activity,idref,resolved_local,resolved_local,"('UML_Metamodel', 'Activity-maximum_one_parameter_node', 1)|constrainedElement|idref",None,None,None,None,None,None,None,None,None,None,constrainedElement,idref,Activity
1,XMIREF_00001,constrainedElement,UML_Metamodel,Activity-maximum_two_parameter_nodes,UML::Activities::Activity::maximum_two_parameter_nodes,UML_Metamodel,Activity,UML::Activities::Activity,idref,resolved_local,resolved_local,"('UML_Metamodel', 'Activity-maximum_two_parameter_nodes', 1)|constrainedElement|idref",None,None,None,None,None,None,None,None,None,None,constrainedElement,idref,Activity
2,XMIREF_00002,subsettedProperty,UML_Metamodel,Activity-edge,UML::Activities::Activity::edge,UML_Metamodel,Element-ownedElement,UML::CommonStructure::Element::ownedElement,idref,resolved_local,resolved_local,"('UML_Metamodel', 'Activity-edge', 1)|subsettedProperty|idref",None,None,None,None,None,None,None,None,None,None,subsettedProperty,idref,Element-ownedElement
3,XMIREF_00003,subsettedProperty,UML_Metamodel,Activity-group,UML::Activities::Activity::group,UML_Metamodel,Element-ownedElement,UML::CommonStructure::Element::ownedElement,idref,resolved_local,resolved_local,"('UML_Metamodel', 'Activity-group', 1)|subsettedProperty|idref",None,None,None,None,None,None,None,None,None,None,subsettedProperty,idref,Element-ownedElement
4,XMIREF_00004,type,UML_Metamodel,Activity-isReadOnly,UML::Activities::Activity::isReadOnly,None,None,None,href,missing_external_target,missing_external_target,"('UML_Metamodel', 'Activity-isReadOnly', 1)|type|href",None,None,None,None,None,None,None,None,None,None,type,href,http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi#Boolean


In [231]:
# Combine XMI references and mapping dependencies.
relationships = pd.concat(
    [xmi_relationships, dependency_relationships],
    ignore_index=True,
    sort=False,
)

# Validate unified relationship identity.
print("Unified relationship table:", relationships.shape)

print(
    "Unique relationship IDs:",
    relationships["relationship_id"].nunique()
)

print(
    "Duplicate relationship IDs:",
    relationships["relationship_id"].duplicated().sum()
)

print("\nRelationship type distribution:")
print(relationships["relationship_type"].value_counts(dropna=False))

print("\nResolution distribution:")
print(
    relationships["effective_resolution_status"]
    .value_counts(dropna=False)
)

print("\nMissing endpoints:")
print(
    relationships[
        ["source_element", "target_element"]
    ].isna().sum()
)

display(relationships.head())

Unified relationship table: (19773, 25)
Unique relationship IDs: 19773
Duplicate relationship IDs: 0

Relationship type distribution:
relationship_type
type                  4701
memberEnd             3558
association           2722
general               1834
constrainedElement    1334
redefinedProperty     1259
ownedRule             1213
subsettedProperty     1103
redefinedOperation    1073
mapping_dependency     854
navigableOwnedEnd       53
precondition            23
importedPackage         22
instance                 7
base_DataType            7
base_Package             4
appliedProfile           2
base_Class               2
packageImport            1
self_dispatch            1
Name: count, dtype: int64

Resolution distribution:
effective_resolution_status
resolved_local                    15796
missing_external_target            1732
resolved_external                  1212
resolved_suffix                     830
external_resource_unregistered      172
resolved_semantic_alias     

,relationship_id,relationship_type,source_resource,source_element_id,source_element,target_resource,target_element_id,target_element,resolution_method,resolution_status,effective_resolution_status,provenance,operation_id,operation,constraint_id,constraint_name,call_expression,semantic_alias_rule,evidence,confidence,expected_target_role,original_target,reference_element_tag,reference_attribute,reference_value
0,XMIREF_00000,constrainedElement,UML_Metamodel,Activity-maximum_one_parameter_node,UML::Activities::Activity::maximum_one_parameter_node,UML_Metamodel,Activity,UML::Activities::Activity,idref,resolved_local,resolved_local,"('UML_Metamodel', 'Activity-maximum_one_parameter_node', 1)|constrainedElement|idref",None,None,None,None,None,None,None,None,None,None,constrainedElement,idref,Activity
1,XMIREF_00001,constrainedElement,UML_Metamodel,Activity-maximum_two_parameter_nodes,UML::Activities::Activity::maximum_two_parameter_nodes,UML_Metamodel,Activity,UML::Activities::Activity,idref,resolved_local,resolved_local,"('UML_Metamodel', 'Activity-maximum_two_parameter_nodes', 1)|constrainedElement|idref",None,None,None,None,None,None,None,None,None,None,constrainedElement,idref,Activity
2,XMIREF_00002,subsettedProperty,UML_Metamodel,Activity-edge,UML::Activities::Activity::edge,UML_Metamodel,Element-ownedElement,UML::CommonStructure::Element::ownedElement,idref,resolved_local,resolved_local,"('UML_Metamodel', 'Activity-edge', 1)|subsettedProperty|idref",None,None,None,None,None,None,None,None,None,None,subsettedProperty,idref,Element-ownedElement
3,XMIREF_00003,subsettedProperty,UML_Metamodel,Activity-group,UML::Activities::Activity::group,UML_Metamodel,Element-ownedElement,UML::CommonStructure::Element::ownedElement,idref,resolved_local,resolved_local,"('UML_Metamodel', 'Activity-group', 1)|subsettedProperty|idref",None,None,None,None,None,None,None,None,None,None,subsettedProperty,idref,Element-ownedElement
4,XMIREF_00004,type,UML_Metamodel,Activity-isReadOnly,UML::Activities::Activity::isReadOnly,None,None,None,href,missing_external_target,missing_external_target,"('UML_Metamodel', 'Activity-isReadOnly', 1)|type|href",None,None,None,None,None,None,None,None,None,None,type,href,http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi#Boolean


In [232]:
# Summarize missing target endpoints by relationship type and resolution.
missing_target_summary = (
    relationships.loc[relationships["target_element"].isna()]
    .groupby(
        ["relationship_type", "effective_resolution_status"],
        dropna=False,
    )
    .size()
    .rename("count")
    .reset_index()
    .sort_values("count", ascending=False)
)

display(missing_target_summary)

# Validate source endpoints and expected missing targets.
assert relationships["source_element"].notna().all()

assert (
    relationships["target_element"].isna().sum()
    == 1911
)

# The only mapping relationship without a target should be self-dispatch.
missing_mapping_targets = relationships.loc[
    relationships["relationship_id"].str.startswith("CALL_")
    & relationships["target_element"].isna()
]

assert missing_mapping_targets["relationship_type"].tolist() == [
    "self_dispatch"
]

print("Endpoint validation passed.")

,relationship_type,effective_resolution_status,count
12,type,missing_external_target,1719
11,type,external_resource_unregistered,150
7,redefinedProperty,external_resource_unregistered,11
2,general,external_resource_unregistered,5
10,subsettedProperty,external_resource_unregistered,5
1,constrainedElement,ambiguous_target,4
3,general,missing_external_target,4
5,importedPackage,missing_external_target,4
8,redefinedProperty,missing_external_target,4
6,precondition,ambiguous_target,2


Endpoint validation passed.


In [233]:
elements = list(registry.elements.values())

elements_df = pd.DataFrame([
    {
        "resource": e.resource,
        "xmi_id": e.xmi_id,
        "tag": e.tag,
        "metaclass": e.metaclass,
        "name": e.name,
        "qualified_name": e.qualified_name,
        "namespace": e.namespace,
        "element_category": e.element_category,
    }
    for e in elements
])

print("Element inventory:", elements_df.shape)

print("\nElements by resource:")
print(elements_df["resource"].value_counts())

print("\nElements by category:")
print(elements_df["element_category"].value_counts(dropna=False))

print("\nMissing identifiers and qualified names:")
print(
    elements_df[
        ["resource", "xmi_id", "qualified_name"]
    ].isna().sum()
)

display(elements_df.head())

Element inventory: (25837, 8)

Elements by resource:
resource
SysML_v1_to_v2            14689
UML_Metamodel              5201
SysML_v2                   2463
KerML                      2294
SysML_v1                    682
UML_DiagramInterchange      319
UML_StandardProfile         177
UML_PrimitiveTypes           12
Name: count, dtype: int64

Elements by category:
element_category
None    25837
Name: count, dtype: int64

Missing identifiers and qualified names:
resource          0
xmi_id            0
qualified_name    0
dtype: int64


,resource,xmi_id,tag,metaclass,name,qualified_name,namespace,element_category
0,UML_Metamodel,_0,Package,uml:Package,UML,UML,None,None
1,UML_Metamodel,_packageImport.0,packageImport,uml:PackageImport,None,UML,UML,None
2,UML_Metamodel,_packageImport.1,packageImport,uml:PackageImport,None,UML,UML,None
3,UML_Metamodel,_packageImport.2,packageImport,uml:PackageImport,None,UML,UML,None
4,UML_Metamodel,_packageImport.3,packageImport,uml:PackageImport,None,UML,UML,None


In [234]:
# Check stable element identity.
print(
    "Duplicate (resource, xmi_id) pairs:",
    elements_df.duplicated(["resource", "xmi_id"]).sum()
)

print(
    "Duplicate qualified names:",
    elements_df.duplicated(["qualified_name"]).sum()
)

# Qualified names shared by multiple elements.
qname_collisions = (
    elements_df.groupby("qualified_name")
    .agg(
        element_count=("xmi_id", "size"),
        resources=("resource", lambda x: sorted(x.unique())),
        xmi_ids=("xmi_id", list),
    )
    .query("element_count > 1")
    .sort_values("element_count", ascending=False)
    .reset_index()
)

print("Qualified names shared by multiple elements:", len(qname_collisions))
display(qname_collisions.head(10))

Duplicate (resource, xmi_id) pairs: 7
Duplicate qualified names: 13667
Qualified names shared by multiple elements: 9221


,qualified_name,element_count,resources,xmi_ids
0,UML,16,[UML_Metamodel],"[_0, _packageImport.0, _packageImport.1, _packageImport.2, _packageImport.3, _packageImport.4, _packageImport.5, _packageImport.6, _packageImport.7, _packageImport.8, _packageImport.9, _packageImport.10, _packageImport.11, _packageImport.12, _packageImport.13, _packageImport.14]"
1,,13,[SysML_v1],"[StandardProfile.ModelLibrary_appliedOn_SysML.Libraries.UnitAndQuantityKind, SysML.ValueType_appliedOn_SysML_dataType.String, SysML.ValueType_appliedOn_SysML_dataType.Boolean, SysML.ValueType_appliedOn_SysML_dataType.Real, SysML.ValueType_appliedOn_SysML_dataType.ControlValue, SysML.ValueType_appliedOn_SysML_dataType.Number, StandardProfile.ModelLibrary_appliedOn_SysML.Libraries, SysML.Block_appliedOn_SysML.QuantityKind, StandardProfile.ModelLibrary_appliedOn_SysML.Libraries.PrimitiveValueTypes, SysML.ValueType_appliedOn_SysML_dataType.Integer, StandardProfile.ModelLibrary_appliedOn_SysML.Libraries.ControlValues, SysML.ValueType_appliedOn_SysML_dataType.Complex, SysML.Block_appliedOn_SysML.Unit]"
2,SysMLv1Tov2::Mappings,11,[SysML_v1_to_v2],"[Mappings, Mappings-_packageImport.Mappings-UML4SysML-Actions, Mappings-_packageImport.Mappings-UML4SysML-StructuredClassifiers, Mappings-_packageImport.Mappings-UML4SysML-Values, Mappings-_packageImport.Mappings-UML4SysML-CommonBehavior, Mappings-_packageImport.Mappings-UML4SysML-Packages, Mappings-_packageImport.Mappings-SysMLv1-Blocks, Mappings-_packageImport.Mappings-UML4SysML-Classification, Mappings-_packageImport.Mappings-UML4SysML-CommonStructure, Mappings-_packageImport.Mappings-UML4SysML-Activities, Mappings-_packageImport.Mappings-UML4SysML-SimpleClassifiers]"
3,org.omg.xmi.nsPrefix,8,"[KerML, SysML_v1, SysML_v1_to_v2, SysML_v2, UML_DiagramInterchange, UML_Metamodel, UML_PrimitiveTypes, UML_StandardProfile]","[UML_mofTag0, _1, _1, _12, SysML_mofTag0, KerML_mofTag0, SysML_mofTag0, SysMLv1Tov2_mofTag0]"
4,UML::InformationFlows,8,[UML_Metamodel],"[InformationFlows, InformationFlows-_packageImport.0, InformationFlows-_packageImport.1, InformationFlows-_packageImport.2, InformationFlows-_packageImport.3, InformationFlows-_packageImport.4, InformationFlows-_packageImport.5, InformationFlows-_packageImport.6]"
5,KerML::Kernel::Expressions::InstantiationExpression::instantiatedType,7,[KerML],"[Kernel-Expressions-InstantiationExpression-instantiatedType, Kernel-Expressions-InstantiationExpression-instantiatedType-lowerValue, Kernel-Expressions-InstantiationExpression-instantiatedType-upperValue, Kernel-Expressions-InstantiationExpression-instantiatedType_, Kernel-Expressions-InstantiationExpression-instantiatedType_-, Kernel-Expressions-InstantiationExpression-instantiatedType_--lowerValue, Kernel-Expressions-InstantiationExpression-instantiatedType_--upperValue]"
6,UML::Actions,6,[UML_Metamodel],"[Actions, Actions-_packageImport.0, Actions-_packageImport.1, Actions-_packageImport.2, Actions-_packageImport.3, Actions-_packageImport.4]"
7,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped::result,6,[SysML_v1_to_v2],"[Mappings-Foundations-Mapping-getMapped_Element-result, Mappings-Foundations-Mapping-getMapped_Element-result-lowerValue, Mappings-Foundations-Mapping-getMapped_Element-result-upperValue, Mappings-Foundations-Mapping-getMapped_Element_Element-result, Mappings-Foundations-Mapping-getMapped_Element_Element-result-lowerValue, Mappings-Foundations-Mapping-getMapped_Element_Element-result-upperValue]"
8,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped::fromVar,6,[SysML_v1_to_v2],"[Mappings-Foundations-Mapping-getMapped_Element-fromVar, Mappings-Foundations-Mapping-getMapped_Element-fromVar-lowerValue, Mappings-Foundations-Mapping-getMapped_Element-fromVar-upperValue, Mappings-Foundations-Mapping-getMapped_Element_Element-fromVar, Mappings-Foundations-Mapping-getMapped_Element_Element-fromVar-lowerValue, Mappings-Foundations-Mapping-getMapped_Element_Element-fromVar-upperValue]"
9,SysML,6,"[SysML_v1, SysML_v

In [ ]:
# Inspect elements sharing the same resource and XMI ID.
duplicate_element_ids = elements_df.loc[
    elements_df.duplicated(["resource", "xmi_id"], keep=False)
].sort_values(["resource", "xmi_id", "qualified_name"])

display(duplicate_element_ids)

# Confirm occurrence-aware registry keys distinguish every element.
registry_keys = list(registry.elements.keys())

print("Registry elements:", len(registry_keys))
print("Unique registry keys:", len(set(registry_keys)))


,resource,xmi_id,tag,metaclass,name,qualified_name,namespace,element_category
1424,UML_Metamodel,State-isConsistentWith,ownedOperation,uml:Operation,isConsistentWith,UML::StateMachines::State::isConsistentWith,UML::StateMachines::State,None
1589,UML_Metamodel,State-isConsistentWith,ownedOperation,uml:Operation,isConsistentWith,UML::StateMachines::Vertex::isConsistentWith,UML::StateMachines::Vertex,None
1427,UML_Metamodel,State-isConsistentWith-pre,ownedRule,uml:Constraint,pre,UML::StateMachines::State::isConsistentWith::pre,UML::StateMachines::State::isConsistentWith,None
1590,UML_Metamodel,State-isConsistentWith-pre,ownedRule,uml:Constraint,pre,UML::StateMachines::Vertex::isConsistentWith::pre,UML::StateMachines::Vertex::isConsistentWith,None
1428,UML_Metamodel,State-isConsistentWith-pre-_specification,specification,uml:OpaqueExpression,None,UML::StateMachines::State::isConsistentWith::pre,UML::StateMachines::State::isConsistentWith::pre,None
1591,UML_Metamodel,State-isConsistentWith-pre-_specification,specification,uml:OpaqueExpression,None,UML::StateMachines::Vertex::isConsistentWith::pre,UML::StateMachines::Vertex::isConsistentWith::pre,None
1430,UML_Metamodel,State-isConsistentWith-redefiningElement,ownedParameter,uml:Parameter,redefiningElement,UML::StateMachines::State::isConsistentWith::redefiningElement,UML::StateMachines::State::isConsistentWith,None
1595,UML_Metamodel,State-isConsistentWith-redefiningElement,ownedParameter,uml:Parameter,redefiningElement,UML::StateMachines::Vertex::isConsistentWith::redefiningElement,UML::StateMachines::Vertex::isConsistentWith,None
1429,UML_Metamodel,State-isConsistentWith-result,ownedParameter,uml:Parameter,result,UML::StateMachines::State::isConsistentWith::result,UML::StateMachines::State::isConsistentWith,None
1594,UML_Metamodel,State-isConsistentWith-result,ownedParameter,uml:Parameter,result,UML::StateMachines::Vertex::isConsistentWith::result,UML::StateMachines::Vertex::isConsistentWith,None


Registry elements: 25837
Unique registry keys: 25837


KeyboardInterrupt: 

In [236]:
elements_df = pd.DataFrame([
    {
        "element_id": key,
        "resource": element.resource,
        "xmi_id": element.xmi_id,
        "occurrence": key[2],
        "tag": element.tag,
        "metaclass": element.metaclass,
        "name": element.name,
        "qualified_name": element.qualified_name,
        "namespace": element.namespace,
    }
    for key, element in registry.elements.items()
])

print("Element inventory:", elements_df.shape)
print("Unique element IDs:", elements_df["element_id"].nunique())
print("Missing element IDs:", elements_df["element_id"].isna().sum())

display(elements_df.head())

Element inventory: (25837, 9)
Unique element IDs: 25837
Missing element IDs: 0


,element_id,resource,xmi_id,occurrence,tag,metaclass,name,qualified_name,namespace
0,"(UML_Metamodel, _0, 1)",UML_Metamodel,_0,1,Package,uml:Package,UML,UML,None
1,"(UML_Metamodel, _packageImport.0, 1)",UML_Metamodel,_packageImport.0,1,packageImport,uml:PackageImport,None,UML,UML
2,"(UML_Metamodel, _packageImport.1, 1)",UML_Metamodel,_packageImport.1,1,packageImport,uml:PackageImport,None,UML,UML
3,"(UML_Metamodel, _packageImport.2, 1)",UML_Metamodel,_packageImport.2,1,packageImport,uml:PackageImport,None,UML,UML
4,"(UML_Metamodel, _packageImport.3, 1)",UML_Metamodel,_packageImport.3,1,packageImport,uml:PackageImport,None,UML,UML


In [237]:
# Build exact endpoint lookup for XMI references.
xmi_endpoint_lookup = {
    f"XMIREF_{i:05d}": {
        "source_element_id": ref.source,
        "target_element_id": ref.target,
    }
    for i, ref in enumerate(registry.references)
}

# Add exact source and target IDs for XMI relationships.
relationships["source_element_id"] = relationships["relationship_id"].map(
    lambda rid: xmi_endpoint_lookup.get(rid, {}).get("source_element_id")
)

relationships["target_element_id"] = relationships["relationship_id"].map(
    lambda rid: xmi_endpoint_lookup.get(rid, {}).get("target_element_id")
)

# Resolve mapping dependency endpoints using the verified mapping qname index.
mapping_mask = relationships["relationship_id"].str.startswith("CALL_")

relationships.loc[mapping_mask, "source_element_id"] = (
    relationships.loc[mapping_mask, "source_element"]
    .map(lambda name: elements_by_qname[name].registry_key)
)

relationships.loc[mapping_mask, "target_element_id"] = (
    relationships.loc[mapping_mask, "target_element"]
    .map(
        lambda name: (
            elements_by_qname[name].registry_key
            if pd.notna(name) and name in elements_by_qname
            else None
        )
    )
)

print("Missing source IDs:", relationships["source_element_id"].isna().sum())
print("Missing target IDs:", relationships["target_element_id"].isna().sum())

display(
    relationships[
        [
            "relationship_id",
            "relationship_type",
            "source_element",
            "source_element_id",
            "target_element",
            "target_element_id",
            "effective_resolution_status",
        ]
    ].head(10)
)

Missing source IDs: 0
Missing target IDs: 1914


,relationship_id,relationship_type,source_element,source_element_id,target_element,target_element_id,effective_resolution_status
0,XMIREF_00000,constrainedElement,UML::Activities::Activity::maximum_one_parameter_node,"(UML_Metamodel, Activity-maximum_one_parameter_node, 1)",UML::Activities::Activity,"(UML_Metamodel, Activity, 1)",resolved_local
1,XMIREF_00001,constrainedElement,UML::Activities::Activity::maximum_two_parameter_nodes,"(UML_Metamodel, Activity-maximum_two_parameter_nodes, 1)",UML::Activities::Activity,"(UML_Metamodel, Activity, 1)",resolved_local
2,XMIREF_00002,subsettedProperty,UML::Activities::Activity::edge,"(UML_Metamodel, Activity-edge, 1)",UML::CommonStructure::Element::ownedElement,"(UML_Metamodel, Element-ownedElement, 1)",resolved_local
3,XMIREF_00003,subsettedProperty,UML::Activities::Activity::group,"(UML_Metamodel, Activity-group, 1)",UML::CommonStructure::Element::ownedElement,"(UML_Metamodel, Element-ownedElement, 1)",resolved_local
4,XMIREF_00004,type,UML::Activities::Activity::isReadOnly,"(UML_Metamodel, Activity-isReadOnly, 1)",None,None,missing_external_target
5,XMIREF_00005,type,UML::Activities::Activity::isSingleExecution,"(UML_Metamodel, Activity-isSingleExecution, 1)",None,None,missing_external_target
6,XMIREF_00006,subsettedProperty,UML::Activities::Activity::node,"(UML_Metamodel, Activity-node, 1)",UML::CommonStructure::Element::ownedElement,"(UML_Metamodel, Element-ownedElement, 1)",resolved_local
7,XMIREF_00007,subsettedProperty,UML::Activities::Activity::partition,"(UML_Metamodel, Activity-partition, 1)",UML::Activities::Activity::group,"(UML_Metamodel, Activity-group, 1)",resolved_local
8,XMIREF_00008,subsettedProperty,UML::Activities::Activity::structuredNode,"(UML_Metamodel, Activity-structuredNode, 1)",UML::Activities::Activity::group,"(UML_Metamodel, Activity-group, 1)",resolved_local
9,XMIREF_00009,subsettedProperty,UML::Activities::Activity::structuredNode,"(UML_Metamodel, Activity-structuredNode, 1)",UML::Activities::Activity::node,"(UML_Metamodel, Activity-node, 1)",resolved_local


In [238]:
# Inspect missing target IDs by relationship and resolution status.
missing_id_summary = (
    relationships.loc[relationships["target_element_id"].isna()]
    .groupby(
        ["relationship_type", "effective_resolution_status"],
        dropna=False,
    )
    .size()
    .rename("count")
    .reset_index()
)

display(missing_id_summary)

# Every resolved relationship should have a target registry key.
resolved_mask = relationships["effective_resolution_status"].isin([
    "resolved_local",
    "resolved_external",
    "resolved_suffix",
    "resolved_semantic_alias",
])

missing_resolved_targets = relationships.loc[
    resolved_mask & relationships["target_element_id"].isna()
]

# Every relationship should have a source registry key.
missing_sources = relationships.loc[
    relationships["source_element_id"].isna()
]

print("Resolved relationships missing target IDs:", len(missing_resolved_targets))
print("Relationships missing source IDs:", len(missing_sources))

assert missing_resolved_targets.empty
assert missing_sources.empty
assert relationships["target_element_id"].isna().sum() == 1914

print("Endpoint identity validation passed.")

,relationship_type,effective_resolution_status,count
0,appliedProfile,missing_external_target,1
1,constrainedElement,ambiguous_target,4
2,general,external_resource_unregistered,5
3,general,missing_external_target,4
4,importedPackage,external_resource_unregistered,1
5,importedPackage,missing_external_target,4
6,mapping_dependency,unresolved,3
7,precondition,ambiguous_target,2
8,redefinedProperty,external_resource_unregistered,11
9,redefinedProperty,missing_external_target,4


Resolved relationships missing target IDs: 0
Relationships missing source IDs: 0
Endpoint identity validation passed.


In [239]:
element_ids = set(elements_df["element_id"])

source_ids = set(relationships["source_element_id"].dropna())
target_ids = set(relationships["target_element_id"].dropna())

missing_source_elements = source_ids - element_ids
missing_target_elements = target_ids - element_ids

print("Unique source IDs:", len(source_ids))
print("Unique target IDs:", len(target_ids))

print("Source IDs absent from inventory:", len(missing_source_elements))
print("Target IDs absent from inventory:", len(missing_target_elements))

assert not missing_source_elements
assert not missing_target_elements

print("Referential integrity passed.")

Unique source IDs: 11501
Unique target IDs: 8353
Source IDs absent from inventory: 0
Target IDs absent from inventory: 0
Referential integrity passed.


In [240]:
# Collect all elements participating in relationships.
participating_ids = (set(relationships["source_element_id"].dropna()) | set(relationships["target_element_id"].dropna()))

elements_df["participates_in_relationship"] = (elements_df["element_id"].isin(participating_ids))

# Summarize participation by resource.
element_coverage = (
    elements_df.groupby("resource")
    .agg(total_elements=("element_id", "size"), participating_elements=("participates_in_relationship", "sum"),)
    .reset_index()
)
element_coverage["nonparticipating_elements"] = (element_coverage["total_elements"] - element_coverage["participating_elements"])
element_coverage["coverage_pct"] = (100 * element_coverage["participating_elements"] / element_coverage["total_elements"])
element_coverage = element_coverage.sort_values("coverage_pct", ascending=False).reset_index(drop=True)

display(element_coverage)

,resource,total_elements,participating_elements,nonparticipating_elements,coverage_pct
0,SysML_v1,682,418,264,61.290323
1,SysML_v1_to_v2,14689,8965,5724,61.032065
2,KerML,2294,1173,1121,51.133391
3,SysML_v2,2463,1241,1222,50.385708
4,UML_Metamodel,5201,2493,2708,47.933090
5,UML_DiagramInterchange,319,66,253,20.689655
6,UML_StandardProfile,177,35,142,19.774011
7,UML_PrimitiveTypes,12,0,12,0.000000


In [241]:
# Isolate unresolved XMI references.
unresolved_xmi = relationships.loc[
    relationships["relationship_id"].str.startswith("XMIREF_")
    & relationships["effective_resolution_status"].isin([
        "missing_external_target",
        "external_resource_unregistered",
        "ambiguous_target",
    ])
].copy()

# Summary by source resource and resolution status.
unresolved_by_resource = (unresolved_xmi.groupby(["source_resource", "effective_resolution_status"]) .size() .unstack(fill_value=0))
unresolved_by_resource["total_unresolved"] = (unresolved_by_resource.sum(axis=1))
display(unresolved_by_resource.sort_values("total_unresolved", ascending=False))

# Summary by relationship type and resolution status.
unresolved_by_type = (
    unresolved_xmi.groupby(["relationship_type", "effective_resolution_status"])
    .size()
    .unstack(fill_value=0)
)
unresolved_by_type["total_unresolved"] = (unresolved_by_type.sum(axis=1))
display(unresolved_by_type.sort_values("total_unresolved", ascending=False))
print("Total unresolved XMI references:", len(unresolved_xmi))

effective_resolution_status,ambiguous_target,external_resource_unregistered,missing_external_target,total_unresolved
source_resource,,,,
SysML_v1_to_v2,0,150,1192,1342
UML_Metamodel,6,0,200,206
SysML_v1,0,0,127,127
KerML,0,0,102,102
UML_DiagramInterchange,0,22,45,67
UML_StandardProfile,0,0,35,35
SysML_v2,0,0,31,31


effective_resolution_status,ambiguous_target,external_resource_unregistered,missing_external_target,total_unresolved
relationship_type,,,,
type,0,150,1719,1869
redefinedProperty,0,11,4,15
general,0,5,4,9
importedPackage,0,1,4,5
subsettedProperty,0,5,0,5
constrainedElement,4,0,0,4
precondition,2,0,0,2
appliedProfile,0,0,1,1


Total unresolved XMI references: 1910


In [242]:
# Focus on unresolved type references in the transformation resource.
unresolved_types = unresolved_xmi.loc[
    (unresolved_xmi["source_resource"] == "SysML_v1_to_v2")
    & (unresolved_xmi["relationship_type"] == "type")
].copy()

print("Unresolved transformation type references:", len(unresolved_types))

display(unresolved_types[["source_element", "source_element_id", "reference_attribute", "reference_value", "effective_resolution_status", "provenance",]].head(30))

print("\nResolution distribution:")
print(unresolved_types["effective_resolution_status"].value_counts())

Unresolved transformation type references: 1342


,source_element,source_element_id,reference_attribute,reference_value,effective_resolution_status,provenance
6982,SysMLv1Tov2::Mappings::Foundations::Mapping::from,"(SysML_v1_to_v2, Mappings-Foundations-Mapping-from, 1)",href,https://www.omg.org/spec/UML/20161101/UML.xmi#Element,missing_external_target,"('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-from', 1)|type|href"
6983,SysMLv1Tov2::Mappings::Foundations::Mapping::filter::result,"(SysML_v1_to_v2, Mappings-Foundations-Mapping-filter_Element-result, 1)",href,https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi#Boolean,missing_external_target,"('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-filter_Element-result', 1)|type|href"
6984,SysMLv1Tov2::Mappings::Foundations::Mapping::filter::src,"(SysML_v1_to_v2, Mappings-Foundations-Mapping-filter_Element-src, 1)",href,https://www.omg.org/spec/UML/20161101/UML.xmi#Element,missing_external_target,"('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-filter_Element-src', 1)|type|href"
6986,SysMLv1Tov2::Mappings::Foundations::Mapping::getMappedColl::fromColl,"(SysML_v1_to_v2, Mappings-Foundations-Mapping-getMappedColl_Element-fromColl, 1)",href,https://www.omg.org/spec/UML/20161101/UML.xmi#Element,missing_external_target,"('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-getMappedColl_Element-fromColl', 1)|type|href"
6989,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped::fromVar,"(SysML_v1_to_v2, Mappings-Foundations-Mapping-getMapped_Element-fromVar, 1)",href,https://www.omg.org/spec/UML/20161101/UML.xmi#Element,missing_external_target,"('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-getMapped_Element-fromVar', 1)|type|href"
6992,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped::fromVar,"(SysML_v1_to_v2, Mappings-Foundations-Mapping-getMapped_Element_Element-fromVar, 1)",href,https://www.omg.org/spec/UML/20161101/UML.xmi#Element,missing_external_target,"('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-getMapped_Element_Element-fromVar', 1)|type|href"
6993,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped::qual,"(SysML_v1_to_v2, Mappings-Foundations-Mapping-getMapped_Element_Element-qual, 1)",href,https://www.omg.org/spec/UML/20161101/UML.xmi#Element,missing_external_target,"('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-getMapped_Element_Element-qual', 1)|type|href"
7015,SysMLv1Tov2::Mappings::Factories::StringParameterFeatureValue_Factory::string,"(SysML_v1_to_v2, Mappings-Factories-StringParameterFeatureValue_Factory-string, 1)",href,https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi#String,missing_external_target,"('SysML_v1_to_v2', 'Mappings-Factories-StringParameterFeatureValue_Factory-string', 1)|type|href"
7017,SysMLv1Tov2::Mappings::Factories::StringParameterFeatureValue_Factory::create::string,"(SysML_v1_to_v2, Mappings-Factories-StringParameterFeatureValue_Factory-create_String-string, 1)",href,https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi#String,missing_external_target,"('SysML_v1_to_v2', 'Mappings-Factories-StringParameterFeatureValue_Factory-create_String-string', 1)|type|href"
7039,SysMLv1Tov2::Mappings::Factories::DirectedReferenceUsage_Factory::create::result,"(SysML_v1_to_v2, Mappings-Factories-DirectedReferenceUsage_Factory-create_FeatureDirectionKind-result, 1)",href,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-DefinitionAndUsage-ReferenceUsage,external_resource_unregistered,"('SysML_v1_to_v2', 'Mappings-Factories-DirectedReferenceUsage_Factory-create_FeatureDirectionKind-result', 1)|type|href"



Resolution distribution:
effective_resolution_status
missing_external_target           1192
external_resource_unregistered     150
Name: count, dtype: int64


In [243]:
# Frequency of unresolved type-reference values.
unresolved_type_values = (
    unresolved_types.groupby(
        ["reference_value", "effective_resolution_status"],
        dropna=False,
    )
    .agg(
        reference_count=("relationship_id", "size"),
        source_elements=("source_element", "nunique"),
        source_ids=("source_element_id", "unique"),
    )
    .reset_index()
    .sort_values("reference_count", ascending=False)
)

display(unresolved_type_values.head(30))

reference_value  \
72                https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi#Boolean   
121                          https://www.omg.org/spec/UML/20161101/UML.xmi#Element   
75                 https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi#String   
212                          https://www.omg.org/spec/UML/20161101/UML.xmi#Trigger   
99                             https://www.omg.org/spec/UML/20161101/UML.xmi#Class   
158                     https://www.omg.org/spec/UML/20161101/UML.xmi#NamedElement   
76                       https://www.omg.org/spec/UML/20161101/UML.xmi#Abstraction   
166                        https://www.omg.org/spec/UML/20161101/UML.xmi#OutputPin   
175                         https://www.omg.org/spec/UML/20161101/UML.xmi#Property   
105                          https://www.omg.org/spec/UML/20161101/UML.xmi#Comment   
78                 https://www.omg.org/spec/UML/20161101/UML.xmi#AcceptEventAction   
160                       https://www.omg.org/spec/UML/20161101/UML.xmi#ObjectFlow   
86   https://www.omg.org/spec/UML/20161101/UML.xmi#AddStructuralFeatureValueAction   
100                       https://www.omg.org/spec/UML/20161101/UML.xmi#Classifier   
73                https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi#Integer   
133                  https://www.omg.org/spec/UML/20161101/UML.xmi#InformationFlow   
164                 https://www.omg.org/spec/UML/20161101/UML.xmi#OpaqueExpression   
143                 https://www.omg.org/spec/UML/20161101/UML.xmi#InvocationAction   
205                       https://www.omg.org/spec/UML/20161101/UML.xmi#Stereotype   
184      https://www.omg.org/spec/UML/20161101/UML.xmi#ReadStructuralFeatureAction   
87            https://www.omg.org/spec/UML/20161101/UML.xmi#AddVariableValueAction   
172                             https://www.omg.org/spec/UML/20161101/UML.xmi#Port   
84                      https://www.omg.org/spec/UML/20161101/UML.xmi#ActivityNode   
169                        https://www.omg.org/spec/UML/20161101/UML.xmi#Parameter   
213                     https://www.omg.org/spec/UML/20161101/UML.xmi#TypedElement   
156                            https://www.omg.org/spec/UML/20161101/UML.xmi#Model   
119              https://www.omg.org/spec/UML/20161101/UML.xmi#DestroyObjectAction   
88                       https://www.omg.org/spec/UML/20161101/UML.xmi#Association   
167                          https://www.omg.org/spec/UML/20161101/UML.xmi#Package   
179     https://www.omg.org/spec/UML/20161101/UML.xmi#ReadIsClassifiedObjectAction   

    effective_resolution_status  reference_count  source_elements  \
72      missing_external_target              174              174   
121     missing_external_target              161              160   
75      missing_external_target               98               98   
212     missing_external_target               52               52   
99      missing_external_target               44               44   
158     missing_external_target               40               40   
76      missing_external_target               36               36   
166     missing_external_target               27               27   
175     missing_external_target               23               23   
105     missing_external_target               23               23   
78      missing_external_target               22               22   
160     missing_external_target               20               20   
86      missing_external_target               19               19   
100     missing_external_target               18               18   
73      missing_external_target               17               17   
133     missing_external_target               15               15   
164     missing_external_target               15               15   
143     missing_external_target               14               14   
205     missing_external_target               13               13   
184     missing_exte

In [244]:
for status, group in unresolved_types.groupby(
    "effective_resolution_status"
):
    print(f"\n--- {status}: {len(group)} references ---")

    display(
        group[
            [
                "source_element",
                "reference_value",
                "reference_attribute",
                "provenance",
            ]
        ].head(10)
    )


--- external_resource_unregistered: 150 references ---


,source_element,reference_value,reference_attribute,provenance
7039,SysMLv1Tov2::Mappings::Factories::DirectedReferenceUsage_Factory::create::result,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-DefinitionAndUsage-ReferenceUsage,href,"('SysML_v1_to_v2', 'Mappings-Factories-DirectedReferenceUsage_Factory-create_FeatureDirectionKind-result', 1)|type|href"
7096,SysMLv1Tov2::Mappings::Factories::AssignmentActionUsage_Factory::create::result,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-Actions-AssignmentActionUsage,href,"('SysML_v1_to_v2', 'Mappings-Factories-AssignmentActionUsage_Factory-create_-result', 1)|type|href"
7110,SysMLv1Tov2::Mappings::Factories::ReferenceUsage_Factory::create::result,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-DefinitionAndUsage-ReferenceUsage,href,"('SysML_v1_to_v2', 'Mappings-Factories-ReferenceUsage_Factory-create_-result', 1)|type|href"
7158,SysMLv1Tov2::Mappings::Factories::AssignmentActionUsageTargetReferenceUsageIn3_Factory::create::result,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-DefinitionAndUsage-ReferenceUsage,href,"('SysML_v1_to_v2', 'Mappings-Factories-AssignmentActionUsageTargetReferenceUsageIn3_Factory-create_-result', 1)|type|href"
7302,SysMLv1Tov2::Mappings::Factories::SubjectMembership_Factory::create::result,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-Requirements-SubjectMembership,href,"('SysML_v1_to_v2', 'Mappings-Factories-SubjectMembership_Factory-create_Type-result', 1)|type|href"
7315,SysMLv1Tov2::Mappings::Factories::FlowUsage_Factory::create,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-Flows-FlowUsage,href,"('SysML_v1_to_v2', 'Mappings-Factories-FlowUsage_Factory-create_InformationFlow-', 1)|type|href"
7328,SysMLv1Tov2::Mappings::Factories::AssignmentActionUsageTargetReferenceUsageIn2_Factory::create::result,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-DefinitionAndUsage-ReferenceUsage,href,"('SysML_v1_to_v2', 'Mappings-Factories-AssignmentActionUsageTargetReferenceUsageIn2_Factory-create_-result', 1)|type|href"
7347,SysMLv1Tov2::Mappings::Factories::AssignmentActionUsageReferenceUsageIn1_Factory::create::result,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-DefinitionAndUsage-ReferenceUsage,href,"('SysML_v1_to_v2', 'Mappings-Factories-AssignmentActionUsageReferenceUsageIn1_Factory-create_-result', 1)|type|href"
7380,SysMLv1Tov2::Mappings::Factories::EmptyRequirementUsage_Factory::create::result,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-Requirements-RequirementUsage,href,"('SysML_v1_to_v2', 'Mappings-Factories-EmptyRequirementUsage_Factory-create_-result', 1)|type|href"
7403,SysMLv1Tov2::Mappings::Factories::InformationFlowEventOccurrenceUsage_Factory::create,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-Occurrences-EventOccurrenceUsage,href,"('SysML_v1_to_v2', 'Mappings-Factories-InformationFlowEventOccurrenceUsage_Factory-create_InformationFlow_NamedElement-', 1)|type|href"



--- missing_external_target: 1192 references ---


,source_element,reference_value,reference_attribute,provenance
6982,SysMLv1Tov2::Mappings::Foundations::Mapping::from,https://www.omg.org/spec/UML/20161101/UML.xmi#Element,href,"('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-from', 1)|type|href"
6983,SysMLv1Tov2::Mappings::Foundations::Mapping::filter::result,https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi#Boolean,href,"('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-filter_Element-result', 1)|type|href"
6984,SysMLv1Tov2::Mappings::Foundations::Mapping::filter::src,https://www.omg.org/spec/UML/20161101/UML.xmi#Element,href,"('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-filter_Element-src', 1)|type|href"
6986,SysMLv1Tov2::Mappings::Foundations::Mapping::getMappedColl::fromColl,https://www.omg.org/spec/UML/20161101/UML.xmi#Element,href,"('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-getMappedColl_Element-fromColl', 1)|type|href"
6989,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped::fromVar,https://www.omg.org/spec/UML/20161101/UML.xmi#Element,href,"('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-getMapped_Element-fromVar', 1)|type|href"
6992,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped::fromVar,https://www.omg.org/spec/UML/20161101/UML.xmi#Element,href,"('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-getMapped_Element_Element-fromVar', 1)|type|href"
6993,SysMLv1Tov2::Mappings::Foundations::Mapping::getMapped::qual,https://www.omg.org/spec/UML/20161101/UML.xmi#Element,href,"('SysML_v1_to_v2', 'Mappings-Foundations-Mapping-getMapped_Element_Element-qual', 1)|type|href"
7015,SysMLv1Tov2::Mappings::Factories::StringParameterFeatureValue_Factory::string,https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi#String,href,"('SysML_v1_to_v2', 'Mappings-Factories-StringParameterFeatureValue_Factory-string', 1)|type|href"
7017,SysMLv1Tov2::Mappings::Factories::StringParameterFeatureValue_Factory::create::string,https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi#String,href,"('SysML_v1_to_v2', 'Mappings-Factories-StringParameterFeatureValue_Factory-create_String-string', 1)|type|href"
7053,SysMLv1Tov2::Mappings::Factories::FeatureTyping_Factory::type,https://www.omg.org/spec/UML/20161101/UML.xmi#NamedElement,href,"('SysML_v1_to_v2', 'Mappings-Factories-FeatureTyping_Factory-type', 1)|type|href"


In [245]:
# Inspect URI aliases used by the registry.
print("URI aliases:")
for uri, target in registry.uri_aliases.items():
    print(uri, "->", target)

print("\nRegistered resource names:")
print(sorted({element.resource for element in registry.elements.values()}))

URI aliases:
http://www.omg.org/spec/UML/20161101/UML.xmi -> UML_20161101
https://www.omg.org/spec/UML/20161101/UML.xmi -> UML_20161101
https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi -> UML_PrimitiveTypes_20161101
http://www.omg.org/spec/UML/20131001/UML.xmi -> UML_20131001
http://www.omg.org/spec/UML/20131001/PrimitiveTypes.xmi -> UML_PrimitiveTypes_20131001
http://www.omg.org/spec/UML/20131001/StandardProfile.xmi -> UML_StandardProfile_20131001
https://www.omg.org/spec/KerML/20250201/KerML.xmi -> KerML

Registered resource names:
['KerML', 'SysML_v1', 'SysML_v1_to_v2', 'SysML_v2', 'UML_DiagramInterchange', 'UML_Metamodel', 'UML_PrimitiveTypes', 'UML_StandardProfile']


In [246]:
# Representative UML and SysML references.
samples = unresolved_types.loc[
    unresolved_types["reference_value"].str.contains(
        "UML.xmi#Element|PrimitiveTypes.xmi#Boolean|SysML.xmi#Systems-",
        regex=True,
        na=False,
    ),
    ["reference_value", "effective_resolution_status"],
].drop_duplicates()

display(samples.head(15))

,reference_value,effective_resolution_status
6982,https://www.omg.org/spec/UML/20161101/UML.xmi#Element,missing_external_target
6983,https://www.omg.org/spec/UML/20161101/PrimitiveTypes.xmi#Boolean,missing_external_target
7039,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-DefinitionAndUsage-ReferenceUsage,external_resource_unregistered
7096,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-Actions-AssignmentActionUsage,external_resource_unregistered
7302,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-Requirements-SubjectMembership,external_resource_unregistered
7315,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-Flows-FlowUsage,external_resource_unregistered
7380,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-Requirements-RequirementUsage,external_resource_unregistered
7403,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-Occurrences-EventOccurrenceUsage,external_resource_unregistered
7471,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-Cases-ObjectiveMembership,external_resource_unregistered
7727,https://www.omg.org/spec/SysML/20250201/SysML.xmi#Systems-Attributes-AttributeDefinition,external_resource_unregistered


In [247]:
from urllib.parse import urlparse, unquote

# Extract representative fragments from unresolved UML references.
uml_refs = unresolved_types.loc[
    unresolved_types["reference_value"].str.contains(
        "/UML/20161101/", na=False
    ),
    "reference_value",
].drop_duplicates()

requested_fragments = {
    unquote(urlparse(uri).fragment)
    for uri in uml_refs
}

for resource in ["UML_Metamodel", "UML_PrimitiveTypes"]:
    resource_elements = [
        element
        for element in registry.elements.values()
        if element.resource == resource
    ]

    resource_ids = {
        element.xmi_id
        for element in resource_elements
        if element.xmi_id is not None
    }

    matches = requested_fragments & resource_ids

    print(f"\n{resource}")
    print("Registered elements:", len(resource_elements))
    print("Requested fragments found:", len(matches))
    print("Example matches:", sorted(matches)[:15])


UML_Metamodel
Registered elements: 5201
Requested fragments found: 145
Example matches: ['Abstraction', 'AcceptCallAction', 'AcceptEventAction', 'Action', 'ActionExecutionSpecification', 'Activity', 'ActivityEdge', 'ActivityFinalNode', 'ActivityNode', 'Actor', 'AddStructuralFeatureValueAction', 'AddVariableValueAction', 'Association', 'AssociationClass', 'Behavior']

UML_PrimitiveTypes
Registered elements: 12
Requested fragments found: 4
Example matches: ['Boolean', 'Integer', 'Real', 'String']


In [ ]:


registry.resolve_references()

In [250]:
display(
    relationships.loc[
        relationships["source_resource"] == "SysML_v1_to_v2"
    ]["effective_resolution_status"].value_counts()
)

effective_resolution_status
resolved_local                    9615
missing_external_target           1192
resolved_external                  994
resolved_suffix                    830
external_resource_unregistered     150
resolved_semantic_alias             21
unresolved                           3
not_applicable                       1
Name: count, dtype: int64

In [251]:
print("Elements:", elements_df.shape)
print("Relationships:", relationships.shape)

print("\nRelationship types:")
print(relationships["relationship_type"].value_counts(dropna=False))

print("\nResolution status:")
print(relationships["effective_resolution_status"].value_counts(dropna=False))

print("\nColumns:")
print(relationships.columns.tolist())

Elements: (25837, 10)
Relationships: (19773, 25)

Relationship types:
relationship_type
type                  4701
memberEnd             3558
association           2722
general               1834
constrainedElement    1334
redefinedProperty     1259
ownedRule             1213
subsettedProperty     1103
redefinedOperation    1073
mapping_dependency     854
navigableOwnedEnd       53
precondition            23
importedPackage         22
instance                 7
base_DataType            7
base_Package             4
appliedProfile           2
base_Class               2
packageImport            1
self_dispatch            1
Name: count, dtype: int64

Resolution status:
effective_resolution_status
resolved_local                    15796
missing_external_target            1732
resolved_external                  1212
resolved_suffix                     830
external_resource_unregistered      172
resolved_semantic_alias              21
ambiguous_target                      6
unresolved        

In [252]:
# Element identity
print("Element IDs unique:", elements_df["element_id"].is_unique)
print("Element IDs missing:", elements_df["element_id"].isna().sum())

# Relationship identity
print("Relationship IDs unique:", relationships["relationship_id"].is_unique)
print("Relationship IDs missing:", relationships["relationship_id"].isna().sum())

# Endpoint completeness
print(
    "Missing source IDs:",
    relationships["source_element_id"].isna().sum()
)
print(
    "Missing target IDs:",
    relationships["target_element_id"].isna().sum()
)

# Referential integrity
element_ids = set(elements_df["element_id"])

invalid_sources = (
    set(relationships["source_element_id"].dropna()) - element_ids
)
invalid_targets = (
    set(relationships["target_element_id"].dropna()) - element_ids
)

print("Invalid source references:", len(invalid_sources))
print("Invalid target references:", len(invalid_targets))

Element IDs unique: True
Element IDs missing: 0
Relationship IDs unique: True
Relationship IDs missing: 0
Missing source IDs: 0
Missing target IDs: 1914
Invalid source references: 0
Invalid target references: 0


In [253]:
missing_targets = relationships.loc[
    relationships["target_element_id"].isna()
].copy()

display(
    missing_targets.groupby(
        ["relationship_type", "effective_resolution_status"],
        dropna=False,
    )
    .size()
    .rename("count")
    .reset_index()
    .sort_values("count", ascending=False)
)

print("\nMissing target labels:")
print(missing_targets["target_element"].isna().value_counts(dropna=False))

,relationship_type,effective_resolution_status,count
13,type,missing_external_target,1719
12,type,external_resource_unregistered,150
8,redefinedProperty,external_resource_unregistered,11
2,general,external_resource_unregistered,5
11,subsettedProperty,external_resource_unregistered,5
1,constrainedElement,ambiguous_target,4
3,general,missing_external_target,4
5,importedPackage,missing_external_target,4
9,redefinedProperty,missing_external_target,4
6,mapping_dependency,unresolved,3



Missing target labels:
target_element
True     1911
False       3
Name: count, dtype: int64


In [254]:
representation_summary = {
    "num_elements": len(elements_df),
    "num_relationships": len(relationships),
    "num_element_resources": elements_df["resource"].nunique(),
    "num_relationship_types": relationships["relationship_type"].nunique(),
    "num_unique_sources": relationships["source_element_id"].nunique(),
    "num_unique_targets": relationships["target_element_id"].nunique(),
    "num_missing_source_ids": relationships["source_element_id"].isna().sum(),
    "num_missing_target_ids": relationships["target_element_id"].isna().sum(),
    "num_unresolved_relationships": relationships[
        "effective_resolution_status"
    ].isin(["missing_external_target",
            "external_resource_unregistered",
            "ambiguous_target",
            "unresolved"]).sum(),
    "element_ids_unique": elements_df["element_id"].is_unique,
    "relationship_ids_unique": relationships["relationship_id"].is_unique,
}

representation_summary_df = pd.DataFrame(
    representation_summary.items(),
    columns=["metric", "value"],
)

display(representation_summary_df)

,metric,value
0,num_elements,25837
1,num_relationships,19773
2,num_element_resources,8
3,num_relationship_types,20
4,num_unique_sources,11501
5,num_unique_targets,8353
6,num_missing_source_ids,0
7,num_missing_target_ids,1914
8,num_unresolved_relationships,1913
9,element_ids_unique,True


In [256]:
relationships.to_csv("C:/Users/Drew/Documents/MyFiles/repo/sysml/data/relationships.csv", index=False)
elements_df.to_csv("C:/Users/Drew/Documents/MyFiles/repo/sysml/data/elements.csv", index=False)